# 가지 열매 YOLO 학습 → CVAT 자동 라벨(track) → 검증 → 백업 → 작업 대장·보드 (Colab · Kaggle) — v4.31.9

> **웹 관리 화면 (v4.14)**: 28번 셀이 `web/state.json`을 만들고, 웹 화면의 요청은 5·9·13·14·15번 셀을 `FROM_REQUESTS = True`로 실행합니다 → README '웹 관리 화면'.

> **Kaggle에서 쓸 때 (v4.12~)**: 이 노트북을 그대로 Kaggle에 올리면 0번 셀이 Kaggle을 알아보고 Drive를 rclone으로 받고 올립니다.
> 처음 한 번 Kaggle 보안 비밀 `RCLONE_CONF`·`CVAT_*`를 넣고 Internet을 켜야 합니다 → **KAGGLE.md**.
> 긴 학습은 `eggplant_kaggle_train_v4.31.0.ipynb`를 **Save & Run All**로 돌리면 브라우저를 닫아도 계속됩니다. 23번 셀은 Colab에서만.

## 사용법 — 세션이 끊겨도 이것만 기억하세요
1. **항상 0번 셀부터 실행합니다.** 설정·Drive 연결·설치·데이터셋 준비를 한 번에 하고, 지금 상태를 보여 줍니다.
2. 그다음 **필요한 셀로 바로 건너뛰면 됩니다.** 각 셀은 0번 셀만 있으면 동작합니다.
3. 학습(1번 셀)은 다시 실행해도 안전합니다. 끝났으면 건너뛰고, 중간에 끊겼으면 **이어서** 학습합니다.
4. **학습 잠금 (v4.9)**: 학습 중에는 `runs/<이름>/RUNNING.json`에 누가·언제까지 학습 중인지 기록합니다. 다른 사람이 같은 이름으로 학습하려 하면 멈추고 알려 줍니다. 0번 셀 `OPERATOR`에 자기 이름을 적어 두세요.

설정값은 **0번 셀 맨 위에서만** 고칩니다. 처리할 TASK ID는 각 작업 셀 맨 위에 적습니다.
프로그램(.py)은 모두 **Drive `scripts/`** 에 두고 노트북이 그 파일을 불러 씁니다. 코드를 고치면 Drive 파일만 바꾸면 됩니다.

| 셀 | 하는 일 |
|---|---|
| 0 | 공통 설정·준비 (매번 먼저) |
| 1~3 | 학습, 학습 결과, 검증 1단계 |
| 4~8 | CVAT 접속·TASK 목록, track 생성, 업로드 승인, 검수 수집, 작업 대장 |
| 9~10 | 프레임별 자동 라벨링 (영상이 없을 때) |
| 11 | 진단 (검출이 적을 때) |
| 12 | **Drive `scripts/`의 파이썬 코드 실행** (목록 보기 + 골라서 실행) |
| 13 | **CVAT TASK 백업** (`cvat_backup.py`, 백업 상태를 보드에 기록) |
| 14 | **라벨 export** — TASK 이름 검색으로 골라 원하는 형식으로. 요청과 받기를 나눠 할 수 있음 |
| 15 | **export 폴더로 학습** — 받은 zip 목록 확인 → 데이터셋(이미지는 영상에서) → YOLO 학습 |
| 16 | **TASK 백업 복원** — Drive의 백업 zip으로 대상 서버에 TASK 만들기 (시험용 사본 등) |
| 17 | **영상 TASK 만들기** — Drive `videos/`의 mp4로 대상 계정에 영상 TASK (5번 셀 track용) |
| 23 | **덮어써진 학습 복원** — Drive 버전 기록에서 사용자별 마지막 파일을 `runs/<이름>_A`, `_B`로 |
| 27 | **앙상블 자동 라벨링** — 모델 여러 개가 동의한 박스만 합쳐 올림, 한 모델만 찾은 것은 검토 목록 |
| 26 | **작업 기록 (시계열)** — 작업 목록 · TASK별 변화 이력 · 날짜별 건수 |
| 25 | **업로드 기록 보기** — 누가·언제·어떤 모델·설정으로 올렸고 라벨이 어떻게 바뀌었나 |
| 24 | **수작업 ↔ 자동 라벨 비교** — 평가 표와 대표 화면 |
| 18 | **라벨 확인** — TASK의 현재 라벨을 라벨별·프레임별로 요약하고 그림으로 (읽기만) |

## CVAT 로그인 — 두 방식 모두 지원 (Colab 왼쪽 열쇠 아이콘 = 보안 비밀에 넣고 '노트북 액세스' 켜기)
- **PAT (Google 로그인 계정)**: CVAT 웹에 Google로 로그인 → Profile → Security → Personal access tokens에서
  Read/Write 토큰을 만들어 `CVAT_TOKEN`에 넣음. Google 로그인(OAuth)은 브라우저 전용이라 노트북은 이 토큰으로 접속합니다.
- **공동 아이디 (ID/PASS)**: `CVAT_USER`(아이디 또는 이메일), `CVAT_PASS`

**계정·서버 여러 개**: 0번 셀 `CVAT_PROFILES`에 계정마다 이름을 붙여 서버 주소·로그인 방식·보안 비밀 이름을 적습니다.
같은 방식(PAT)의 계정이 둘이면 보안 비밀 이름을 다르게 합니다. 예: A = `CVAT_TOKEN`, B = `CVAT_TOKEN_B` → 프로필 `'B': {'host': ..., 'login': 'token', 'token': 'CVAT_TOKEN_B'}`
CVAT를 쓰는 셀(4·5·6·7·9·10·13)은 맨 위에 `LOGIN` 줄이 있습니다.
- `LOGIN = None` → 지금 접속 중인 서버 그대로 (처음이면 0번 셀 `CVAT_AUTH`, 기본 `'auto'`)
- `LOGIN = '프로필 이름'` → 그 계정으로 바꿈. 한 번 로그인한 계정은 다시 로그인하지 않고, 보안 비밀 값이 바뀌면 자동으로 다시 로그인합니다. 직접 지우려면 `cvat_forget()`.

셀을 실행하면 `▶ CVAT 서버: 아이디@주소` 줄로 대상이 표시되고, 작업 대장·백업 기록에 서버가 함께 남습니다.

## TASK 이름 검색 (4번 셀 `TASK_QUERY`, 13번 셀 `BK_QUERY`, 14번 셀 `EX_QUERY`)
- 공백 = **AND**, `|` = **OR**, 쉼표 = **묶음 OR**: `'261019 vf_r2'` = 촬영일 261019 이면서 상태 vf_r2 · `'vf_r2df|vf_r2df_ok'` · `'261015 vf|vf_r2, 261019 vf_r2'`
- 일반 단어는 이름의 칸 하나와 **정확히** 같아야 합니다 (`vf_r2`는 `vf_r2df`와 다름, 숫자는 라인 또는 그루)
- 칸 지정: `date=` `row=` `plant=` `state=` `gh=` · 와일드카드 `*`: `'*r2df*'`, `'date=2610*'`
- `검토` = 검토 완료 TASK (0번 셀 `REVIEWED_RULE`과 같은 판단: 상태에 'vf'가 들어 있음), `-검토` = 미검토: `'검토 date=261019..261022'`. 검색어 `vf`는 상태가 **정확히** vf인 것만입니다 (vfdf·vf_r2 등은 빠짐) (v4.24)
- 4번 셀에 검색어를 넣으면 TASK마다 맞음 여부와 단어별 ✓/✗를 보여 주고, 13번 셀은 `BK_PREVIEW = True`(기본)로 대상만 먼저 보여 줍니다.

비밀번호·토큰은 노트북 파일, Drive, 로그에 저장되지 않습니다. 보안 비밀이 없으면 실행할 때 입력창이 뜹니다.

**먼저 할 것**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`

**공유받은 폴더로 쓸 때**: Google Drive **공유 문서함**의 `eggplant_yolo`는 Colab에서 바로 보이지 않습니다.
폴더 오른쪽 클릭 → **정리 → 바로가기 추가 → 내 드라이브**를 한 번 해 두면 `내 드라이브/eggplant_yolo`로 보입니다(편집자 권한이면 읽기·쓰기 모두 됨).
0번 셀은 BASE 폴더가 없으면 새로 만들지 않고 멈춥니다(처음 설치하는 사람만 `CREATE_BASE = True`).

## Drive 폴더
```
내 드라이브/eggplant_yolo/
├─ dataset.zip        ← cvat_to_yolo.py가 만든 dataset 폴더를 압축한 것
├─ scripts/           ← 프로그램: cvat_task_tools.py, cvat_backup.py, cvat_to_yolo.py, yolo_track_to_cvat.py,
│                       compare_tracks.py, eval_occluded_recall.py, cvat_yolo_function.py
├─ config/            ← assignment_rules.csv(구역 담당), board_sheet_id.txt(BK_BOARD='sheet'일 때 작업보드 ID)
├─ videos/<YYMMDD>/   ← CVAT에 올린 것과 같은 영상 (이름 = TASK 기본 이름 또는 CVAT 원본 이름)
├─ datasets/          ← 15번 셀이 만든 데이터셋 zip·통계 리포트
├─ runs/              ← 학습 결과 (에폭마다 저장 → 끊겨도 남음)
├─ tasks/<TASK_ID>/   ← TASK별 track 작업: 라벨 export, track XML, 검증 결과, 실행 로그
├─ backups/<기본 이름>/{original,work,revised}/ ← TASK 백업(영상+라벨), 시각별
├─ logs/              ← 12번 셀로 실행한 코드의 로그, versions.csv
├─ exports/<시각>_<형식>/ ← 14번 셀로 받은 라벨 zip (요청 목록: exports/pending_exports.csv)
├─ task_registry.csv  ← 작업 대장 (실행 1회 = 1줄)
├─ backup_index.csv   ← 백업 색인 (백업 1회 = 1줄)
└─ backup_status.csv  ← TASK별 최신 백업 상태 (reports/작업보드.xlsx '백업 상태' 탭과 같은 내용)
```
**학습에 쓴 영상으로 검증하면 점수가 부풀려집니다.** 검증용 영상은 dataset에 넣지 마세요.


In [ ]:
# 0. 공통 설정 · 준비 — 세션이 새로 열릴 때마다 가장 먼저 실행 (다시 실행해도 안전) · 앱 버전 4.31.9 (Colab · Kaggle)
APP_VERSION = '4.31.9'             # 이 노트북의 버전 — Drive scripts/의 파일도 같은 버전이어야 함 (바꾸지 마세요)
# ====================== 설정 (여기만 고치세요) ======================
BASE     = '/content/drive/MyDrive/eggplant_yolo'   # 공유받은 폴더는 '내 드라이브'에 바로가기를 만들면 이 경로로 보임
CREATE_BASE = False            # True = BASE가 없을 때 새로 만듦 (처음 설치하는 사람만)
MODEL_CHOICE = 'latest'        # 쓸 학습 모델: 'latest' = runs/에서 가장 최근에 학습을 끝낸 모델 (15번 셀 결과가 자동으로 잡힘)
                               #   또는 아래 목록 번호(1, 2 …) · 또는 runs/ 폴더 이름 그대로 (예: 'bud_r1')
MODEL_NAMES = {                # 자주 쓰는 모델에 번호를 붙여 둠 (선택 사항)
    1: 'fruit_1920',                                   # 예전 dataset.zip으로 학습
    2: 'exp_20261025_081411_cvat_for_images_1_1',      # exports/20261025_081411_cvat_for_images_1_1로 학습 (15번 셀)
    3: 'fruit_r261101_2class_rotate',                  # 11/6 운영판: 밀린 과제 #3
}
MODEL    = 'yolo11s.pt'        # 사전학습 가중치 (s → 더 높이려면 m)
IMGSZ    = 1920                # 변환 때 TRAIN_IMGSZ와 같게
EPOCHS   = 100
PATIENCE = 30                  # 개선이 없으면 일찍 끝냄
BATCH    = -1                  # -1 = GPU 메모리에 맞춰 자동. 메모리 오류 시 2 또는 1
DEVICE   = None                # None = 자동(GPU 있으면 GPU)
TRACK_CONF = 0.25             # track 생성 시 탐지 신뢰도 하한. 검출이 적으면 0.1~0.15로 (추적기 기준도 함께 낮아짐)
TRACK_FILL_GAPS = 3            # 잠깐(태스크 프레임 3개 이하) 안 보인 열매는 track을 끊지 않고 CVAT 보간에 맡김
TRACK_BUFFER = 90              # 추적기가 가려진 열매의 ID를 기억하는 프레임 수 (30fps 기준 3초)
TRACK_DECODER = 'pyav'         # v4.31.4 기본 (밀린 과제 #4) · 영상 읽기: 'opencv' / 'pyav'(CVAT와 같은 방식) — 20번 셀 진단 결과에 따라
AUTO_TASK_REPLACE = True       # v4.31.4 (밀린 과제 #5): 이름이 _uld·_AA로 끝나는 자동 라벨 TASK는 기존 라벨을 백업한 뒤 교체가 기본
                               #   (5·6·10번 셀, 30번 셀) · 손 라벨 TASK는 각 셀의 ALLOW_REPLACE·TRACK_UPLOAD_REPLACE·AA_EXISTING 그대로 · False = 예전처럼
def auto_task(name):
    """자동 라벨 TASK(이름 끝 _uld · _AA)이고 AUTO_TASK_REPLACE가 켜져 있으면 True"""
    n = str(name or '').strip()
    return bool(globals().get('AUTO_TASK_REPLACE', False)) and (n.endswith('_uld') or bool(re.search(r'_AA(_r\d+)?$', n)))   # v4.31.5: AA_r1 = AA
TRACK_FRAME_OFFSET = 0         # 영상 프레임 + 이 값 = CVAT 프레임 (20번 셀 진단이 알려 줌)
TRACK_BOX_SOURCE = 'detection'  # 'detection' = 모델이 그 프레임에서 찾은 박스 (권장) / 'tracker' = 추적기 추정 박스(이전 방식, 늦게 따라와 잘림)
TRACK_NEAR = 1                 # 태스크 프레임에 검출이 없을 때 앞뒤 몇 프레임까지 빌려올지 (크면 카메라 이동만큼 밀림)
UPLOAD_MIN_PRECISION = 0.8     # 업로드 권장 기준 (검증 2단계 정밀도) — 대장에 '권장/재검토'로 표시
AA_CONF    = 0.25              # 프레임별 자동 라벨링 신뢰도 하한
REVIEWED_RULE = 'vf'          # 검토 완료 TASK = 이름의 상태에 이 글자가 있음 (vf, vf_r2, vf_r2df_ok, …_vf_AA). '' = 모두 검토 완료로 봄
                              #   검색어에서는 '검토'로 씀: '검토 date=261019..261022' (검색어 'vf'는 상태가 정확히 vf인 것만 — 다름)
VAL_RULE   = 'plant=3|plant=7'      # val(평가) TASK = 이 검색어에 맞는 TASK — 모든 온실·라인의 3번·7번 그루. 15번이 TASK 단위로 train/val을 나눔
                              #   '' = 예전처럼 무작위 영상(CONVERT의 VAL_RATIO) · 예: 'gh=G4', 'row=2 plant=6..10' (웹 학습 요청이 바꿀 수 있음)
CVAT_PROFILES = {             # 계정(프로필) 목록. 이름은 자유 — 셀 맨 위 LOGIN에 이 이름을 씀
    'token':    {'host': 'https://app.cvat.ai', 'login': 'token',    'token': 'CVAT_TOKEN'},                    # PAT (Google 로그인 계정)
    'password': {'host': 'https://app.cvat.ai', 'login': 'password', 'user': 'CVAT_USER', 'pass': 'CVAT_PASS'}, # 공동 아이디 (ID/PASS)
    'B':        {'host': 'https://app.cvat.ai', 'login': 'token',    'token': 'CVAT_TOKEN_B'},                  # B 계정 PAT (보안 비밀 CVAT_TOKEN_B)
}                              # 'token' / 'user' / 'pass' 값 = Colab 보안 비밀 이름 (계정마다 다른 이름으로)
CVAT_SDK_VERSION = 'auto'      # 'auto' = 처음 접속할 서버와 같은 X.Y 버전의 cvat-cli/cvat-sdk 설치 (SaaS 자동 업그레이드 대응)
                               # 문제가 생기면 확인된 버전으로 고정: 예 '2.79.0'
CVAT_AUTH  = 'auto'            # 처음 접속할 프로필: 'auto' = 보안 비밀이 있는 첫 프로필 · 또는 프로필 이름
                               # 셀마다 맨 위 LOGIN = '프로필 이름'으로 계정·서버를 바꿀 수 있음
CVAT_ORG     = 'horti'        # 기본 조직: CVAT 조직의 slug 또는 이름 (CVAT 화면 왼쪽 위 조직 메뉴에 보이는 것)
                               #   '' = 개인 작업공간 · None = 구분 안 함 (보이는 모든 조직의 TASK — v4.18까지와 같음)
CVAT_PROJECT = ''              # v4.31.4 기본 (밀린 과제 #2): eggplant·eggplant_AA 모두 — CPU 마스터는 config/app_settings.json (같게 둘 것) · 기본 프로젝트: 이름 또는 ID (그 조직 안). '' = 모든 프로젝트 · '-' = 프로젝트 없는 TASK만
                               #   정하면 셀의 TASK 목록·검색·백업·export가 이 범위만 보고, 16·17번 셀의 새 TASK도 이 조직·프로젝트에 만듦
                               #   TASK ID를 직접 적으면 프로젝트는 안 따짐(조직은 따짐). 프로필마다 다르게: CVAT_PROFILES 항목에 'org': …, 'project': …
                               #   웹 관리 화면의 요청은 그 TASK들의 조직으로 실행 (요청이 끝나면 다시 이 기본값)
DS_ROOT    = '/content/ds'     # 데이터셋을 풀 곳 (Colab 로컬 디스크 — 빠름)
OPERATOR   = ''               # 학습 잠금·완료 기록에 남길 내 이름 (예: 'A-재웅', 'B-james'). 사람마다 다르게
LOCK_STALE_MIN = 45           # 학습 잠금이 이 시간(분) 넘게 갱신되지 않으면 끊긴 학습으로 보고 넘겨받을 수 있음
LOCK_CONFIRM_SEC = 30         # 잠금을 잡은 뒤 이만큼 기다렸다 다시 확인 (두 컴퓨터가 거의 동시에 시작한 경우를 잡음)
# ---- Kaggle에서 실행할 때만 (v4.12) — Colab에서는 무시됨. 준비 방법은 KAGGLE.md ----
KAGGLE_REMOTE   = 'gdrive:'              # rclone 원격. root_folder_id = eggplant_yolo 폴더 ID로 만들었으면 'gdrive:' · 아니면 'gdrive:eggplant_yolo'
KAGGLE_BASE     = '/tmp/eggplant_yolo'   # Kaggle 안의 작업 폴더 (세션이 끝나면 사라짐 — 원본은 언제나 Drive)
KAGGLE_PULL     = ['exports']            # 0번 셀이 처음에 받아 둘 폴더. 영상·백업·TASK 폴더는 셀이 필요할 때 자동으로 받음
KAGGLE_PULL_DATASET = False              # True = dataset.zip · datasets/<선택 모델>.zip도 받음 (2·3번 셀용, 클 수 있음)
KAGGLE_PULL_ALL_BEST = False             # v4.31.6: False = 선택 모델(MODEL_CHOICE)의 best.pt만 받음, 다른 모델은 셀이 쓸 때 받음 · True = 예전처럼 모든 모델의 best.pt
KAGGLE_VIDEOS   = ''                     # 미리 받아 둘 영상 (TASK 이름 검색 규칙, 예 'G6 date=261019'). '' = 필요할 때만
KAGGLE_AUTO_SYNC = True                  # 셀마다: 실행 전 공유 기록 받기 → 실행 후 바뀐 파일을 Drive에 올리기 (끄면 drive_sync_now()로 직접)
# ====================================================================
import os, sys, glob, re, json, zipfile, shutil, subprocess, importlib, importlib.util
from pathlib import Path
from urllib.parse import urlparse
import time

# ---------- Kaggle에서 실행 (v4.12) — Drive를 붙일 수 없으므로 rclone으로 받고 올림. Colab에서는 아무것도 안 함 ----------
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
KAGGLE_BATCH = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'      # Save & Run All(커밋)로 실행 중
LOCAL_TMP = '/tmp' if ON_KAGGLE else '/content'                         # 잠깐 쓰는 로컬 폴더
_KG = globals().get('_KG') or {}                                        # 이 세션의 동기화 상태 (0번 셀을 다시 실행해도 유지)
for _k, _v in {'ls': {}, 'media': set(), 'tasks': set(), 'lock_t': {}, 'bg': None, 'secrets': {}}.items():
    _KG.setdefault(_k, _v)
_KG_SHARED = ['logs/**', 'task_registry.csv', 'backup_index.csv', 'backup_status.csv',
              'backups/pending_backups.csv', 'exports/*.csv', 'requests/*.json']   # 셀마다 실행 전에 받는 공유 기록·웹 요청 (작음)
_KG_NOSYNC = ('raw_images/', 'cvat_frames/')                            # 프레임 캐시: 올리지도 받지도 않음 (다시 만들 수 있음)
_KG_NODELETE = ('videos/', 'backups/')                                  # Kaggle에서 지워도 Drive에서는 지우지 않음

def _kg_remote(rel=''):
    r = KAGGLE_REMOTE if KAGGLE_REMOTE.endswith(':') else KAGGLE_REMOTE.rstrip('/') + '/'
    return r + rel.strip('/')

_KG_RC_LOCK = globals().get('_KG_RC_LOCK') or __import__('threading').RLock()

def _kg_rc(args, quiet=False):
    # rclone 실행. 실패하면 마지막 오류만 보여 줌 (값·토큰은 출력에 나오지 않음)
    # v4.28.1: Drive에 쓰는 rclone은 한 번에 하나 — 작업자 셀의 심장 박동 스레드와 본 작업이 동시에 새 폴더를 만들면
    #          Google Drive에 같은 이름 폴더가 둘 생겨(relabel/ 등) 앱과 노트북이 서로 다른 쪽을 봄
    with _KG_RC_LOCK:
        r = subprocess.run(['rclone', *args], capture_output=True, text=True)
    if r.returncode and not quiet:
        err = '\n'.join(l for l in r.stderr.strip().splitlines() if 'ERROR' in l or 'Failed' in l or 'error' in l)[-600:]
        print(f'⚠️ rclone {args[0]} 실패 (코드 {r.returncode}): {err or r.stderr.strip()[-300:]}')
    return r

_KG_SHARDED = ('task_registry.csv', 'backup_index.csv', 'backups/pending_backups.csv', 'exports/pending_exports.csv',
               'logs/uploads.csv', 'logs/activity.csv', 'logs/train_tasks.csv', 'logs/model_compare.csv')

def _kg_filters(paths):
    # 'runs/a' → 파일 또는 폴더 전체 · '*'가 있으면 rclone 규칙 그대로 ('*'는 / 를 넘지 않음, '**'는 넘음)
    out = []
    for p in paths:
        p = '/' + p.strip('/')
        out += ['--include', p] if '*' in p else ['--include', p, '--include', p + '/**']
    if any(str(p).strip('/') in _KG_SHARDED for p in paths):     # v4.31.0: 그 기록의 세션 파일도 함께 (logs/shards/)
        out += ['--include', '/logs/shards/**']
    return out

def drive_pull(*paths, quiet=False):
    # Drive → Kaggle (BASE 기준 경로·패턴). 로컬이 더 새것이면 건너뜀. Colab에서는 아무것도 안 함
    if not ON_KAGGLE or not paths:
        return
    flt = _kg_filters(paths) + [x for n in _KG_NOSYNC for x in ('--exclude', f'/{n}**')]
    r = _kg_rc(['copy', _kg_remote(), BASE, '--update', '--transfers', '8', '--checkers', '16', '-v', '--stats', '0', *flt], quiet=quiet)
    for _l in (r.stderr or '').splitlines():       # 셀 안에서 Drive에서 받은 파일 → 셀이 끝날 때 다시 올리지 않음 (v4.24.2)
        _m = re.search(r'INFO\s+:\s+(.+?): Copied', _l)
        if _m:
            try:                                   # 받은 그대로인지 알려고 크기·수정 시각을 기억 (셀이 그 뒤에 고치면 올림)
                _st = os.stat(os.path.join(BASE, _m.group(1).strip()))
                _KG.setdefault('pulled', {})[_m.group(1).strip()] = (_st.st_mtime_ns, _st.st_size)
            except OSError:
                pass

def kg_sync_exports():
    # Kaggle: Drive exports/에서 새로 생긴 zip을 받아 둠 — 0번 셀 뒤에 다른 세션·컴퓨터가 받은 export도 보이게 (v4.23)
    if ON_KAGGLE:
        drive_pull('exports', quiet=True)

def drive_push(*paths, quiet=False):
    # Kaggle → Drive (BASE 기준 경로·패턴). Drive가 더 새것이면 건너뜀. 지우지는 않음
    if not ON_KAGGLE or not paths:
        return
    _kg_rc(['copy', BASE, _kg_remote(), '--update', '--transfers', '8', '--checkers', '16', *_kg_filters(paths)], quiet=quiet)

def drive_push_bg(*paths):
    # 학습 중 에폭마다: 앞의 올리기가 끝났을 때만 백그라운드로 올림 (학습을 멈추지 않음)
    import threading
    t = _KG.get('bg')
    if not ON_KAGGLE or (t and t.is_alive()):
        return
    _KG['bg'] = threading.Thread(target=drive_push, args=paths, kwargs={'quiet': True}, daemon=True)
    _KG['bg'].start()

def _kg_files(prefix):
    # Drive의 폴더 목록 (경로, 크기, 수정 시각) — 세션마다 한 번만 읽음
    if prefix not in _KG['ls']:
        r = _kg_rc(['lsf', '-R', '--files-only', '--format', 'pst', '--separator', '\t', _kg_remote(prefix)], quiet=True)
        _KG['ls'][prefix] = [tuple(l.split('\t')) for l in r.stdout.splitlines() if l.count('\t') == 2]
    return _KG['ls'][prefix]

_KG_VEXT = ('.mp4', '.mov', '.avi', '.mkv')

def drive_get_videos(query='', files=()):
    # Drive videos/에서 영상을 받음: query = TASK 이름 검색 규칙(파일 이름에 적용) · files = videos/ 기준 경로
    if not ON_KAGGLE:
        return []
    allv = [(p, s) for p, s, t in _kg_files('videos') if p.lower().endswith(_KG_VEXT)]
    pick = [p for p, s in allv if p in set(files)]
    if query:
        pick += [p for p, s in allv if ctt.match_name(os.path.splitext(os.path.basename(p))[0], query)[0]]
    pick = sorted(set(pick))
    if pick:
        mb = sum(int(s) for p, s in allv if p in pick) / 2**20
        print(f'⬇️ 영상 {len(pick)}개 받기 ({mb:.0f} MB) …')
        drive_pull(*[f'videos/{p}' for p in pick])
    return pick

def kg_file_index(*prefixes):
    # Kaggle: Drive 파일 목록 {BASE 기준 경로: (크기, 수정 시각 epoch)} — 새로 읽음 (모델 정리·웹 상태, v4.24.8). Colab은 None
    if not ON_KAGGLE:
        return None
    import datetime as _dt
    out = {}
    for pre in prefixes:
        _KG['ls'].pop(pre, None)
        for p, s, t in _kg_files(pre):
            try:
                ts = _dt.datetime.strptime(t[:19], '%Y-%m-%d %H:%M:%S').timestamp()
            except ValueError:
                ts = 0
            out[f'{pre}/{p}'] = (int(s), ts)
    return out

def kaggle_media_on_drive(source='', tname=''):
    # TASK 원본 영상이 Drive에 있는지만 봄 (받지 않음, v4.24.7): 'videos/<경로>' · 'backups/<경로>' · ''
    if not ON_KAGGLE:
        return ''
    base = ctt.base_name(tname) if (ctt is not None and tname) else ''
    want = {x.lower() for x in (base, os.path.splitext(os.path.basename(source or ''))[0], tname or '') if x}
    vids = [p for p, s, t in _kg_files('videos')
            if p.lower().endswith(_KG_VEXT) and os.path.splitext(os.path.basename(p))[0].lower() in want]
    if vids:
        return f'videos/{vids[0]}'
    bk = sorted((t, p) for p, s, t in _kg_files('backups') if base and p.startswith(base + '/') and p.endswith('.zip'))
    return f'backups/{bk[-1][1]}' if bk else ''

def kaggle_fetch_media(source='', tname=''):
    # TASK 하나의 원본 영상을 Drive에서 받음: videos/(같은 이름) → 없으면 backups/<기본 이름>/의 가장 최근 백업
    key = (source or '', tname or '')
    if not ON_KAGGLE or key in _KG['media'] or _KG.get('no_fetch'):
        return
    _KG['media'].add(key)
    base = ctt.base_name(tname) if (ctt is not None and tname) else ''
    want = {x.lower() for x in (base, os.path.splitext(os.path.basename(source or ''))[0], tname or '') if x}
    vids = [(p, s) for p, s, t in _kg_files('videos')
            if p.lower().endswith(_KG_VEXT) and os.path.splitext(os.path.basename(p))[0].lower() in want]
    if vids:
        todo = [p for p, s in vids if not os.path.exists(f'{BASE}/videos/{p}')]
        if todo:
            print(f"⬇️ 영상 받기: {', '.join(todo)} ({sum(int(s) for p, s in vids if p in todo) / 2**20:.0f} MB)")
            drive_pull(*[f'videos/{p}' for p in todo])
        return
    if base:
        bk = sorted((t, p, s) for p, s, t in _kg_files('backups') if p.startswith(base + '/') and p.endswith('.zip'))
        if bk and not os.path.exists(f'{BASE}/backups/{bk[-1][1]}'):
            print(f'⬇️ 백업 받기 (영상 포함): backups/{bk[-1][1]} ({int(bk[-1][2]) / 2**20:.0f} MB)')
            drive_pull(f'backups/{bk[-1][1]}')

def _kg_lock_mirror(run_dir, force=False):
    # Drive의 RUNNING.json을 그대로 비춤 (없으면 로컬도 지움) — 학습 잠금이 다른 컴퓨터와 통하도록
    rel = os.path.relpath(os.path.join(run_dir, 'RUNNING.json'), BASE)
    if not force and time.time() - _KG['lock_t'].get(rel, 0) < 15:
        return rel
    _KG['lock_t'][rel] = time.time()
    r = _kg_rc(['cat', _kg_remote(rel)], quiet=True)
    p = f'{BASE}/{rel}'
    if r.returncode == 0 and r.stdout.strip():
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, 'w', encoding='utf-8').write(r.stdout)
    elif os.path.exists(p):
        os.remove(p)
    return rel

def _kg_lock_publish(rel):
    p = f'{BASE}/{rel}'
    if os.path.exists(p):
        _kg_rc(['copyto', p, _kg_remote(rel)], quiet=True)
    else:
        _kg_rc(['deletefile', _kg_remote(rel)], quiet=True)
    _KG['lock_t'][rel] = time.time()

def _kg_patch_ctt():
    # ctt(공용 도구)의 잠금·영상 찾기 함수에 Drive 동기화를 덧붙임 (0번 셀이 ctt를 다시 불러올 때마다)
    if not ON_KAGGLE or ctt is None:
        return
    if hasattr(ctt, 'find_video'):
        _fv = ctt.find_video
        def find_video(*a, **k):
            try:
                kaggle_fetch_media(k.get('source', a[1] if len(a) > 1 else ''),
                                   k.get('task_name', k.get('tname', a[2] if len(a) > 2 else '')))
            except Exception as e:
                print(f'(Drive에서 영상 받기 실패: {e})')
            return _fv(*a, **k)
        ctt.find_video = find_video
    if hasattr(ctt, 'video_from_backups'):
        _vb = ctt.video_from_backups
        def video_from_backups(*a, **k):
            try:
                kaggle_fetch_media('', k.get('task_name', a[1] if len(a) > 1 else ''))
            except Exception as e:
                print(f'(Drive에서 백업 받기 실패: {e})')
            return _vb(*a, **k)
        ctt.video_from_backups = video_from_backups
    if all(hasattr(ctt, f) for f in ('acquire_run_lock', 'touch_run_lock', 'release_run_lock', 'read_run_lock')):
        _acq, _touch, _rel, _read = ctt.acquire_run_lock, ctt.touch_run_lock, ctt.release_run_lock, ctt.read_run_lock
        def read_run_lock(run_dir, *a, **k):
            _kg_lock_mirror(run_dir)
            return _read(run_dir, *a, **k)
        def acquire_run_lock(run_dir, *a, **k):
            rel = _kg_lock_mirror(run_dir, force=True)
            drive_pull(os.path.relpath(run_dir, BASE), quiet=True)     # 이어서 학습할 last.pt·results.csv도 받음
            res = _acq(run_dir, *a, **k)
            if res and res[0]:
                _kg_lock_publish(rel)
            return res
        def touch_run_lock(run_dir, *a, **k):
            rel = os.path.relpath(os.path.join(run_dir, 'RUNNING.json'), BASE)
            due = time.time() - _KG['lock_t'].get(rel, 0) >= 60          # Drive와 주고받기는 1분에 한 번
            if due:
                _kg_lock_mirror(run_dir, force=True)
            res = _touch(run_dir, *a, **k)
            if due and res:
                _kg_lock_publish(rel)
            return res
        def release_run_lock(run_dir, *a, **k):
            rel = _kg_lock_mirror(run_dir, force=True)
            res = _rel(run_dir, *a, **k)
            _kg_lock_publish(rel)
            return res
        ctt.read_run_lock, ctt.acquire_run_lock = read_run_lock, acquire_run_lock
        ctt.touch_run_lock, ctt.release_run_lock = touch_run_lock, release_run_lock

def _kg_snapshot():
    snap = {}
    for root, dirs, files in os.walk(BASE):
        rel_root = os.path.relpath(root, BASE)
        if rel_root != '.' and (rel_root + '/').startswith(_KG_NOSYNC):
            dirs[:] = []
            continue
        for f in files:
            p = os.path.join(root, f)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            snap[os.path.relpath(p, BASE)] = (st.st_mtime_ns, st.st_size)
    return snap

def _kg_pre_cell(*_a):
    if not (ON_KAGGLE and KAGGLE_AUTO_SYNC):
        return
    drive_pull(*_KG_SHARED, quiet=True)
    _KG['snap'], _KG['snap_t'] = _kg_snapshot(), time.time()

def drive_sync_now(echo=True):
    # 지난 스냅숏 뒤에 바뀐 파일을 Drive에 올리고, 셀에서 지운 파일은 Drive에서도 지움(휴지통). 셀이 끝날 때 자동 실행
    snap0 = _KG.pop('snap', None)
    if not ON_KAGGLE or snap0 is None:
        return
    import tempfile
    snap1 = _kg_snapshot()
    _KG.pop('snap_t', 0)
    pulled = _KG.pop('pulled', {})               # 셀 안에서 Drive에서 받은 뒤 그대로인 파일은 빼고
    mine = lambda p: not p.endswith(('.partial', 'RUNNING.json'))   # 학습 잠금은 잠금 함수가 따로 주고받음
    # 셀이 새로 만들거나 바꾼 파일 전부 — 수정 시각을 보존한 복사(shutil.copy2: 14번 셀 export 재사용)도 올림 (v4.24.2, 예전엔 빠졌음)
    changed = sorted(p for p, v in snap1.items() if snap0.get(p) != v and pulled.get(p) != v and mine(p))
    deleted = sorted(p for p in snap0 if p not in snap1 and not p.startswith(_KG_NODELETE) and mine(p))
    for lst, args in ((changed, ['copy', BASE, _kg_remote(), '--update', *(['--no-traverse'] if len(changed) <= 200 else []), '--transfers', '8']),
                      (deleted, ['delete', _kg_remote()])):
        if lst:
            with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False, encoding='utf-8') as fh:
                fh.write('\n'.join(lst) + '\n')
            _kg_rc(args + ['--files-from-raw', fh.name])
            os.remove(fh.name)
    if echo and (changed or deleted):
        mb = sum(snap1[p][1] for p in changed) / 2**20
        print('☁️ Drive 반영: ' + ' · '.join(x for x in (f'올림 {len(changed)}개 ({mb:.1f} MB)' if changed else '',
                                                    f'지움 {len(deleted)}개 (Drive 휴지통)' if deleted else '') if x))

def _kg_post_cell(*_a):
    if ON_KAGGLE and KAGGLE_AUTO_SYNC:
        try:
            drive_sync_now()
        except Exception as e:
            print(f'⚠️ Drive 올리기 실패: {e} → drive_sync_now() 또는 drive_push(경로)로 다시')

def _kg_secret(name):
    # 읽은 값만 기억 — 없던 비밀을 나중에 붙이면 0번 셀을 다시 실행하는 것만으로 읽힘 (v4.13)
    if not _KG['secrets'].get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            _KG['secrets'][name] = UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return _KG['secrets'][name]

def _kg_fix_conf(conf):
    # Kaggle 보안 비밀의 rclone.conf를 되살림 (v4.31.4-1: CPU 마스터와 같은 처리기)
    # 한 줄로 붙은 것 · 글자 그대로 남은 백슬래시-n · 토큰 JSON 뒤에 붙은 team_drive 등 · 원격 이름이 gdrive가 아닌 것
    NL, CR, BS = chr(10), chr(13), chr(92)
    s = conf.replace(CR + NL, NL).replace(CR, NL).replace(BS + 'n', NL).strip()
    if s[:1] in ('"', "'") and s[-1:] == s[:1]:
        s = s[1:-1].strip()
    blobs = []
    def _mask_all(t):                       # 괄호 짝을 세어 JSON({...}, 안의 [...] 포함)을 통째로 가림 (v4.31.4-2)
        out_, i, n = [], 0, len(t)
        while i < n:
            if t[i] != '{':
                out_.append(t[i]); i += 1; continue
            d, j, q = 0, i, None
            while j < n:
                c = t[j]
                if q:
                    if c == BS:
                        j += 1
                    elif c == q:
                        q = None
                elif c == '"':
                    q = c
                elif c == '{':
                    d += 1
                elif c == '}':
                    d -= 1
                    if d == 0:
                        break
                j += 1
            blobs.append(t[i:j + 1]); out_.append(f'__JSON_BLOB_{len(blobs) - 1}__'); i = j + 1
        return ''.join(out_)
    s = _mask_all(s)
    out, sec = [], None
    for part in re.split(r'(?:^|(?<=\s))(?=\[[A-Za-z0-9_.-]+\])', s):
        part = part.strip()
        m = re.match(r'^\[([A-Za-z0-9_.-]+)\]\s*(.*)$', part, flags=re.S)
        if not m:
            continue
        sec, body = m.group(1), m.group(2)
        kv = []
        for ch in re.split(r'(?:^|\s+)(?=[A-Za-z0-9_]+\s*=)', body.strip()):
            ch = ch.strip()
            if '=' not in ch:
                continue
            k, v = (x.strip() for x in ch.split('=', 1))
            for i, bl in enumerate(blobs):
                v = v.replace(f'__JSON_BLOB_{i}__', bl)
            if k == 'token' and v:
                obj, _ = json.JSONDecoder().raw_decode(v[v.index('{'):])   # 뒤에 남은 '}' 등은 버림
                v = json.dumps(obj, separators=(',', ':'))
            kv.append(f'{k} = {v}')
        out += [f'[{sec}]'] + kv + ['']
        if sec != 'gdrive' and len([x for x in out if x.startswith('[')]) == 1:
            out += ['[gdrive]'] + kv + ['']       # KAGGLE_REMOTE 'gdrive:'로도 찾게
    if not out:
        raise SystemExit("⛔ RCLONE_CONF에서 [원격 이름] 줄을 찾지 못했습니다 → 보안 비밀 값이 '[gdrive]'로 시작하는지 확인 (KAGGLE.md 2장)")
    return NL.join(out).strip() + NL

def _kg_install_rclone():
    # v4.31.5: ① 노트북에 붙인 Kaggle 데이터셋의 rclone 파일(/kaggle/input/**/rclone)을 복사 (1~2초)
    #          ② 없으면 내려받기 (브라우저 이름을 붙이고 60초 제한) ③ 그것도 안 되면 GitHub 최신판 ④ apt
    if shutil.which('rclone'):
        return
    import urllib.request, io, glob as _g
    _t0 = time.time()
    dst = '/usr/local/bin/rclone' if os.access('/usr/local/bin', os.W_OK) else os.path.expanduser('~/.local/bin/rclone')
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    def _done(how):
        os.chmod(dst, 0o755)
        os.environ['PATH'] = os.path.dirname(dst) + ':' + os.environ['PATH']
        print(f'rclone 설치: {how} ({time.time() - _t0:.0f}초)')
    src = next((p for p in _g.glob('/kaggle/input/**/rclone', recursive=True) if os.path.isfile(p)), None)
    if src:
        shutil.copy(src, dst)
        return _done(f'붙인 데이터셋 {os.path.dirname(src)}')
    def _get(url, timeout):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/126.0 Safari/537.36'})
        data = urllib.request.urlopen(req, timeout=timeout).read()
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            open(dst, 'wb').write(z.read([n for n in z.namelist() if n.endswith('/rclone')][0]))
    try:
        _get('https://downloads.rclone.org/rclone-current-linux-amd64.zip', 60)
        return _done('downloads.rclone.org')
    except Exception as e:
        print(f'(rclone 받기 실패 downloads.rclone.org: {e})')
    try:                                          # GitHub는 첫 주소가 안 될 때만 물어봄
        tag = json.load(urllib.request.urlopen('https://api.github.com/repos/rclone/rclone/releases/latest', timeout=20))['tag_name']
        _get(f'https://github.com/rclone/rclone/releases/download/{tag}/rclone-{tag}-linux-amd64.zip', 120)
        return _done('github.com')
    except Exception as e:
        print(f'(rclone 받기 실패 github.com: {e})')
    if shutil.which('apt-get'):
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'rclone'], capture_output=True)
        if shutil.which('rclone'):
            print(f'rclone 설치: apt ({time.time() - _t0:.0f}초)')
            return
    raise SystemExit('⛔ rclone을 설치하지 못했습니다 → Kaggle Session options에서 Internet이 On인지 확인하거나, '
                     'rclone 파일을 Kaggle 데이터셋으로 올려 이 노트북에 붙이세요 (Add Input)')

if ON_KAGGLE:
    BASE = KAGGLE_BASE
    if DS_ROOT.startswith('/content'):
        DS_ROOT = f'{LOCAL_TMP}/ds'
    os.makedirs(BASE, exist_ok=True)
    _kg_install_rclone()
    _conf = os.environ.get('RCLONE_CONF') or _kg_secret('RCLONE_CONF')
    if not _conf:
        raise SystemExit('⛔ Kaggle 보안 비밀 RCLONE_CONF가 없습니다 → Add-ons → Secrets에 rclone.conf 내용을 넣고, 이 노트북에 체크해 붙이세요 (KAGGLE.md 2장)')
    _conf = _kg_fix_conf(_conf)
    _cp = Path.home() / '.config/rclone/rclone.conf'
    _cp.parent.mkdir(parents=True, exist_ok=True)
    if not _cp.exists() or _cp.read_text() != _conf:
        _cp.write_text(_conf)
    os.chmod(_cp, 0o600)
    _t0 = time.time()
    _kg_t = []                                     # v4.31.6: 단계별 걸린 시간 (0번 셀이 느릴 때 어디서인지)
    _r = _kg_rc(['lsf', '--max-depth', '1', _kg_remote('scripts')], quiet=True)
    _kg_t.append(f'Drive 확인 {time.time() - _t0:.0f}초')
    if _r.returncode != 0:
        _names = re.findall(r'^\[(.+?)\]', _conf, re.M)
        raise SystemExit(f'⛔ Drive {_kg_remote("scripts")} 를 읽지 못했습니다.\n'
                         f'   rclone 오류: {_r.stderr.strip()[-700:]}\n'
                         f'   설정의 원격 이름 {_names} · KAGGLE_REMOTE {KAGGLE_REMOTE!r}\n'
                         '   → notFound: root_folder_id 확인 · accessNotConfigured: GCP에서 Drive API 사용 · '
                         'invalid_grant: 토큰 만료(OAuth 앱을 프로덕션으로, PC에서 rclone config reconnect gdrive:) (KAGGLE.md 2장)')
    for _what, _ps in (('scripts·config', ('scripts', 'config')),
                       ('학습 기록', ('runs/*/*', 'datasets/*.sig', 'web/label_stats.json')),   # 가중치는 받지 않음 (선택 모델 best.pt는 아래에서)
                       ('모든 best.pt', ('runs/*/weights/best.pt',) if KAGGLE_PULL_ALL_BEST else ()),
                       ('공유 기록', tuple(_KG_SHARED)), ('KAGGLE_PULL', tuple(KAGGLE_PULL)),
                       ('dataset.zip', ('dataset.zip',) if KAGGLE_PULL_DATASET else ())):
        if _ps:
            _ts = time.time()
            drive_pull(*_ps)
            _kg_t.append(f'{_what} {time.time() - _ts:.0f}초')
    if KAGGLE_VIDEOS:
        _ts = time.time()
        drive_get_videos(KAGGLE_VIDEOS)
        _kg_t.append(f'영상 {time.time() - _ts:.0f}초')
    print(f'Kaggle: Drive에서 받기 {time.time() - _t0:.0f}초 ({" · ".join(_kg_t)}) → {BASE}')
    _KG['snap'], _KG['snap_t'] = _kg_snapshot(), time.time()   # 0번 셀이 바꾸는 파일(logs/versions.csv 등)도 끝에 올림
    try:
        _ip = get_ipython()
        for _ev, _fn in (('pre_run_cell', _kg_pre_cell), ('post_run_cell', _kg_post_cell)):
            for _old in [f for f in _ip.events.callbacks[_ev] if getattr(f, '__name__', '') == _fn.__name__]:
                _ip.events.unregister(_ev, _old)
            _ip.events.register(_ev, _fn)
    except NameError:
        pass


# Drive 연결
if not ON_KAGGLE and not os.path.exists('/content/drive/MyDrive') and BASE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
def _find_base(base):
    # BASE가 없으면 공유 바로가기·공유 드라이브에서 같은 이름의 폴더를 찾음
    name = os.path.basename(base.rstrip('/'))
    cands = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=False) + \
            glob.glob(f'/content/drive/.shortcut-targets-by-id/*/{name}') + \
            glob.glob(f'/content/drive/Shareddrives/*/{name}')
    return [c for c in cands if os.path.isdir(f'{c}/scripts')]
if not os.path.isdir(BASE):
    found = _find_base(BASE)
    if found:
        print(f'BASE {BASE} 가 없어 찾은 폴더를 씁니다: {found[0]}')
        BASE = found[0]
    elif CREATE_BASE:
        print(f'새 BASE 폴더를 만듭니다: {BASE}')
    else:
        raise SystemExit(
            f'⛔ {BASE} 폴더가 없습니다.\n'
            '   공유받은 폴더라면: Google Drive → 공유 문서함 → eggplant_yolo 오른쪽 클릭 → 정리 → 바로가기 추가 → 내 드라이브\n'
            '   그다음 메뉴 런타임 → 세션 다시 시작 → 0번 셀 다시. (처음 설치하는 사람이면 CREATE_BASE = True)')
if os.path.isdir(BASE) and not glob.glob(f'{BASE}/scripts/*.py'):
    found = [c for c in _find_base(BASE) if glob.glob(f'{c}/scripts/*.py') and os.path.realpath(c) != os.path.realpath(BASE)]
    msg = (f'⚠️ {BASE}/scripts 에 .py가 없습니다. 공유받은 폴더가 아니라 새로 생긴 빈 폴더일 수 있습니다.\n'
           '   빈 eggplant_yolo 폴더를 지우거나 이름을 바꾼 뒤, 공유받은 폴더의 바로가기를 내 드라이브에 추가하세요.')
    if found:
        msg += f'\n   코드가 있는 같은 이름의 폴더: {found[0]} → BASE를 이 경로로 바꿔도 됩니다.'
    print(msg)
for d in ['scripts', 'config', 'videos', 'runs', 'tasks', 'backups', 'logs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# 패키지: 없을 때만 설치 (세션마다 확인)
from importlib import metadata
def _ver(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

CVAT_HOST = CVAT_PROFILES[CVAT_AUTH if CVAT_AUTH in CVAT_PROFILES else next(iter(CVAT_PROFILES))]['host']   # 처음 접속할 서버 (셀에서 바뀜)

def cvat_server_version(host=None):
    # CVAT 서버 버전 (/api/server/about, 로그인 불필요). 실패하면 None
    try:
        import requests
        r = requests.get(f'{host or CVAT_HOST}/api/server/about', timeout=15)
        return r.json().get('version') if r.ok else None
    except Exception:
        return None

CVAT_LAST_KNOWN = '2.79.0'                   # 서버 버전을 못 읽을 때 쓸, 마지막으로 시험한 버전
SERVER_VER = cvat_server_version()
if CVAT_SDK_VERSION == 'auto':
    _base = SERVER_VER or CVAT_LAST_KNOWN
    _mj, _mn = _base.split('.')[:2]
    cvat_spec, cvat_prefix = f'cvat-cli~={_mj}.{_mn}.0', f'{_mj}.{_mn}.'   # 같은 X.Y의 최신 패치
else:
    cvat_spec, cvat_prefix = f'cvat-cli=={CVAT_SDK_VERSION}', CVAT_SDK_VERSION
_cli, _sdk = _ver('cvat-cli'), _ver('cvat-sdk')
need = [p for p, mod in [('ultralytics', 'ultralytics'), ('lap', 'lap'), ('av', 'av')] if importlib.util.find_spec(mod) is None]
if not (_cli and _sdk and _cli == _sdk and _cli.startswith(cvat_prefix)):
    need.append(cvat_spec)
if need and not os.environ.get('SKIP_INSTALL'):
    print('설치:', need)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need])
    if r.returncode != 0 and cvat_spec in need:
        # SaaS가 PyPI보다 먼저 올라간 경우 등: 같은 X.Y가 아직 없으면 최신 cvat-cli
        print(f'⚠️ {cvat_spec} 설치 실패 → 최신 cvat-cli 설치')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'cvat-cli',
                        *[p for p in need if p != cvat_spec]], check=True)
    importlib.invalidate_caches()
    if 'cvat_sdk' in sys.modules and sys.modules['cvat_sdk'].__version__ != _ver('cvat-sdk'):
        print('⚠️ 이 세션에 이전 cvat-sdk가 이미 불러져 있습니다 → 메뉴 런타임 → 세션 다시 시작 후 0번 셀을 다시 실행하세요')
elif need:
    print('설치 필요(SKIP_INSTALL이라 건너뜀):', need)
import torch, ultralytics
CVAT_CLI_VER, CVAT_SDK_VER = _ver('cvat-cli'), _ver('cvat-sdk')

def cvat_compat(server_ver=None):
    # cvat-sdk는 서버가 SDK와 같은 X.Y 또는 한 단계 위(X.Y+1)일 때 공식 지원
    server_ver = server_ver or SERVER_VER
    if not (server_ver and CVAT_SDK_VER):
        return '확인 불가'
    s, k = [int(x) for x in server_ver.split('.')[:2]], [int(x) for x in CVAT_SDK_VER.split('.')[:2]]
    if s == k:
        return '일치'
    if s[0] == k[0] and s[1] == k[1] + 1:
        return '지원 범위 (서버가 한 단계 위)'
    return '⚠️ 지원 범위 밖 — CVAT_SDK_VERSION을 서버 버전에 맞추거나 auto로'

# 버전이 바뀔 때만 logs/versions.csv에 한 줄 (문제가 생겼을 때 언제부터인지 추적)
_vrow = [SERVER_VER or '?', CVAT_SDK_VER or '?', CVAT_CLI_VER or '?', ultralytics.__version__]
_vlog = f'{BASE}/logs/versions.csv'
os.makedirs(f'{BASE}/logs', exist_ok=True)
_last = open(_vlog, encoding='utf-8').read().strip().splitlines()[-1].split(',')[1:] if os.path.exists(_vlog) else None
if _last != _vrow:
    import datetime
    with open(_vlog, 'a', encoding='utf-8') as fh:
        if _last is None:
            fh.write('time,cvat_server,cvat_sdk,cvat_cli,ultralytics\n')
        from zoneinfo import ZoneInfo
        fh.write(','.join([datetime.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d %H:%M')] + _vrow) + '\n')

# 데이터셋: dataset.zip이 바뀌었을 때만 다시 풂
DS, yaml_path = None, None
zpath = f'{BASE}/dataset.zip'
stamp = f'{DS_ROOT}/.source_mtime'
if os.path.exists(zpath):
    mt = str(os.path.getmtime(zpath))
    if not (os.path.exists(stamp) and open(stamp).read() == mt):
        shutil.rmtree(DS_ROOT, ignore_errors=True)
        zipfile.ZipFile(zpath).extractall(DS_ROOT)
        open(stamp, 'w').write(mt)
        print('데이터셋 풀기 완료')
    yaml_path = glob.glob(f'{DS_ROOT}/**/data.yaml', recursive=True)[0]
    DS = os.path.dirname(yaml_path)
    txt = open(yaml_path, encoding='utf-8').read()
    txt = re.sub(r'^path:.*$', f'path: {DS}', txt, flags=re.M)     # PC 경로 → Colab 경로
    open(yaml_path, 'w', encoding='utf-8').write(txt)

# 쓸 모델 정하기 (MODEL_CHOICE)
_trained = sorted(glob.glob(f'{BASE}/runs/*/TRAINING_DONE'), key=os.path.getmtime)
if MODEL_CHOICE == 'latest':
    NAME = os.path.basename(os.path.dirname(_trained[-1])) if _trained else MODEL_NAMES.get(1, 'fruit_1920')
elif isinstance(MODEL_CHOICE, int):
    assert MODEL_CHOICE in MODEL_NAMES, f'MODEL_CHOICE {MODEL_CHOICE}가 MODEL_NAMES 번호 {list(MODEL_NAMES)}에 없습니다'
    NAME = MODEL_NAMES[MODEL_CHOICE]
else:
    NAME = str(MODEL_CHOICE)

# 그 모델을 15번 셀로 학습했다면, 그때 만든 데이터셋(datasets/<NAME>.zip)을 DS로 → 2·3번 셀이 같은 데이터로 확인
if ON_KAGGLE and KAGGLE_PULL_DATASET:
    drive_pull(f'datasets/{NAME}.zip')           # Kaggle: 선택 모델의 데이터셋
_ds_zip = f'{BASE}/datasets/{NAME}.zip'
DS_SOURCE = 'dataset.zip' if DS else '없음'
if os.path.exists(_ds_zip):
    _ds_dir = f'{DS_ROOT}_exp/{NAME}'
    if not os.path.exists(f'{_ds_dir}/data.yaml'):
        shutil.rmtree(_ds_dir, ignore_errors=True)
        zipfile.ZipFile(_ds_zip).extractall(_ds_dir)
    yaml_path, DS = f'{_ds_dir}/data.yaml', _ds_dir
    _t = open(yaml_path, encoding='utf-8').read()
    open(yaml_path, 'w', encoding='utf-8').write(re.sub(r'^path:.*$', f'path: {DS}', _t, flags=re.M))
    DS_SOURCE = f'datasets/{NAME}.zip'

# 경로와 상태
RUN_DIR = f'{BASE}/runs/{NAME}'
BEST, LAST, DONE = f'{RUN_DIR}/weights/best.pt', f'{RUN_DIR}/weights/last.pt', f'{RUN_DIR}/TRAINING_DONE'
if ON_KAGGLE and not os.path.exists(BEST):        # v4.31.6: 선택 모델의 best.pt만 받음 (다른 모델은 셀이 쓸 때 need_file로)
    _ts = time.time()
    drive_pull(f'runs/{NAME}/weights/best.pt', quiet=True)
    print(f'Kaggle: 선택 모델 {NAME}/best.pt 받기 {time.time() - _ts:.0f}초' + ('' if os.path.exists(BEST) else ' — Drive에 없음'))
SCRIPTS = f'{BASE}/scripts'
VIDEOS, TASKS_DIR, REGISTRY = f'{BASE}/videos', f'{BASE}/tasks', f'{BASE}/task_registry.csv'
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
try:
    import cvat_task_tools as ctt
    importlib.reload(ctt)
except ImportError:
    ctt = None
if ctt is not None and hasattr(ctt, 'REVIEW_WORDS'):
    ctt.REVIEWED_RULE = REVIEWED_RULE                # 검색어 '검토' = 이 규칙의 검토 완료 TASK (v4.24)
try:                                            # 웹 관리 화면 연동 (v4.14): 상태 파일·웹 요청
    import web_bridge as wb
    importlib.reload(wb)
except ImportError:
    wb = None
_kg_patch_ctt()                                 # Kaggle: 영상 찾기·학습 잠금이 Drive와 통하게 (Colab에서는 아무것도 안 함)
if ON_KAGGLE and KAGGLE_VIDEOS and ctt is not None:
    drive_get_videos(KAGGLE_VIDEOS)

def epochs_done():
    p = f'{RUN_DIR}/results.csv'
    return max(0, sum(1 for _ in open(p)) - 1) if os.path.exists(p) else 0

def train_status():
    if os.path.exists(DONE):
        return '완료'
    if os.path.exists(LAST):
        return f'중단됨 ({epochs_done()}/{EPOCHS} 에폭) → 1번 셀이 이어서 학습'
    return '시작 전'

def _secret(name):
    # Colab 보안 비밀(왼쪽 열쇠 아이콘) → 없으면 None. 값은 출력하지 않음
    # Kaggle에서는 Kaggle 보안 비밀(Add-ons → Secrets), 둘 다 없으면 환경 변수
    # v4.31.9 (F11): Colab은 Colab 보안 비밀만 · Kaggle은 Kaggle 보안 비밀만 (환경 변수는 둘 다 아닌 로컬 실행에서만)
    v = None
    if ON_KAGGLE:
        v = _kg_secret(name)
    else:
        try:
            from google.colab import userdata
            try:
                v = userdata.get(name)
            except Exception:
                v = None
        except ImportError:
            v = os.environ.get(name)
    return v.strip().strip("'\"") if v else None

def _may_prompt():
    # v4.31.9 (F11): 보안 비밀이 없을 때 입력창으로 받는 것은 Colab에서 사람이 셀을 직접 실행할 때만 (Kaggle·작업자 실행 중에는 멈춤)
    w = globals().get('_W')
    return not ON_KAGGLE and not (isinstance(w, dict) and not w.get('stop', True))

def _need(what, m):
    if not _may_prompt():
        raise SystemExit(f"⛔ [{m}] {what}이(가) 보안 비밀에 없습니다 → {'Kaggle Add-ons → Secrets' if ON_KAGGLE else 'Colab 보안 비밀'}에 넣고 이 노트북에 붙이세요 "
                         "(Kaggle·작업자 실행에서는 직접 입력을 받지 않음)")

def prof_login(name):
    # 프로필의 로그인 방식 ('token' 또는 'password')
    p = CVAT_PROFILES[name]
    return p.get('login') or ('token' if 'token' in p else 'password')

def _has_secret(name):
    p = CVAT_PROFILES[name]
    return bool(_secret(p['token'])) if prof_login(name) == 'token' else bool(_secret(p['user']) and _secret(p['pass']))

def cvat_mode(mode=None):
    # 쓸 프로필: 셀의 LOGIN → 지금 접속 중인 프로필 → 0번 셀 CVAT_AUTH ('auto' = 보안 비밀이 있는 첫 프로필)
    m = mode or globals().get('CVAT_LOGIN_MODE') or CVAT_AUTH
    if m == 'auto':
        m = next((k for k in CVAT_PROFILES if _has_secret(k)), next(iter(CVAT_PROFILES)))
    assert m in CVAT_PROFILES, f"LOGIN은 'auto' 또는 0번 셀 CVAT_PROFILES의 이름 {list(CVAT_PROFILES)} 중 하나 (지금 값: {m!r})"
    return m

_CRED_TYPED = globals().get('_CRED_TYPED', {})   # 보안 비밀이 없어 입력창으로 받은 값 (이 세션 메모리에만, 0번 셀을 다시 실행해도 유지)

def cvat_auth(mode=None):
    # 프로필의 (로그인 방식, 로그인 정보): ('password', (아이디, 비밀번호)) 또는 ('token', PAT)
    from getpass import getpass
    m = cvat_mode(mode)
    prof, kind = CVAT_PROFILES[m], prof_login(m)
    if kind == 'password':
        u = _secret(prof['user']) or _CRED_TYPED.get((m, 'u')) or (_need(prof['user'], m) or input(f"[{m}] CVAT 아이디 ({prof['host']}): "))
        p = _secret(prof['pass']) or _CRED_TYPED.get((m, 'p')) or (_need(prof['pass'], m) or getpass(f'[{m}] CVAT 비밀번호: '))
        _CRED_TYPED[(m, 'u')], _CRED_TYPED[(m, 'p')] = u, p
        return kind, (u, p)
    t = _secret(prof['token']) or _CRED_TYPED.get((m, 't')) or (_need(prof['token'], m) or getpass(f"[{m}] CVAT Personal Access Token ({prof['host']}): "))
    _CRED_TYPED[(m, 't')] = t
    return kind, re.sub(r'^(Token|Bearer)\s+', '', t.strip(), flags=re.IGNORECASE)

def cvat_client(mode=None):
    # cvat-sdk 클라이언트 (그 프로필의 서버·계정으로 로그인)
    m = cvat_mode(mode)
    kind, v = cvat_auth(m)
    host = CVAT_PROFILES[m]['host']
    cl = ctt.connect(host, credentials=v) if kind == 'password' else ctt.connect(host, v)
    apply_scope(cl, m)                          # 기본 조직·프로젝트 (v4.19)
    return cl

_SCOPE_TMP = {}                                  # 웹 요청이 정한 이번 작업만의 조직 (요청이 끝나면 비움)
CVAT_SCOPE = globals().get('CVAT_SCOPE') or {}   # 지금 쓰는 조직·프로젝트 (use_cvat가 정함)

def scope_want(mode=None):
    # 쓸 조직·프로젝트: 웹 요청 → 프로필의 'org'·'project' → 0번 셀 CVAT_ORG·CVAT_PROJECT
    p = CVAT_PROFILES.get(cvat_mode(mode), {})
    if 'org' in _SCOPE_TMP:
        return _SCOPE_TMP['org'], _SCOPE_TMP.get('project', '')
    return p.get('org', globals().get('CVAT_ORG')), p.get('project', globals().get('CVAT_PROJECT', ''))

def apply_scope(cl, mode=None):
    # 클라이언트에 조직·프로젝트 범위를 붙임 (못 찾으면 있는 조직·프로젝트 목록과 함께 멈춤)
    org, proj = scope_want(mode)
    if not hasattr(ctt, 'set_scope'):
        return None
    try:
        return ctt.set_scope(cl, org, proj)
    except ValueError as e:
        raise SystemExit(f'⛔ {e}\n→ 0번 셀 CVAT_ORG·CVAT_PROJECT (또는 프로필의 org·project)를 고치고 0번 셀을 다시 실행하세요')

def cvat_cli_auth(mode=None):
    # cvat-cli에 넘길 인증 인자와 환경 변수 (비밀번호·토큰은 환경 변수로만 전달)
    kind, v = cvat_auth(mode)
    return (['--auth', v[0]], {'PASS': v[1]}) if kind == 'password' else ([], {'CVAT_ACCESS_TOKEN': v})

_CVAT_SESSIONS = globals().get('_CVAT_SESSIONS', {})   # 프로필별 접속 (0번 셀을 다시 실행해도 유지, 로그인 정보가 바뀌면 다시 로그인)

def cvat_forget(name=None):
    # 저장된 접속을 지움 (name=None이면 전부) → 다음 셀에서 다시 로그인
    for k in ([name] if name else list(_CVAT_SESSIONS)):
        _CVAT_SESSIONS.pop(k, None)
    for k in [k for k in _CRED_TYPED if name is None or k[0] == name]:
        _CRED_TYPED.pop(k, None)

def use_cvat(login=None):
    # 이 셀이 쓸 계정·서버를 정함: LOGIN=None이면 지금 그대로, 프로필 이름이면 그 계정으로 바꿈
    global CLIENT, CVAT_LOGIN_MODE, CVAT_HOST, CVAT_SERVER, CVAT_SCOPE
    import hashlib
    m = cvat_mode(login)
    host, kind = CVAT_PROFILES[m]['host'], prof_login(m)
    cred_sig = hashlib.sha256(repr((host,) + cvat_auth(m)).encode()).hexdigest()[:16]   # 값 자체는 저장 안 함
    sess = _CVAT_SESSIONS.get(m)
    if sess is None or sess[3] != cred_sig:
        if sess is not None:
            print(f'[{m}] 로그인 정보가 바뀌어 다시 로그인합니다.')
        def _login():
            cl_ = cvat_client(m)
            return cl_, cl_.api_client.users_api.retrieve_self()[0].username     # 로그인이 실제로 되는지 확인
        try:                                     # v4.31.4: 503·429·연결 오류는 1·2·4·8분 뒤 다시 (토큰 문제 아님)
            cl, user = ctt.with_retry(_login, what=f'CVAT 접속 [{m}]') if hasattr(ctt, 'with_retry') else _login()
        except Exception as e:
            e = e.__cause__ or e
            body = str(getattr(e, 'body', '') or '')[:200]
            _sh = ctt.cvat_error_hint(e) if hasattr(ctt, 'cvat_error_hint') else ''
            hint = _sh if _sh and '인증' not in _sh else (f"PAT가 만료됐거나 Read only인지, 보안 비밀 {CVAT_PROFILES[m].get('token')} 이름·값이 맞는지 확인하세요."
                    if kind == 'token' else
                    f"아이디·비밀번호가 틀렸습니다(보안 비밀 {CVAT_PROFILES[m].get('user')}·{CVAT_PROFILES[m].get('pass')} 값, 앞뒤 공백, "
                    f"비밀번호 변경 여부). 입력창으로 넣었다면 cvat_forget('{m}') 후 다시. "
                    'Google 로그인으로 만든 계정은 비밀번호가 없으므로 PAT 프로필로 접속하세요.')
            status = getattr(e, 'status', '')
            raise RuntimeError(f'CVAT 접속 실패 [{m} → {host}] ({type(e).__name__} {status}: {body or str(e)[:200]})\n→ {hint}') from None
        sv = cvat_server_version(host)
        _CVAT_SESSIONS[m] = (cl, f'{user}@{urlparse(host).hostname}', sv, cred_sig)
        if sv and cvat_compat(sv).startswith('⚠️'):
            print(f'⚠️ 이 서버({sv})는 설치된 cvat-sdk {CVAT_SDK_VER}의 지원 범위 밖입니다. 일부 기능이 안 될 수 있습니다.')
    CLIENT, CVAT_SERVER, _sv, _ = _CVAT_SESSIONS[m]
    CVAT_LOGIN_MODE, CVAT_HOST = m, host
    CVAT_SCOPE = apply_scope(CLIENT, m) or {}
    print(f"▶ CVAT 서버: {CVAT_SERVER} · 프로필 {m} ({'PAT' if kind == 'token' else 'ID/PASS'}) · 버전 {_sv or '?'}")
    if hasattr(ctt, 'scope_text'):
        print(f"  범위: {ctt.scope_text(CVAT_SCOPE)}" + (' ← 웹 요청의 조직 (이 작업만)' if 'org' in _SCOPE_TMP else ''))
    return CLIENT

def task_dir(tid):
    # TASK별 작업 폴더. app.cvat.ai가 아닌 서버는 앞에 서버 이름을 붙여 TASK ID가 겹쳐도 섞이지 않게
    host = urlparse(CVAT_HOST).hostname or ''
    tag = '' if host == 'app.cvat.ai' else re.sub(r'[^A-Za-z0-9]+', '-', host) + '_'
    d = f'{TASKS_DIR}/{tag}{tid}'
    if ON_KAGGLE and d not in _KG['tasks']:      # Kaggle: 이 TASK의 작업 폴더를 처음 쓸 때 Drive에서 받음
        _KG['tasks'].add(d)
        drive_pull(os.path.relpath(d, BASE), quiet=True)
    return d

def need_file(p):
    # v4.31.6 Kaggle: BASE 안의 파일(다른 모델의 best.pt 등)이 없으면 그때 Drive에서 받음 → p · Colab은 그대로
    if ON_KAGGLE and p and str(p).startswith(BASE + '/') and not os.path.exists(str(p)):
        drive_pull(os.path.relpath(str(p), BASE), quiet=True)
    return p

def task_reviewed(name):
    # 검토 완료 TASK인가 (0번 셀 REVIEWED_RULE: 이름의 상태에 그 글자가 있으면)
    return ctt.is_reviewed(name, REVIEWED_RULE) if ctt is not None else True

def task_role(name):
    # 'val' / 'train' / '규칙 밖'(이름 규칙에 안 맞음 → 학습에서는 train) / ''(VAL_RULE 없음)
    return ctt.split_role(name, VAL_RULE) if ctt is not None else ''

def load_script(name):
    # Drive scripts/의 .py를 노트북 안으로 불러옴 (자동 실행 안 됨 → 설정을 바꾼 뒤 함수 호출)
    spec = importlib.util.spec_from_file_location(os.path.splitext(name)[0], f'{SCRIPTS}/{name}')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod

def run(cmd, log=None, env=None):
    # 명령 실행: 화면에 보여 주고, log 경로가 있으면 같은 내용을 파일에도 남김. env = 추가 환경 변수(화면에 안 나옴)
    print('$', ' '.join(map(str, cmd)))
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})   # 진행 출력을 바로바로 보여 줌
    fh = open(log, 'a', encoding='utf-8') if log else None
    if fh:
        fh.write('$ ' + ' '.join(map(str, cmd)) + '\n')
    for line in p.stdout:
        print(line, end='')
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    return p.returncode

def model_label():
    # 대장에 남길 모델 표시: runs/<NAME> + best.pt 수정 시각
    if not os.path.exists(BEST):
        return ''
    import datetime
    return f"{NAME}/best.pt ({datetime.datetime.fromtimestamp(os.path.getmtime(BEST)).strftime('%m-%d %H:%M')})"

# ---------- 학습 잠금 (v4.9): 같은 runs/<이름>에 두 사람이 동시에 쓰지 않게 ----------
import uuid, time
if 'SESSION_ID' not in globals():
    SESSION_ID = uuid.uuid4().hex[:8]          # 이 Colab 세션의 표시 (0번 셀을 다시 실행해도 그대로)
if ctt is not None and hasattr(ctt, 'set_shard'):   # v4.31.0: 함께 쓰는 기록은 세션마다 따로 (logs/shards/<이름>/<세션>.csv) → 여러 세션이 동시에 써도 줄이 사라지지 않음
    ctt.set_shard(SESSION_ID)
TAKEOVER_LOCK = globals().get('TAKEOVER_LOCK', False)

def who_am_i():
    return OPERATOR or f'이름없음-{SESSION_ID}'

def lock_run(run_dir, stage):
    # 잠금을 잡음. 다른 사람이 학습 중이면 멈춤(SystemExit)
    held = (ctt.read_run_lock(run_dir) or {}).get('session') == SESSION_ID   # 이미 이 세션이 잡고 있음
    ok, msg = ctt.acquire_run_lock(run_dir, SESSION_ID, who_am_i(), stage, LOCK_STALE_MIN, TAKEOVER_LOCK)
    if not ok:
        raise SystemExit(msg)
    if msg:
        print('ℹ️', msg)
    if LOCK_CONFIRM_SEC and not held:
        print(f'학습 잠금: {os.path.basename(run_dir)} ({who_am_i()}, 세션 {SESSION_ID}) — {LOCK_CONFIRM_SEC}초 뒤 다시 확인 …')
        time.sleep(LOCK_CONFIRM_SEC)
        cur = ctt.read_run_lock(run_dir)
        if cur and cur.get('session') != SESSION_ID:
            raise SystemExit(f"⛔ 거의 같은 때 {cur.get('who')}(세션 {cur.get('session')})도 이 이름으로 시작했습니다 → 이 세션은 멈춥니다. "
                             '다른 TRAIN_NAME을 쓰세요.')

def unlock_run(run_dir):
    ctt.release_run_lock(run_dir, SESSION_ID)

def _lock_callbacks(model, run_dir):
    # 에폭마다 잠금 갱신. 다른 세션이 잠금을 가져갔으면 즉시 멈춤 (결과가 섞이지 않게)
    def hb(trainer):
        ep = int(getattr(trainer, 'epoch', -1)) + 1
        if not ctt.touch_run_lock(run_dir, SESSION_ID, who=who_am_i(), stage='train', epoch=ep):
            raise RuntimeError(f'⛔ 다른 세션이 runs/{os.path.basename(run_dir)} 잠금을 가져갔습니다 → 이 학습을 멈춥니다 (결과 섞임 방지)')
    for ev in ('on_pretrain_routine_end', 'on_train_epoch_start', 'on_fit_epoch_end'):
        model.add_callback(ev, hb)
    if ON_KAGGLE:                                # Kaggle: 에폭마다 runs/<이름>을 Drive에 올림 (세션이 끊겨도 이어서 학습)
        model.add_callback('on_fit_epoch_end', lambda tr: drive_push_bg(os.path.relpath(run_dir, BASE)))

def _train_multi_gpu(m, kw, run_dir, n_done, start):
    # v4.30: GPU 여럿으로 동시 학습 (Ultralytics DDP). 학습은 다른 프로세스에서 돌아 에폭 콜백(잠금 갱신·Drive 올리기)이 닿지 않음
    #   → 여기서 1분마다 잠금을 갱신하고 (Kaggle) runs/<이름>을 Drive로. 시작하지 못하면(에폭 0) GPU 하나로 다시 학습
    import threading as _th
    stop = _th.Event()
    def beat():
        while not stop.wait(60):
            try:
                ctt.touch_run_lock(run_dir, SESSION_ID, who=who_am_i(), stage='train', epoch=n_done())
                if ON_KAGGLE:
                    drive_push_bg(os.path.relpath(run_dir, BASE))
            except Exception:
                pass
    _th.Thread(target=beat, daemon=True).start()
    print(f"GPU {kw['device'].count(',') + 1}개로 동시 학습 (device={kw['device']}, batch={kw.get('batch')}) — 에폭마다 결과는 runs/{os.path.basename(run_dir)}/results.csv")
    try:
        m.train(**kw)
    except Exception as e:
        if n_done() > 0:
            raise
        print(f"⚠️ GPU 여럿으로 학습을 시작하지 못했습니다 ({type(e).__name__}: {str(e)[:200]}) → GPU 하나로 다시 (batch = 0번 셀 BATCH)")
        kw.update(device=0, batch=BATCH)
        from ultralytics import YOLO
        m2 = YOLO(start or MODEL); _lock_callbacks(m2, run_dir); m2.train(**kw)
    finally:
        stop.set()

def run_training(name, data_yaml, data_sig='', start=None, train_args=None):
    # start = 시작 가중치 (None = 0번 셀 MODEL) · train_args = Ultralytics 학습 설정 덮어쓰기 (lr0·optimizer·epochs 등, v4.24.3)
    # 잠금을 잡고 학습 (완료=건너뜀, 중단=이어서, 처음=새로). 1번·15번 셀이 함께 씀
    from ultralytics import YOLO
    run_dir = f'{BASE}/runs/{name}'
    last, done = f'{run_dir}/weights/last.pt', f'{run_dir}/TRAINING_DONE'
    if os.path.exists(done):
        print('이미 학습 완료 → 건너뜁니다.', open(done, encoding='utf-8').read())
        return False
    lock_run(run_dir, 'train')
    try:
        sp = f'{run_dir}/DATASET_SIG'
        if data_sig:
            old = open(sp).read().strip() if os.path.exists(sp) else ''
            if old and old != data_sig:
                raise SystemExit(f'⛔ runs/{name}은(는) 다른 데이터({old})로 학습하던 이름입니다 (지금 {data_sig}) → TRAIN_NAME을 바꾸세요')
            open(sp, 'w').write(data_sig)
        n_done = lambda: max(0, sum(1 for _ in open(f'{run_dir}/results.csv')) - 1) if os.path.exists(f'{run_dir}/results.csv') else 0
        try:
            if os.path.exists(last):
                print(f'이어서 학습: {n_done()}/{EPOCHS} 에폭부터')
                m = YOLO(last); _lock_callbacks(m, run_dir); m.train(resume=True)
            else:
                m = YOLO(start or MODEL); _lock_callbacks(m, run_dir)
                _kw = dict(data=data_yaml, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                           device=DEVICE, project=f'{BASE}/runs', name=name, exist_ok=True)
                _kw.update(train_args or {})
                if isinstance(_kw.get('device'), str) and ',' in _kw['device']:
                    _train_multi_gpu(m, _kw, run_dir, n_done, start)       # v4.30: GPU 여럿 (Kaggle T4×2)
                else:
                    m.train(**_kw)
        except AssertionError as e:            # 이미 끝난 학습을 resume하려 할 때
            if 'nothing to resume' not in str(e) and 'is finished' not in str(e):
                raise
            print('학습이 이미 끝나 있었습니다.')
        import datetime as _d
        open(done, 'w', encoding='utf-8').write(f'{n_done()} epochs · data {name} · {who_am_i()} · 세션 {SESSION_ID} · '
                                                f'{_d.datetime.now().strftime("%Y-%m-%d %H:%M")}')
        return True
    finally:
        unlock_run(run_dir)

def upload_log(tid, method, model_path=None, echo=print, **ctx):
    # 업로드 한 건의 상세 기록 시작 → .say(메시지), .set(값=…), .finish(상태, 오류)
    #   tasks/<ID>/upload_<시각>.json·.log + logs/uploads.csv (한 줄씩 누적)
    mp = model_path or BEST
    mt, md5 = ctt.file_fingerprint(mp) if mp and os.path.exists(mp) else ('', '')
    return ctt.UploadLog(BASE, task_dir(tid), tid, method, echo=echo, server=CVAT_SERVER, profile=CVAT_LOGIN_MODE,
                         cvat_user=str(CVAT_SERVER).split('@')[0], operator=who_am_i(), session=SESSION_ID,
                         app_version=APP_VERSION, model=os.path.relpath(mp, BASE) if mp else '',
                         model_time=mt, model_md5=md5, **ctx)

def track_settings_of(xml_rel):
    # 5번 셀이 track을 만들 때 남긴 설정 (track_settings.json). 없으면 {}
    p = os.path.join(os.path.dirname(f'{BASE}/{xml_rel}'), 'track_settings.json')
    try:
        return json.load(open(p, encoding='utf-8'))
    except Exception:
        return {}

def change_report(task, before_zip, tag='after', show=True, max_frames=40):
    # 업로드 뒤 라벨을 export해 업로드 전 백업과 비교 → 라벨별·프레임별 '기존→새' 박스 수. CSV는 tasks/<ID>/change_<시각>.csv
    import pandas as pd
    from IPython.display import display
    folder = task_dir(task.id)
    after = ctt.export_annotations(task, f'{folder}/{tag}_{ctt.stamp()}.zip')
    rows, frows, text = ctt.label_change_report(before_zip, after)
    csv_path = ctt.save_change_report(rows, frows, f'{folder}/change_{ctt.stamp()}.csv')
    res = {'text': text, 'rows': rows, 'frames': frows, 'csv': os.path.relpath(csv_path, BASE), 'after': os.path.relpath(after, BASE)}
    if show:
        show_change(task.id, res, max_frames)
    return res

def show_change(tid, res, max_frames=40):
    import pandas as pd
    from IPython.display import display
    print(f"■ TASK {tid} 라벨 변화: {res['text']}")
    display(pd.DataFrame(res['rows']).set_index('라벨'))
    fr = res['frames']
    if fr:
        print(f"프레임별 박스 수 (기존→새){'' if len(fr) <= max_frames else f' — {len(fr)}프레임 중 앞 {max_frames}개, 전체는 CSV'}:")
        display(pd.DataFrame(fr[:max_frames]).set_index('프레임'))
    print(f"   저장: {res['csv']} (라벨별·프레임별 전체) · 업로드 후 라벨 {res['after']}")

_REQ = globals().get('_REQ') or {'next': None}   # 이 세션에서 받은 웹 요청 → 다음 작업 기록(job)에 묶임

def req_use(kind):
    # 웹 요청 받기 (셀의 FROM_REQUESTS = True): 대기 요청을 '진행 중'으로 바꾸고 TASK ID 목록(학습은 설정 dict)을 돌려줌.
    # 그 셀의 작업 기록이 끝날 때(job_end) 요청이 '완료'/'실패'로 바뀜 → 웹 화면 '요청' 탭에 보임
    if wb is None:
        raise SystemExit('⛔ scripts/web_bridge.py가 없습니다 (v4.14 scripts로 바꾸세요)')
    reqs = wb.take(BASE, kind, who_am_i(), SESSION_ID, limit=1 if kind in ('train', 'cleanup') else None,
                   **({'host': globals().get('REQ_HOST') or ('Kaggle' if ON_KAGGLE else 'Colab')} if 'host' in wb.take.__code__.co_varnames else {}))
    if not reqs:
        raise SystemExit(f"⛔ 받을 웹 요청이 없습니다 ({wb.KINDS[kind][0]}) → 웹 화면에서 요청하거나 FROM_REQUESTS = False")
    _REQ['next'] = (kind, [r['id'] for r in reqs])
    _REQ['params'] = dict(reqs[0].get('params') or {})   # 요청의 설정 (export 형식, 학습 규칙 등)
    for r in reqs:
        print(f"📨 웹 요청 {r['id']} · {wb.KINDS[kind][0]} · {r.get('by', '')} · TASK {len(r.get('task_ids') or [])}개"
              + (f" · {r['note']}" if r.get('note') else ''))
    _SCOPE_TMP.clear()
    if kind != 'train' and _REQ['params'].get('org') is not None:      # 웹에서 고른 TASK들의 조직으로 (프로젝트는 안 따짐) — v4.19
        _SCOPE_TMP.update(org=_REQ['params']['org'], project='')
        print(f"   → 조직: {_REQ['params']['org'] or '개인 작업공간'} (웹 요청의 TASK 조직, 이 작업이 끝나면 0번 셀 기본값으로)")
    if kind in ('train', 'cleanup'):                 # 설정 dict (모델 정리는 v4.24.8)
        return dict(reqs[0].get('params') or {})
    ids = wb.task_ids_of(reqs)
    print(f"   → TASK {len(ids)}개: {ids[:10]}{' …' if len(ids) > 10 else ''}")
    return ids

def _req_bind(job, action):
    nxt = _REQ.get('next')
    if job is not None and nxt and wb is not None and wb.KINDS[nxt[0]][2] == action:
        job['req'], _REQ['next'] = nxt, None

def _req_finish(job, status, summary):
    req = (job or {}).get('req')
    if req:
        _SCOPE_TMP.clear()                       # 웹 요청의 조직 → 다시 0번 셀 기본값
    if req and wb is not None:
        ok = status in ('끝', '완료')
        try:
            done = wb.finish(BASE, req[1], ok=ok, summary=summary or status, job_id=job.get('job_id', ''), who=who_am_i())
            _w = sum(1 for d in done if d.get('status') == 'waiting')
            print(f"📨 웹 요청 {len(done)}건 → " + ('실패' if not ok else '완료' if not _w else
                  f"완료 {len(done) - _w} · 받는 중 {_w} (CVAT가 아직 준비 중 → 다 받으면 완료)") + " (웹 화면 '요청' 탭)")
        except Exception as e:
            print(f'(웹 요청 완료 표시 실패: {e})')

def job_start(cell, action, settings=None, summary=''):
    # 시계열 작업 기록(logs/activity.csv)에 작업 시작 — 이 셀의 TASK별 결과가 이 작업에 묶임 (26번 셀로 보기)
    if ctt is not None and hasattr(ctt, 'start_job'):
        try:
            _j = ctt.start_job(BASE, cell, action, who=who_am_i(), session=SESSION_ID,
                               server=globals().get('CVAT_SERVER', ''), model=model_label() if os.path.exists(BEST) else '',
                               settings=settings, summary=summary)
            _req_bind(_j, action)
            return _j
        except Exception as e:
            print(f'(작업 기록 시작 실패: {e})')

def job_end(status='끝', summary='', **kw):
    if ctt is not None and hasattr(ctt, 'current_job') and ctt.current_job():
        try:
            _j = ctt.current_job()
            ctt.end_job(status, summary, **kw)
            _req_finish(_j, status, summary)
        except Exception as e:
            print(f'(작업 기록 끝 실패: {e})')

def lock_status(name=None):
    # 학습 잠금 상태 한 줄
    d = ctt.read_run_lock(f'{BASE}/runs/{name or NAME}') if ctt else None
    if not d:
        return '없음'
    mine = ' (이 세션)' if d.get('session') == SESSION_ID else ''
    return (f"{d.get('who')}{mine} · 단계 {d.get('stage')} {d.get('epoch', '')} · 마지막 갱신 {d['age_min']:.0f}분 전"
            + ('  ← 오래됨(끊긴 학습)' if d['age_min'] >= LOCK_STALE_MIN else ''))

if ctt is not None and hasattr(ctt, 'import_old_records'):   # 처음 한 번: 예전 기록을 작업 기록(logs/activity.csv)으로
    try:
        ctt.import_old_records(BASE)
    except Exception as _e:
        print(f'(예전 기록 옮기기 실패: {_e})')
# ---------------- v4.31.5: ULD(CVAT_UpLoader) 폴더 읽기 · 라벨 업로드 뒤 이름 규칙 ----------------
# ULD 노트북이 내 드라이브 최상위 CVAT_UpLoader/ 에 남긴 기록(registry/tasks/<TASK ID>.json)과 clip(clips/…mp4)을 읽기만 함.
# 가지 앱은 그 폴더에 아무것도 쓰지 않음. 받은 파일은 작업 폴더 밖(임시 폴더)에 두어 eggplant_yolo/로 올라가지 않음.
ULD_REMOTE = globals().get('ULD_REMOTE') or 'gdrive,root_folder_id=root:CVAT_UpLoader'   # Kaggle: 같은 rclone 설정, 시작 폴더만 내 드라이브 최상위로
ULD_COLAB = globals().get('ULD_COLAB') or '/content/drive/MyDrive/CVAT_UpLoader'         # Colab: Drive 경로 그대로
ULD_DIR = f'{LOCAL_TMP}/_uld' if ON_KAGGLE else ULD_COLAB
_ULD = globals().get('_ULD') or {'rec': {}, 'media': None, 'media_t': 0}

def uld_pull(*rels, quiet=True):
    # ULD 폴더 → Kaggle 임시 폴더 (받기만 함). Colab에서는 Drive를 바로 읽으므로 아무것도 안 함
    if not ON_KAGGLE or not rels:
        return
    flt = [x for r in rels for x in ('--include', '/' + r.strip('/'))]
    _kg_rc(['copy', ULD_REMOTE, ULD_DIR, '--update', *flt], quiet=quiet)

def uld_task(tid, pull=True):
    # ULD가 만든 TASK면 그 기록(dict), 아니면 None (한 번 읽은 것은 기억)
    tid = str(tid)
    if tid in _ULD['rec']:
        return _ULD['rec'][tid]
    if pull:
        uld_pull(f'registry/tasks/{tid}.json')
    p = f'{ULD_DIR}/registry/tasks/{tid}.json'
    try:
        rec = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None
    except Exception as e:
        print(f'⚠️ ULD 기록을 읽지 못함 ({tid}): {e}')
        rec = None
    _ULD['rec'][tid] = rec
    return rec

def uld_is_images(rec):
    return bool(rec) and rec.get('upload_as') == 'images' and bool(rec.get('frame_map'))

def uld_index(max_age=600):
    # v4.31.6: ULD가 만든 TASK {TASK ID: 'images'(사진 TASK) | 'video'} — 두 앱의 '사진(ULD)'·'ULD clip' 표시와 백업 판단용 (10분마다 새로 읽음)
    # v4.31.9 (F15): 읽기·교체 연결은 scripts의 wb.uld_scan (CPU 마스터와 같음) — 교체는 ULD가 '바꾸기'로 만든 기록만, 시각은 시간대를 맞춰 비교
    if _ULD.get('idx') is not None and time.time() - _ULD.get('idx_t', 0) < max_age:
        return _ULD['idx']
    sc = {'index': {}, 'lineage': {}, 'recs': {}}
    try:
        uld_pull('registry/tasks/*.json')
        sc = wb.uld_scan(f'{ULD_DIR}/registry/tasks')
        _ULD['rec'].update(sc['recs'])
    except Exception as e:
        print(f'(ULD 기록 목록을 읽지 못함 — 앱에 ULD 표시만 빠짐: {e})')
    _ULD['idx'], _ULD['idx_t'], _ULD['lineage'] = sc['index'], time.time(), sc['lineage']
    return sc['index']

def uld_lineage():
    # ULD가 바꿔 만든 TASK {옛 번호: 새 번호} (v4.31.8)
    uld_index()
    return dict(_ULD.get('lineage') or {})

def uld_video(tid):
    # ULD가 '영상'으로 올린 TASK의 원본 = ULD clip (v4.31.8) → clip 경로 또는 None (사진 TASK는 uld_clip + frame_map)
    rec = uld_task(tid)
    if not rec or uld_is_images(rec):
        return None
    return uld_clip(rec)

def uld_clip(rec):
    # 그 TASK의 clip 경로 (Kaggle은 받아 옴). 없거나 크기가 기록과 다르면 None
    rel = str(rec.get('clip_path') or '').split('CVAT_UpLoader/', 1)[-1]
    if not rel:
        return None
    uld_pull(rel, quiet=False)
    p = f'{ULD_DIR}/{rel}'
    if not os.path.exists(p):
        return None
    if rec.get('clip_bytes') not in (None, '') and os.path.getsize(p) != int(rec['clip_bytes']):
        print(f"⚠️ ULD clip 크기가 기록과 다름: {os.path.getsize(p)} ≠ {rec['clip_bytes']} ({rel})")
        return None
    return p

def uld_clip_done(path):
    # Kaggle: 다 쓴 clip을 지워 디스크를 아낌 (Colab은 Drive의 원본이므로 절대 지우지 않음)
    if ON_KAGGLE and path and str(path).startswith(ULD_DIR + '/') and os.path.exists(path):
        os.remove(path)

def uld_mismatch(task, rec):
    # 사진 TASK와 ULD 기록이 맞는지: 사진 수 · 사진 이름 끝(_f<프레임 6자리>) · 사진 크기. 맞으면 '' , 아니면 이유
    fm = list(rec.get('frame_map') or [])
    try:
        frames = list(task.get_meta().frames)
    except Exception as e:
        return f'CVAT 사진 목록을 읽지 못함: {ctt.api_error(e) if hasattr(ctt, "api_error") else e}'
    if len(frames) != len(fm):
        return f'사진 수 {len(frames)} ≠ ULD frame_map {len(fm)}'
    for k, fr in enumerate(frames):
        stem = os.path.splitext(str(fr.name))[0]
        if not stem.endswith(f'_f{int(fm[k]):06d}'):
            return f'사진 {k}번 이름 {fr.name} ≠ _f{int(fm[k]):06d} (ULD frame_map)'
        if rec.get('clip_w') and rec.get('clip_h') and (int(fr.width), int(fr.height)) != (int(rec['clip_w']), int(rec['clip_h'])):
            return f"사진 크기 {fr.width}x{fr.height} ≠ clip {rec['clip_w']}x{rec['clip_h']}"
    return ''

def uld_images(tid, out_dir, names=None):
    # 사진 TASK의 사진을 clip에서 다시 뽑아 out_dir에 CVAT 사진 이름으로 저장 (15번 셀 학습 데이터 — 라벨만 받은 export용)
    # names = CVAT 사진 이름 목록(순서대로). 없으면 '<만들 때 이름>_f<프레임 6자리>.jpg'. 반환: 새로 저장한 장 수 (ULD 사진 TASK가 아니면 None)
    rec = uld_task(tid)
    if not uld_is_images(rec):
        return None
    fm = [int(x) for x in rec['frame_map']]
    names = list(names) if names else [f"{rec.get('task_name_initial', '')}_f{n:06d}.jpg" for n in fm]
    os.makedirs(out_dir, exist_ok=True)
    todo = {n: nm for n, nm in zip(fm, names) if not os.path.exists(f'{out_dir}/{nm}')}
    if not todo:
        return 0
    clip = uld_clip(rec)
    if clip is None:
        print(f'⚠️ ULD clip을 받지 못해 TASK {tid}의 사진을 만들지 못함')
        return 0
    import av
    n_saved, last = 0, max(todo)
    with av.open(clip) as c:
        st = c.streams.video[0]
        for i, fr in enumerate(c.decode(st)):
            if i in todo:
                img = ctt.upright_image(fr.to_image(), ctt.pyav_rotation(fr, st)) if hasattr(ctt, 'upright_image') else fr.to_image()
                img.save(f'{out_dir}/{todo[i]}', quality=95)
                n_saved += 1
            if i >= last:
                break
    uld_clip_done(clip)
    return n_saved

def has_source_video(task):
    # 이 TASK의 원본 영상이 확보돼 있나: Drive videos/ · TASK 백업 · ULD clip (v4.31.5 — 없을 때만 백업 요청)
    name, tid = task.name, task.id
    try:
        if uld_task(tid):
            return True
    except Exception:
        pass
    try:
        if _ULD['media'] is None or time.time() - _ULD['media_t'] > 600:
            if ON_KAGGLE:
                _ls = lambda sub: [l for l in _kg_rc(['lsf', '-R', '--files-only', _kg_remote(sub)], quiet=True).stdout.splitlines() if l]
                _ULD['media'] = wb.media_names(BASE, remote_videos=_ls('videos'), remote_backups=_ls('backups'))
            else:
                _ULD['media'] = wb.media_names(BASE)
            _ULD['media_t'] = time.time()
        vn, bn = _ULD['media']
        b = ctt.base_name(name)
        return b.lower() in vn or b in bn
    except Exception as e:
        print(f'(영상 확보 여부를 확인하지 못함: {e})')
        return True                                  # 모르면 요청하지 않음

AA_REPLACED_NOTE = '_AA의 annotation export, backup하고 새로운 annotation으로 교체함'

def label_uploaded(task, replaced=False, before_zip=None, say=print, rename=True):
    # 라벨을 CVAT에 올린 뒤 (v4.31.5 이름 규칙 — 5·6·10·27·30번 셀 공통)
    #   …_uld → …_uld_AA 로 이름을 바꿈 · …_AA는 이름 그대로, 기존 라벨을 바꿨으면 그 사실을 기록:
    #   올리기 전 라벨(before_zip)을 exports/_aa_replaced/에 남기고, 원본 영상이 없으면(videos/·백업·ULD clip) 백업 요청
    # 반환 {'name': 지금 이름, 'renamed': bool, 'note': 기록에 붙일 말}
    old = task.name
    out = {'name': old, 'renamed': False, 'note': ''}
    new = ctt.aa_name_after_upload(old) if rename else old
    if new != old:
        try:
            task.update({'name': new})
            out.update(name=new, renamed=True, note=f'이름 변경 {old} → {new}')
            say(f'이름 변경: {old} → {new}')
        except Exception as e:
            out['note'] = f'이름 변경 실패 {ctt.api_error(e)}'
            say(f'⚠️ 이름 변경 실패 (라벨은 올라감): {ctt.api_error(e)}')
        return out
    if ctt.is_aa_name(old) and replaced:
        notes = [AA_REPLACED_NOTE]
        try:
            if before_zip and os.path.exists(str(before_zip)):
                d = f'{BASE}/exports/_aa_replaced'
                os.makedirs(d, exist_ok=True)
                dst = f"{d}/{ctt.export_file_stem(old)}__{task.id}__{ctt.stamp()}.zip"
                shutil.copy2(str(before_zip), dst)
                notes.append(f'이전 라벨 {os.path.relpath(dst, BASE)}')
        except Exception as e:
            notes.append(f'이전 라벨 보관 실패 {e}')
        try:
            if not has_source_video(task):
                r = wb.new_request(BASE, 'backup', [task.id], {'mode': 'all', 'export_reviewed': False, 'auto': 'aa_replace'},
                                   by=f'{who_am_i()} (AA 교체)', note='AA 라벨 교체 — 원본 영상 없음', task_names=[old])
                if r:
                    notes.append(f"영상 없음 → 백업 요청 {r['id']}")
        except Exception as e:
            notes.append(f'백업 요청 실패 {e}')
        out['note'] = ' · '.join(notes)
        say(out['note'])
    return out

if ON_KAGGLE:
    _free = shutil.disk_usage(BASE).free / 2**30
    print(f"실행 환경  : Kaggle ({'Save & Run All' if KAGGLE_BATCH else '대화형'}) · 작업 폴더 {BASE} ↔ Drive {_kg_remote() or KAGGLE_REMOTE} · "
          f"디스크 여유 {_free:.0f} GB · 자동 동기화 {'켬' if KAGGLE_AUTO_SYNC else '끔 → drive_sync_now()'}")
else:
    print('실행 환경  : Colab (Drive 연결)')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ("없음 (오른쪽 Session options → Accelerator → GPU)" if ON_KAGGLE else "없음 (런타임 유형을 T4 GPU로)")}'
      + (f' × {torch.cuda.device_count()} (15번 셀 학습은 모두 씀 — TRAIN_GPUS)' if torch.cuda.device_count() > 1 else ''))
print(f'패키지     : ultralytics {ultralytics.__version__}, cvat-sdk {CVAT_SDK_VER}, cvat-cli {CVAT_CLI_VER}')
print(f'CVAT 서버  : {SERVER_VER or "버전 확인 불가"} → SDK 호환: {cvat_compat()}')
print(f"CVAT 범위  : 기본 조직 {'구분 안 함 (모든 조직)' if CVAT_ORG is None else (CVAT_ORG or '개인 작업공간')} · "
      f"기본 프로젝트 {CVAT_PROJECT or '전부'}" + (' — 셀이 CVAT에 접속할 때 확인' if CVAT_ORG is not None or CVAT_PROJECT else ''))
if DS:
    print(f'데이터셋   : {DS_SOURCE} → {DS} (train {len(os.listdir(f"{DS}/train/images"))} / val {len(os.listdir(f"{DS}/val/images"))})  ← 2·3번 셀이 쓰는 데이터')
else:
    print('데이터셋   : dataset.zip 없음 (학습을 안 할 거면 괜찮음)')
print(f'선택 모델  : {NAME} (MODEL_CHOICE={MODEL_CHOICE!r})' + ('' if os.path.isdir(RUN_DIR) else '  ⚠️ runs/에 이 폴더가 없습니다 (이름 확인 또는 15번 셀로 학습)'))
print(f'학습 상태  : {train_status()}  [{RUN_DIR}]')
print(f'학습 잠금  : {lock_status()}')
print(f'내 이름    : {who_am_i()} (세션 {SESSION_ID})' + ('' if OPERATOR else "  ← 0번 셀 OPERATOR에 이름을 적으면 잠금 안내에 표시됨"))
print(f'best.pt    : {"있음 — " + model_label() if os.path.exists(BEST) else "없음"}')
if _trained:
    import datetime as _dt
    print('학습된 모델 (최근순, 쓰려면 MODEL_CHOICE에 이름 또는 번호):')
    _num = {v: k for k, v in MODEL_NAMES.items()}
    for d in _trained[::-1][:8]:
        n = os.path.basename(os.path.dirname(d))
        print(f"   {'▶' if n == NAME else ' '} {n:<45} 완료 {_dt.datetime.fromtimestamp(os.path.getmtime(d)).strftime('%m-%d %H:%M')}"
              + (f'  (번호 {_num[n]})' if n in _num else ''))
print(f'videos/    : {sum(1 for f in Path(VIDEOS).rglob("*") if f.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv"))}개 영상 (하위 폴더 포함)')
_m = cvat_mode()
for _k, _p in CVAT_PROFILES.items():
    print(f"CVAT {_k:<8}: {_p['host']} · {'PAT' if prof_login(_k) == 'token' else 'ID/PASS'} · 보안 비밀 "
          f"{'있음' if _has_secret(_k) else '없음(실행 때 입력창)'}" + ('  ← 처음 접속' if _k == _m else ''))
print(f"학습 데이터 : 검토 완료 = 상태에 '{REVIEWED_RULE}' · val = '{VAL_RULE or '무작위(VAL_RATIO)'}' · "
      f"TASK 학습 대장 {len(ctt.read_train_tasks(BASE)) if ctt is not None else 0}줄"
      + (f" · 기록 세션 파일 {ctt.SHARD['id']}" if ctt is not None and hasattr(ctt, 'SHARD') else ''))
if wb is not None:
    _open = [r for r in wb.list_requests(BASE) if r['status'] in ('pending', 'taken')]
    print(f"웹 요청    : 대기·진행 중 {len(_open)}건" + (' → 28번 셀에서 보기, 해당 셀 FROM_REQUESTS = True로 실행' if _open else ''))
print(f'작업 대장  : {REGISTRY if os.path.exists(REGISTRY) else "아직 없음 (첫 작업 때 생성)"}')
_need = ['cvat_task_tools.py', 'cvat_backup.py', 'cvat_to_yolo.py', 'yolo_track_to_cvat.py',
         'eval_occluded_recall.py', 'compare_tracks.py', 'cvat_yolo_function.py', 'diagnose_pipeline.py', 'drive_revisions.py', 'ensemble_annotate.py', 'web_bridge.py', 'relabel_bridge.py']
missing = [f for f in _need if not os.path.exists(f'{SCRIPTS}/{f}')]
def _file_ver(path):
    m = re.search(r'^__version__\s*=\s*"([^"]+)"', open(path, encoding='utf-8').read(), re.M)
    return m.group(1) if m else '버전 없음(예전 파일)'
_wrong = {f: _file_ver(f'{SCRIPTS}/{f}') for f in _need if f not in missing}
SCRIPTS_OK = (APP_VERSION,)                   # v4.31.0: scripts도 바뀜 (세션별 기록 — cvat_task_tools·web_bridge·cvat_backup) → scripts/를 4.30.1로
_wrong = {f: v for f, v in _wrong.items() if v not in SCRIPTS_OK}
_dups = sorted(os.path.basename(f) for f in glob.glob(f'{SCRIPTS}/*(*).py'))
print(f'앱 버전    : {APP_VERSION}')
print('scripts/   : ' + ('모두 있음, 버전 일치' if not missing and not _wrong else
                         ('없음 → ' + ', '.join(missing) + ' ' if missing else '')
                         + ('⚠️ 버전 다름 → ' + ', '.join(f'{f}({v})' for f, v in _wrong.items())
                            + f'  ← Drive scripts/를 v{APP_VERSION} 파일로 바꾸고 세션 다시 시작' if _wrong else '')))
if _dups:
    print(f'⚠️ scripts/에 이름이 겹친 파일: {", ".join(_dups)} → 지우세요 (Drive가 같은 이름 업로드를 새 파일로 만든 것)')
if ON_KAGGLE:                                   # scripts/ 안에 Colab 전용 코드가 남아 있는지 (Kaggle에서 안 될 수 있음)
    _colab_only = sorted(os.path.basename(f) for f in glob.glob(f'{SCRIPTS}/*.py')
                         if os.path.basename(f) != 'drive_revisions.py'
                         and re.search(r'google\.colab|/content/', open(f, encoding='utf-8').read()))
    if _colab_only:
        print(f"⚠️ Colab 전용 코드가 있는 scripts: {', '.join(_colab_only)} → 이 파일을 쓰는 셀은 Kaggle에서 확인 필요")
    print('Kaggle     : 23번 셀(Drive 버전 복원)은 Colab에서만 · 영상·백업은 쓰는 셀이 Drive에서 자동으로 받음 · 직접: drive_pull(경로), drive_get_videos(검색어)')


## 1. 학습

다시 실행해도 안전합니다: **완료** → 건너뜀 / **중단됨** → 마지막 에폭부터 이어서 / **시작 전** → 새로 학습.
결과는 에폭마다 Drive `runs/<NAME>/`에 저장되므로 연결이 끊겨도 잃지 않습니다. 끊기면 0번 → 1번을 다시 실행하세요.

학습 시간을 줄이려면(0번 셀 설정): `EPOCHS`를 줄이거나 `PATIENCE`를 20 정도로, `MODEL`을 `yolo11n.pt`로, `IMGSZ`를 1600으로.
처음부터 다시 학습하려면 아래 `FORCE_NEW = True`.

In [ ]:
# 1. 학습 — 다시 실행해도 안전 (완료=건너뜀, 중단=이어서, 처음=새로) · 학습 잠금으로 다른 사람과 겹치지 않음
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

FORCE_NEW = False
if FORCE_NEW:
    lock_run(RUN_DIR, 'delete')                # 다른 사람이 학습 중인 폴더는 지우지 않음
    shutil.rmtree(RUN_DIR, ignore_errors=True)
assert DS, 'dataset.zip이 없습니다 (Drive eggplant_yolo/에 올리기)'
assert torch.cuda.is_available() or os.environ.get('ALLOW_CPU'), \
    'GPU가 없습니다 → 메뉴 런타임 → 런타임 유형 변경 → T4 GPU 후 0번 셀부터 다시 (CPU로 하려면 os.environ["ALLOW_CPU"]="1")'
if os.path.exists(f'{RUN_DIR}/TRAINING_DONE'):
    run_training(NAME, yaml_path)              # 이미 완료 → 건너뜀 안내만 (작업 기록 안 남김)
else:
    _ok = None
    job_start(1, 'train', {'name': NAME, 'data': DS_SOURCE, 'force_new': FORCE_NEW})   # 작업 기록 (v4.21)
    try:
        _ok = run_training(NAME, yaml_path)
    finally:
        _fin = os.path.exists(f'{RUN_DIR}/TRAINING_DONE')
        job_end('끝' if _fin else '중단?', f'runs/{NAME} · 데이터 {DS_SOURCE} · ' + ('학습 완료' if _fin else '중간에 멈춤 — 다시 실행하면 이어서'))
    if _ok:
        print('학습 완료:', BEST)


In [ ]:
# 2. 학습 결과 보기 — 마지막 줄의 precision / recall / mAP50 확인
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

import pandas as pd
from IPython.display import Image, display
df = pd.read_csv(f'{RUN_DIR}/results.csv')
df.columns = [c.strip() for c in df.columns]
display(df[['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(5))
for f in ['results.png', 'val_batch0_pred.jpg']:
    if os.path.exists(f'{RUN_DIR}/{f}'):
        display(Image(f'{RUN_DIR}/{f}', width=900))

In [ ]:
# 3. 검증 1단계 — 가림/크기별 재현율 (val 이미지 기준)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

assert os.path.exists(BEST) and DS, 'best.pt와 dataset이 필요합니다'
run([sys.executable, f'{SCRIPTS}/eval_occluded_recall.py', '--model', BEST, '--dataset', DS, '--imgsz', str(IMGSZ)])

## CVAT TASK 작업 — TASK ID만 적으면 됩니다

| 단계 | 셀 | 하는 일 | 사람이 할 일 |
|---|---|---|---|
| 접속 | 4 | 대상 서버(PAT 또는 공동 아이디)로 접속, TASK 이름 검색 확인, 내 TASK 목록(ID·이름·상태·프레임 수) 표시. 검색어가 있으면 TASK별 프레임·클래스별 박스 수 통계와 묶음 합계(v4.17) | 목록에서 TASK ID 고르기 |
| track 생성 | 5 | TASK 정보 조회 → 현재 라벨 자동 백업(export) → 원본 mp4 찾기 → 추적 → 검증 2단계 → 대장 기록 | `TRACK_IDS` 입력, 원본 mp4를 `videos/`에 |
| 업로드 승인 | 6 | 업로드 직전 다시 백업 → track을 TASK에 업로드 → 대장 기록 | 대장을 보고 `UPLOAD_IDS` 입력 |
| 검수 수집 | 7 | CVAT에서 고친 라벨을 export → 검증 3단계 → 대장 기록 | CVAT에서 검수 후 `REVIEW_IDS` 입력 |
| 대장 보기 | 8 | `task_registry.csv` 표시 | — |

- 추적에는 **Drive `videos/`의 영상**을 씁니다(하위 폴더 가능). 이름은 TASK 기본 이름(`eggplant_G1_261019_1_1.mp4`)이나 CVAT에 올린 원본 이름이면 찾습니다.
- **업로드는 TASK의 라벨을 바꿉니다.** 노트북이 업로드 직전에 항상 백업하고, 손 라벨이 있는 TASK는 `ALLOW_REPLACE = True`가 아니면 올리지 않습니다.
- TASK마다 결과가 `tasks/<TASK_ID>/`에 모이고, 실행 1회마다 대장에 한 줄이 쌓입니다(실행 시각, TASK 이름, 영상, 모델, track 수, 검증 수치, 백업 파일, 상태).

In [ ]:
# 4. CVAT 접속 · TASK 목록 · TASK 이름 검색 확인 — PAT 서버 또는 공동 아이디 서버
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

LOGIN = None        # None = 지금 접속 중인 서버(처음이면 0번 셀 CVAT_AUTH) · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)
TASK_QUERY = ''     # TASK 이름 검색어 확인 (비우면 전체). 예: '261019 vf_r2', 'G6 uld -tag=AA*', 'G2 plant=6..10 uld', '*AA_r1*'
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
TASK_STATS = True   # 검색 결과에 프레임 수·클래스별 박스 수 통계 (검색어가 있을 때, uld로 끝나는 TASK는 세지 않음)
STATS_CVAT = 20     # 통계가 없거나 export 뒤에 바뀐 TASK를 CVAT에서 셀 수 (0 = 28번 셀이 세어 둔 것만)
STATS_BY   = 'gh date'   # 합계를 묶을 칸: 'gh' · 'gh date' · 'date' · 'row' · 'plant' · 'state' … ('' = 전체 합계만)
STATS_SAVE = False  # True = 표를 Drive reports/task_stats_<시각>.csv로 저장

assert ctt, 'scripts/cvat_task_tools.py가 없습니다 (Drive scripts/에 올린 뒤 0번 셀 다시)'
import pandas as pd
from IPython.display import display
use_cvat(LOGIN)                              # 이후 셀은 LOGIN=None이면 이 서버를 그대로 씀
tasks = ctt.list_tasks(CLIENT, limit=5000)
if not tasks:
    print('태스크가 없습니다 → 0번 셀 CVAT_ORG·CVAT_PROJECT(기본 조직·프로젝트)를 확인하세요')
elif TASK_QUERY:
    pd.set_option('display.max_rows', 300)
    _hits = ctt.select_tasks(tasks, TASK_QUERY, display=None if TASK_STATS else display, show_all=not TASK_STATS,
                             label='검색어 확인')   # 통계를 볼 때는 맞는 TASK만 (아래 표)
    if TASK_STATS and _hits:
        assert wb is not None, 'scripts/web_bridge.py가 없습니다 (v4.16 이상 scripts)'
        kg_sync_exports()                             # Kaggle: 다른 세션이 받은 export도 (v4.23)
        _st = wb.compute_stats(BASE, _hits, client=CLIENT if STATS_CVAT else None, cvat_max=STATS_CVAT)
        try:                                          # 28번 셀이 만든 '같은 온실·촬영일과 비교' 결과가 있으면 함께
            _peer = {str(t['id']): t.get('peer') or {} for t in json.load(open(f'{BASE}/web/state.json', encoding='utf-8'))['tasks']}
        except Exception:
            _peer = {}
        _labels = sorted({k for s in _st.values() for k in (s.get('by_label') or {})})
        _rows = []
        for t in _hits:
            s, f = _st.get(str(t['task_id'])) or {}, ctt.parse_task_name(t['name']) or {}
            r = {'task_id': t['task_id'], 'TASK': t['name'], **{k: f.get(k, '') for k in ('gh', 'date', 'row', 'plant', 'state')},
                 '프레임': s.get('frames', t.get('frames', '')), '박스 있는 프레임': s.get('labeled', ''), '빈 프레임': s.get('zero', '')}
            for lab in _labels:
                r[lab] = (s.get('by_label') or {}).get(lab, 0) if s.get('frames') else ''
            r.update({'박스 합': s.get('total', ''), '프레임당': s.get('total_mean', ''), '프레임당 최대': s.get('total_max', ''),
                      '최대 프레임': (f"{s['max_idx']} (원본 {s['max_frame']})" if s.get('max_idx') is not None else ''),
                      '변동': s.get('cv', ''), '가림': s.get('occluded', ''),
                      '출처': ('uld (세지 않음)' if t['name'].lower().endswith('uld') else
                              s.get('error') or ('CVAT' if s.get('src') == 'cvat' else 'export' + (' (오래됨)' if s.get('stale') else '')) if s else '아직 못 셈'),
                      '같은 온실·촬영일과': ' · '.join((_peer.get(str(t['task_id'])) or {}).get('flags') or [])})
            _rows.append(r)
        TASK_STATS_DF = pd.DataFrame(_rows)
        print(f"\n■ TASK별 통계 ({len(_rows)}개) — 클래스 열 = 박스 수 (track은 프레임마다 보이는 박스로 셈)")
        display(TASK_STATS_DF)
        _ok = TASK_STATS_DF[pd.to_numeric(TASK_STATS_DF['프레임'], errors='coerce').notna() & (TASK_STATS_DF['박스 합'] != '')]
        if len(_ok):
            _num = ['프레임', '박스 있는 프레임', '빈 프레임', *_labels, '박스 합']
            _ok = _ok.assign(**{c: pd.to_numeric(_ok[c], errors='coerce').fillna(0).astype(int) for c in _num})
            _by = [c for c in STATS_BY.split() if c in _ok.columns]
            _g = (_ok.groupby(_by)[_num].sum() if _by else _ok[_num].sum().to_frame('합계').T)
            _g.insert(0, 'TASK', _ok.groupby(_by).size() if _by else len(_ok))
            for lab in _labels:
                _g[f'{lab}/프레임'] = (_g[lab] / _g['프레임'].where(_g['프레임'] > 0)).round(2)
            _g['박스/프레임'] = (_g['박스 합'] / _g['프레임'].where(_g['프레임'] > 0)).round(2)
            print(f"■ 합계{' — ' + ' × '.join(_by) + '별' if _by else ''} (통계가 있는 TASK {len(_ok)}개)")
            display(_g)
        _miss = [r['task_id'] for r in _rows if r['출처'] == '아직 못 셈']
        if _miss:
            print(f"아직 못 센 TASK {len(_miss)}개 — export가 없고 CVAT로 셀 수({STATS_CVAT})를 넘음 → 셀을 다시 실행하거나 STATS_CVAT를 늘리세요")
        if STATS_SAVE:
            os.makedirs(f'{BASE}/reports', exist_ok=True)
            _p = f'{BASE}/reports/task_stats_{ctt.stamp()}.csv'
            TASK_STATS_DF.to_csv(_p, index=False, encoding='utf-8-sig')
            print('저장:', os.path.relpath(_p, BASE))
else:
    display(pd.DataFrame(tasks))


In [ ]:
# 5. track 생성 — TASK ID 목록을 적고 실행 (업로드는 하지 않음)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

TRACK_IDS = []          # 예: [2687720, 2687801]
TRACK_QUERY = ''        # 또는 TASK 이름 검색 (TRACK_IDS가 비었을 때). 예: 'G6 uld tag=, G2 plant=6..10 uld tag='
FROM_REQUESTS = False   # True = 웹 화면의 요청(track 생성)을 받아 그 TASK로 실행 (끝나면 요청이 '완료'로)
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
LOGIN = None            # None = 지금 접속 중인 서버 그대로 · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)
TRACK_SKIP_DONE = False # True: 이미 track을 만든 TASK는 건너뜀 / False: 다시 실행 (이전 결과는 그대로 두고 새 폴더에 저장)
TRACK_AUTO_UPLOAD = False  # True = TASK마다 track이 끝나는 대로 뒤에서 바로 CVAT에 올림(6번 셀 일을 동시에), 다음 TASK 추적은 계속
                           #        올리는 조건: track 1개 이상 + 기존 라벨 없음(또는 아래 True) + 검증 2단계가 있으면 정밀도 ≥ UPLOAD_MIN_PRECISION
                           # False = track만 만들고, 6번 셀에서 미리 보기 후 올림 (처음 몇 TASK는 이쪽 권장)
TRACK_UPLOAD_REPLACE = False   # True = 기존 라벨이 있으면 백업한 뒤 지우고 자동 track으로 교체 (6번 셀 ALLOW_REPLACE와 같음)
TRACK_UPLOAD_TAG = 'AA'        # 올린 뒤 이름 규칙 적용: …_uld → …_uld_AA, …_AA는 그대로 (v4.31.5) · '' = 이름을 바꾸지 않음

if FROM_REQUESTS:
    TRACK_IDS, TRACK_QUERY = req_use('track'), ''
TRACK_IDS = ctt.as_task_ids(TRACK_IDS)          # 2696411 · '2696411' · '2696411, 2696412' 모두 됨
use_cvat(LOGIN)
assert os.path.exists(BEST), 'best.pt가 없습니다 (1번 셀 학습)'
if not TRACK_IDS and TRACK_QUERY:
    from IPython.display import display
    _hits = ctt.select_tasks(ctt.list_tasks(CLIENT, limit=5000), TRACK_QUERY, display=display, label='track 대상')
    TRACK_IDS = [t['task_id'] for t in _hits]
    print(f'TRACK_IDS = {TRACK_IDS}')
import json, threading, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display
done, up_done, up_futs = [], [], []
_reg_lock = threading.Lock()
_up_pool = ThreadPoolExecutor(max_workers=1) if TRACK_AUTO_UPLOAD else None

def _upload_one(tid, xml_rel, tracks):
    # 6번 셀과 같은 순서: 올리기 직전 라벨 백업 → 기존 라벨 확인 → 업로드(교체) → 전·후 비교 → 이름에 _AA · 상세 로그
    ts_ = track_settings_of(xml_rel)
    L = upload_log(tid, 'track', model_path=f"{BASE}/{ts_['model']}" if ts_.get('model') else None,
                   echo=lambda m: print(f'   [업로드 {tid}] {m}'), result_file=xml_rel,
                   source=ts_.get('video', ''), settings=ts_.get('settings', {}), track_made=ts_.get('made', ''))
    say = L.say
    row = dict(run_time=ctt.now(), action='upload', task_id=tid, model=model_label(), server=CVAT_SERVER,
               tracks=tracks, result_file=xml_rel)
    err = None
    try:
        cl = cvat_client(CVAT_LOGIN_MODE)                    # 추적과 따로 접속 (동시 실행에 안전)
        task = cl.tasks.retrieve(tid)
        row['task_name'] = task.name
        L.set(task_name=task.name)
        b = ctt.export_annotations(task, f'{task_dir(tid)}/before_upload_{ctt.stamp()}.zip')
        meta_ = ctt.read_export(b)
        row.update(existing_labels=meta_['existing_boxes'], backup_file=os.path.relpath(b, BASE))
        L.set(backup_file=os.path.relpath(b, BASE), mode='replace' if meta_['existing_boxes'] else 'new')
        L.counts(before_zip=b)
        say(f"백업 {os.path.basename(b)} · 기존 라벨 박스 {L.rec.get('before_total', 0)}개 {L.rec.get('before_by_label', {})}")
        if meta_['existing_boxes'] > 0 and not (TRACK_UPLOAD_REPLACE or auto_task(task.name)):   # v4.31.4: uld·AA는 백업 뒤 교체
            row.update(status='보류', note=f"기존 라벨 {meta_['existing_boxes']}개 → 6번 셀에서 확인 후 ALLOW_REPLACE")
            L.set(mode='보류')
            say(row['note']); return row
        ctt.import_tracks(task, f'{BASE}/{xml_rel}', replace=meta_['existing_boxes'] > 0, backup=b, log=say)
        row.update(status='업로드 완료', note=('기존 라벨 교체(백업 있음) · ' if meta_['existing_boxes'] > 0 else '') + 'CVAT에서 검수 후 7번 셀로 수집')
        L.set(uploaded=f'track {tracks}개')
        _lu = label_uploaded(task, replaced=meta_['existing_boxes'] > 0, before_zip=b, say=say, rename=bool(TRACK_UPLOAD_TAG))   # v4.31.5 이름 규칙
        if _lu['note']:
            row['note'] += ' · ' + _lu['note']
        if _lu['renamed']:
            row.update(task_name=_lu['name']); L.set(task_name_after=_lu['name'])
        say(f"✅ 업로드 완료 (track {tracks}개) {row['task_name']}")
        try:
            _ch = change_report(task, b, 'after_upload', show=False)
            say(f"라벨 변화: {_ch['text']} · 프레임별 표 {_ch['csv']}")
            row['note'] += f" · {_ch['text']} · 비교 {_ch['csv']}"
            L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
        except Exception as _e:
            say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
    except Exception as e:
        err = e
        row.update(status='오류', note=ctt.api_error(e)[:300]); say(f"❌ {row['note']}")
    finally:
        with _reg_lock:
            ctt.append_registry(REGISTRY, **row)
        try:
            p_ = L.finish(row.get('status', '?'), err)
            print(f'   [업로드 {tid}] 상세 로그: {os.path.relpath(str(p_), BASE)}')
        except Exception as _e:
            print(f'   [업로드 {tid}] (상세 로그 저장 실패: {_e})')
    return row
if TRACK_IDS:                                      # v4.31.6: 자동 업로드 설정을 처음에 보여 줌 (LC앱 요청은 셀 기본값 — 31번이 아니라 30번 셀 LC_SET으로 바꿈)
    print(f"자동 업로드: {'켬 — track이 끝난 TASK부터 뒤에서 올림' if TRACK_AUTO_UPLOAD else '끔 — 6번 셀에서 미리 보기 후 올림'}"
          + (f" · 기존 라벨 교체 {'함' if TRACK_UPLOAD_REPLACE else '안 함 (uld·AA TASK는 백업 뒤 교체)'} · 정밀도 기준 {UPLOAD_MIN_PRECISION:.0%}" if TRACK_AUTO_UPLOAD else '')
          + (' · (LC앱 요청으로 실행 — 셀 기본값)' if FROM_REQUESTS else ''))
if TRACK_IDS:
    job_start(5, 'track', {'ids': TRACK_IDS, 'query': TRACK_QUERY, 'auto_upload': TRACK_AUTO_UPLOAD, 'replace': TRACK_UPLOAD_REPLACE,
                           'conf': TRACK_CONF, 'imgsz': IMGSZ, 'buffer': TRACK_BUFFER, 'fill_gaps': TRACK_FILL_GAPS,
                           'box_source': TRACK_BOX_SOURCE, 'near': TRACK_NEAR, 'decoder': TRACK_DECODER, 'offset': TRACK_FRAME_OFFSET})
for tid in TRACK_IDS:
    ts = ctt.stamp()
    folder = task_dir(tid)
    os.makedirs(folder, exist_ok=True)
    log = f'{folder}/run_{ts}_track.log'
    row = dict(run_time=ctt.now(), action='track', task_id=tid, model=model_label(), server=CVAT_SERVER)
    print('=' * 70, f'\nTASK {tid}')
    hist = [r for r in ctt.read_registry(REGISTRY) if str(r['task_id']) == str(tid) and ctt.same_server(r, CVAT_SERVER)]
    prev_tracks = [r for r in hist if r['action'] == 'track 생성' and r['status'].startswith('완료')]
    uploaded = any(r['action'] == '업로드' and r['status'] == '업로드 완료' for r in hist)
    if prev_tracks:
        print(f"이전 track 생성 {len(prev_tracks)}회 (마지막 {prev_tracks[-1]['run_time']}, {prev_tracks[-1]['status']})"
              + (' · 자동 track을 업로드한 적 있음' if uploaded else ''))
        if TRACK_SKIP_DONE:
            print('→ TRACK_SKIP_DONE=True 라서 건너뜁니다 (대장에 기록하지 않음)')
            continue
    try:
        task = CLIENT.tasks.retrieve(tid)
        info = ctt.task_basic(task)
        row.update(task_name=info['task_name'], media=info['media'], frames=info['frames'])
        print(f"이름 {info['task_name']} / 종류 {info['media']} / 프레임 {info['frames']}")
        _uld, _fm = None, None
        if not info['is_video']:
            _uld = uld_task(tid)                           # v4.31.5: ULD가 영상에서 프레임을 잘라 만든 사진 TASK면 clip으로 track
            _why = ('' if uld_is_images(_uld) else 'ULD 기록 없음') or uld_mismatch(task, _uld)
            if _why:
                row.update(status='건너뜀', note=f"이미지 태스크 → track 불가 ({_why} · 프레임별 자동 라벨링 9~10번 셀 사용) [{info['media_detail']}]")
                print(f"건너뜀: 이미지로 만든 TASK입니다 — {_why} ({info['media_detail']})")
                continue
            _fm = [int(x) for x in _uld['frame_map']]
            print(f"ULD 사진 TASK: 사진 {len(_fm)}장 = clip 프레임 {_fm} → clip 전체를 추적해 이 프레임만 올림")
        backup = ctt.export_annotations(task, f'{folder}/backup_{ts}.zip')          # 현재 라벨 백업 + 기준 정보
        meta = ctt.read_export(backup)
        row.update(step=meta['step'], existing_labels=meta['existing_boxes'], backup_file=os.path.relpath(backup, BASE))
        print(f"원본 영상 {meta['source']} / step {meta['step']} / 기존 라벨 {meta['existing_boxes']}개 / 백업 {os.path.basename(backup)}")
        if _fm:
            _clip = uld_clip(_uld)
            if _clip is None:
                row.update(status='영상 없음', note='ULD clip을 받지 못함 (CVAT_UpLoader/clips에 없거나 크기가 기록과 다름)')
                print('❌ ULD clip을 받지 못했습니다'); continue
            video = Path(_clip)
        else:
            video = ctt.find_video(VIDEOS, meta['source'], info['task_name'])
        _uvid = False
        if video is None and not _fm:                      # v4.31.8: ULD가 영상으로 올린 TASK → 원본은 ULD clip
            _uc = uld_video(tid)
            if _uc:
                video, _uvid = Path(_uc), True
                print(f'원본 영상: ULD clip {os.path.basename(_uc)}')
        if video is None:                                  # 없으면 13번 셀 TASK 백업(영상 포함)에서 꺼냄
            video, bz = ctt.video_from_backups(f'{BASE}/backups', info['task_name'], VIDEOS)
            if video is not None:
                print(f'영상을 TASK 백업에서 꺼냄: {os.path.relpath(str(bz), BASE)} → {os.path.relpath(str(video), BASE)}')
        if video is None:
            row.update(status='영상 없음', note=f"Drive videos/에 {ctt.base_name(info['task_name'])}.mp4 (또는 {meta['source']}) 올리기")
            print(f"❌ videos/에 {ctt.base_name(info['task_name'])}.mp4 또는 {meta['source']}가 없습니다 → 올린 뒤 다시 실행")
            continue
        row['video'] = video.name
        out_dir = f'{folder}/track_{ts}'                       # 실행마다 새 폴더 → 이전 결과를 덮어쓰지 않음
        rc = run([sys.executable, f'{SCRIPTS}/yolo_track_to_cvat.py', '--model', BEST, '--video', video,
                  '--reference', backup, '--imgsz', IMGSZ, '--conf', TRACK_CONF, '--fill-gaps', TRACK_FILL_GAPS,
                  '--track-buffer', TRACK_BUFFER, '--decoder', 'pyav' if (_fm or _uvid) else TRACK_DECODER,
                  '--frame-offset', 0 if (_fm or _uvid) else TRACK_FRAME_OFFSET,
                  *(['--frame-map', ','.join(map(str, _fm))] if _fm else []),
                  '--box-source', TRACK_BOX_SOURCE, '--near', TRACK_NEAR, '--out', out_dir], log=log)
        if rc != 0:
            raise RuntimeError(f'track 생성 실패 (로그: {os.path.relpath(log, BASE)})')
        _mt, _md5 = ctt.file_fingerprint(BEST)
        json.dump({'made': ctt.now(), 'operator': who_am_i(), 'session': SESSION_ID, 'server': CVAT_SERVER,
                   'model': os.path.relpath(BEST, BASE), 'model_time': _mt, 'model_md5': _md5,
                   'video': os.path.relpath(str(video), BASE), 'video_bytes': os.path.getsize(video),
                   **({'uld': {'clip': _uld.get('clip_path'), 'frame_map': _fm, 'clip_sha256': _uld.get('clip_sha256')}} if _fm else {}),
                   'reference_backup': os.path.relpath(backup, BASE), 'app_version': APP_VERSION,
                   'settings': {'imgsz': IMGSZ, 'conf': TRACK_CONF, 'fill_gaps': TRACK_FILL_GAPS, 'track_buffer': TRACK_BUFFER,
                                'decoder': 'pyav' if (_fm or _uvid) else TRACK_DECODER, 'frame_offset': 0 if (_fm or _uvid) else TRACK_FRAME_OFFSET,   # v4.31.9 (F20): 실제로 쓴 값
                                'box_source': TRACK_BOX_SOURCE, 'near': TRACK_NEAR}},
                  open(f'{out_dir}/track_settings.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        tj = json.load(open(f'{out_dir}/tracks_{video.stem}.json', encoding='utf-8'))
        xml_path = f'{out_dir}/tracks_{video.stem}.xml'
        row.update(tracks=tj['tracks'], result_file=os.path.relpath(xml_path, BASE))
        if _fm or _uvid:
            uld_clip_done(video)                           # Kaggle: 다 쓴 clip은 지움 (결과 XML은 TASK 폴더에 있음)
        if tj.get('detections_all_frames', 1) == 0:
            row['note'] = '검출 0개 → 11번 셀(진단)로 원인 확인'
            print('⚠️ 영상 전체에서 검출이 하나도 없습니다 → 11번 셀(진단)을 실행하세요')
        # 검증 2단계 정답: 자동 track을 업로드한 적이 있으면 지금 TASK 라벨은 '이전 자동 track'이므로
        # 맨 처음 백업(원래 라벨)을 정답으로 쓴다
        gt, gt_note = backup, ''
        if uploaded and prev_tracks and prev_tracks[0].get('backup_file') and os.path.exists(f"{BASE}/{prev_tracks[0]['backup_file']}"):
            gt = f"{BASE}/{prev_tracks[0]['backup_file']}"
            gt_note = f"검증 기준: 최초 백업 {os.path.basename(gt)} (현재 TASK 라벨은 이전 자동 track)"
            print('검증 2단계 정답 →', gt_note)
        gt_boxes = ctt.read_export(gt)['existing_boxes']
        if gt_boxes > 0:                                                              # 검증 2단계
            cj = f'{out_dir}/compare_L2.json'
            run([sys.executable, f'{SCRIPTS}/compare_tracks.py', '--pred', xml_path, '--gt', gt,
                 '--json', cj, '--csv', f'{out_dir}/compare_L2.csv'], log=log)
            m = json.load(open(cj, encoding='utf-8'))
            row.update(precision=m['precision'], recall=m['recall'], f1=m['f1'])
            verdict = '업로드 권장' if m['precision'] >= UPLOAD_MIN_PRECISION else '재검토 (정밀도 낮음)'
            row.update(status=f'완료 · {verdict}',
                       note=' / '.join(x for x in [gt_note, '기존 라벨이 있는 TASK → 업로드 시 ALLOW_REPLACE 필요'] if x))
        else:
            row.update(status='완료 · 업로드 대기' if tj['tracks'] else '완료 · track 없음',
                       note=row.get('note') or ('현재 TASK 라벨은 이전 자동 track → 업로드 시 ALLOW_REPLACE=True'
                                                if meta['existing_boxes'] else '기존 라벨 없음 → 검증 2단계 생략'))
    except Exception as e:
        row.update(status='오류', note=str(e)[:300])
        print('❌ 오류:', e)
    finally:
        with _reg_lock:
            ctt.append_registry(REGISTRY, **row)
        done.append(row)
    # 끝난 TASK는 뒤에서 바로 업로드 (다음 TASK 추적과 동시에) · v4.31.6: 올리지 않으면 그 이유를 TASK마다 출력
    if TRACK_AUTO_UPLOAD:
        _st = str(row.get('status', ''))
        try:
            _ntr = int(row.get('tracks') or 0)
        except (TypeError, ValueError):
            _ntr = 0
        if not _st.startswith('완료'):
            print(f"   → 자동 업로드 안 함: 상태 '{_st or '기록 없음'}'" + (f" — {str(row.get('note', ''))[:120]}" if row.get('note') else ''))
        elif _ntr <= 0:
            print('   → 자동 업로드 안 함: track 0개')
        elif not (row.get('precision') in (None, '') or float(row['precision']) >= UPLOAD_MIN_PRECISION):
            print(f"   → 자동 업로드 안 함: 정밀도 {float(row['precision']):.0%} < {UPLOAD_MIN_PRECISION:.0%} (6번 셀에서 확인)")
        else:
            print(f'   → 업로드를 뒤에서 시작합니다 (다음 TASK 추적은 계속)')
            up_futs.append(_up_pool.submit(_upload_one, tid, row['result_file'], row['tracks']))
if up_futs:
    print('\n남은 업로드를 기다립니다 …')
    up_done = []
    for _f in up_futs:                                 # v4.31.6: 뒤에서 난 오류도 보이게 (한 건이 실패해도 나머지 결과는 보여 줌)
        try:
            up_done.append(_f.result())
        except Exception as _e:
            print(f'❌ 업로드 줄 오류: {type(_e).__name__}: {_e}')
    _up_pool.shutdown()
    print('업로드 결과:')
    display(pd.DataFrame(up_done)[['task_id', 'task_name', 'tracks', 'status', 'note']])
elif _up_pool:
    _up_pool.shutdown()
if TRACK_IDS:
    job_end('끝', f"track {sum(1 for r in done if str(r.get('status', '')).startswith('완료'))}/{len(done)}개"
            + (f" · 업로드 {sum(1 for r in up_done if r.get('status') == '업로드 완료')}개" if up_done else ''))
TRACK_LAST_DONE = [dict(r, uploaded=any(u.get('task_id') == r.get('task_id') and u.get('status') == '업로드 완료' for u in up_done)) for r in done]   # v4.31.6: 6번 셀이 이어 받음
if done:
    display(pd.DataFrame(done)[[c for c in ['task_id', 'task_name', 'video', 'tracks', 'precision', 'recall', 'status', 'note'] if c in pd.DataFrame(done)]])
    print(f'작업 대장에 {len(done)}줄 기록: {REGISTRY}')
else:
    print('TRACK_IDS가 비어 있습니다.')

In [ ]:
# 6. 업로드 승인 — 대장을 보고 올릴 TASK ID를 적고 실행
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

UPLOAD_IDS = []         # 예: [2687801] · v4.31.6: 비워 두면 이 세션의 5번 셀 결과 중 track이 있고 아직 올리지 않은 TASK (미리 보기부터)
UPLOAD_FROM_TRACK = True   # False = 비어 있으면 아무것도 안 함 (예전처럼)
LOGIN = None            # None = 지금 접속 중인 서버 그대로 · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)
ALLOW_REPLACE = False   # True: 기존 라벨이 있는 TASK는 백업(tasks/<ID>/before_upload_*.zip)한 뒤 지우고 자동 track으로 교체
UPLOAD_PREVIEW = True   # True = 올리지 않고 올릴 내용(라벨별 track·박스 수, 가림, 그림)만 확인 → 확인 후 False
PREVIEW_IMAGES = 6      # 미리 보기 그림 수
UPLOAD_NAME_TAG = 'AA'  # 업로드 뒤 이름 규칙 (v4.31.5): …_uld → …_uld_AA · …_AA는 이름 그대로(교체 기록만) · '' = 이름을 바꾸지 않음

use_cvat(LOGIN)
import pandas as pd
from IPython.display import display
if not UPLOAD_IDS and UPLOAD_FROM_TRACK and globals().get('TRACK_LAST_DONE'):
    _ups = {}
    for _r in ctt.read_registry(REGISTRY):             # 5번 셀 뒤에 다른 곳(6번 셀·작업자)에서 올린 것도 뺌
        if _r.get('action') == '업로드' and _r.get('status') == '업로드 완료' and ctt.same_server(_r, CVAT_SERVER):
            _ups[str(_r.get('task_id'))] = max(_ups.get(str(_r.get('task_id')), ''), str(_r.get('run_time', '')))
    _cand = []
    for _r in TRACK_LAST_DONE:
        try:
            _ok = str(_r.get('status', '')).startswith('완료') and int(_r.get('tracks') or 0) > 0
        except (TypeError, ValueError):
            _ok = False
        if _ok and not _r.get('uploaded') and _ups.get(str(_r['task_id']), '') < str(_r.get('run_time', '')):
            _cand.append(_r['task_id'])
    UPLOAD_IDS = _cand
    print(f"UPLOAD_IDS가 비어 있어 5번 셀 결과 중 아직 올리지 않은 TASK {len(UPLOAD_IDS)}개를 대상으로 합니다: {UPLOAD_IDS}"
          + (" (정밀도가 낮은 TASK도 들어 있으니 미리 보기로 확인)" if any('재검토' in str(r.get('status', '')) for r in TRACK_LAST_DONE if r['task_id'] in UPLOAD_IDS) else ''))
if UPLOAD_PREVIEW:
    for tid in ctt.as_task_ids(UPLOAD_IDS):
        last = ctt.last_row(REGISTRY, tid, 'track', server=CVAT_SERVER)
        print('=' * 70, f'\nTASK {tid} — 올릴 내용 미리 보기 (아직 올리지 않음)')
        if not last or not last.get('result_file'):
            print('  track 생성 기록 없음 → 5번 셀 먼저'); continue
        xml_path = f"{BASE}/{last['result_file']}"
        ann = ctt.load_annotations(xml_path)
        label_rows, fsum, _ = ctt.annotation_tables(ann)
        print(f"  파일: {last['result_file']} · 만든 시각 {last['run_time']} · 모델 {last.get('model', '')}")
        if UPLOAD_NAME_TAG and last.get('task_name'):
            print(f"  올리면 이름: {last['task_name']} → {ctt.aa_name_after_upload(last['task_name'])}")
        print('  ' + ' · '.join(f'{k} {v}' for k, v in fsum.items()))
        display(pd.DataFrame(label_rows))
        vid = ctt.find_video(VIDEOS, last.get('video') or ann['meta']['source'], last.get('task_name')) or \
              ctt.video_from_backups(f'{BASE}/backups', last.get('task_name', ''), VIDEOS)[0]
        if vid and PREVIEW_IMAGES:
            ctt.draw_annotations(ann, ctt.video_frame_getter(vid), n=PREVIEW_IMAGES,
                                 title=f"TASK {tid} auto tracks (yellow=healthy_fruit, red=disease_fruit, cyan=healthy_bud, magenta=disease_bud, white inner line=occluded, *=keyframe)")
        elif not vid:
            print('  (영상을 찾지 못해 그림은 생략)')
    print('\n내용이 맞으면 UPLOAD_PREVIEW = False로 바꿔 다시 실행하면 올립니다.')
if UPLOAD_IDS and not UPLOAD_PREVIEW:
    job_start(6, 'upload', {'ids': UPLOAD_IDS, 'replace': ALLOW_REPLACE, 'name_tag': UPLOAD_NAME_TAG})
for tid in (UPLOAD_IDS if not UPLOAD_PREVIEW else []):
    ts = ctt.stamp()
    folder = task_dir(tid)
    row = dict(run_time=ctt.now(), action='upload', task_id=tid, model=model_label(), server=CVAT_SERVER)
    print('=' * 70, f'\nTASK {tid}')
    L, err = None, None
    try:
        last = ctt.last_row(REGISTRY, tid, 'track', server=CVAT_SERVER)
        if not last or not last.get('result_file'):
            row.update(status='보류', note='track 생성 기록 없음 → 5번 셀 먼저')
            continue
        xml_path = f"{BASE}/{last['result_file']}"
        ts_ = track_settings_of(last['result_file'])
        L = upload_log(tid, 'track', model_path=f"{BASE}/{ts_['model']}" if ts_.get('model') else None,
                       result_file=last['result_file'], source=ts_.get('video') or last.get('video', ''),
                       settings=ts_.get('settings') or {'track_run': last.get('run_time'), 'model_at_track': last.get('model')},
                       track_made=ts_.get('made') or last.get('run_time'))
        task = CLIENT.tasks.retrieve(tid)
        L.set(task_name=task.name)
        row.update(task_name=task.name, video=last.get('video', ''), tracks=last.get('tracks', ''),
                   precision=last.get('precision', ''), result_file=last['result_file'])
        backup = ctt.export_annotations(task, f'{folder}/before_upload_{ts}.zip')      # 업로드 직전 백업
        meta = ctt.read_export(backup)
        row.update(step=meta['step'], existing_labels=meta['existing_boxes'], backup_file=os.path.relpath(backup, BASE))
        L.set(backup_file=os.path.relpath(backup, BASE), mode='replace' if meta['existing_boxes'] else 'new')
        L.counts(before_zip=backup)
        L.say(f"백업 {os.path.basename(backup)} · 기존 라벨 박스 {L.rec.get('before_total', 0)}개 {L.rec.get('before_by_label', {})}")
        if meta['existing_boxes'] > 0 and not (ALLOW_REPLACE or auto_task(task.name)):   # v4.31.4: uld·AA는 백업 뒤 교체
            row.update(status='보류', note=f"기존 라벨 {meta['existing_boxes']}개 → 손 라벨 TASK인지 확인 후 ALLOW_REPLACE=True")
            L.set(mode='보류'); L.say(f"⚠️ 기존 라벨 {meta['existing_boxes']}개가 있어 올리지 않았습니다.")
            continue
        ctt.import_tracks(task, xml_path, replace=meta['existing_boxes'] > 0, backup=backup, log=L.say)
        L.set(uploaded=f"track {last.get('tracks', '')}개")
        row.update(status='업로드 완료', note=(f"기존 라벨 {meta['existing_boxes']}개 교체(백업 {os.path.basename(backup)}) · " if meta['existing_boxes'] > 0 else '') + 'CVAT에서 검수 후 7번 셀로 수집')
        L.say(f'✅ 업로드 완료: {os.path.basename(xml_path)} → TASK {tid} (백업 {os.path.basename(backup)})')
        try:
            _ch = change_report(task, backup, 'after_upload')
            row['note'] += f" · {_ch['text']} · 비교 {_ch['csv']}"
            L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
            L.say(f"라벨 변화: {_ch['text']}")
        except Exception as _e:
            L.say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
        _lu = label_uploaded(task, replaced=meta['existing_boxes'] > 0, before_zip=backup, say=L.say, rename=bool(UPLOAD_NAME_TAG))   # v4.31.5 이름 규칙
        if _lu['note']:
            row['note'] = f"{row['note']} · {_lu['note']}"
        if _lu['renamed']:
            row.update(task_name=_lu['name']); L.set(task_name_after=_lu['name'])
    except Exception as e:
        err = e
        row.update(status='오류', note=str(e)[:300])
        (L.say if L else print)(f'❌ 오류: {ctt.api_error(e)}')
    finally:
        ctt.append_registry(REGISTRY, **row)
        if L is not None:
            try:
                print(f'   상세 로그: {os.path.relpath(str(L.finish(row.get("status", "?"), err)), BASE)}')
            except Exception as _e:
                print(f'   (상세 로그 저장 실패: {_e})')
if UPLOAD_IDS and not UPLOAD_PREVIEW:
    job_end()
if not UPLOAD_IDS and not UPLOAD_PREVIEW:
    print('UPLOAD_IDS가 비어 있습니다 — 8번 셀에서 대장을 보고 결정하세요.')

In [ ]:
# 7. 검수 결과 수집 — CVAT에서 검수(Merge·추가·삭제)를 끝낸 TASK ID를 적고 실행 → 검증 3단계
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

REVIEW_IDS = []         # 예: [2687801]
LOGIN = None            # None = 지금 접속 중인 서버 그대로 · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)

use_cvat(LOGIN)
import json
if ctt.as_task_ids(REVIEW_IDS):
    job_start(7, 'review', {'ids': REVIEW_IDS})
for tid in ctt.as_task_ids(REVIEW_IDS):
    ts = ctt.stamp()
    folder = task_dir(tid)
    log = f'{folder}/run_{ts}_review.log'
    row = dict(run_time=ctt.now(), action='review', task_id=tid, server=CVAT_SERVER)
    print('=' * 70, f'\nTASK {tid}')
    try:
        # 비교 대상: 실제로 CVAT에 올린 track (다시 생성만 하고 안 올린 결과가 아니라)
        ups = [r for r in ctt.read_registry(REGISTRY) if str(r['task_id']) == str(tid) and ctt.same_server(r, CVAT_SERVER)
               and r['action'] == '업로드' and r['status'] == '업로드 완료' and r.get('result_file')]
        last = ups[-1] if ups else ctt.last_row(REGISTRY, tid, 'track', server=CVAT_SERVER)
        if not last or not last.get('result_file'):
            row.update(status='보류', note='track 생성·업로드 기록 없음')
            continue
        print('비교할 자동 track:', last['result_file'], '(업로드한 것)' if ups else '(업로드 기록 없음 → 마지막 생성본)')
        task = CLIENT.tasks.retrieve(tid)
        reviewed = ctt.export_annotations(task, f'{folder}/reviewed_{ts}.zip')
        row.update(task_name=task.name, video=last.get('video', ''), model=last.get('model', ''),
                   backup_file=os.path.relpath(reviewed, BASE), result_file=last['result_file'])
        cj = f'{folder}/compare_L3_{ts}.json'
        run([sys.executable, f'{SCRIPTS}/compare_tracks.py', '--pred', f"{BASE}/{last['result_file']}", '--gt', reviewed,
             '--json', cj, '--csv', f'{folder}/compare_L3_{ts}.csv'], log=log)
        m = json.load(open(cj, encoding='utf-8'))
        row.update(tracks=m.get('pred_tracks', ''), precision=m['precision'], recall=m['recall'], f1=m['f1'],
                   count_error=m.get('count_error', ''), fragments=m.get('fragments', ''),
                   id_switches=m.get('id_switches', ''), idf1=m.get('idf1', ''))
        row.update(status='검수 수집 완료' if m.get('gt_is_track') else '검수 수집 (track 형식 아님)',
                   note='검수본은 다음 학습 데이터로 사용 (이미지 포함 export는 CVAT에서)')
    except Exception as e:
        row.update(status='오류', note=str(e)[:300])
        print('❌ 오류:', e)
    finally:
        ctt.append_registry(REGISTRY, **row)
if not REVIEW_IDS:
    print('REVIEW_IDS가 비어 있습니다.')
if ctt.as_task_ids(REVIEW_IDS):
    job_end()


In [ ]:
# 8. 작업 대장 보기 (Drive의 task_registry.csv — Google 스프레드시트로도 열림)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

SHOW_TASK = None        # 특정 TASK만 보려면 ID 입력
import pandas as pd
from IPython.display import display
rows = ctt.read_registry(REGISTRY) if ctt else []
df = pd.DataFrame(rows)
if len(df) and SHOW_TASK:
    df = df[df['task_id'].astype(str) == str(SHOW_TASK)]
pd.set_option('display.max_colwidth', 60)
display(df.tail(30) if len(df) else '기록 없음')

## 프레임별 자동 라벨링 (원본 mp4 없이도 가능, track 아님)
CVAT 서버에서 TASK 프레임을 받아 모델을 돌리고 박스를 그 TASK에 **덧붙입니다**. 9번 셀 점검에 ❌가 없을 때만 10번 셀을 실행하세요.

In [ ]:
# 9. 자동 라벨링 사전 점검 — TASK 여러 개를 한 번에 점검 (통과한 TASK만 10번 셀이 실행)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

AA_TASK_IDS = []        # ← 자동 라벨링할 TASK ID 목록, 예: [2700651, 2700652, 2700653]
AA_QUERY    = ''        # 또는 TASK 이름 검색 (AA_TASK_IDS가 비었을 때). 예: 'G6 261019 row=2 uld tag='
FROM_REQUESTS = False   # True = 웹 화면의 요청(자동 라벨 shape — 10번 셀이 끝나면 완료)을 받아 그 TASK로 실행 (끝나면 요청이 '완료'로)
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
LOGIN = None            # None = 지금 접속 중인 서버 그대로 · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)
AA_EXISTING = 'skip'    # 이미 라벨이 있는 TASK: 'skip' = 빼기 · 'replace' = 기존 라벨을 백업한 뒤 지우고 자동 라벨로 교체
                        #   · 'add' = 기존 라벨에 덧붙임 (같은 열매에 박스가 겹칠 수 있음)
if 'AA_SKIP_LABELED' in globals() and AA_SKIP_LABELED is False and AA_EXISTING == 'skip':
    AA_EXISTING = 'add'                 # 예전 설정 이름 호환
assert AA_EXISTING in ('skip', 'replace', 'add'), "AA_EXISTING은 'skip' / 'replace' / 'add' 중 하나"
AA_SOURCE   = 'auto'    # 프레임을 어디서 읽을지: 'auto' = 원본 영상이 있으면 영상(원본 화질), 없으면 CVAT
                        #   · 'video' = 영상만 (없으면 그 TASK는 뺌) · 'cvat' = 예전처럼 CVAT에서 받기
AA_DECODER  = 'pyav'    # 영상 읽기: 'pyav' = CVAT와 같은 방식으로 프레임을 셈 (권장) · 'opencv'
AA_VIDEO_OFFSET = None  # 영상 프레임 + 이 값 = CVAT 프레임. None = 0번 셀 TRACK_FRAME_OFFSET (20번 셀 진단값)
AA_CHECK_ALIGN = True   # 영상 프레임과 CVAT 프레임이 같은 장면인지 미리 확인 (어긋나면 CVAT 프레임으로)
if FROM_REQUESTS:
    AA_TASK_IDS, AA_QUERY = req_use('aa'), ''

import pandas as pd
from IPython.display import display
ok = True
def chk(cond, good, bad, warn_only=False):
    global ok
    print(('✅ ' + good) if cond else (('⚠️ ' if warn_only else '❌ ') + bad))
    if not cond and not warn_only:
        ok = False
FUNC = f'{SCRIPTS}/cvat_yolo_function.py'
AA_DEVICE = '0' if torch.cuda.is_available() else 'cpu'
chk(torch.cuda.is_available(), 'GPU 사용 가능', 'GPU 없음 → CPU로 실행(매우 느림)', warn_only=True)
chk(os.path.exists(BEST), f'모델: {model_label()}', f'모델 파일 없음: {BEST}')
chk(os.path.exists(FUNC), '함수 파일: cvat_yolo_function.py', f'{FUNC} 없음 → Drive scripts/에 올리기')
model_names = []
if os.path.exists(BEST):
    from ultralytics import YOLO
    model_names = list(YOLO(BEST).names.values())
# split 학습 모델의 'healthy_fruit_occluded'는 CVAT 라벨 healthy_fruit + 가림(occluded) 표시로 올라감
base_names = list(dict.fromkeys(n[:-len('_occluded')] if n.endswith('_occluded') else n for n in model_names))
if base_names != model_names:
    print(f"ℹ️ 가림 클래스 {[n for n in model_names if n.endswith('_occluded')]} → 같은 이름의 라벨 + 가림 표시로 올림")
AA_SERVER_MODE = cvat_mode(LOGIN)      # 10번 셀이 같은 계정을 쓰도록 기억
AA_READY, aa_names, rows = [], {}, []
AA_AUTO_REPLACE = set()               # v4.31.4: uld·AA TASK — AA_EXISTING='skip'이어도 백업 뒤 교체 (10번 셀이 씀)
AA_VIDEO = {}                          # TASK → (원본 영상 경로, 보정값) — 10번 셀이 씀
_aa_off = TRACK_FRAME_OFFSET if AA_VIDEO_OFFSET is None else int(AA_VIDEO_OFFSET)
assert AA_SOURCE in ('auto', 'video', 'cvat'), "AA_SOURCE는 'auto' / 'video' / 'cvat' 중 하나"
if ok:
    cl = use_cvat(LOGIN)
    ids = ctt.as_task_ids(AA_TASK_IDS)
    if not ids and AA_QUERY:
        ids = [t['task_id'] for t in ctt.select_tasks(ctt.list_tasks(cl, limit=5000), AA_QUERY, label='자동 라벨링 대상')]
    chk(bool(ids), f'대상 TASK {len(ids)}개', 'AA_TASK_IDS 또는 AA_QUERY를 입력하세요')
    for tid in ids:
        r = {'task_id': tid}
        try:
            task = cl.tasks.retrieve(tid)
            ann = task.get_annotations()
            labels = [l.name for l in task.get_labels()]
            n_exist = len(ann.shapes) + len(ann.tracks)
            miss = [n for n in base_names if n not in labels]
            r.update(name=task.name, frames=task.size, 기존라벨=n_exist, 없는라벨=', '.join(miss))
            if miss:
                r['판정'] = '❌ 모델 클래스가 TASK 라벨에 없음'
            elif n_exist and AA_EXISTING == 'skip' and not (auto_task(task.name) and globals().get('AA_FRAMES') is None):   # v4.31.7: 10번 셀 전에도
                r['판정'] = "⏭ 이미 라벨 있음 → 뺌 (바꾸려면 AA_EXISTING = 'replace')"
            else:
                src, vid = 'CVAT', None
                if AA_SOURCE != 'cvat':
                    if not ctt.task_basic(task)['is_video']:
                        src = 'CVAT (사진 TASK)'
                    else:
                        vid = ctt.find_video(VIDEOS, '', task.name) or \
                              ctt.video_from_backups(f'{BASE}/backups', task.name, VIDEOS)[0]
                        if vid is None:
                            src = 'CVAT (영상 못 찾음 → videos/에 올리거나 13번 셀 백업)'
                        elif AA_CHECK_ALIGN:
                            al = ctt.check_video_alignment(task, vid, _aa_off, AA_DECODER)
                            if al['ok']:
                                src = f'영상 {os.path.relpath(str(vid), BASE)} (프레임 일치 확인)'
                            else:
                                src = (f"CVAT (영상과 프레임이 {al['best_shift']:+d} 어긋남 → 20번 셀로 진단, "
                                       f"AA_VIDEO_OFFSET = {_aa_off - (al['best_shift'] or 0)} 시도)")
                                vid = None
                        else:
                            src = f'영상 {os.path.relpath(str(vid), BASE)} (일치 확인 안 함)'
                r['프레임 출처'] = src
                if AA_SOURCE == 'video' and vid is None:
                    r['판정'] = "❌ 영상으로 못 함 (AA_SOURCE = 'video')"
                else:
                    _auto_rep = bool(n_exist) and AA_EXISTING == 'skip'          # 위에서 걸러지지 않은 skip = uld·AA 자동 교체
                    if _auto_rep:
                        AA_AUTO_REPLACE.add(tid)
                    r['판정'] = '✅ 실행' + ((' (기존 라벨 백업 후 교체)' if AA_EXISTING == 'replace' or _auto_rep else ' (기존 라벨에 덧붙임)') if n_exist else '') + (' · uld·AA' if _auto_rep else '')
                    AA_READY.append(tid); aa_names[tid] = task.name
                    if vid is not None:
                        AA_VIDEO[tid] = (str(vid), _aa_off)
        except Exception as e:
            r['판정'] = f'❌ 조회 실패: {ctt.api_error(e)}'
        rows.append(r)
    if rows:
        display(pd.DataFrame(rows))
aa_task_name = aa_names.get(AA_READY[0], '') if AA_READY else ''
AA_TASK_ID = AA_READY[0] if AA_READY else 0          # (예전 10번 셀 호환)
ok = ok and bool(AA_READY)
print(f'\n점검 통과 TASK {len(AA_READY)}개 → 10번 셀 실행' if ok else '\n❌ 실행할 TASK가 없습니다 — 위 표의 판정을 확인하세요')


In [ ]:
# 10. 프레임별 자동 라벨링 실행 — RUN_AA = True 일 때만 (9번 셀 통과 후, 통과한 TASK 전부)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'

RUN_AA = False
AA_WORKERS = 2          # 동시에 처리할 TASK 수 (T4 GPU 기준 2~3 권장. 1 = 하나씩)
AA_PREFETCH = 4         # TASK마다 미리 받아 둘 프레임 수 (프레임 받기와 탐지를 겹쳐 빠르게)
AA_FRAMES  = None       # None = TASK 전체 · 시험할 때는 range(0, 20) 처럼 일부 (TASK 프레임 번호 0,1,2…)
AA_QUALITY = 'original' # CVAT에서 받을 때 화질: 'original' (고해상도라 서버가 못 만들면 자동으로 'compressed') / 'compressed'
                        #   원본 영상을 쓰는 TASK(9번 셀 '프레임 출처'가 영상)는 이 값과 상관없이 원본 화질
AA_TILE    = False      # True = 위·중간·아래로 잘라 추론 (작은 열매·꽃봉오리)

import threading, time as _time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from IPython.display import display
if not RUN_AA:
    print('RUN_AA = False — 실행하지 않았습니다.')
elif not globals().get('ok') or not globals().get('AA_READY'):
    print('9번 셀 점검을 먼저 통과하세요.')
elif globals().get('AA_EXISTING') == 'replace' and AA_FRAMES is not None:
    print("⛔ 교체(AA_EXISTING = 'replace')는 TASK 전체에서만 됩니다 → AA_FRAMES = None (일부 프레임이면 나머지 프레임 라벨이 사라짐)")
else:
    use_cvat(AA_SERVER_MODE)                # 9번 셀에서 점검한 계정
    _lock = threading.Lock()
    _changes = {}                            # TASK별 업로드 전·후 라벨 비교 (끝난 뒤 한꺼번에 표로)
    _nv = sum(1 for t in AA_READY if t in globals().get('AA_VIDEO', {}))
    print(f"TASK {len(AA_READY)}개 (원본 영상 {_nv} · CVAT 프레임 {len(AA_READY) - _nv}) · 동시 {AA_WORKERS}개 · conf {AA_CONF} · 모델 {model_label()}")

    job_start(10, 'auto', {'ids': AA_READY, 'existing': AA_EXISTING, 'source': globals().get('AA_SOURCE', 'cvat'),
                           'conf': AA_CONF, 'imgsz': IMGSZ, 'tile': AA_TILE, 'quality': AA_QUALITY,
                           'frames': 'all' if AA_FRAMES is None else str(AA_FRAMES)})

    def _one(tid):
        say = lambda m: print(f'[{tid}] {m.strip()}')
        folder = task_dir(tid); os.makedirs(folder, exist_ok=True)
        ts = ctt.stamp()
        row = dict(run_time=ctt.now(), action='auto', task_id=tid, task_name=aa_names.get(tid, ''), model=model_label(), server=CVAT_SERVER)
        t0 = _time.time()
        L, err = None, None
        try:
            cl = cvat_client(AA_SERVER_MODE)          # 작업마다 따로 접속 (동시 실행에 안전)
            task = cl.tasks.retrieve(tid)
            _vid, _off = globals().get('AA_VIDEO', {}).get(tid, (None, 0))
            L = upload_log(tid, 'shape', echo=say, task_name=task.name,
                           source=(f"원본 영상 {os.path.relpath(_vid, BASE)} ({globals().get('AA_DECODER', 'pyav')}, 보정 {_off:+d})" if _vid
                                   else f'CVAT 프레임 ({AA_QUALITY}, 원본 불가 시 compressed)'),
                           settings={'imgsz': IMGSZ, 'conf': AA_CONF, 'tile': AA_TILE, 'quality': AA_QUALITY,
                                     'frames': 'all' if AA_FRAMES is None else str(AA_FRAMES), 'existing': AA_EXISTING,
                                     'workers': AA_WORKERS, 'prefetch': AA_PREFETCH, 'function': os.path.basename(FUNC)})
            backup = ctt.export_annotations(task, f'{folder}/before_auto_{ts}.zip')     # 올리기 전 라벨 백업 (교체·덧붙임 모두)
            row['backup_file'] = os.path.relpath(backup, BASE)
            _replace = (AA_EXISTING == 'replace' or tid in globals().get('AA_AUTO_REPLACE', set())) and ctt.annotation_count(task) > 0
            L.counts(before_zip=backup)
            L.set(backup_file=os.path.relpath(backup, BASE),
                  mode='replace' if _replace else ('add' if L.rec.get('before_total') else 'new'))
            L.say(f"백업 {os.path.basename(backup)} · 기존 라벨 박스 {L.rec.get('before_total', 0)}개 {L.rec.get('before_by_label', {})}")
            say(f"시작 '{task.name}' · 프레임 {task.size}장" + (' · 기존 라벨 교체' if _replace else ''))
            r = ctt.auto_annotate_frames(task, FUNC, BEST, imgsz=IMGSZ, device=AA_DEVICE, conf=AA_CONF, tile=AA_TILE,
                                         quality=AA_QUALITY, frames=AA_FRAMES, log=L.say, prefetch=AA_PREFETCH,
                                         replace=_replace, backup=backup,
                                         video=_vid, video_offset=_off, decoder=globals().get('AA_DECODER', 'pyav'))
            row.update(status='완료(교체)' if r.get('replaced') is not None else '완료', note=(f"기존 라벨 {r['replaced']}개 교체(백업 {os.path.basename(backup)}) · " if r.get('replaced') is not None else '') + f"프레임 {r['frames']}장 · 박스 {r['boxes']}개 {r['per_label']} · {r['quality']}"
                       + (f" · TASK에 없는 라벨이라 버림 {r['skipped_labels']}" if r['skipped_labels'] else ''))
            L.set(uploaded=f"박스 {r['boxes']}개 {r['per_label']}", frames_done=r['frames'], quality_used=r['quality'],
                  skipped_labels=r['skipped_labels'])
            L.say(f"✅ 완료: 박스 {r['boxes']}개 {r['per_label']} ({_time.time() - t0:.0f}초)")
            _lu = label_uploaded(task, replaced=r.get('replaced') is not None, before_zip=backup, say=L.say)   # v4.31.5 이름 규칙
            if _lu['note']:
                row['note'] += ' · ' + _lu['note']
            if _lu['renamed']:
                row.update(task_name=_lu['name']); L.set(task_name_after=_lu['name'])
            try:
                _ch = change_report(task, backup, 'after_auto', show=False)
                _changes[tid] = _ch
                row['note'] += f" · {_ch['text']} · 비교 {_ch['csv']}"
                L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
                L.say(f"라벨 변화: {_ch['text']}")
            except Exception as _e:
                say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
        except Exception as e:
            err = e
            row.update(status='오류', note=ctt.api_error(e)[:300])
            (L.say if L else say)(f'❌ 오류: {ctt.api_error(e)}')
        finally:
            with _lock:
                ctt.append_registry(REGISTRY, **row)
            if L is not None:
                try:
                    say(f'상세 로그: {os.path.relpath(str(L.finish(row.get("status", "?"), err)), BASE)}')
                except Exception as _e:
                    say(f'(상세 로그 저장 실패: {_e})')
        return row

    done = []
    with ThreadPoolExecutor(max_workers=max(1, int(AA_WORKERS))) as ex:
        for f in as_completed([ex.submit(_one, t) for t in AA_READY]):
            done.append(f.result())
    job_end('끝', f"완료 {sum(1 for r in done if str(r.get('status', '')).startswith('완료'))}/{len(done)}개")
    display(pd.DataFrame(done)[['task_id', 'task_name', 'status', 'note']])
    for _tid, _ch in _changes.items():
        print('=' * 70)
        show_change(_tid, _ch)
    print('CVAT에서 TASK를 열어 확인하거나 18·21번 셀로 요약을 보세요. (프레임별 박스라 track이 아님)')
    print('업로드 상세 로그: tasks/<ID>/upload_<시각>.log·.json, 전체 목록 logs/uploads.csv (25번 셀로 보기)')
    print('교체 전 라벨은 tasks/<ID>/before_auto_<시각>.zip — 되돌리려면 CVAT 작업 화면 Upload annotations(CVAT 1.1)로 그 zip을 올림')


## 진단 — 검출이 적거나 없을 때
TASK의 영상에서 프레임 몇 장을 골라 **신뢰도 하한을 아주 낮춰(0.01)** 모델이 무엇을 보는지 보여 줍니다.
- 낮은 신뢰도에서는 열매를 잡는데 0.25에서 사라진다 → 모델이 자신 없어 함: `TRACK_CONF`를 낮추고, 데이터를 더 모아 재학습
- 낮춰도 열매에 박스가 거의 없다 → 모델이 이 장면을 처음 봄(조명·거리·품종 차이): 이 영상을 라벨링해 학습에 추가
- 영상 프레임과 CVAT 프레임이 다르게 보인다(회전·다른 장면) → 같은 mp4가 아님: `videos/` 파일 확인

In [ ]:
# 11. 진단 — 모델이 이 TASK 영상에서 무엇을 보는지
DIAG_TASK_ID = 0        # ← 진단할 TASK ID (5번 셀을 한 번 실행한 TASK)
DIAG_FRAMES  = 4        # 볼 프레임 수
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
import cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics import YOLO
DIAG_TASK_ID = ctt.as_task_id(DIAG_TASK_ID)          # 2696411 또는 '2696411' 모두 됨
last = ctt.last_row(REGISTRY, DIAG_TASK_ID, 'track', server=globals().get('CVAT_SERVER'))
assert last and last.get('backup_file'), '이 TASK의 track 생성 기록이 없습니다 (5번 셀 먼저)'
meta = ctt.read_export(f"{BASE}/{last['backup_file']}")
video = ctt.find_video(VIDEOS, meta['source'], last.get('task_name')) or \
        ctt.video_from_backups(f'{BASE}/backups', last.get('task_name', ''), VIDEOS)[0]
assert video, f"videos/에 {meta['source']}가 없습니다"
model = YOLO(BEST)
cap = cv2.VideoCapture(str(video))
vw, vh = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vn, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
print(f"영상 {video.name}: {vw}x{vh}, {fps:.1f}fps, {vn}프레임 / TASK 프레임 {meta['start']}~{meta['stop']} step {meta['step']}")
print(f"모델: {model_label()} / 클래스 {list(model.names.values())} / IMGSZ {IMGSZ}")
task_frames = list(range(meta['start'], meta['stop'] + 1, meta['step']))
picks = [task_frames[int(i)] for i in np.linspace(0, len(task_frames) - 1, DIAG_FRAMES)]
cvat_task = CLIENT.tasks.retrieve(DIAG_TASK_ID) if 'CLIENT' in globals() else None
buckets = {'0.5 이상': 0, '0.25~0.5': 0, '0.1~0.25': 0, '0.01~0.1': 0}
fig, axes = plt.subplots(2 if cvat_task else 1, DIAG_FRAMES, figsize=(4 * DIAG_FRAMES, 14 if cvat_task else 7), squeeze=False)
for j, f in enumerate(picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok_, img = cap.read()
    if not ok_:
        axes[0][j].set_title(f'video frame {f}: read failed'); continue
    r = model.predict(img, imgsz=IMGSZ, conf=0.01, verbose=False)[0]
    vis = img[:, :, ::-1].copy()
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
        buckets['0.5 이상' if c >= 0.5 else '0.25~0.5' if c >= 0.25 else '0.1~0.25' if c >= 0.1 else '0.01~0.1'] += 1
        if c < 0.05:
            continue                                   # 너무 낮은 것은 그리지 않음 (숫자에는 포함)
        col = (0, 255, 0) if c >= TRACK_CONF else ((255, 200, 0) if c >= 0.1 else (255, 0, 0))
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), col, 6)
        cv2.putText(vis, f'{c:.2f}', (int(x1), max(30, int(y1) - 10)), cv2.FONT_HERSHEY_SIMPLEX, 2, col, 4)
    axes[0][j].imshow(vis); axes[0][j].set_title(f'video frame {f}: {int((r.boxes.conf >= TRACK_CONF).sum())} boxes >= {TRACK_CONF} / {len(r.boxes)} >= 0.01'); axes[0][j].axis('off')
    if cvat_task:
        try:
            k = task_frames.index(f)
            from PIL import Image as PILImage
            try:
                im = PILImage.open(cvat_task.get_frame(k, quality='original'))
            except Exception as e:                     # 고해상도 영상: 원본 프레임 불가(cvat#7425) → 압축 프레임
                if 'resolution' not in str(e):
                    raise
                im = PILImage.open(cvat_task.get_frame(k, quality='compressed'))
            axes[1][j].imshow(im); axes[1][j].set_title(f'CVAT frame {f} ({im.size[0]}x{im.size[1]})')
        except Exception as e:
            axes[1][j].set_title(f'CVAT frame failed: {str(e)[:40]}')
        axes[1][j].axis('off')
cap.release()
plt.tight_layout()
out_png = f'{task_dir(DIAG_TASK_ID)}/diag_{ctt.stamp()}.png'
plt.savefig(out_png, dpi=60); plt.show()
print('신뢰도 분포:', buckets)
print('그림: 초록 = TRACK_CONF 이상, 주황 = 0.1 이상, 빨강 = 0.05~0.1 (위 = 영상 프레임 + 모델 예측, 아래 = 같은 번호의 CVAT 프레임)')
print('저장:', os.path.relpath(out_png, BASE))


## Drive의 파이썬 코드 실행
Drive `scripts/`의 `.py` 목록(수정 시각, 첫 줄 설명)을 보여 주고, 고른 파일을 실행합니다.
- **cli**: 별도 프로세스로 실행하고 명령줄 인자를 넘깁니다(`compare_tracks.py`, `eval_occluded_recall.py`, `yolo_track_to_cvat.py` 등). 출력은 화면과 `logs/`에 남습니다.
- **module**: 노트북 안으로 불러와 파일 맨 위의 설정값을 `SETTINGS`로 바꾼 뒤 함수(`ENTRY`)를 실행합니다(`cvat_to_yolo.py` 등 설정을 파일 안에서 고치는 코드). Colab 보안 비밀이 필요한 코드도 이 방식으로 실행합니다.


In [ ]:
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
# 12. Drive scripts/의 파이썬 코드 실행 — RUN_FILE을 비우면 목록만 보여 줌
RUN_FILE = ''          # 예: 'cvat_to_yolo.py', 'compare_tracks.py'
RUN_MODE = 'cli'       # 'cli' = 별도 프로세스 + RUN_ARGS / 'module' = 노트북 안에서 SETTINGS 적용 후 ENTRY() 실행
RUN_ARGS = []          # cli 예: ['--pred', f'{BASE}/tasks/123/track_.../tracks_x.xml', '--gt', f'{BASE}/tasks/123/backup_....zip']
SETTINGS = {}          # module 예: {'INPUT_DIR': f'{BASE}/zips', 'VIDEOS_DIR': VIDEOS, 'OUT_DIR': f'{LOCAL_TMP}/dataset'}
ENTRY    = 'main'      # module: 실행할 함수 이름 (cvat_backup.py는 'run_backup')

import ast, datetime
import pandas as pd
from IPython.display import display

def _first_doc_line(path):
    try:
        d = ast.get_docstring(ast.parse(open(path, encoding='utf-8').read()))
        return d.strip().splitlines()[0] if d else ''
    except Exception as e:
        return f'(읽기 오류: {type(e).__name__})'

rows = [{'파일': os.path.basename(f),
         '수정 시각': datetime.datetime.fromtimestamp(os.path.getmtime(f)).strftime('%Y-%m-%d %H:%M'),
         'KB': round(os.path.getsize(f) / 1024, 1), '설명': _first_doc_line(f)}
        for f in sorted(glob.glob(f'{SCRIPTS}/*.py'))]
display(pd.DataFrame(rows) if rows else f'{SCRIPTS}에 .py 파일이 없습니다 → Drive scripts/에 올리세요')

if RUN_FILE:
    path = f'{SCRIPTS}/{RUN_FILE}'
    assert os.path.exists(path), f'{path} 가 없습니다'
    log = f"{BASE}/logs/{ctt.stamp()}_{os.path.splitext(RUN_FILE)[0]}.log"        # 한국 시각
    if RUN_MODE == 'cli':
        rc = run([sys.executable, path, *RUN_ARGS], log=log)
        print(f'\n종료 코드 {rc} · 로그 {os.path.relpath(log, BASE)}')
    elif RUN_MODE == 'module':
        mod = load_script(RUN_FILE)
        for k, v in SETTINGS.items():
            if not hasattr(mod, k):
                print(f'⚠️ {RUN_FILE}에 {k} 설정이 없습니다 (그래도 넣음)')
            setattr(mod, k, v)
        assert hasattr(mod, ENTRY), f'{RUN_FILE}에 {ENTRY}() 함수가 없습니다'
        print(f'▶ {RUN_FILE}.{ENTRY}()  설정 {list(SETTINGS) or "기본값"}')
        RESULT = getattr(mod, ENTRY)()
    else:
        print("RUN_MODE는 'cli' 또는 'module'")


## CVAT TASK 백업
Drive `scripts/cvat_backup.py`를 불러 실행합니다. 로그인은 0번 셀의 `CVAT_AUTH`(공동 아이디 또는 PAT)를 따릅니다.
- TASK 백업(영상+라벨)을 `backups/<기본 이름>/{original,work,revised}/`에 시각별로 저장합니다. 이름과 수정 시각이 마지막 백업과 같으면 건너뜁니다.
- 결과: `backup_index.csv`(백업 1회 = 1줄), `backup_status.csv`(TASK별 최신 상태), 작업보드 `reports/작업보드.xlsx`의 **'백업 상태'·'백업 색인'** 탭 (eggplant_yolo 폴더 안, 추가 승인 없음 — `BK_BOARD = 'sheet'`이면 Google 스프레드시트).
- 처음 보드를 쓸 때 Google 계정 권한 승인 창이 한 번 뜹니다.


In [ ]:
# 13. CVAT TASK 백업 — 요청(request)과 받기(download)를 나눠서 할 수 있음 · 백업 상태를 Drive CSV와 작업보드에 기록
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN           = None     # None = 지금 접속 중인 서버 · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)
BK_QUERY        = ''       # TASK 이름 검색 (비우면 전체). 예: '261019 vf_r2', 'vf_r2df|vf_r2df_ok', 'G6 uld tag='
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
BK_TASK_IDS     = None     # None = 검색어에 맞는 전체, 예: [123, 456]
FROM_REQUESTS = False   # True = 웹 화면의 요청(백업)을 받아 그 TASK로 실행 (끝나면 요청이 '완료'로)
BK_PREVIEW      = True     # True = 백업하지 않고 대상만 보여 줌 → 확인 후 False로 바꿔 실행
BK_FORCE        = False    # True = 바뀐 것이 없어도 다시 백업
BK_MODE         = 'all'    # 'all' = 요청 → 준비되는 대로 받기 · 'request' = 요청만 · 'download' = 요청해 둔 것 받기 (한 번 확인)
BK_MAX_WAIT     = 1800     # 'all'에서 기다리는 최대 시간(초). 넘으면 남은 것은 나중에 'download'로 (요청 목록 backups/pending_backups.csv)
BK_SKIP_ULD     = False    # v4.31.8: True = ULD clip(원본)이 있는 TASK는 백업에서 뺌 (LC앱 요청으로 실행할 때는 요청대로 백업)
BK_BOARD        = 'file'   # 작업보드('백업 상태'·'백업 색인' 탭): 'file' = eggplant_yolo/reports/작업보드.xlsx (Drive 연결만으로, 추가 승인 없음)
                           #   · 'sheet' = Google 스프레드시트 (Colab에서 Google 계정 승인을 따로 받음, eggplant_yolo 폴더 안에 둠) · '' = 안 만듦

assert os.path.exists(f'{SCRIPTS}/cvat_backup.py'), 'Drive scripts/에 cvat_backup.py를 올리세요'
if FROM_REQUESTS:
    BK_TASK_IDS, BK_QUERY = req_use('backup'), ''
    BK_MODE, BK_PREVIEW = _REQ['params'].get('mode') or 'all', False      # 웹 요청: 방식은 요청대로, 대상 확인은 웹에서 했으므로 미리 보기 건너뜀 (v4.22)
    if BK_MODE != 'download' and not BK_TASK_IDS:
        raise SystemExit('⛔ 웹 요청에 TASK가 없습니다')
    print(f"   → 방식: {wb.MODES.get(BK_MODE, BK_MODE)} (BK_MODE = '{BK_MODE}') · 미리 보기 건너뜀")
use_cvat(LOGIN)                              # 대상 서버 확인 (로그인 정보 검사 포함)
cb = load_script('cvat_backup.py')
prof = CVAT_PROFILES[CVAT_LOGIN_MODE]
cb.HOST, cb.AUTH_MODE, cb.BASE = CVAT_HOST, prof_login(CVAT_LOGIN_MODE), Path(BASE)
cb.CREDENTIALS = cvat_auth(CVAT_LOGIN_MODE)[1]   # 같은 로그인 정보를 넘김 (파일·로그에 남지 않음)
cb.ORGANIZATION_SLUG, cb.PROJECT_ID = CVAT_SCOPE.get('org'), CVAT_SCOPE.get('project_id')   # 기본 조직·프로젝트 (v4.19)
_bk_ids = ctt.as_task_ids(BK_TASK_IDS) or None
if BK_SKIP_ULD and not FROM_REQUESTS and not _bk_ids:   # v4.31.9 (F19): 검색어로 고를 때도 ULD를 빼려면 대상을 먼저 ID로 정함
    _bk_ids = [t['task_id'] for t in ctt.select_tasks(ctt.list_tasks(CLIENT, limit=5000), BK_QUERY, show=lambda *a: None, label='백업 대상')] or None
    BK_QUERY = ''
if _bk_ids:                                      # v4.31.8: ULD clip이 원본으로 있는 TASK — 백업이 꼭 필요하지 않음
    _bk_uld = [t for t in _bk_ids if uld_task(t)]
    if _bk_uld:
        print(f"ℹ️ ULD clip(원본)이 있는 TASK {len(_bk_uld)}개: {_bk_uld[:10]}" + (" → 백업에서 뺌 (BK_SKIP_ULD)" if BK_SKIP_ULD and not FROM_REQUESTS
              else " — 요청대로 백업합니다 (원본은 ULD clip이 있으므로 꼭 필요하지는 않음)"))
        if BK_SKIP_ULD and not FROM_REQUESTS:
            _bk_ids = [t for t in _bk_ids if t not in _bk_uld] or None
            assert _bk_ids, '백업할 TASK가 없습니다 (모두 ULD clip이 있음)'
cb.NAME_QUERY, cb.TASK_IDS, cb.PREVIEW = BK_QUERY, _bk_ids, (BK_PREVIEW and BK_MODE != 'download')
if globals().get('BK_UPDATE_BOARD') is False:   # 예전 설정 이름 호환
    BK_BOARD = ''
cb.FORCE, cb.UPDATE_BOARD, cb.ONLY_STATES = BK_FORCE, bool(BK_BOARD), None
cb.BOARD_MODE = BK_BOARD
cb.MODE, cb.MAX_WAIT = BK_MODE, BK_MAX_WAIT
_bk_job = not (BK_PREVIEW and BK_MODE != 'download')
if _bk_job:
    job_start(13, 'backup', {'query': BK_QUERY, 'ids': BK_TASK_IDS, 'mode': BK_MODE, 'force': BK_FORCE})
try:
    BK_STATUS = cb.run_backup()
finally:
    if _bk_job:
        from collections import Counter as _C
        job_end('끝', f"모드 {BK_MODE} · 백업 상태: " + ', '.join(f'{k} {v}' for k, v in _C(r['backup_state'] for r in (globals().get('BK_STATUS') or [])).items()))
if BK_PREVIEW and BK_MODE != 'download':
    print('\n대상이 맞으면 BK_PREVIEW = False로 바꿔 다시 실행하세요.')
elif BK_MODE == 'request':
    print("\n요청만 했습니다. 30분~1시간 뒤 BK_MODE = 'download'로 받으세요 (같은 LOGIN으로) — 웹 '요청' 탭의 '백업 받기'로 요청해도 됩니다.")
if wb is not None and hasattr(wb, 'settle'):        # '받는 중'이던 웹 요청: 다 받았으면 완료로 (v4.22)
    for _r in wb.settle(BASE, ('backup',), who=who_am_i()):
        print(f"📨 웹 요청 {_r['id']} ({wb.KINDS[_r['kind']][0]}) → {wb.STATUS_KO.get(_r['status'], _r['status'])} · {_r.get('progress', '')}")


## 라벨 export — TASK 이름으로 골라, 요청과 받기를 나눠서
CVAT는 export를 **요청 → 서버가 파일 준비(백그라운드) → 받기** 순서로 처리합니다. 이 셀은 세 방식으로 씁니다.
- `EX_MODE = 'all'`(기본): 대상을 **한꺼번에 요청**한 뒤, 준비되는 대로 받습니다. `EX_MAX_WAIT`가 지나면 남은 것은 다음에 받습니다.
- `EX_MODE = 'request'`: 요청만 하고 끝냅니다. 서버가 준비하는 동안 세션을 닫아도 됩니다.
- `EX_MODE = 'download'`: 앞서 요청한 것 중 준비된 것을 받습니다(검색어 무시). **요청한 것과 같은 서버·계정**으로 실행하세요.

요청 목록은 `exports/pending_exports.csv`에 남아 세션이 바뀌어도 이어서 받을 수 있습니다. 서버의 준비 파일은 만료 시각(`expiry`)이 지나면 사라지므로, 그 전에 받으세요. 만료되면 다시 요청하면 됩니다.

- 형식 예: `'CVAT for images 1.1'`, `'CVAT for video 1.1'`, `'YOLO 1.1'`, `'COCO 1.0'`
- 저장: `exports/<요청 시각>_<형식>/<TASK 이름>.zip` — 이 폴더를 `cvat_to_yolo.py`의 `INPUT_DIR`로 쓰면 바로 학습 데이터로 변환됩니다.
- 검색어: 공백 = AND, `|` = OR, **쉼표 = 묶음 OR**. 예: `'261015 vf|vf_r2, 261019 vf_r2'`
- 처음에는 `EX_PREVIEW = True`로 대상만 확인하세요.
- **학습 스냅숏 (v4.16)**: `EX_FOLDER = '_train_<학습 이름>'`(웹 '학습 준비'의 스냅숏 요청이 넣음)이면 그 폴더에 대상 TASK를 모두 모읍니다 — 바뀌지 않은 TASK는 이전 export를 복사, 바뀐 TASK는 새로 받음. 모두 받으면 `_manifest.json`·`_manifest.csv`에 TASK마다 클래스별 박스 수와 **정기 export·마지막 백업·마지막 학습과 달라졌는지**를 남깁니다. 정기 export(시각_형식 폴더)와 섞이지 않습니다.


In [ ]:
# 14. 라벨 export — 요청(request)과 받기(download)를 나눠서 할 수 있음
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN      = None                             # None = 지금 접속 중인 서버 · 또는 0번 셀 CVAT_PROFILES의 이름 ('token', 'password', 'B' …)
EX_QUERY   = '261015 vf|vf_r2, 261019 vf_r2'  # (261015 이면서 vf 또는 vf_r2) 또는 (261019 이면서 vf_r2) · 예: '*AA_r1*', 'vf* -tag=AA*'
EX_FOLDER  = ''                               # 받을 폴더 이름 ('' = 시각_형식). 학습 스냅숏은 '_train_<학습 이름>' — 다 받으면 매니페스트 (v4.24.3: 위치를 여기로)
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
EX_FORMAT  = 'CVAT for images 1.1'            # export 형식 (CVAT 화면의 Export 형식 이름 그대로)
EX_IMAGES  = False                            # 이미지 포함 (무료 플랜은 거부됨 → 이미지는 13번 셀 백업이나 Drive videos/로)
EX_IMAGES_FALLBACK = True                     # 이미지 포함이 거부(403)되거나 실패(고해상도 영상 등)하면 라벨만으로 다시 요청
                                              # → 이미지는 15번 셀이 TASK 백업의 원본 영상에서 뽑음
EX_MODE    = 'all'                            # 'all' = 요청 → 준비되는 대로 받기 · 'request' = 요청만 · 'download' = 준비된 것 받기
EX_MAX_WAIT = 1800                            # 'all'에서 기다리는 최대 시간(초). 넘으면 남은 것은 나중에 'download'로
EX_POLL    = 10                               # 준비 상태 확인 간격(초)
EX_PREVIEW = True                             # True = 요청하지 않고 대상만 보여 줌 → 확인 후 False
EX_IDS     = []                               # 또는 TASK ID 직접 (22번 셀의 EXPORT_TODO_IDS 등). 넣으면 EX_QUERY 대신
FROM_REQUESTS = False   # True = 웹 화면의 요청(export)을 받아 그 TASK로 실행 (끝나면 요청이 '완료'로)
EX_REUSE   = True                             # True = 이미 받은 export가 지금 CVAT 라벨과 같으면(같은 형식·이미지 조건) 서버에 요청하지 않고
                                              #        새 폴더로 복사 → 폴더 하나에 대상 TASK가 모두 모임. False = 전부 새로 요청

if FROM_REQUESTS:                                  # 웹 요청 받기 — 작업 기록보다 먼저 (끝날 때 요청이 '완료'로)
    EX_IDS, EX_QUERY = req_use('export'), ''
    EX_MODE, EX_PREVIEW = _REQ['params'].get('mode') or 'all', False      # 웹 요청: 방식은 요청대로, 미리 보기 건너뜀 (v4.22)
    if EX_MODE != 'download' and not EX_IDS:
        raise SystemExit('⛔ 웹 요청에 TASK가 없습니다')
    print(f"   → 방식: {wb.MODES.get(EX_MODE, EX_MODE)} (EX_MODE = '{EX_MODE}') · 미리 보기 건너뜀")
    EX_FORMAT = _REQ['params'].get('format') or EX_FORMAT         # 웹 요청의 형식 (기본 CVAT for images 1.1, 라벨만)
    EX_IMAGES = bool(_REQ['params'].get('images', EX_IMAGES))
    if _REQ['params'].get('snapshot'):
        EX_FOLDER, EX_REUSE = wb.snapshot_folder(_REQ['params']['snapshot']), True
    print(f'   → EX_FORMAT={EX_FORMAT!r} · EX_IMAGES={EX_IMAGES}' + (f' · 학습 스냅숏 폴더 exports/{EX_FOLDER}' if EX_FOLDER else ''))
print(f"▶ 이번 실행: 검색어 {('TASK ID ' + str(len(EX_IDS)) + '개') if EX_IDS else repr(EX_QUERY)} · 형식 {EX_FORMAT}"
      f"{' + 이미지' if EX_IMAGES else ' (라벨만)'} · 폴더 exports/{EX_FOLDER or '<시각_형식>'} · 방식 {EX_MODE}"
      f" · 재사용 {'켬' if EX_REUSE else '끔'}{' · 미리 보기' if EX_PREVIEW and EX_MODE != 'download' else ''}")
import datetime, time
import pandas as pd
from IPython.display import display
assert EX_MODE in ('all', 'request', 'download'), "EX_MODE는 'all', 'request', 'download' 중 하나"
use_cvat(LOGIN)
_ex_job = not (EX_PREVIEW and EX_MODE != 'download')
if _ex_job:
    job_start(14, 'export', {'query': EX_QUERY, 'ids': EX_IDS, 'mode': EX_MODE, 'format': EX_FORMAT, 'images': EX_IMAGES, 'reuse': EX_REUSE})
PENDING = f'{BASE}/exports/pending_exports.csv'
kg_sync_exports()                                 # Kaggle: 다른 세션·컴퓨터가 받은 export zip도 받아 둠 → '파일 없음' 잘못 판단 막기 (v4.23)
if wb is not None and hasattr(wb, 'repair_pending'):
    wb.repair_pending(BASE)                       # 파일이 있는데 '만료'로 적힌 줄 → '받음' (v4.22까지 Kaggle에서 생김)
pending = ctt.read_pending(PENDING)

# ---------- 1) 요청 ----------
if EX_MODE in ('all', 'request'):
    tasks = ctt.list_tasks(CLIENT, limit=5000)
    pd.set_option('display.max_rows', 300)
    targets = pd.DataFrame(ctt.select_tasks(tasks, EX_QUERY, ids=ctt.as_task_ids(EX_IDS) or None, display=display, label='export 대상'))
    _kind = 'video' if 'video' in EX_FORMAT.lower() else 'images'
    _live_ids = {int(r['task_id']) for _, r in targets.iterrows()} if len(targets) else set()
    _missing_ids = [int(x) for x in ctt.as_task_ids(EX_IDS) if int(x) not in _live_ids] if EX_IDS else []
    if EX_REUSE and (len(targets) or _missing_ids):
        print('이미 받은 export 확인 중 (대상 TASK 이름의 zip만, 한 번 읽은 것은 기억) …')
    _scan = ctt.scan_exports(f'{BASE}/exports', names=None if _missing_ids else list(targets['name'])) if EX_REUSE and (len(targets) or _missing_ids) else []
    _reuse = {}
    for _, _t in targets.iterrows():
        _ex, _st = ctt.best_export(_scan, _t['task_id'], _t.get('updated_utc'), kind=_kind, need_images=EX_IMAGES) if EX_REUSE else (None, '')
        if _st == '최신':
            _reuse[int(_t['task_id'])] = _ex
    if EX_REUSE and _missing_ids:
        _ghost_rows = []
        for _mid in _missing_ids:
            _ex, _st = ctt.best_export(_scan, _mid, None, kind=_kind, need_images=EX_IMAGES)
            if _ex:
                _reuse[_mid] = _ex
                _ghost_rows.append({'task_id': _mid, 'name': _ex.get('name') or f'task_{_mid}', '같은 이름': ''})
        if _ghost_rows:
            targets = pd.concat([targets, pd.DataFrame(_ghost_rows)], ignore_index=True)
            print(f"📦 CVAT에서 지워진 TASK {len(_ghost_rows)}개 → Drive의 이전 export에서 스냅숏으로 복사합니다")
    if EX_REUSE and len(targets):
        print(f"이미 받은 export 그대로 사용(복사) {len(_reuse)}개 · 서버에 새로 요청 {len(targets) - len(_reuse)}개")
    print(f"형식 {EX_FORMAT}" + (' · 이미지 포함' if EX_IMAGES else '')
          + (' · 같은 이름의 TASK는 파일 이름 끝에 TASK ID를 붙여 따로 저장' if len(targets) and (targets['같은 이름'] != '').any() else ''))
    if EX_PREVIEW:
        print('대상이 맞으면 EX_PREVIEW = False로 바꿔 다시 실행하세요.')
    else:
        stamp_ = ctt.stamp()                                   # 한국 시각 (Colab 기본 시계는 UTC)
        out_dir = (f"{BASE}/exports/{EX_FOLDER}" if EX_FOLDER else
                   f"{BASE}/exports/{stamp_}_{re.sub(r'[^A-Za-z0-9]+', '_', EX_FORMAT).strip('_').lower()}")
        cur_images, fails_in_row, n_req = EX_IMAGES, 0, 0
        _stem = lambda n: re.sub(r'[^0-9A-Za-z가-힣_.-]+', '_', str(n))
        _dupn = targets['name'].map(_stem).value_counts()          # v4.31.4: 대상 안(지운 TASK 포함)에서 이름이 겹치면 → __ID (요청 8cjj 실패)
        for _, t in targets.iterrows():
            tid, name = int(t['task_id']), t['name']
            fname = _stem(name) + (f'__{tid}' if t['같은 이름'] != '' or _dupn.get(_stem(name), 0) > 1 else '')
            prow = dict(request_time=ctt.now(), server=CVAT_SERVER, task_id=tid, task_name=name, format=EX_FORMAT,
                        out_path=os.path.relpath(f'{out_dir}/{fname}.zip', BASE),
                        org=ctt.org_ctx_value(getattr(CLIENT, 'organization_slug', None)))   # 받을 때 같은 조직 범위로 (v4.22)
            if tid in _reuse:                                   # 받은 것과 같음 → 복사 (서버 요청 없음)
                _src = _reuse[tid]['zip']
                os.makedirs(out_dir, exist_ok=True)
                shutil.copy2(_src, f"{BASE}/{prow['out_path']}")
                prow.update(images='예' if _reuse[tid]['images'] else '', status='받음', result_file=prow['out_path'],
                            note=f"복사: {os.path.relpath(_src, BASE)} (CVAT 라벨이 그 뒤로 바뀌지 않음)")
                pending.append(prow)
                ctt.append_registry(REGISTRY, run_time=ctt.now(), action='export', task_id=tid, task_name=name, server=CVAT_SERVER,
                                    status='완료(복사)', result_file=prow['out_path'], note=prow['note'])
                print(f'📋 TASK {tid} {name} — 이전 export 복사 ({os.path.basename(os.path.dirname(_src))})')
                continue
            try:
                try:
                    rq = ctt.request_export(CLIENT, tid, EX_FORMAT, cur_images)
                except Exception as e:
                    if cur_images and EX_IMAGES_FALLBACK and 'resolution' in str(e):      # 이 TASK만 라벨로
                        rq = ctt.request_export(CLIENT, tid, EX_FORMAT, False)
                        prow.update(images='', rq_id=rq, status='요청됨', note='고해상도 → 라벨만 (이미지는 백업 영상에서)')
                        pending.append(prow); n_req += 1
                        print(f'📨 TASK {tid} {name} — 고해상도 영상이라 이미지 포함 불가 → 라벨만 요청됨')
                        continue
                    if getattr(e, 'status', None) == 403 and cur_images and EX_IMAGES_FALLBACK:
                        rq = ctt.request_export(CLIENT, tid, EX_FORMAT, False)     # 라벨만은 되는지 확인
                        cur_images = False
                        print('⚠️ 이미지 포함 export가 거부되었습니다(403) — 라벨만은 됩니다. 나머지도 라벨만 요청합니다.\n'
                              '   이미지는 13번 셀 TASK 백업(영상 포함) 또는 Drive videos/의 영상에서 얻으세요.')
                    else:
                        raise
                prow.update(images='예' if cur_images else '', rq_id=rq, status='요청됨')
                pending.append(prow)
                n_req, fails_in_row = n_req + 1, 0
                print(f'📨 TASK {tid} {name} — 요청됨')
            except Exception as e:
                msg = ctt.api_error(e)
                extra = ''
                if getattr(e, 'status', None) == 403:
                    try:
                        owner, assignee = ctt.task_people(CLIENT.tasks.retrieve(tid))
                        extra = f' (TASK 소유자 {owner}, 담당자 {assignee})'
                    except Exception:
                        pass
                print(f'❌ TASK {tid} {name}: {msg}{extra}')
                prow.update(status='실패', note=f'요청 실패: {msg}'[:200])      # 대기 목록에도 남김 → 웹 요청이 '실패'를 알 수 있게 (v4.22)
                pending.append(prow)
                ctt.append_registry(REGISTRY, run_time=ctt.now(), action='export', task_id=tid, task_name=name,
                                    server=CVAT_SERVER, status='오류', note=f'요청 실패 {EX_FORMAT}: {msg}')
                fails_in_row += 1
                if fails_in_row >= 3 and n_req == 0:
                    print('\n⛔ 처음 3개가 모두 실패해 멈춥니다. 403이면 이 계정(' + CVAT_SERVER + ')에 TASK export 권한이 없습니다 '
                          '— TASK 소유자·담당자 또는 조직 관리자 권한이 필요합니다. 다른 계정은 LOGIN을 바꿔 다시 실행하세요.')
                    break
        ctt.write_pending(PENDING, pending)
        print(f'\n요청 {n_req}개 기록: {os.path.relpath(PENDING, BASE)}')
        print(f'받을 폴더: {out_dir}')
        if EX_MODE == 'request' and n_req:
            print("서버가 파일을 준비하는 동안 다른 일을 해도 됩니다. 나중에 EX_MODE = 'download'로 실행해 받으세요.")

# ---------- 2) 받기 ----------
if EX_MODE in ('all', 'download') and not (EX_PREVIEW and EX_MODE == 'all'):
    # '받음'인데 Drive에 파일이 없으면 다시 받기 대상으로 (만료 전이면 다시 받아짐)
    _lost, _recopy = [], 0
    for r in pending:
        if r['server'] != CVAT_SERVER:
            continue
        _f = r.get('result_file') or r.get('out_path') or ''
        if r['status'] == '받음' and not os.path.exists(f"{BASE}/{_f}"):
            _m = re.match(r'복사: (\S+)', r.get('note') or '')
            if _m and os.path.exists(f"{BASE}/{_m.group(1)}"):          # 이전 export를 복사했던 줄 → 다시 복사 (v4.24.2)
                os.makedirs(os.path.dirname(f"{BASE}/{_f}"), exist_ok=True)
                shutil.copy(f"{BASE}/{_m.group(1)}", f"{BASE}/{_f}")
                _recopy += 1
            elif not r.get('rq_id'):                                       # 서버에 요청한 적이 없는 줄 → 다시 받을 수 없음
                r.update(status='만료', note='받은 기록만 있고 파일이 없음 → 14번 셀을 같은 검색어로 다시 실행')
                _lost.append(r)
            else:
                print(f"⚠️ '받음'으로 기록됐지만 파일이 없음 → 다시 받기: {BASE}/{_f}")
                r.update(status='요청됨', note='파일이 Drive에 없어 다시 받기')
        elif r['status'] in ('요청됨', '준비 중') and not r.get('rq_id'):    # v4.24까지 복사본을 '다시 받기'로 바꿔 둔 줄
            r.update(status='만료', note='요청 번호 없음 (복사본이 Drive에 올라가지 않았음) → 14번 셀을 같은 검색어로 다시 실행')
            _lost.append(r)
    if _recopy:
        print(f'📋 이전 export에서 다시 복사: {_recopy}개 (받은 기록은 있는데 파일이 없던 것)')
    if _lost:
        print(f"⚠️ 받은 기록만 있고 파일이 없는 줄 {len(_lost)}개 → '만료'로 정리 (CVAT에 다시 묻지 않음). "
              "같은 검색어로 14번 셀을 다시 실행하면 이전 export에서 다시 복사하거나 새로 받습니다: "
              + ', '.join(sorted({x['task_name'] for x in _lost})[:5]) + (' …' if len(_lost) > 5 else ''))
    open_rows = [r for r in pending if r['server'] == CVAT_SERVER and r['status'] in ('요청됨', '준비 중')]
    others = sum(1 for r in pending if r['server'] != CVAT_SERVER and r['status'] in ('요청됨', '준비 중'))
    if others:
        print(f'참고: 다른 서버·계정으로 요청한 것 {others}개는 그 LOGIN으로 받아야 합니다.')
    print(f'\n받을 것 {len(open_rows)}개 ({CVAT_SERVER})')
    _org_keep = getattr(CLIENT, 'organization_slug', None)
    deadline = time.time() + (EX_MAX_WAIT if EX_MODE == 'all' else 0)
    while open_rows:
        for r in list(open_rows):
            r['checked_time'] = ctt.now()
            CLIENT.organization_slug = ctt.org_ctx_slug(r.get('org'))      # 요청했던 조직 범위로 확인·받기 (v4.22)
            try:
                st = ctt.export_status(CLIENT, r['rq_id'])
            except Exception as e:
                if getattr(e, 'status', None) in (404, 410):
                    r.update(status='만료', note='서버에 요청이 없음(만료) → 다시 요청하세요')
                else:
                    r['note'] = f'상태 확인 실패: {ctt.api_error(e)}'
                    continue
                open_rows.remove(r)
                print(f"⌛ TASK {r['task_id']} {r['task_name']} — 만료됨, 다시 요청 필요")
                continue
            r['expiry'] = st['expiry'] or r.get('expiry', '')
            if st['status'] == 'finished':
                try:
                    path = ctt.download_result(CLIENT, st['result_url'], f"{BASE}/{r['out_path']}")
                    mb = os.path.getsize(path) / 2**20
                    r.update(status='받음', result_file=r['out_path'], progress='100%', note=f'{mb:.2f} MB')
                    ctt.append_registry(REGISTRY, run_time=ctt.now(), action='export', task_id=r['task_id'],
                                        task_name=r['task_name'], server=CVAT_SERVER, status='완료', result_file=r['out_path'],
                                        note=f"{r['format']}{' + 이미지' if r.get('images') else ' (라벨만)'} ({mb:.2f} MB)")
                    print(f"✅ TASK {r['task_id']} {r['task_name']} → {os.path.basename(path)} ({mb:.2f} MB)")
                except Exception as e:
                    code = getattr(e, 'status', None)
                    r.update(status='만료' if code in (404, 410) else '준비 중', note=f'받기 실패: {ctt.api_error(e)}')
                    print(f"❌ TASK {r['task_id']} 받기 실패: {ctt.api_error(e)}")
                    if code not in (404, 410):
                        continue
                open_rows.remove(r)
            elif st['status'] == 'failed' and r.get('images') and EX_IMAGES_FALLBACK:
                # 이미지 포함 export 실패(고해상도 영상의 프레임 인코딩 등) → 같은 TASK를 라벨만으로 다시 요청
                try:
                    rq = ctt.request_export(CLIENT, int(r['task_id']), r['format'], False)
                    r.update(images='', rq_id=rq, status='요청됨', progress='',
                             note=f"이미지 포함 실패({st['message'][:80]}) → 라벨만 다시 요청 (이미지는 백업 영상에서)")
                    print(f"↻ TASK {r['task_id']} {r['task_name']} — 이미지 포함 실패 → 라벨만 다시 요청")
                except Exception as e:
                    r.update(status='실패', note=f"이미지 포함 실패 후 라벨만 요청도 실패: {ctt.api_error(e)}")
                    open_rows.remove(r)
            elif st['status'] == 'failed':
                r.update(status='실패', note=st['message'][:200])
                ctt.append_registry(REGISTRY, run_time=ctt.now(), action='export', task_id=r['task_id'],
                                    task_name=r['task_name'], server=CVAT_SERVER, status='오류', note=f"서버 준비 실패: {st['message'][:200]}")
                print(f"❌ TASK {r['task_id']} {r['task_name']} — 서버 준비 실패: {st['message'][:120]}")
                open_rows.remove(r)
            else:
                r.update(status='준비 중', progress=st['progress'])
        ctt.write_pending(PENDING, pending)                 # 중간 저장 (세션이 끊겨도 남음)
        if not open_rows or time.time() >= deadline:
            break
        print(f"  … 준비 중 {len(open_rows)}개, {EX_POLL}초 뒤 다시 확인 (최대 {max(0, int(deadline - time.time()))}초)")
        time.sleep(EX_POLL)
    CLIENT.organization_slug = _org_keep
    if open_rows:
        print(f"\n아직 준비 중인 {len(open_rows)}개는 나중에 EX_MODE = 'download'로 받으세요 (만료 전에) — 웹 '요청' 탭의 'export 받기'로 요청해도 됩니다.")
    ctt.write_pending(PENDING, pending)
    try:
        os.sync()                                            # Drive로 쓰기 마무리 (웹에 보이기까지 몇 분 걸릴 수 있음)
    except Exception:
        pass
    got = [r for r in pending if r['server'] == CVAT_SERVER and r['status'] == '받음']
    if got:
        dirs = sorted({os.path.dirname(f"{BASE}/{r['result_file']}") for r in got})
        ok_files = sum(os.path.exists(f"{BASE}/{r['result_file']}") for r in got)
        print(f'\n받은 파일 {ok_files}/{len(got)}개가 Drive에 있음. 폴더:')
        for d in dirs[-5:]:
            print('  ', d, f'({len(glob.glob(d + "/*.zip"))}개 zip)')
    mine = pd.DataFrame([r for r in pending if r['server'] == CVAT_SERVER])
    if len(mine):
        print('\n이 서버의 export 요청 현황 (최근 30개):')
        display(mine[['request_time', 'task_id', 'task_name', 'format', 'images', 'status', 'progress', 'expiry', 'result_file', 'note']].tail(30))
# 학습 스냅숏(exports/_train_*): 그 폴더의 파일을 모두 받았으면 매니페스트 (정기 export·백업·마지막 학습과 비교)
if wb is not None and not (EX_PREVIEW and EX_MODE != 'download'):
    _snap_dirs = sorted({r['out_path'].split('/')[1] for r in pending if r.get('out_path', '').startswith('exports/' + wb.SNAP_PREFIX)}
                        | ({EX_FOLDER} if EX_FOLDER.startswith(wb.SNAP_PREFIX) else set()))
    for _sd in _snap_dirs:
        _left = [r for r in pending if r.get('out_path', '').startswith(f'exports/{_sd}/') and r['status'] in ('요청됨', '준비 중')]
        if _left:
            print(f"스냅숏 {_sd}: 아직 받을 것 {len(_left)}개 → EX_MODE = 'download'로 받은 뒤 매니페스트를 만듭니다")
        elif os.path.isdir(f'{BASE}/exports/{_sd}'):
            _man = wb.write_snapshot_manifest(BASE, _sd)
            display(pd.DataFrame([{'비교': {'vs_export': '정기 export와', 'vs_backup': '백업과', 'vs_train': '마지막 학습과'}[k], **v}
                                  for k, v in _man['summary'].items()]).set_index('비교').fillna(0).astype(int))
if _ex_job:
    job_end('끝', f'모드 {EX_MODE} · 형식 {EX_FORMAT}' + (f' · 스냅숏 {EX_FOLDER}' if EX_FOLDER else ''))
if wb is not None and hasattr(wb, 'settle'):        # '받는 중'이던 웹 요청: 다 받았으면 완료로 (v4.22)
    for _r in wb.settle(BASE, ('export',), who=who_am_i()):
        print(f"📨 웹 요청 {_r['id']} ({wb.KINDS[_r['kind']][0]}) → {wb.STATUS_KO.get(_r['status'], _r['status'])} · {_r.get('progress', '')}")

## export 폴더로 학습 — zip 목록 확인 → 데이터셋 → 학습
14번 셀로 받은 `exports/<폴더>/`의 zip을 읽어 바로 YOLO 학습까지 합니다.
- zip마다 형식·TASK·프레임 수·박스 수·**이미지를 어디서 얻는지**(zip 안 / Drive `videos/`의 영상 / 없음)를 먼저 보여 줍니다.
  무료 플랜 export는 라벨만 있으므로, 이미지는 `videos/`(하위 폴더 포함)에서 같은 이름의 영상을 찾아 프레임을 뽑습니다.
- 만든 데이터셋은 `datasets/<이름>.zip`으로 Drive에 저장됩니다. 세션이 끊기면 0번 → 15번 셀을 다시 실행하면 데이터셋은 저장본을 풀고, 학습은 이어서 합니다.
- 학습 결과는 `runs/<TRAIN_NAME>/`. 이 셀을 실행한 뒤 2번(결과)·3번(검증 1단계) 셀은 이 학습을 기준으로 동작합니다.
- **v4.13**: 0번 셀 `REVIEWED_RULE`(기본 `'vf'`)로 **검토 완료 TASK만** 쓰고, `VAL_RULE`(기본 `'row=2 plant=6..10'`)에 맞는 TASK를 **val**, 나머지를 train으로 나눕니다(TASK 단위). 목록 아래 train/val 비율 표를 보고 학습하세요. 어떤 TASK가 어느 학습의 train/val에 들어갔는지는 `logs/train_tasks.csv`에 남고 22번 셀에서 봅니다.

**v4.30.0 — 추가 학습 계획 · GPU 여럿 · 데이터셋 동시 읽기**
- `VAL_IDS`: val TASK를 ID로 고정합니다 (VAL_RULE 대신, 나머지는 모두 train). LC앱 '추가 학습' 탭이 기준 모델의 val을 그대로 넣어 보냅니다 — 같은 val로 채점해야 새 모델이 나아졌는지 비교됩니다.
- `TRAIN_PATIENCE`: 조기 종료. 이 에폭 동안 val mAP가 나아지지 않으면 멈춥니다. 이어 학습은 이미 수렴한 가중치에서 시작하므로 짧게(에폭 60 · patience 15 정도).
- `TRAIN_GPUS = 'auto'`: GPU가 여럿이면(Kaggle T4×2) 모두 써서 동시에 학습합니다(DDP). 이때 자동 배치(BATCH = -1)는 안 되므로 GPU 하나당 `TRAIN_GPU_BATCH`(4)장. 동시 학습을 시작하지 못하면 GPU 하나로 다시 합니다. 동시 학습 중에는 1분마다 잠금을 갱신하고 (Kaggle) runs/<이름>을 Drive로 올립니다.
- 데이터셋을 만들 때 TASK zip 읽기·영상에서 프레임 꺼내기를 4개씩 동시에 합니다 (`cvat_to_yolo.LOAD_WORKERS`).


In [ ]:
# 15. export 폴더로 학습 — zip 목록 확인 → 데이터셋 만들기 → 학습
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
EXP_DIR     = ''        # '' = exports/ 안 가장 최근 폴더 · 폴더 이름 · 여러 폴더 ['20261025_1640_...', '20261025_1702_...']
EXP_DEDUP   = True      # 여러 폴더에 같은 TASK(같은 TASK ID)가 있으면 가장 나중 폴더의 것만 씀 (같은 이미지가 두 번 학습되지 않게)
EXP_QUERY   = ''        # 폴더 안에서 쓸 TASK만 고르기 (모든 셀 같은 검색 규칙, TASK 이름 기준). '' = 폴더 안 전부. 예: 'G1|G3 date=261019.. vf*'
# 검토 완료만 쓰기·train/val 나누기는 0번 셀 REVIEWED_RULE·VAL_RULE (v4.13) — 미검토 TASK는 자동으로 빠지고, VAL_RULE에 맞는 TASK가 val
TRAIN_NAME  = ''        # 학습 결과 이름 runs/<이름> ('' = exp_<폴더 이름>, 다른 데이터로 이미 쓴 이름이면 뒤에 _<서명>을 붙임)
DATASET_NAME = ''       # 데이터셋 이름 datasets/<이름>.zip ('' = TRAIN_NAME과 같음). 이미 만든 데이터셋을 다른 학습 이름으로 쓸 때 그 이름을 적음 (v4.24.6)
                        #   예: TRAIN_NAME = 'fruit_1920_ft261101', DATASET_NAME = 'fruit_r261101_2class_rotate' — EXP_*·CONVERT는 그 데이터셋을 만들 때와 같게
FROM_REQUESTS = False   # True = 웹 화면의 학습 요청을 받아 실행 — 요청의 export 폴더·TASK·검토/val 규칙·학습 이름 (끝나면 요청이 '완료'로)
EXP_IDS     = []        # 쓸 TASK ID만 (비우면 폴더·검색어로 고른 전부). 웹의 '학습 준비'가 준비된 TASK를 넣어 보냄
LIST_ONLY   = False     # True = zip 목록만 보고 끝
BUILD_ONLY  = False     # True = 데이터셋만 만들고 학습은 안 함
REBUILD     = False     # True = 저장된 데이터셋이 있어도 다시 만듦
FORCE_NEW   = False     # True = 같은 이름의 이전 학습(runs/<이름>)과 데이터셋을 지우고 처음부터 (다른 사람이 학습 중이면 안 됨)
USE_TEST_SET = True     # v4.31.8: 고정 시험 TASK(config/test_tasks.json)와 같은 그루·촬영일 TASK는 학습·val에서 뺌 (33번 셀에서 정함)
ALLOW_CPU   = False     # GPU가 없으면 학습하지 않고 멈춤 (CPU로 imgsz 1920 학습은 며칠 걸림)
CONVERT = {'MERGE_HEALTH': True, 'KEEP_CLASSES': ['healthy_fruit'], 'OCCLUDED_POLICY': 'keep', 'VAL_RATIO': 0.2}   # cvat_to_yolo 설정
# 학습 설정 (v4.24.3) — 같은 데이터셋으로 학습만 달리할 때: TRAIN_NAME은 새 이름, DATASET_NAME = 처음 만든 데이터셋 이름 (v4.24.6)
TRAIN_FROM  = ''        # 시작 가중치: '' = 0번 셀 MODEL(yolo11s.pt)에서 처음부터 · runs/ 이름(예 'fruit_r261101') = 그 모델의 best.pt에서 이어 학습 · 또는 .pt 경로
TRAIN_EPOCHS = None     # None = 0번 셀 EPOCHS · 이어 학습은 보통 짧게 (30~60)
TRAIN_LR0   = None      # 학습률. None = Ultralytics 자동 · 이어 학습 권장 0.001~0.002 · 처음부터는 보통 그대로
TRAIN_OPTIMIZER = 'auto'  # 'auto' · 'SGD' · 'AdamW' — 'auto'는 학습률을 무시하므로 TRAIN_LR0를 정하면 AdamW로 바꿈
TRAIN_ARGS  = {}        # 그 밖의 Ultralytics 학습 설정, 예: {'warmup_epochs': 1, 'lrf': 0.1, 'freeze': 10}
TRAIN_PATIENCE = None   # v4.30: 조기 종료 — 이 에폭 동안 val이 나아지지 않으면 멈춤. None = 0번 셀 PATIENCE · 이어 학습은 10~20
VAL_IDS     = []        # v4.30: val TASK를 ID로 고정 (추가 학습 계획: 기준 모델의 val 그대로) — 정하면 VAL_RULE 대신, 나머지는 모두 train
TRAIN_GPUS  = 'auto'    # v4.30: 'auto' = GPU가 여럿이면(Kaggle T4×2) 모두 써서 동시 학습(DDP) · 1 = GPU 하나만
TRAIN_GPU_BATCH = 4     # GPU 여럿일 때 GPU 하나당 이미지 수 (자동 배치는 GPU 하나에서만 됨) · 메모리 오류면 2
_PLAN = None            # (LC앱 '추가 학습' 탭의 계획 — 웹 요청이 넣음)

if FROM_REQUESTS:                                  # 웹 요청(학습)의 설정
    _p = req_use('train')
    EXP_DIR = _p.get('exp_dirs') or _p.get('exp_dir') or EXP_DIR    # 여러 폴더면 시간 순 (같은 TASK는 나중 폴더 것)
    EXP_IDS = _p.get('task_ids') or EXP_IDS
    if _p.get('reviewed_rule') is not None:
        REVIEWED_RULE = _p['reviewed_rule']
        ctt.REVIEWED_RULE = REVIEWED_RULE
    if _p.get('val_rule') is not None:
        VAL_RULE = _p['val_rule']
    print(f"   → 규칙: 검토 = 상태에 '{REVIEWED_RULE}' · val = '{VAL_RULE}' · TASK {len(EXP_IDS) or '폴더 전부'}개 · 폴더 "
          f"{len(EXP_DIR) if isinstance(EXP_DIR, (list, tuple)) else 1}개 (이 학습에만, 0번 셀 값은 그대로 두려면 0번 셀을 다시 실행)")
    EXP_QUERY = _p.get('exp_query', EXP_QUERY) or ''
    TRAIN_NAME = _p.get('train_name') or TRAIN_NAME
    DATASET_NAME = _p.get('dataset_name') or DATASET_NAME
    BUILD_ONLY = bool(_p.get('build_only', BUILD_ONLY))
    TRAIN_FROM = _p.get('train_from') or TRAIN_FROM                  # 웹 요청의 시작 모델·에폭·학습률 (v4.24.3)
    TRAIN_EPOCHS = _p.get('epochs') or TRAIN_EPOCHS
    TRAIN_LR0 = _p.get('lr0') if _p.get('lr0') not in (None, '') else TRAIN_LR0
    TRAIN_OPTIMIZER = _p.get('optimizer') or TRAIN_OPTIMIZER
    TRAIN_PATIENCE = _p.get('patience') or TRAIN_PATIENCE           # v4.30: 추가 학습 계획
    VAL_IDS = [str(x) for x in (_p.get('val_ids') or VAL_IDS)]
    _PLAN = _p.get('plan') or _PLAN
    if VAL_IDS:
        print(f"   → val = TASK ID {len(VAL_IDS)}개로 고정 (VAL_RULE 대신)" + (f" · 추가 학습 계획 {_PLAN.get('name')} (기준 {_PLAN.get('base')})" if isinstance(_PLAN, dict) else ''))
    print(f'   → EXP_DIR={EXP_DIR!r} · EXP_QUERY={EXP_QUERY!r} · TRAIN_NAME={TRAIN_NAME!r} · BUILD_ONLY={BUILD_ONLY}')
# 학습 설정 정리: 시작 가중치 · Ultralytics에 넘길 값
_TA = dict(TRAIN_ARGS or {})
if TRAIN_EPOCHS:
    _TA['epochs'] = int(TRAIN_EPOCHS)
if TRAIN_LR0 is not None:
    _TA['lr0'] = float(TRAIN_LR0)
    _TA['optimizer'] = TRAIN_OPTIMIZER if TRAIN_OPTIMIZER != 'auto' else 'AdamW'
elif TRAIN_OPTIMIZER != 'auto':
    _TA['optimizer'] = TRAIN_OPTIMIZER
if TRAIN_PATIENCE:
    _TA['patience'] = int(TRAIN_PATIENCE)
import torch as _torch
_NGPU = _torch.cuda.device_count() if _torch.cuda.is_available() else 0
if _NGPU > 1 and str(TRAIN_GPUS) not in ('1', '0', 'None') and 'device' not in _TA:   # v4.30: GPU 여럿으로 동시 학습
    _TA['device'] = ','.join(str(i) for i in range(_NGPU))
    if 'batch' not in _TA and (BATCH is None or int(BATCH) < 1):
        _TA['batch'] = int(TRAIN_GPU_BATCH) * _NGPU
if not TRAIN_FROM:
    _START, _START_TXT = MODEL, f'{MODEL} (처음부터)'
else:
    _t = str(TRAIN_FROM)
    _START = need_file(_t if _t.endswith('.pt') else f'{BASE}/runs/{_t}/weights/best.pt')   # v4.31.6 Kaggle: 그때 받음
    if not os.path.isabs(_START) and os.path.exists(f'{BASE}/{_START}'):
        _START = f'{BASE}/{_START}'
    _START_TXT = f"{_t if not _t.endswith('.pt') else os.path.relpath(_START, BASE)} best.pt에서 이어 학습"
print(f"▶ 이번 실행: 폴더 {EXP_DIR or '(가장 최근)'} · 검색어 {EXP_QUERY or '(폴더 전부)'}"
      f"{' · TASK ID ' + str(len(EXP_IDS)) + '개' if EXP_IDS else ''} · 이름 {TRAIN_NAME or '(자동)'} · "
      f"{('데이터셋 ' + DATASET_NAME + ' (있는 것 사용) · ') if DATASET_NAME and DATASET_NAME != TRAIN_NAME else ''}"
      f"{'목록만' if LIST_ONLY else '데이터셋만' if BUILD_ONLY else '데이터셋 + 학습'}"
      + ('' if LIST_ONLY or BUILD_ONLY else f" · 시작 {_START_TXT} · 에폭 {_TA.get('epochs', EPOCHS)} · "
         f"학습률 {_TA.get('lr0', '자동')} ({_TA.get('optimizer', 'auto')})" + (f" · 조기 종료 {_TA['patience']}" if 'patience' in _TA else '') + (f" · GPU {_NGPU}개 동시 (batch {_TA.get('batch')})" if ',' in str(_TA.get('device', '')) else '')
         + (f" · 그 밖 {({k: v for k, v in _TA.items() if k not in ('epochs', 'lr0', 'optimizer')})}" if set(_TA) - {'epochs', 'lr0', 'optimizer'} else '')))
if TRAIN_FROM and not (LIST_ONLY or BUILD_ONLY):
    assert os.path.exists(_START), (f"시작 모델이 없습니다: {_START}\n"
                                    f"runs/ 이름을 확인하세요 (학습을 끝낸 모델: {[os.path.basename(os.path.dirname(os.path.dirname(p))) for p in sorted(glob.glob(f'{BASE}/runs/*/weights/best.pt'))][:12]})")
import xml.etree.ElementTree as ET, hashlib, datetime
from collections import Counter
import pandas as pd
from IPython.display import display

# ---------- 1) 폴더 고르기 ----------
kg_sync_exports()                                 # Kaggle: 다른 세션이 받은 export도 (v4.23)
EXP_ROOT = f'{BASE}/exports'
all_dirs = sorted([d for d in glob.glob(f'{EXP_ROOT}/*') if os.path.isdir(d)], key=os.path.getmtime)
assert all_dirs, f'{EXP_ROOT}에 export 폴더가 없습니다 (14번 셀로 먼저 받기)'
if not EXP_DIR:
    sel = [all_dirs[-1]]
else:
    names = [EXP_DIR] if isinstance(EXP_DIR, str) else list(EXP_DIR)
    sel = [d if os.path.isabs(d) else f'{EXP_ROOT}/{d}' for d in names]
    for d in sel:
        assert os.path.isdir(d), f'폴더 없음: {d}\n있는 폴더(최근 5개): {[os.path.basename(x) for x in all_dirs[-5:]]}'
print('export 폴더:', ', '.join(os.path.basename(d) for d in sel))

# ---------- 2) zip 목록과 이미지 출처 ----------
rows, zips = [], []
_KG['no_fetch'] = ON_KAGGLE                    # Kaggle: 목록 단계에서는 영상을 받지 않음 — Drive에 있는지만 보고, 데이터셋을 새로 만들 때 쓸 TASK 것만 받음 (v4.24.7)
for d in sel:
    for z in sorted(glob.glob(f'{d}/*.zip')):
        zips.append(z)
        r = {'폴더': os.path.basename(d), '파일': os.path.basename(z)}
        try:
            with zipfile.ZipFile(z) as zf:
                xn = [n for n in zf.namelist() if n.replace('\\', '/').split('/')[-1] == 'annotations.xml']
                n_img_in_zip = sum(1 for n in zf.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
                root = ET.fromstring(zf.read(xn[0])) if xn else None
        except Exception as e:
            r.update(형식='읽기 실패', 비고=str(e)[:80]); rows.append(r); continue
        if root is None:
            r.update(형식='annotations.xml 없음'); rows.append(r); continue
        t = root.find('meta/task') if root.find('meta/task') is not None else root.find('meta/job')
        tname = (t.findtext('name') if t is not None else '') or ''
        source = (t.findtext('source') if t is not None else '') or ''
        r['TASK ID'] = (t.findtext('id') if t is not None else '') or ''
        r['_updated'] = (t.findtext('updated') if t is not None else '') or ''   # 이 export가 반영한 CVAT 라벨 시각
        r['_zip'] = z
        if root.find('image') is not None:
            fmt, n_fr = 'images', len(root.findall('image'))
            labels = Counter(b.get('label') for b in root.iter('box'))
        else:
            fmt = 'video'
            ff = (t.findtext('frame_filter') or '') if t is not None else ''
            step = int(ff.split('=')[1]) if ff.startswith('step=') else 1
            n_fr = len(range(int(t.findtext('start_frame') or 0), int(t.findtext('stop_frame') or 0) + 1, step)) if t is not None else 0
            labels = Counter(tr.get('label') for tr in root.findall('track'))
        video = ctt.find_video(VIDEOS, source, tname) if ctt else None
        bk = sorted(glob.glob(f"{BASE}/backups/{ctt.base_name(tname)}/**/*.zip", recursive=True)) if (ctt and tname) else []
        _on_drive = '' if (n_img_in_zip or video or bk) else kaggle_media_on_drive(source, tname)
        src = ('zip 안' if n_img_in_zip else f'영상 {os.path.relpath(str(video), VIDEOS)}' if video
               else f'TASK 백업 영상 ({len(bk)}개 중 최신)' if bk
               else f'{_on_drive} (Drive — 필요할 때 받음)' if _on_drive else '❌ 없음')
        r['_source'], r['_media'] = source, (not n_img_in_zip)
        r.update(형식=fmt, TASK=tname, 프레임=n_fr, 박스_또는_track=sum(labels.values()),
                 라벨=', '.join(f'{k} {v}' for k, v in labels.most_common()), 이미지=src,
                 비고='' if (n_img_in_zip or video or bk or _on_drive) else f"13번 셀로 TASK 백업 또는 videos/에 {ctt.base_name(tname) if ctt else tname}.mp4 (또는 {source}) 올리기")
        rows.append(r)
# 검색어(EXP_QUERY) 밖의 TASK는 뺌
if EXP_QUERY:
    _out = []
    for r in rows:
        if r.get('TASK') and not ctt.match_name(r['TASK'], EXP_QUERY)[0]:
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + '검색어 밖 → 제외'
            _out.append(r['_zip'])
    zips = [z for z in zips if z not in _out]
    rows_in = [r for r in rows if r.get('_zip') not in _out]
    print(f"검색어 '{EXP_QUERY}' → 폴더 안 zip {len(rows)}개 중 {len(rows_in)}개 사용")
else:
    rows_in = rows
if EXP_IDS:                                        # TASK ID로 고르기 (웹 '학습 준비'의 준비된 TASK)
    _want = {str(x) for x in ctt.as_task_ids(EXP_IDS)}
    _drop = [r['_zip'] for r in rows_in if r.get('_zip') and str(r.get('TASK ID')) not in _want]
    for r in rows_in:
        if r.get('_zip') in _drop:
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + 'TASK ID 목록 밖 → 제외'
    zips = [z for z in zips if z not in _drop]
    rows_in = [r for r in rows_in if r.get('_zip') not in _drop]
    _have = {str(r.get('TASK ID')) for r in rows_in}
    print(f"TASK ID {len(_want)}개 중 이 폴더들에 export가 있는 것 {len(_want & _have)}개"
          + (f" · 없는 것 {len(_want - _have)}개 (export부터): {sorted(_want - _have)[:10]}" if _want - _have else ''))
# 검토 여부·역할 (0번 셀 REVIEWED_RULE·VAL_RULE) — 미검토 TASK는 뺌
for r in rows:
    if r.get('TASK'):
        r['검토'] = '✅' if task_reviewed(r['TASK']) else '⏳ 미검토'
        r['역할'] = task_role(r['TASK']) or '무작위'
if VAL_IDS:                                        # v4.30: val TASK를 ID로 고정 (VAL_RULE 대신)
    _VS = {str(x) for x in VAL_IDS}
    for r in rows:
        if r.get('TASK'):
            r['역할'] = 'val' if str(r.get('TASK ID')) in _VS else 'train'
if REVIEWED_RULE:
    _unrev = [r['_zip'] for r in rows_in if r.get('TASK') and r.get('_zip') and not task_reviewed(r['TASK'])]
    for r in rows_in:
        if r.get('_zip') in _unrev:
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + '미검토 → 제외'
    zips = [z for z in zips if z not in _unrev]
    rows_in = [r for r in rows_in if r.get('_zip') not in _unrev]
    if _unrev:
        print(f"미검토 TASK {len(_unrev)}개를 뺐습니다 (상태에 '{REVIEWED_RULE}'가 없음 — 0번 셀 REVIEWED_RULE).")
_TEST = ctt.read_test_tasks(BASE) if hasattr(ctt, 'read_test_tasks') else None   # v4.31.8: 고정 시험 TASK
if USE_TEST_SET and _TEST and _TEST['ids']:
    _tx = {}
    for r in rows_in:
        _why = ctt.test_conflict(_TEST, r.get('TASK ID'), r.get('TASK', '')) if r.get('TASK') else ''
        if _why and r.get('_zip'):
            _tx[r['_zip']] = _why
            r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + f'{_why} → 제외 (고정 시험)'
    if _tx:
        zips = [z for z in zips if z not in _tx]
        rows_in = [r for r in rows_in if r.get('_zip') not in _tx]
        print(f"고정 시험 TASK {sum(v == '시험 TASK' for v in _tx.values())}개 · 같은 그루·촬영일 {sum(v != '시험 TASK' for v in _tx.values())}개를 뺐습니다 "
              f"(config/test_tasks.json · {_TEST['note']}) — 빼지 않으려면 USE_TEST_SET = False")
elif not USE_TEST_SET and _TEST and _TEST['ids']:
    print('⚠️ USE_TEST_SET = False — 고정 시험 TASK도 학습에 들어갈 수 있습니다 (그 뒤 시험 성적은 믿을 수 없음)')
# 같은 TASK가 여러 폴더에 있으면: 폴더 순서(EXP_DIR에 적은 순서, 기본은 시간순)상 가장 나중 것만 사용
latest = {}
for r in rows_in:
    if r.get('TASK ID'):
        latest[r['TASK ID']] = r['_zip']          # TASK ID 기준 (상태가 바뀌어 이름이 달라도 같은 TASK)
dups = []
for r in rows_in:
    k = r.get('TASK ID')
    if EXP_DEDUP and r.get('TASK ID') and latest.get(k) != r['_zip']:
        r['비고'] = (r.get('비고', '') + ' · ' if r.get('비고') else '') + f"중복 → 제외 (나중 폴더 {os.path.basename(os.path.dirname(latest[k]))} 것 사용)"
        dups.append(r['_zip'])
if dups:
    zips = [z for z in zips if z not in dups]
    print(f'같은 TASK가 여러 폴더에 있어 이전 것 {len(dups)}개를 뺐습니다 (EXP_DEDUP = False면 모두 사용).')
_KG['no_fetch'] = False
lst = pd.DataFrame(rows).drop(columns=['_zip', '_source', '_media', '_updated'], errors='ignore')
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 50)
display(lst)
n_ok = int(((~lst.get('이미지', pd.Series(dtype=str)).fillna('').str.startswith('❌'))
             & (~lst.get('비고', pd.Series(dtype=str)).fillna('').str.contains('중복|검색어 밖'))).sum()) if len(lst) else 0
print(f'zip {len(zips)}개 사용 · 학습에 쓸 수 있는 것 {n_ok}개 (이미지 없는 것은 자동으로 빠짐)')
_use = [r for r in rows_in if r.get('_zip') in zips and not str(r.get('이미지', '')).startswith('❌')]
if (VAL_RULE or VAL_IDS) and _use:             # train/val 비율 미리 보기 (TASK·프레임)
    _s = pd.DataFrame(_use).assign(역할=lambda d: d['역할'].replace({'규칙 밖': 'train (규칙 밖)'}))
    _s = _s.groupby('역할').agg(TASK=('TASK', 'count'), 프레임=('프레임', 'sum'))
    _s['프레임 비율'] = (100 * _s['프레임'] / max(1, _s['프레임'].sum())).round(1).astype(str) + '%'
    print("■ train / val (val = " + (f"TASK ID {len(VAL_IDS)}개 고정" if VAL_IDS else f"'{VAL_RULE}'") + ")")
    display(_s)
    _out = sorted(r['TASK'] for r in _use if r.get('역할') == '규칙 밖')
    if _out:
        print(f'참고: 이름 규칙 밖 TASK {len(_out)}개는 val 규칙을 볼 수 없어 train으로 씁니다: {_out[:5]}')
    if not any(r.get('역할') == 'val' for r in _use):
        print(f"⚠️ val TASK가 없습니다 → 0번 셀 VAL_RULE '{VAL_RULE}'를 확인하세요 (학습하면 멈춤)")

# ---------- 3) 데이터셋 ----------
if not LIST_ONLY and zips:
    job_start(15, 'train', {'exp_dir': [os.path.basename(d) for d in sel], 'query': EXP_QUERY, 'dedup': EXP_DEDUP,
                            'train_name': TRAIN_NAME, 'dataset_name': DATASET_NAME or TRAIN_NAME, 'convert': CONVERT, 'model': MODEL, 'imgsz': IMGSZ, 'epochs': EPOCHS,
                            'batch': BATCH, 'patience': PATIENCE, 'build_only': BUILD_ONLY,
                            'reviewed_rule': REVIEWED_RULE, 'val_rule': VAL_RULE, 'val_ids': len(VAL_IDS),
                            'plan': (_PLAN.get('name') if isinstance(_PLAN, dict) else _PLAN), 'gpus': _TA.get('device', ''),
                            'train_from': (os.path.relpath(_START, BASE) if TRAIN_FROM else MODEL), 'train_args': _TA})
    for r in rows:                                             # 어떤 TASK가 학습 데이터로 쓰였나 (TASK별 기록)
        if not r.get('TASK ID'):
            continue
        _used = r.get('_zip') in zips and not str(r.get('이미지', '')).startswith('❌')
        ctt.log_task('train_data', r['TASK ID'], r.get('TASK', ''), '사용' if _used else '제외', frames=r.get('프레임', ''),
                     boxes=r.get('박스_또는_track', ''), by_label=r.get('라벨', ''),
                     file=os.path.relpath(r['_zip'], BASE) if r.get('_zip') else '', source=r.get('이미지', ''),
                     summary=r.get('비고', ''))
    tag = os.path.basename(sel[-1]) if len(sel) == 1 else f'{len(sel)}dirs_{os.path.basename(sel[-1])}'
    sig = hashlib.md5(json.dumps([[os.path.basename(z), os.path.getsize(z)] for z in zips] + [CONVERT, IMGSZ, EXP_QUERY]
                                 + ([REVIEWED_RULE, VAL_RULE] if (REVIEWED_RULE or VAL_RULE) else [])
                                 + ([sorted(str(x) for x in VAL_IDS)] if VAL_IDS else []),
                                 sort_keys=True).encode()).hexdigest()[:12]
    def _used_sig(n):
        # 그 이름이 이미 쓰인 데이터 서명 (runs/<n>/DATASET_SIG → datasets/<n>.sig). 없으면 ''
        for p in (f'{BASE}/runs/{n}/DATASET_SIG', f'{BASE}/datasets/{n}.sig'):
            if os.path.exists(p):
                return open(p).read().strip()
        return 'unknown' if os.path.exists(f'{BASE}/runs/{n}/weights') else ''
    if not TRAIN_NAME:                                         # 자동 이름: 다른 데이터로 쓴 이름이면 서명을 붙여 새 이름
        TRAIN_NAME = f'exp_{tag}'
        if _used_sig(TRAIN_NAME) not in ('', sig):
            TRAIN_NAME = f'exp_{tag}_{sig[:6]}'
            print(f'같은 폴더 이름의 이전 학습과 데이터가 달라 새 이름을 씁니다: {TRAIN_NAME}')
    _u = _used_sig(TRAIN_NAME)
    if _u not in ('', sig) and not FORCE_NEW:
        raise SystemExit(f"⛔ TRAIN_NAME '{TRAIN_NAME}'은(는) 다른 데이터(서명 {_u})로 이미 쓴 이름입니다 (지금 데이터 서명 {sig}).\n"
                         f"   덮어쓰면 그 결과가 사라집니다 → TRAIN_NAME을 새 이름으로 바꾸세요 (예: '{TRAIN_NAME}_2').\n"
                         '   정말 지우고 다시 하려면 FORCE_NEW = True')
    RUN_LOCK_DIR = f'{BASE}/runs/{TRAIN_NAME}'
    if FORCE_NEW or not os.path.exists(f'{RUN_LOCK_DIR}/TRAINING_DONE'):
        lock_run(RUN_LOCK_DIR, 'dataset')                      # 데이터셋을 만드는 동안에도 다른 사람이 같은 이름을 못 씀
    if FORCE_NEW and _u not in ('', sig):
        for _p in (RUN_LOCK_DIR + '/weights', RUN_LOCK_DIR + '/results.csv', RUN_LOCK_DIR + '/TRAINING_DONE', RUN_LOCK_DIR + '/DATASET_SIG'):
            shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) else (os.remove(_p) if os.path.exists(_p) else None)
        print(f'FORCE_NEW: runs/{TRAIN_NAME}의 이전 학습을 지웠습니다')
    DS_NAME = DATASET_NAME or TRAIN_NAME                       # v4.24.6: 데이터셋 이름 (학습 이름과 따로 둘 수 있음)
    _other_ds = DS_NAME != TRAIN_NAME
    ds_zip, ds_sig = f'{BASE}/datasets/{DS_NAME}.zip', f'{BASE}/datasets/{DS_NAME}.sig'
    ds_json = f'{BASE}/datasets/{DS_NAME}.json'                # 데이터셋을 만든 설정 (v4.24.6~)
    ds_dir = f'{DS_ROOT}_exp/{DS_NAME}'
    os.makedirs(f'{BASE}/datasets', exist_ok=True)
    if _other_ds:
        drive_pull(f'datasets/{DS_NAME}.sig', f'datasets/{DS_NAME}.json', quiet=True)
    if ON_KAGGLE and not REBUILD and os.path.exists(ds_sig) and open(ds_sig).read().strip() == sig:
        drive_pull(f'datasets/{DS_NAME}.zip')   # Kaggle: 같은 데이터로 만든 데이터셋이 Drive에 있으면 받아서 씀
    saved_ok = os.path.exists(ds_zip) and os.path.exists(ds_sig) and open(ds_sig).read().strip() == sig
    if _other_ds and not saved_ok:                             # 남의 이름의 데이터셋은 새로 만들거나 덮어쓰지 않음
        unlock_run(RUN_LOCK_DIR)
        if not os.path.exists(ds_zip) and not os.path.exists(ds_sig):   # v4.31.5: Kaggle은 서명이 다르면 zip을 받지 않음 → '없음'이 아니라 아래 설정 비교로
            raise SystemExit(f'⛔ DATASET_NAME 데이터셋이 없습니다: datasets/{DS_NAME}.zip → 이름을 확인하거나 DATASET_NAME = \'\'로 새로 만드세요')
        _was = json.load(open(ds_json, encoding='utf-8')) if os.path.exists(ds_json) else None
        _now = {'EXP_DIR': [os.path.basename(d) for d in sel], 'EXP_QUERY': EXP_QUERY, 'EXP_DEDUP': EXP_DEDUP, 'EXP_IDS': len(EXP_IDS),
                'CONVERT': CONVERT, 'IMGSZ': IMGSZ, 'REVIEWED_RULE': REVIEWED_RULE, 'VAL_RULE': VAL_RULE, 'zip 수': len(zips)}
        print('데이터셋을 만든 설정과 지금 설정:')
        for _k in _now:
            _w = (_was or {}).get(_k, '(기록 없음)')
            print(f"  {'  ' if _w == _now[_k] or _w == '(기록 없음)' else '≠ '}{_k}: 만들 때 {_w!r} · 지금 {_now[_k]!r}")
        raise SystemExit(f'⛔ datasets/{DS_NAME}.zip은 지금 설정과 다른 데이터로 만든 것입니다 (서명 {open(ds_sig).read().strip() if os.path.exists(ds_sig) else "없음"} ≠ 지금 {sig}).\n'
                         '   EXP_DIR·EXP_QUERY·EXP_DEDUP·EXP_IDS·CONVERT와 0번 셀 IMGSZ·REVIEWED_RULE·VAL_RULE을 만들 때와 같게 하세요'
                         + ('' if _was else ' (이 데이터셋은 v4.24.6 이전에 만들어 설정 기록이 없음 → 그때 15번 셀 설정 그대로)') + '.')
    if _other_ds and REBUILD:
        unlock_run(RUN_LOCK_DIR)
        raise SystemExit(f'⛔ REBUILD = True와 DATASET_NAME을 함께 쓸 수 없습니다 (다른 학습의 데이터셋 datasets/{DS_NAME}.zip을 덮어씀)')
    if saved_ok and not REBUILD:
        if not os.path.exists(f'{ds_dir}/data.yaml'):
            shutil.rmtree(ds_dir, ignore_errors=True)
            zipfile.ZipFile(ds_zip).extractall(ds_dir)
        print(f'저장된 데이터셋 사용: datasets/{DS_NAME}.zip (zip 목록·설정 같음)' + (f' → 학습 이름 {TRAIN_NAME}' if _other_ds else ''))
    else:
        in_dir = f'{LOCAL_TMP}/exp_in/{DS_NAME}'                  # 여러 폴더의 zip을 한곳에 (복사, 라벨이라 작음)
        shutil.rmtree(in_dir, ignore_errors=True); os.makedirs(in_dir)
        for z in zips:
            dst = f'{in_dir}/{os.path.basename(z)}'
            if os.path.exists(dst):
                dst = f'{in_dir}/{os.path.basename(os.path.dirname(z))}__{os.path.basename(z)}'
            shutil.copy2(z, dst)
        shutil.rmtree(ds_dir, ignore_errors=True)
        if ON_KAGGLE:                                  # v4.24.7: 이번 데이터셋에 쓰는 TASK 중 영상에서 프레임을 꺼낼 것만 Drive에서 받음
            _need = [r for r in _use if r.get('_media')]
            if _need:
                print(f'영상에서 프레임을 꺼낼 TASK {len(_need)}개의 영상을 Drive에서 받습니다 (이미 받은 것은 건너뜀)')
            for r in _need:
                kaggle_fetch_media(r.get('_source', ''), r.get('TASK', ''))
        _n_uld = 0                                     # v4.31.5: ULD 사진 TASK는 라벨만 받은 export라 사진을 clip에서 다시 뽑아 raw_images/에
        try:
            uld_pull('registry/tasks/*.json')
            for r in _use:
                _t = str(r.get('TASK ID') or '')
                if _t and uld_is_images(uld_task(_t, pull=False)):
                    _n_uld += uld_images(_t, f'{BASE}/raw_images') or 0
            if _n_uld:
                print(f'ULD 사진 TASK의 사진 {_n_uld}장을 clip에서 뽑았습니다 → raw_images/')
        except Exception as _e:
            print(f'⚠️ ULD 사진을 준비하지 못함 (그 TASK는 이미지가 없으면 빠짐): {_e}')
        conv = load_script('cvat_to_yolo.py')
        conv.INPUT_DIR, conv.VIDEOS_DIR, conv.OUT_DIR = Path(in_dir), Path(VIDEOS), Path(ds_dir)
        conv.BACKUPS_DIR = Path(f'{BASE}/backups')     # 라벨만 받은 zip: videos/에 없으면 TASK 백업의 원본 영상에서 프레임 추출
        conv.DECODER = 'auto'                          # PyAV(CVAT와 같은 디코딩)로 프레임 번호를 맞춤
        conv.IMAGES_DIR, conv.CVAT_FRAMES_DIR = Path(f'{BASE}/raw_images'), Path(f'{BASE}/cvat_frames')
        conv.TRAIN_IMGSZ = IMGSZ
        _lost = []
        conv.PROGRESS_HOOK = lambda: ctt.touch_run_lock(RUN_LOCK_DIR, SESSION_ID, stage='dataset') or _lost.append(1)
        for k, v in CONVERT.items():
            setattr(conv, k, v)
        if VAL_RULE or VAL_IDS:                        # v4.13: VAL_RULE에 맞는 TASK(의 영상 전체)를 val로 · v4.30: VAL_IDS
            conv.VAL_TASKS = {str(r['TASK ID']): r.get('역할') == 'val' for r in _use if r.get('TASK ID')}
            if not any(conv.VAL_TASKS.values()):
                unlock_run(RUN_LOCK_DIR)
                raise SystemExit(f"⛔ VAL_RULE '{VAL_RULE}'에 맞는 TASK가 없습니다 → 0번 셀 VAL_RULE을 고치거나 '' (무작위)")
        else:
            conv.VAL_TASKS = None
        conv.main()
        if _lost:
            raise SystemExit('⛔ 데이터셋을 만드는 동안 다른 세션이 이 이름의 잠금을 가져갔습니다 → 저장하지 않고 멈춥니다')
        print('데이터셋 저장 중 → Drive datasets/ ...')
        base_noext = ds_zip[:-4]
        shutil.make_archive(base_noext, 'zip', ds_dir)
        open(ds_sig, 'w').write(sig)
        json.dump({'EXP_DIR': [os.path.basename(d) for d in sel], 'EXP_QUERY': EXP_QUERY, 'EXP_DEDUP': EXP_DEDUP, 'EXP_IDS': len(EXP_IDS),
                   'CONVERT': CONVERT, 'IMGSZ': IMGSZ, 'REVIEWED_RULE': REVIEWED_RULE, 'VAL_RULE': VAL_RULE, 'zip 수': len(zips),
                   'VAL_IDS': sorted(str(x) for x in VAL_IDS), 'sig': sig, 'made_by': TRAIN_NAME, 'time': ctt.now(), 'app_version': APP_VERSION},
                  open(ds_json, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)   # v4.24.6: 이 데이터셋을 만든 설정
        if os.path.exists(f'{ds_dir}/stats_report.html'):
            shutil.copy2(f'{ds_dir}/stats_report.html', f'{BASE}/datasets/{DS_NAME}_report.html')
        print(f'저장: datasets/{DS_NAME}.zip · 통계 리포트 datasets/{DS_NAME}_report.html · 설정 datasets/{DS_NAME}.json')
    yaml_path = f'{ds_dir}/data.yaml'
    txt = open(yaml_path, encoding='utf-8').read()
    open(yaml_path, 'w', encoding='utf-8').write(re.sub(r'^path:.*$', f'path: {ds_dir}', txt, flags=re.M))
    DS = ds_dir
    n_tr, n_va = len(os.listdir(f'{DS}/train/images')), len(os.listdir(f'{DS}/val/images'))
    print(f'데이터셋: train {n_tr}장 / val {n_va}장')
    ctt.log_task('dataset', '', TRAIN_NAME, '준비', frames=n_tr + n_va, summary=f'train {n_tr} / val {n_va} · 서명 {sig}'
                 + (f' · 데이터셋 {DS_NAME}' if _other_ds else ''),
                 file=os.path.relpath(ds_zip, BASE))
    _split_csv = f'{ds_dir}/split_tasks.csv'          # v4.13: TASK 학습 대장 (logs/train_tasks.csv)
    if os.path.exists(_split_csv):
        _info = {str(r['TASK ID']): {'state': (ctt.parse_task_name(r.get('TASK', '')) or {'state': '규칙 밖'}).get('state', ''),
                                     'label_updated': r.get('_updated', ''), 'export_zip': os.path.relpath(r['_zip'], BASE)}
                 for r in rows if r.get('TASK ID') and r.get('_zip') in zips}
        _n = ctt.append_train_tasks(BASE, TRAIN_NAME, sig, _split_csv, info=_info, who=who_am_i(), session=SESSION_ID,
                                    app_version=APP_VERSION)
        _sp = pd.read_csv(_split_csv)
        if 'frames_from' in _sp:                    # v4.24.4: 학습 이미지 출처·디코더 (TASK 학습 대장 frames_from 열)
            print('학습 이미지 출처: ' + ', '.join(f'{k} {n}개 TASK' for k, n in _sp['frames_from'].fillna('?').value_counts().items()))
        print(f"TASK 학습 대장: {'새로 ' + str(_n) + '줄' if _n else '이미 기록됨'} · TASK train {int((_sp['split'] == 'train').sum())}개 / "
              f"val {int((_sp['split'] == 'val').sum())}개 (logs/train_tasks.csv, 22번 셀에서 보기)")
    else:
        print('참고: 이 데이터셋은 v4.13 이전에 만들어 TASK별 train/val 기록(split_tasks.csv)이 없습니다 → REBUILD = True로 다시 만들면 기록됨')
    if n_tr == 0 or n_va == 0:
        unlock_run(RUN_LOCK_DIR)
    assert n_tr > 0, '학습 이미지가 0장입니다 — 위 표의 이미지 열(❌ 없음)을 확인하세요'
    assert n_va > 0, 'val 이미지가 0장입니다 — 이미지가 있는 TASK가 너무 적습니다 (VAL_RATIO 또는 zip 확인)'

    # 이후 1·2·3번 셀도 이 학습을 기준으로
    NAME = TRAIN_NAME
    RUN_DIR = f'{BASE}/runs/{NAME}'
    BEST, LAST, DONE = f'{RUN_DIR}/weights/best.pt', f'{RUN_DIR}/weights/last.pt', f'{RUN_DIR}/TRAINING_DONE'
    print(f'학습 이름: {NAME} · 상태: {train_status()}')

    # ---------- 4) 학습 ----------
    if not BUILD_ONLY:
        import torch
        from ultralytics import YOLO
        if not torch.cuda.is_available() and not ALLOW_CPU:
            unlock_run(RUN_LOCK_DIR)
            raise SystemExit('⛔ GPU가 없습니다. 메뉴 런타임 → 런타임 유형 변경 → T4 GPU → 0번 셀 → 15번 셀을 다시 실행하세요.\n'
                             '   (데이터셋은 Drive에 저장돼 있어 다시 만들지 않습니다. 정말 CPU로 하려면 ALLOW_CPU = True)')
        print(f"학습 장치: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (느림)'}")
        if FORCE_NEW and os.path.exists(LAST) and _u in ('', sig):
            for _p in (RUN_DIR + '/weights', RUN_DIR + '/results.csv', RUN_DIR + '/TRAINING_DONE'):
                shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) else (os.remove(_p) if os.path.exists(_p) else None)
            print(f'이전 학습 삭제: runs/{NAME}')
        if os.path.exists(DONE):
            print('이미 학습 완료 → 건너뜁니다. (다시 하려면 TRAIN_NAME을 바꾸거나 FORCE_NEW = True)')
        elif run_training(NAME, yaml_path, data_sig=sig, start=_START, train_args=_TA):
            print('학습 완료:', BEST, '→ 2번 셀(결과), 3번 셀(검증 1단계)')
    unlock_run(RUN_LOCK_DIR)                                   # 데이터셋만 만들었거나 건너뛴 경우에도 잠금을 풂
    _res = f'{BASE}/runs/{TRAIN_NAME}/results.csv'
    _best = ''
    try:
        _df = pd.read_csv(_res); _df.columns = [c.strip() for c in _df.columns]
        _k = 'metrics/mAP50-95(B)'
        if _k in _df:
            _i = _df[_k].idxmax()
            _best = f"에폭 {len(_df)}개 · 최고 mAP50-95 {_df[_k].max():.3f} (에폭 {int(_df.loc[_i, 'epoch'])}) · mAP50 {_df.loc[_i, 'metrics/mAP50(B)']:.3f}"
    except Exception:
        pass
    job_end('끝' if os.path.exists(DONE) or BUILD_ONLY else '중단?', f"runs/{TRAIN_NAME} · " + (_best or ('데이터셋만' if BUILD_ONLY else '결과 없음')),
            file=f'runs/{TRAIN_NAME}')


## TASK 백업 복원 — Drive의 백업 zip으로 CVAT에 TASK 만들기
CVAT 웹의 'Create from backup'은 PC 파일이나 클라우드 스토리지(AWS S3·Azure·Google Cloud Storage)만 받고 **Google Drive는 고를 수 없습니다.**
이 셀은 Colab이 Drive의 백업 zip을 읽어 대상 CVAT 서버에 바로 올립니다(PC로 내려받을 필요 없음).
- 13번 셀로 만든 **TASK 백업**(영상 + 라벨)을 씁니다. 'project from backup'이 아니라 TASK로 복원됩니다.
- 시험용 사본이면 이름 끝에 `_test`를 붙여 원본과 구분하세요(`RS_SUFFIX`). 기본 이름은 그대로라 영상 찾기에 문제가 없습니다.
- 무료 플랜 계정이면 TASK 3개·저장 1GB 한도가 있습니다. 백업에는 영상이 들어 있어 큽니다.


In [ ]:
# 16. TASK 백업 복원 — Drive backups/의 zip → 대상 CVAT 서버에 새 TASK
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN         = None       # 복원할 서버: None = 지금 접속 중인 서버 · 또는 0번 셀 CVAT_PROFILES의 이름
RS_QUERY      = ''         # 백업한 TASK 이름 검색. 예: '20261013 row=2 plant=3 vf_r2'
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
RS_FILES      = []         # 또는 백업 파일 경로를 직접 (BASE 기준), 예: ['backups/eggplant_G3_20261013_2_3/original/....zip']
RS_LATEST_ONLY = True      # TASK마다 가장 최근 백업만
RS_SUFFIX     = '_test'    # 복원한 TASK 이름 끝에 붙일 글자 ('' = 원래 이름 그대로)
RS_PROJECT    = None       # 넣을 프로젝트 (ID 숫자 또는 이름). 라벨 이름이 같아야 함. None = 0번 셀 기본 프로젝트(CVAT_PROJECT) · '' = 프로젝트 없이
                           # 조직은 0번 셀 CVAT_ORG (그 조직에 만듦)
RS_PREVIEW    = True       # True = 복원하지 않고 대상만 보여 줌 → 확인 후 False

import pandas as pd
from IPython.display import display
idx_csv = f'{BASE}/backup_index.csv'
rows = []
if RS_FILES:
    rows = [{'task_name': os.path.basename(f), 'file': f, 'kind': '', 'run_time': '', 'size_mb': '', 'server': ''} for f in RS_FILES]
else:
    allb = [r for r in ctt.read_registry(idx_csv) if r.get('result') == '성공']    # v4.31.0: 세션 파일 합침 (read_registry = 세션별 기록 읽기)
    assert allb or os.path.exists(idx_csv), 'backup_index.csv가 없습니다 (13번 셀로 먼저 백업)'
    rows = [r for r in allb if ctt.match_name(r['task_name'], RS_QUERY)[0]] if RS_QUERY else allb
    if RS_LATEST_ONLY:
        last = {}
        for r in rows:
            last[(r.get('server', ''), r['task_id'])] = r
        rows = list(last.values())
if ON_KAGGLE and (RS_QUERY or RS_FILES):
    drive_pull(*[r['file'] for r in rows])          # Kaggle: 고른 백업 zip만 Drive에서 받음
elif ON_KAGGLE:
    print('Kaggle: RS_QUERY 또는 RS_FILES를 정하면 그 백업만 Drive에서 받습니다 (지금은 목록만)')
rows = [dict(r, exists=os.path.exists(f"{BASE}/{r['file']}")) for r in rows]
for r in rows:
    k, v = ctt.backup_content(f"{BASE}/{r['file']}") if r['exists'] else ('?', '')
    r['내용'] = f'영상 TASK ({v})' if k == 'video' else (f'이미지 TASK ({v}장)' if k == 'images' else '알 수 없음')
tbl = pd.DataFrame(rows)
print(f"백업 {len(rows)}개" + (f" (검색어 '{RS_QUERY}')" if RS_QUERY else ''))
if len(tbl):
    display(tbl[[c for c in ['run_time', 'server', 'task_id', 'task_name', 'kind', '내용', 'size_mb', 'file', 'exists'] if c in tbl]])
    if (tbl['내용'].str.startswith('이미지')).any():
        print('참고: 이미지 TASK 백업은 복원해도 이미지 TASK입니다 → 5번 셀(track) 대신 9~10번 셀(프레임별 자동 라벨링). track은 17번 셀로 영상 TASK를 만들어서.')
    mb = sum(float(r.get('size_mb') or 0) for r in rows)
    print(f'합계 약 {mb:.0f} MB — 무료 플랜 저장 한도 1GB에 주의')
missing = [r['file'] for r in rows if not r['exists']]
if missing:
    print('⚠️ Drive에 없는 파일:', missing[:5])

use_cvat(LOGIN)
pid = None
if RS_PROJECT is None and (CVAT_SCOPE.get('project_id') or 0) > 0:     # 기본 프로젝트 (v4.19)
    pid = CVAT_SCOPE['project_id']
    print(f"넣을 프로젝트: {CVAT_SCOPE.get('project_name')} (ID {pid}) — 0번 셀 기본 프로젝트")
elif RS_PROJECT:
    projs = list(CLIENT.projects.list())
    hit = [p for p in projs if str(p.id) == str(RS_PROJECT) or p.name == str(RS_PROJECT)]
    assert hit, f"프로젝트 '{RS_PROJECT}'를 찾지 못했습니다. 있는 프로젝트: {[(p.id, p.name) for p in projs][:10]}"
    pid = hit[0].id
    print(f'넣을 프로젝트: {hit[0].name} (ID {pid})')
print(f'지금 이 서버의 TASK 수: {len(list(CLIENT.tasks.list()))}개 (무료 플랜은 최대 3개)')

if RS_PREVIEW:
    print('\n대상이 맞으면 RS_PREVIEW = False로 바꿔 다시 실행하세요.')
else:
    job_start(16, 'restore', {'query': RS_QUERY, 'files': RS_FILES, 'suffix': RS_SUFFIX, 'project': pid})   # 작업 기록 (v4.21)
    _rs_n = {'완료': 0, '오류': 0}
    try:
      for r in rows:
        if not r['exists']:
            continue
        path = f"{BASE}/{r['file']}"
        row = dict(run_time=ctt.now(), action='restore', task_name=r['task_name'], server=CVAT_SERVER,
                   backup_file=r['file'])
        try:
            print(f"⏫ {os.path.basename(path)} 올리는 중 …")
            task = CLIENT.tasks.create_from_backup(path)
            changes = {}
            if RS_SUFFIX and not task.name.endswith(RS_SUFFIX):
                changes['name'] = task.name + RS_SUFFIX
            if pid is not None:
                changes['project_id'] = pid
            if changes:
                task = task.update(changes)
            row.update(task_id=task.id, task_name=task.name, status='완료',
                       note=f"원본 TASK {r.get('task_id', '')} → 새 TASK {task.id}" + (f' · 프로젝트 {pid}' if pid else ''))
            print(f"✅ 새 TASK {task.id} '{task.name}' — {CVAT_HOST}/tasks/{task.id}")
        except Exception as e:
            row.update(status='오류', note=ctt.api_error(e))
            print(f"❌ {os.path.basename(path)}: {ctt.api_error(e)}")
        finally:
            ctt.append_registry(REGISTRY, **row)       # 작업 대장 + 작업 기록(TASK 줄)
            _rs_n[row.get('status', '오류')] = _rs_n.get(row.get('status', '오류'), 0) + 1
    finally:
        job_end('끝' if not _rs_n.get('오류') else '일부 실패', f"새 TASK {_rs_n['완료']}개" + (f" · 오류 {_rs_n['오류']}개" if _rs_n.get('오류') else ''))
    print('\n다음: 4번 셀 TASK_QUERY로 새 TASK ID 확인 → 5번 셀(track) · 9~10번 셀(프레임별 자동 라벨링)')


## 영상 TASK 만들기 — Drive의 mp4로 CVAT에 새 TASK
Drive `videos/`(하위 폴더 포함)의 mp4를 골라 대상 계정(LOGIN)에 **영상 TASK**를 만듭니다. 만든 TASK는 5번 셀(track)에 바로 쓸 수 있습니다.
- 이름: 영상 파일 이름(기본 이름) + `_uld` + `VC_SUFFIX` (예: `eggplant_G6_20261013_2_4_uld_test`)
- 라벨: `VC_PROJECT`를 주면 그 프로젝트의 라벨, 아니면 `VC_LABELS`(기본 4개)
- `VC_FRAME_STEP`: TASK에 넣을 프레임 간격(10 = 10프레임마다 1장). 원래 TASK와 같게 하면 비교가 쉽습니다.
- 무료 플랜 계정이면 TASK 3개·저장 1GB 한도에 주의하세요(4K 영상은 큼).


In [ ]:
# 17. 영상 TASK 만들기 — Drive videos/의 mp4 → 대상 계정에 새 영상 TASK
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN         = None       # 만들 계정: None = 지금 접속 중인 계정 · 또는 0번 셀 CVAT_PROFILES의 이름 (예: 'B')
VC_QUERY      = ''         # videos/의 영상 파일 이름 검색 (TASK 이름과 같은 규칙). 예: 'G6 20261013 row=2 plant=4'
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
VC_FILES      = []         # 또는 경로 직접 (videos/ 기준), 예: ['261013/eggplant_G6_20261013_2_4.mp4']
VC_STATE      = 'uld'      # 이름에 붙일 상태
VC_SUFFIX     = '_test'    # 이름 끝에 붙일 글자 ('' = 없음)
VC_FRAME_STEP = 10         # 프레임 간격
VC_PROJECT    = None       # 넣을 프로젝트 (ID 또는 이름). None = 0번 셀 기본 프로젝트(CVAT_PROJECT) · '' = 프로젝트 없이 VC_LABELS로 라벨을 만듦
                           # 조직은 0번 셀 CVAT_ORG (그 조직에 만듦)
VC_LABELS     = ['healthy_fruit', 'disease_fruit', 'healthy_bud', 'disease_bud']
VC_IMAGE_QUALITY = 70      # CVAT 화면용 압축 품질 (원본 해상도는 그대로)
VC_PREVIEW    = True       # True = 만들지 않고 대상만 보여 줌 → 확인 후 False

import pandas as pd
from IPython.display import display
from cvat_sdk.api_client import models as cvat_models
exts = ('.mp4', '.mov', '.avi', '.mkv')
if ON_KAGGLE and (VC_QUERY or VC_FILES):
    drive_get_videos(VC_QUERY, VC_FILES)         # Kaggle: 고른 영상만 Drive에서 받음
if VC_FILES:
    vids = [Path(VIDEOS) / f for f in VC_FILES]
else:
    allv = sorted(p for p in Path(VIDEOS).rglob('*') if p.suffix.lower() in exts and '_from_backup' not in p.parts)
    vids = [p for p in allv if ctt.match_name(p.stem, VC_QUERY)[0]] if VC_QUERY else []    # 파일 이름을 TASK 이름처럼 읽어 같은 규칙으로
    if VC_QUERY:
        print(f"검색어 '{VC_QUERY}' → 영상 {len(vids)}/{len(allv)}개")
    if not VC_QUERY:
        print('VC_QUERY 또는 VC_FILES를 정하세요. videos/의 영상(최근 20개):')
        display(pd.DataFrame([{'파일': str(p.relative_to(VIDEOS)), 'MB': round(p.stat().st_size / 2**20, 1)} for p in allv[-20:]]))
rows = []
for p in vids:
    name = p.stem + (f'_{VC_STATE}' if VC_STATE else '') + VC_SUFFIX
    info = {}
    try:
        import cv2
        cap = cv2.VideoCapture(str(p))
        info = dict(프레임=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), fps=round(cap.get(cv2.CAP_PROP_FPS), 1),
                    크기=f'{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}')
        cap.release()
    except Exception:
        pass
    rows.append(dict(파일=str(p.relative_to(VIDEOS)) if str(p).startswith(VIDEOS) else str(p), 존재=p.exists(),
                     MB=round(p.stat().st_size / 2**20, 1) if p.exists() else '', **info,
                     TASK_프레임=(info['프레임'] + VC_FRAME_STEP - 1) // VC_FRAME_STEP if info.get('프레임') else '',
                     새_TASK_이름=name, 단어별=ctt.format_detail(ctt.match_name(p.stem, VC_QUERY)[1]) if VC_QUERY else ''))
if rows:
    display(pd.DataFrame(rows))
    print(f"합계 {sum(r['MB'] or 0 for r in rows):.0f} MB")

if vids:
    use_cvat(LOGIN)
    pid = None
    if VC_PROJECT is None and (CVAT_SCOPE.get('project_id') or 0) > 0:     # 기본 프로젝트 (v4.19)
        pid = CVAT_SCOPE['project_id']
        print(f"프로젝트: {CVAT_SCOPE.get('project_name')} (ID {pid}) — 0번 셀 기본 프로젝트, 라벨은 프로젝트 것을 씀")
    elif VC_PROJECT:
        projs = list(CLIENT.projects.list())
        hit = [p for p in projs if str(p.id) == str(VC_PROJECT) or p.name == str(VC_PROJECT)]
        assert hit, f"프로젝트 '{VC_PROJECT}'를 찾지 못했습니다. 있는 프로젝트: {[(p.id, p.name) for p in projs][:10]}"
        pid = hit[0].id
        print(f'프로젝트: {hit[0].name} (ID {pid}) — 라벨은 프로젝트 것을 씀')
    existing = {t.name for t in CLIENT.tasks.list()}
    print(f'지금 이 계정의 TASK 수: {len(existing)}개 (무료 플랜은 최대 3개)')
    dup = [r['새_TASK_이름'] for r in rows if r['새_TASK_이름'] in existing]
    if dup:
        print('⚠️ 같은 이름의 TASK가 이미 있습니다:', dup, '→ VC_SUFFIX를 바꾸세요 (그대로 만들면 같은 이름이 둘)')
    if VC_PREVIEW:
        print('\n대상이 맞으면 VC_PREVIEW = False로 바꿔 다시 실행하세요.')
    else:
        job_start(17, 'create', {'query': VC_QUERY, 'files': VC_FILES, 'step': VC_FRAME_STEP, 'project': pid})   # 작업 기록 (v4.21)
        _vc_n = {'완료': 0, '오류': 0}
        try:
          for p, r in zip(vids, rows):
            if not p.exists():
                print('❌ 파일 없음:', p); continue
            row = dict(run_time=ctt.now(), action='create', task_name=r['새_TASK_이름'], server=CVAT_SERVER,
                       video=p.name, step=VC_FRAME_STEP)
            try:
                spec = cvat_models.TaskWriteRequest(name=r['새_TASK_이름'], project_id=pid) if pid else \
                       cvat_models.TaskWriteRequest(name=r['새_TASK_이름'],
                                                    labels=[cvat_models.PatchedLabelRequest(name=n, type='rectangle') for n in VC_LABELS])
                print(f"⏫ {p.name} ({r['MB']} MB) 올리는 중 … 서버가 영상을 처리할 때까지 기다립니다")
                task = CLIENT.tasks.create_from_data(spec=spec, resources=[str(p)],
                                                     data_params={'frame_step': VC_FRAME_STEP, 'image_quality': VC_IMAGE_QUALITY,
                                                                  'use_cache': True})
                row.update(task_id=task.id, media=getattr(task, 'media_type', ''), frames=task.size, status='완료',
                           note=f"영상 TASK · step {VC_FRAME_STEP}" + (f' · 프로젝트 {pid}' if pid else ''))
                print(f"✅ 새 TASK {task.id} '{task.name}' · {getattr(task, 'media_type', '')} · 프레임 {task.size} — {CVAT_HOST}/tasks/{task.id}")
            except Exception as e:
                row.update(status='오류', note=ctt.api_error(e))
                print(f'❌ {p.name}: {ctt.api_error(e)}')
            finally:
                ctt.append_registry(REGISTRY, **row)       # 작업 대장 + 작업 기록(TASK 줄)
                _vc_n[row.get('status', '오류')] = _vc_n.get(row.get('status', '오류'), 0) + 1
        finally:
            job_end('끝' if not _vc_n.get('오류') else '일부 실패', f"새 영상 TASK {_vc_n['완료']}개" + (f" · 오류 {_vc_n['오류']}개" if _vc_n.get('오류') else ''))
        print('\n다음: 5번 셀 TRACK_IDS = [새 TASK ID], LOGIN을 같은 계정으로 → track 생성')


## 라벨 확인 — TASK의 현재 라벨을 라벨별·프레임별로 요약하고 그림으로
CVAT에 있는 지금 라벨(사람이 그린 것, 9~10번 셀 프레임별 자동 라벨링 결과, 업로드한 track, 검토 중인 것)을 받아
**라벨별 track·박스 수, 가림 비율, 평균 크기, 프레임당 박스 수**와 박스를 그린 그림을 보여 줍니다. 라벨은 바꾸지 않습니다.
- 5번 셀에서 만든 track을 **올리기 전에** 확인하려면 6번 셀의 `UPLOAD_PREVIEW = True`(기본)를 쓰세요.


In [ ]:
# 18. 라벨 확인 — CVAT TASK의 현재 라벨 요약 (읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN       = None      # None = 지금 접속 중인 계정 · 또는 0번 셀 CVAT_PROFILES의 이름
CHECK_IDS   = []        # 확인할 TASK ID, 예: [2704421]
SHOW_IMAGES = 6         # 그림 수 (0 = 그림 없이 표만)

import pandas as pd
from IPython.display import display
use_cvat(LOGIN)
for tid in ctt.as_task_ids(CHECK_IDS):
    print('=' * 70)
    try:
        task = CLIENT.tasks.retrieve(tid)
        info = ctt.task_basic(task)
        print(f"TASK {tid} '{task.name}' · {info['media']} · 프레임 {info['frames']} · 상태 {getattr(task, 'status', '')}")
        tmp = f'{task_dir(tid)}/check_{ctt.stamp()}.zip'
        ctt.export_annotations(task, tmp)                    # 현재 라벨 (이 파일이 그 시점 기록으로 남음)
        ann = ctt.load_annotations(tmp)
        label_rows, fsum, counts = ctt.annotation_tables(ann)
        print('  ' + ' · '.join(f'{k} {v}' for k, v in fsum.items()))
        display(pd.DataFrame(label_rows) if label_rows else '라벨 없음')
        if counts and any(counts):
            print('  프레임별 박스 수:', ' '.join(str(c) for c in counts[:60]) + (' …' if len(counts) > 60 else ''))
        if SHOW_IMAGES and label_rows:
            ctt.draw_annotations(ann, ctt.cvat_frame_getter(task, ann['meta']), n=SHOW_IMAGES,
                                 title=f"TASK {tid} current labels (yellow=healthy_fruit, red=disease_fruit, cyan=healthy_bud, magenta=disease_bud, white inner line=occluded, *=keyframe/direct)")
        print(f'  기록: {os.path.relpath(tmp, BASE)}')
    except Exception as e:
        print(f'❌ TASK {tid}: {ctt.api_error(e)}')
if not CHECK_IDS:
    print('CHECK_IDS에 TASK ID를 넣으세요.')


## 자동 라벨링 현황 · 자동 track ↔ CVAT 반영 결과 비교
`LOGIN = 'B'`로 그 계정에서 track 생성·업로드·검수 수집을 몇 개 했는지 표로 보고, `CMP_IDS`의 TASK는 올린 track XML과 지금 CVAT 라벨을 박스 단위로 비교합니다 (같음·가림 다름·수정·라벨 바뀜·없어짐·추가). 읽기만 합니다.

In [ ]:
# 19. 자동 라벨링 현황 · 자동 track ↔ CVAT 반영 결과 비교 (읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN    = None     # None = 지금 접속 중인 계정 · 또는 0번 셀 CVAT_PROFILES의 이름 (예: 'B')
CMP_IDS  = []       # 비교할 TASK ID (비우면 현황표만), 예: [2704421]
SAME_IOU = 0.95     # 이만큼 겹치면 '같음', 0.5~0.95는 '수정'(위치·크기 고침)
SHOW_FRAMES = 20    # 차이가 있는 프레임을 몇 줄까지 보여 줄지

import pandas as pd
from IPython.display import display
use_cvat(LOGIN)

# (1) 이 계정(서버)의 자동 라벨링 현황 — 작업 대장 기준
rows = [r for r in ctt.read_registry(REGISTRY) if ctt.same_server(r, CVAT_SERVER)]
by = {}
for r in rows:
    d = by.setdefault(r['task_id'], {'task_id': r['task_id'], 'task_name': '', 'track 생성': 0, '마지막 track': '',
                                     'track 수': '', '정밀도': '', '업로드': '', '업로드 시각': '', '검수 수집': ''})
    d['task_name'] = r.get('task_name') or d['task_name']
    if r['action'] == 'track 생성' and r['status'].startswith('완료'):
        d['track 생성'] += 1
        d.update({'마지막 track': r['run_time'], 'track 수': r.get('tracks', ''), '정밀도': r.get('precision', '')})
    elif r['action'] == '업로드':
        d.update({'업로드': r['status'], '업로드 시각': r['run_time'] if r['status'] == '업로드 완료' else d['업로드 시각']})
    elif r['action'] == '검수 수집':
        d['검수 수집'] = r['status']
st = pd.DataFrame(by.values())
print(f'▶ 자동 라벨링 현황 · {CVAT_SERVER} · TASK {len(st)}개')
if len(st):
    print(f"  track 생성 {int((st['track 생성'] > 0).sum())}개 · 업로드 완료 {int((st['업로드 시각'] != '').sum())}개 · "
          f"검수 수집 {int((st['검수 수집'] != '').sum())}개")
    display(st)
else:
    print('  이 계정으로 기록된 작업이 없습니다 (5번 셀 track 생성부터).')

# (2) TASK별 비교 — 올린 track XML(로컬) vs 지금 CVAT에 있는 라벨
for tid in ctt.as_task_ids(CMP_IDS):
    print('=' * 70)
    try:
        up = ctt.last_row(REGISTRY, tid, 'upload', server=CVAT_SERVER)
        up = up if up and up['status'] == '업로드 완료' else None
        src = up or ctt.last_row(REGISTRY, tid, 'track', server=CVAT_SERVER)
        if not src or not src.get('result_file'):
            print(f'TASK {tid}: track 기록 없음 → 5번 셀 먼저'); continue
        task = CLIENT.tasks.retrieve(tid)
        snap = f'{task_dir(tid)}/compare_{ctt.stamp()}.zip'
        ctt.export_annotations(task, snap)                       # 지금 CVAT 라벨 (기록으로 남음)
        local, cvat = ctt.load_annotations(f"{BASE}/{src['result_file']}"), ctt.load_annotations(snap)
        rows_l, total, frows = ctt.compare_annotations(local, cvat, same_iou=SAME_IOU)
        print(f"TASK {tid} '{task.name}' · CVAT 마지막 수정 {str(task.updated_date)[:19]} UTC")
        print(f"  로컬: {src['result_file']} ({'업로드한 파일 ' + up['run_time'] if up else '아직 업로드 안 함'})")
        print('  ' + ' · '.join(f'{k} {v}' for k, v in total.items()))
        display(pd.DataFrame(rows_l))
        if frows:
            print(f'  차이가 있는 프레임 {len(frows)}개 (앞 {SHOW_FRAMES}개):')
            display(pd.DataFrame(frows[:SHOW_FRAMES]).fillna(0).astype(int))
        if not up:
            v = '→ 아직 올리지 않았으므로 CVAT 라벨과 다른 것이 정상 (6번 셀로 업로드)'
        elif total['로컬 박스'] and total['같음'] == total['로컬 박스'] == total['CVAT 박스']:
            v = '→ 올린 그대로 CVAT에 있음 (검수로 바뀐 것 없음)'
        elif total['CVAT 박스'] == 0:
            v = '→ CVAT에 라벨이 없음: 업로드 후 지워졌거나 다른 TASK에 올림 — 6번 셀 기록·TASK ID 확인'
        else:
            v = '→ 업로드 후 CVAT에서 바뀜 (검수 수정이면 정상 — 7번 셀로 수집해 검증 3단계)'
        print('  ' + v)
        print(f'  기록: {os.path.relpath(snap, BASE)}')
    except Exception as e:
        print(f'❌ TASK {tid}: {ctt.api_error(e)}')


## 단계별 원인 찾기 — 자동 라벨 박스가 안 맞을 때
사람이 라벨을 끝낸 영상 TASK 하나로 ① CVAT 프레임 ↔ mp4 프레임 번호가 맞는지(OpenCV·PyAV) ② 모델이 CVAT 프레임에서 열매를 찾는지 ③ mp4 프레임에서는 어떤지 ④ 5번 셀 track XML이 정답과 얼마나 맞는지를 차례로 보고, 어디가 문제인지 판정합니다. 흰 박스 = 정답, 노란 박스 = 모델.

In [ ]:
# 20. 단계별 원인 찾기 — 손으로 라벨한 TASK 하나로 ①프레임 맞춤 ②모델 ③영상 프레임 ④track XML 점검 (읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN      = None    # 그 TASK가 있는 계정 · 0번 셀 CVAT_PROFILES의 이름
DG_TASK_ID = 0       # ← 사람이 라벨을 끝낸 영상 TASK (예: vf_r2df_ok). 학습에 쓴 TASK도 됨
DG_CONF    = 0.25    # 모델 신뢰도 하한 (track 생성과 같게)
DG_SAMPLES = 6       # 시험할 프레임 수 (라벨이 있는 프레임에서 고르게)
DG_TRACK   = True    # True: 이 TASK에 5번 셀 track 결과가 있으면 ④도 점검

use_cvat(LOGIN)
DG_TASK_ID = ctt.as_task_id(DG_TASK_ID)              # 2696411 또는 '2696411' 모두 됨
assert DG_TASK_ID, 'DG_TASK_ID를 넣으세요'
dg = load_script('diagnose_pipeline.py')
task = CLIENT.tasks.retrieve(DG_TASK_ID)
folder = task_dir(DG_TASK_ID); os.makedirs(folder, exist_ok=True)
gt = ctt.export_annotations(task, f'{folder}/diag_gt_{ctt.stamp()}.zip')      # 지금 라벨 = 정답(GT)
meta = ctt.read_export(gt)
video = ctt.find_video(VIDEOS, meta['source'], task.name) or ctt.video_from_backups(f'{BASE}/backups', task.name, VIDEOS)[0]
assert video, f"원본 영상을 못 찾음: videos/ 또는 backups/에 {meta['source']}"
last = ctt.last_row(REGISTRY, DG_TASK_ID, 'track', server=CVAT_SERVER) if DG_TRACK else None
txml = f"{BASE}/{last['result_file']}" if last and last.get('result_file') else None
print(f"TASK {DG_TASK_ID} '{task.name}' · 영상 {os.path.relpath(str(video), BASE)} · 모델 {model_label()} · IMGSZ {IMGSZ}")
print(f"track XML: {os.path.relpath(txml, BASE) if txml else '없음 (④ 생략)'}")
DG_RESULT = dg.run_diagnosis(task, gt, video, BEST, imgsz=IMGSZ, conf=DG_CONF, samples=DG_SAMPLES, track_xml=txml,
                             out_png=f'{folder}/diag_pipeline_{ctt.stamp()}.png')


## TASK별 개체 수 통계
CVAT TASK의 지금 라벨을 읽어 TASK마다 healthy/disease fruit·bud 개체 수(track 수)를 표로 보여 주고, 온실·촬영일별 합계와 CSV(reports/)를 만듭니다. 라벨은 바꾸지 않습니다.

In [ ]:
# 21. TASK별 개체 수 통계 — fruit·bud가 TASK마다 몇 개인지 (CVAT 지금 라벨, 읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN      = None     # None = 지금 접속 중인 계정 · 또는 0번 셀 CVAT_PROFILES의 이름 (예: 'B')
CNT_QUERY  = ''       # TASK 이름 검색. '' = 전체. 예: '261019', 'G6 vf_r2df_ok', 'tag=AA*'
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)
CNT_IDS    = []       # 또는 TASK ID 직접, 예: [2700651, 2704421] (넣으면 CNT_QUERY 무시)
CNT_LIMIT  = 500      # 최근 TASK 몇 개까지 볼지
CNT_SAVE   = True     # True = Drive reports/task_counts_<시각>.csv로 저장
CNT_PRINT  = True     # True = 인쇄용 HTML(A4 가로)도 만들어 내려받기 → 브라우저에서 열고 Ctrl+P (PDF 저장도 가능)

import pandas as pd
from IPython.display import display
use_cvat(LOGIN)
if CNT_IDS:
    targets = [{'task_id': t, 'name': ''} for t in ctt.as_task_ids(CNT_IDS)]
else:
    targets = ctt.select_tasks(ctt.list_tasks(CLIENT, limit=CNT_LIMIT), CNT_QUERY, label='집계 대상')
print(f"대상 TASK {len(targets)}개" + (f" (검색어 '{CNT_QUERY}')" if CNT_QUERY and not CNT_IDS else ''))
if not targets:
    raise SystemExit(f"검색어 '{CNT_QUERY}'에 맞는 TASK가 없습니다 → 4번 셀 TASK_QUERY로 먼저 확인 "
                     "(예: 'vf*' = 상태가 vf로 시작, 'vf_r2df_ok', '261019 G6')")
LABELS = ['healthy_fruit', 'disease_fruit', 'healthy_bud', 'disease_bud']
rows = []
for i, t in enumerate(targets, 1):
    try:
        task = CLIENT.tasks.retrieve(t['task_id'])
        per, kind = ctt.count_objects(task)
        f = ctt.parse_task_name(task.name) or {}
        r = {'task_id': task.id, 'TASK 이름': task.name, '온실': f.get('gh', ''), '촬영일': f.get('date', ''),
             '라인': f.get('row', ''), '그루': f.get('plant', ''), '상태': f.get('state', ''), '표시': f.get('tag', ''),
             '라벨 종류': kind, '프레임': task.size}
        # 개체 수: track이면 track 수, 프레임별 박스만 있으면 '한 프레임 최대 박스 수'(하한)
        for lab in sorted(set(LABELS) | set(per)):
            x = per.get(lab, {'track': 0, 'box': 0, 'max_frame': 0, 'occluded_track': 0})
            r[lab] = x['track'] + (x['max_frame'] if x['track'] == 0 else 0)
            r[f'{lab} 박스'] = x['box']
        r['fruit 합계'] = sum(r.get(l, 0) for l in r if l.endswith('_fruit'))
        r['bud 합계'] = sum(r.get(l, 0) for l in r if l.endswith('_bud'))
        r['비고'] = '프레임별 박스 → 한 프레임 최대 수(하한)' if kind == '박스' else ('track + 프레임별 박스 섞임' if kind == '혼합' else '')
        rows.append(r)
    except Exception as e:
        rows.append({'task_id': t['task_id'], 'TASK 이름': t.get('name', ''), '비고': f'❌ {ctt.api_error(e)}'})
    if i % 20 == 0:
        print(f'  {i}/{len(targets)} …')
df = pd.DataFrame(rows)
for c_ in ['비고', '온실', '촬영일'] + LABELS + ['fruit 합계', 'bud 합계']:      # 모두 실패해도 표가 깨지지 않게
    if c_ not in df.columns:
        df[c_] = 0 if c_ in LABELS or c_.endswith('합계') else ''
num = [c for c in df.columns if c == '프레임' or c.endswith(('_fruit', '_bud', '박스', '합계'))]
df[num] = df[num].fillna(0).astype(int)
df = df.fillna('')
main = ['task_id', 'TASK 이름', '온실', '촬영일', '라인', '그루', '상태', '표시', '라벨 종류', '프레임',
        'healthy_fruit', 'disease_fruit', 'fruit 합계', 'healthy_bud', 'disease_bud', 'bud 합계', '비고']
pd.set_option('display.max_rows', 500)
display(df[[c for c in main if c in df.columns]])
ok = df[~df['비고'].astype(str).str.startswith('❌')]
print(f"\n합계 (TASK {len(ok)}개): fruit {int(ok['fruit 합계'].sum())} (healthy {int(ok['healthy_fruit'].sum())} · disease {int(ok['disease_fruit'].sum())}) · "
      f"bud {int(ok['bud 합계'].sum())} (healthy {int(ok['healthy_bud'].sum())} · disease {int(ok['disease_bud'].sum())})")
named = ok[ok['온실'].astype(str) != '']
if len(named):
    print('\n온실·촬영일별 합계:')
    display(named.groupby(['온실', '촬영일'])[['healthy_fruit', 'disease_fruit', 'fruit 합계', 'healthy_bud', 'disease_bud', 'bud 합계']].sum().astype(int))
print('개체 수 = track 수 (track 하나 = 열매/봉오리 하나). 같은 개체가 track 2개로 끊겨 있으면 2로 셈 → CVAT에서 Merge 후 다시 실행')
if CNT_PRINT:
    import html as _h, datetime
    from zoneinfo import ZoneInfo
    cols = [c for c in main if c in df.columns]
    def _tbl(d, index=False):
        return d.to_html(index=index, border=0, classes='t', na_rep='', justify='center')
    grp = (named.groupby(['온실', '촬영일'])[['healthy_fruit', 'disease_fruit', 'fruit 합계', 'healthy_bud', 'disease_bud', 'bud 합계']]
           .sum().astype(int)) if len(named) else None
    cond = f"검색어 '{CNT_QUERY}'" if CNT_QUERY and not CNT_IDS else (f'TASK ID {CNT_IDS}' if CNT_IDS else '전체')
    page = f'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><title>TASK별 개체 수</title>
<style>
@page {{ size: A4 landscape; margin: 10mm; }}
body {{ font-family: "Malgun Gothic", "Apple SD Gothic Neo", "Noto Sans KR", sans-serif; font-size: 10px; color: #000; }}
h1 {{ font-size: 16px; margin: 0 0 4px; }} h2 {{ font-size: 13px; margin: 14px 0 4px; }}
.meta {{ color: #444; margin-bottom: 8px; }}
table.t {{ border-collapse: collapse; width: 100%; }}
table.t th, table.t td {{ border: 1px solid #888; padding: 2px 4px; text-align: center; white-space: nowrap; }}
table.t th {{ background: #e8e8e8; }}
table.t tr {{ page-break-inside: avoid; }} thead {{ display: table-header-group; }}
.sum {{ font-weight: bold; margin: 6px 0; }}
.note {{ color: #444; margin-top: 8px; }}
</style></head><body>
<h1>TASK별 fruit · bud 개체 수</h1>
<div class="meta">CVAT {_h.escape(str(CVAT_SERVER))} · 대상 {_h.escape(cond)} · TASK {len(ok)}개 ·
출력 {datetime.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d %H:%M')}</div>
<div class="sum">합계: fruit {int(ok['fruit 합계'].sum())} (healthy {int(ok['healthy_fruit'].sum())} · disease {int(ok['disease_fruit'].sum())}) ·
bud {int(ok['bud 합계'].sum())} (healthy {int(ok['healthy_bud'].sum())} · disease {int(ok['disease_bud'].sum())})</div>
{_tbl(df[cols])}
{('<h2>온실·촬영일별 합계</h2>' + _tbl(grp.reset_index())) if grp is not None else ''}
<div class="note">개체 수 = track 수 (track 하나 = 열매/봉오리 하나). 라벨 종류가 '박스'인 TASK는 한 프레임 최대 박스 수(하한).</div>
</body></html>'''
    os.makedirs(f'{BASE}/reports', exist_ok=True)
    hp = f'{BASE}/reports/task_counts_{ctt.stamp()}.html'
    open(hp, 'w', encoding='utf-8').write(page)
    print('인쇄용:', os.path.relpath(hp, BASE), '→ 내려받은 파일을 브라우저에서 열고 Ctrl+P (A4 가로, PDF로 저장도 가능)')
    try:
        from google.colab import files
        files.download(hp)
    except Exception:
        if ON_KAGGLE:                                # Kaggle: 오른쪽 Output(/kaggle/working)에 복사 → 거기서 내려받기
            os.makedirs('/kaggle/working', exist_ok=True)
            shutil.copy2(hp, '/kaggle/working/')
            print('Kaggle: 오른쪽 패널 Output → /kaggle/working/' + os.path.basename(hp) + ' 내려받기')
if CNT_SAVE:
    os.makedirs(f'{BASE}/reports', exist_ok=True)
    out = f'{BASE}/reports/task_counts_{ctt.stamp()}.csv'
    df.to_csv(out, index=False, encoding='utf-8-sig')
    print('저장:', os.path.relpath(out, BASE), '(Google 스프레드시트로 열림)')


## export 현황 — 그룹별 TASK 수와 이미 받은 export 재사용
검색어로 고른 TASK가 그룹별로 몇 개인지, 각 TASK의 export가 이미 있고 지금 CVAT 라벨과 같은지(✅ 그대로 사용 / 🔄 다시 받기 / ⬇️ 새로 받기)를 보여 줍니다. 다시 받을 TASK 목록은 14번 셀 `EX_IDS`로, 그대로 쓸 수 있는 것은 14번 `EX_REUSE`(복사)로 이어집니다. 읽기만 합니다.
- **v4.13**: TASK마다 **검토**(✅/⏳), **역할**(val/train, 0번 셀 `VAL_RULE`), **마지막 학습**(학습 이름과 그때 역할), **학습 뒤**(🆕 학습 안 함 / 🔁 라벨 바뀜 / —)를 보여 주고, 그룹 × 역할 표로 train/val 개수를 셉니다. `XS_TRAIN = 'todo'`면 다음 학습에 넣을 TASK만 봅니다.


In [ ]:
# 22. export 현황 — 검색어로 고른 TASK가 몇 개인지, 이미 받은 export가 지금 CVAT 라벨과 같은지 (읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
LOGIN      = None      # TASK가 있는 계정 · 0번 셀 CVAT_PROFILES의 이름
XS_QUERY   = 'G1|G3 date=261019.. vf*, G5|G6 date=261019.. vf*'   # 볼 TASK (모든 셀 같은 규칙). 쉼표로 여러 그룹
XS_GROUPS  = {'G1·G3': 'G1|G3', 'G5·G6': 'G5|G6'}                   # 요약표의 그룹 (이름: 검색어). {} = 온실별
XS_FORMAT  = 'CVAT for video 1.1'   # 쓸 export 형식 (14번 EX_FORMAT과 같게)
XS_IMAGES  = True                   # True = 이미지가 들어 있는 export만 '있음'으로 셈 (이미지 없는 export는 영상에서 뽑을 수 있으면 False)
XS_TRAIN   = 'all'                  # v4.13 학습 상태로 거르기: 'all' · 'new' = 검토 완료인데 학습에 안 씀 · 'changed' = 학습 뒤 라벨 바뀜 · 'todo' = new + changed
# 검색어 규칙(모든 셀 같음): 공백=그리고 · |=또는 · ,=묶음 또는 · 앞에 -=빼기 · 칸=값(gh date row plant state tag) · 칸=a..b 범위 · * 와일드카드 · 검토=검토 완료(0번 셀 REVIEWED_RULE, -검토=미검토)

import pandas as pd
from IPython.display import display
use_cvat(LOGIN)
pd.set_option('display.max_rows', 500)
hits = ctt.select_tasks(ctt.list_tasks(CLIENT, limit=5000), XS_QUERY, label='대상 TASK')
kg_sync_exports()                                 # Kaggle: 다른 세션이 받은 export도 (v4.23)
print('exports/ 폴더 읽는 중 …')
SCAN = ctt.scan_exports(f'{BASE}/exports', names=[t['name'] for t in hits])   # 대상 TASK 이름의 zip만, 한 번 읽은 것은 기억
kind = 'video' if 'video' in XS_FORMAT.lower() else 'images'
rows = []
LAST_TRAIN = ctt.last_trained(BASE)              # TASK → 가장 최근에 들어간 학습 (logs/train_tasks.csv)
for t in hits:
    ex, st = ctt.best_export(SCAN, t['task_id'], t.get('updated_utc'), kind=kind, need_images=XS_IMAGES)
    f = ctt.parse_task_name(t['name']) or {}
    grp = next((g for g, q in XS_GROUPS.items() if ctt.match_name(t['name'], q)[0]), '') if XS_GROUPS else f.get('gh', '')
    rows.append({'그룹': grp, 'task_id': t['task_id'], 'TASK 이름': t['name'], 'CVAT 수정(UTC)': t.get('updated', ''),
                 '상태': {'최신': '✅ 받은 것 그대로 사용', '수정됨': '🔄 다시 받기 (CVAT에서 바뀜)', '없음': '⬇️ 새로 받기'}[st],
                 'export 폴더': ex['folder'] if ex else '', 'export 기준(UTC)': (str(ex['updated'])[:16] or '파일 시각') if ex else '',
                 '이미지': ('있음' if ex['images'] else '없음') if ex else '',
                 '검토': '✅' if task_reviewed(t['name']) else '⏳', '역할': task_role(t['name']) or '무작위',
                 '마지막 학습': (f"{LAST_TRAIN[str(t['task_id'])]['model']} ({LAST_TRAIN[str(t['task_id'])]['role']})"
                                 + ('' if os.path.exists(f"{BASE}/runs/{LAST_TRAIN[str(t['task_id'])]['model']}/TRAINING_DONE") else ' · 학습 전')
                                 if str(t['task_id']) in LAST_TRAIN else ''),
                 '학습 뒤': (ctt.train_change(t.get('updated_utc'), LAST_TRAIN.get(str(t['task_id'])))
                             if task_reviewed(t['name']) else '')})
XS = pd.DataFrame(rows)
if len(XS) and XS_TRAIN != 'all':
    _want = {'new': ['🆕 학습 안 함'], 'changed': ['🔁 라벨 바뀜'], 'todo': ['🆕 학습 안 함', '🔁 라벨 바뀜']}[XS_TRAIN]
    print(f"XS_TRAIN = '{XS_TRAIN}' → {int(XS['학습 뒤'].isin(_want).sum())}/{len(XS)}개")
    XS = XS[XS['학습 뒤'].isin(_want)]
if len(XS):
    print('■ 학습 데이터 상태 (검토 완료 TASK) — 그룹 × 역할')
    _rv = XS[XS['검토'] == '✅']
    if len(_rv):
        display(_rv.pivot_table(index='그룹', columns='역할', values='task_id', aggfunc='count', fill_value=0, margins=True, margins_name='합계'))
        display(_rv.pivot_table(index='역할', columns='학습 뒤', values='task_id', aggfunc='count', fill_value=0))
    print(f"미검토 TASK {int((XS['검토'] != '✅').sum())}개 (학습에 안 씀)")
    print('\n■ 그룹별 요약')
    smry = XS.groupby('그룹')['상태'].value_counts().unstack(fill_value=0)
    smry['TASK 합계'] = smry.sum(axis=1)
    display(smry)
    print('■ TASK별')
    display(XS.sort_values(['그룹', '상태', 'TASK 이름']))
    EXPORT_TODO_IDS = [int(r['task_id']) for _, r in XS.iterrows() if not r['상태'].startswith('✅')]
    TRAIN_TODO_IDS = [int(r['task_id']) for _, r in XS.iterrows() if r['검토'] == '✅' and r['학습 뒤'] in ('🆕 학습 안 함', '🔁 라벨 바뀜')]
    print(f"학습에 새로 넣거나 다시 넣을 TASK {len(TRAIN_TODO_IDS)}개 (🆕·🔁) → TRAIN_TODO_IDS. 다시 받아야 하면 14번 EX_IDS = TRAIN_TODO_IDS")
    REUSE_FOLDERS = sorted({r['export 폴더'] for _, r in XS.iterrows() if r['상태'].startswith('✅')})
    print(f"\n다시 받거나 새로 받을 TASK {len(EXPORT_TODO_IDS)}개 → 14번 셀: EX_IDS = EXPORT_TODO_IDS (이 세션에 저장됨)")
    print(f"그대로 쓸 수 있는 export가 있는 폴더 {len(REUSE_FOLDERS)}개: {REUSE_FOLDERS}")
    print("14번 셀을 EX_REUSE = True로 실행하면, 대상 전체를 새 폴더 하나에 모으면서 '받은 것 그대로'인 TASK는 서버에 요청하지 않고 복사합니다.")
else:
    print('대상 TASK가 없습니다.')


## 덮어써진 학습 복원 — Google Drive 버전 기록에서 사용자별 마지막 파일
두 사람이 같은 학습 이름으로 동시에 학습해 `runs/<이름>`의 파일이 서로 덮어써졌을 때 씁니다. Drive가 파일마다 보관하는 이전 버전 중 **사용자(Google 계정)별 마지막 저장본**을 찾아 `runs/<이름>_A/`, `runs/<이름>_B/`(먼저 시작한 사람이 A)와 `datasets/<이름>_A.zip` 등으로 내려받습니다. 원본은 건드리지 않습니다.

- 처음에는 `RV_MODE = 'list'`로 표만 봅니다 → 맞으면 `'restore'`.
- Drive는 이전 버전을 파일마다 **약 30일 / 100개**까지만 보관합니다. `RV_PIN = True`이면 찾은 버전을 영구 보관으로 표시해 사라지지 않게 합니다(파일 편집 권한 필요).
- 처음 실행할 때 Google 계정 허용 창이 뜹니다 (Drive API 사용).
- 복원한 학습은 **이어서 학습하지 마세요** (체크포인트 안의 저장 위치가 원래 이름을 가리킴). 2·3번 셀이나 평가에는 `MODEL_CHOICE = '<이름>_A'`로 씁니다.


In [ ]:
# 23. 덮어써진 학습 복원 — Drive 버전 기록에서 사용자별 마지막 파일 → runs/<이름>_A, _B (원본은 그대로)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
if ON_KAGGLE:
    raise SystemExit('⛔ 23번 셀은 Google Drive 버전 기록(API)을 쓰므로 Colab에서만 됩니다 → Colab에서 0번 → 23번')
RV_NAME   = ''          # 덮어써진 학습 이름 (runs/ 아래 폴더 이름). 예: 'fruit_r2'
RV_MODE   = 'list'      # 'list' = 사용자별 마지막 버전 표만 · 'restore' = 내려받아 복원
RV_USERS  = 'auto'      # 'auto' = 먼저 저장한 사람부터 A, B … · 직접: {'a@gmail.com': 'A', 'b@gmail.com': 'B'}
RV_UNTIL  = None        # 이 시각(한국 시각) 이전의 마지막 버전만. 예: '2026-10-29 18:30' (같은 사람이 두 번 돌렸을 때)
RV_PIN    = True        # 찾은 버전을 영구 보관(keepForever)으로 → 30일/100개 자동 삭제 막기
RV_OVERWRITE = False    # True = 이미 복원한 파일도 다시 받음
RV_EXTRA  = ['datasets/{name}.zip', 'datasets/{name}.sig', 'datasets/{name}_report.html']   # runs/ 밖에서 함께 볼 파일

import pandas as pd
from IPython.display import display
assert RV_NAME, "RV_NAME에 학습 이름을 적으세요 (runs/ 아래 폴더 이름)"
dr = load_script('drive_revisions.py')
svc = dr.drive_service()
items = dr.collect(svc, BASE, RV_NAME, extra=RV_EXTRA)
rows = dr.pick(items, until=RV_UNTIL)
users = dr.users_of(items)
suffix = dr.auto_suffixes(items) if RV_USERS == 'auto' else dict(RV_USERS)
print(f"runs/{RV_NAME}: 파일 {len(items)}개 · 저장한 사용자 {len(users)}명")
for u, t in users.items():
    print(f"   {suffix.get(u, '-')}  {u}  처음 저장 {dr._kst(t)} (KST)")
if len(users) < 2:
    print('⚠️ 저장한 사용자가 1명뿐입니다 — 같은 Google 계정으로 학습했다면 사용자로는 나눌 수 없습니다 (RV_UNTIL로 시각을 잘라 보세요)')
tbl = pd.DataFrame([{**{k: v for k, v in r.items() if not k.startswith('_')}, '복원 이름': (f"{RV_NAME}_{suffix[r['사용자']]}" if r['사용자'] in suffix else '')} for r in rows])
pd.set_option('display.max_rows', 400)
display(tbl)
os.makedirs(f'{BASE}/reports', exist_ok=True)
dr.save_table(rows, f"{BASE}/reports/revisions_{RV_NAME}_{ctt.stamp() if ctt else ''}.csv")
if RV_PIN:
    print(f'영구 보관 표시: {dr.pin(svc, rows)}개')
if RV_MODE == 'restore':
    job_start(23, 'revision_restore', {'name': RV_NAME, 'users': RV_USERS, 'until': RV_UNTIL, 'pin': RV_PIN})   # 작업 기록 (v4.21)
    try:
        done = dr.restore(svc, BASE, RV_NAME, rows, suffix, overwrite=RV_OVERWRITE)
        for _u, _s in suffix.items():
            if os.path.isdir(f'{BASE}/runs/{RV_NAME}_{_s}'):
                ctt.log_task('revision_restore', '', f'{RV_NAME}_{_s}', '복원', file=f'runs/{RV_NAME}_{_s}', summary=f'Drive 버전 기록 · 사용자 {_u}')
    finally:
        job_end('끝', f'runs/{RV_NAME} → ' + ', '.join(f'{RV_NAME}_{s}' for s in suffix.values()))
    print('\n복원한 모델 확인 (체크포인트 안의 기록):')
    info = []
    for u, s in suffix.items():
        for w in ('best.pt', 'last.pt'):
            p = f'{BASE}/runs/{RV_NAME}_{s}/weights/{w}'
            if os.path.exists(p):
                try:
                    info.append({'복원 이름': f'{RV_NAME}_{s}', '사용자': u, **dr.ckpt_info(p)})
                except Exception as e:
                    info.append({'복원 이름': f'{RV_NAME}_{s}', '사용자': u, '파일': p, '저장 시각': f'읽기 실패: {e}'})
    if info:
        display(pd.DataFrame(info).drop(columns=['파일']))
    print('→ 두 모델을 같은 기준으로 비교: 0번 셀 MODEL_CHOICE = ' + ' / '.join(f"'{RV_NAME}_{s}'" for s in suffix.values())
          + ' → 2·3번 셀. 이어서 학습(resume)은 하지 마세요.')
else:
    print("\n표가 맞으면 RV_MODE = 'restore'로 다시 실행하세요.")


## 수작업 ↔ 자동 라벨 비교 — 평가 수치와 대표 화면
같은 TASK의 수작업 라벨(업로드 직전 백업 `tasks/<ID>/before_upload_*.zip`·`before_auto_*.zip` → 없으면 `exports/`의 이 TASK export zip, 또는 지금 CVAT 라벨·지정 파일)과 자동 라벨(지금 CVAT 라벨 · track XML · 모델 새 탐지)을 비교합니다. 재현율·정밀도·F1, 박스 위·아래 잘림, 경우별(위치·크기 다름/놓침/잘못 그림) 대표 화면. 읽기만 합니다.


In [ ]:
# 24. 수작업 라벨 ↔ 자동 라벨 비교 — 같은 TASK, 평가 수치 + 대표 화면 (CVAT에 아무것도 올리지 않음) · MODELS = 여러 모델을 같은 정답과 비교 (11/6)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
TID   = 0              # 비교할 TASK ID (2696411 또는 '2696411')
LOGIN = None           # 그 TASK가 있는 계정 · 0번 셀 CVAT_PROFILES의 이름
GT    = 'auto'         # 수작업(정답): 'auto' = 업로드 직전 백업 → 없으면 exports/의 이 TASK export zip
                       #   · 'backup' = tasks/<ID>/의 업로드 직전 백업 중 라벨이 든 가장 이른 것
                       #   · 'export' = exports/ 안 이 TASK의 export zip (GT_PICK으로 고름)
                       #   · 'cvat'   = 지금 CVAT 라벨 (아직 자동 라벨을 올리지 않은 TASK를 모델과 비교할 때)
                       #   · 파일 경로(BASE 기준, 예 'exports/20261028_125036_cvat_for_images_1_1/eggplant_G5_261019_3_6_vf.zip')
GT_PICK = -1           # GT='export'에서 쓸 것: -1 = 가장 최근 export, 0 = 가장 이른 것 (목록이 표로 나옴)
PRED  = 'cvat'         # 자동: 'cvat' = 지금 CVAT에 있는 라벨(자동 라벨 업로드 후) · 파일 경로(track XML 등)
                       #   · 'model' = 지금 모델(best.pt)로 프레임마다 새로 탐지
CONF  = 0.25           # PRED='model'일 때 신뢰도 하한
MODEL_PICK = None      # PRED='model'일 때 쓸 모델: None = 0번 셀에서 고른 모델 · runs/ 폴더 이름(예 'fruit_r2_A')
                       #   · 0번 셀 MODEL_NAMES 번호(예 2) · best.pt 경로
MAX_FRAMES = None      # 비교 프레임 수 상한 (정답 라벨이 있는 프레임에서 고르게). None = 전부
SHOW_EACH  = 6         # 경우마다 그림 수 (나쁜 것부터)
NEAR, SAME = 0.5, 0.95 # 짝으로 보는 IoU 하한 / '같음'으로 보는 IoU
EDGE_TOL   = 0.02      # 박스 높이·폭의 2% 이하 차이는 '같음'
LABELS     = None      # 비교할 라벨 목록. None = 자동 쪽에 있는 라벨만 (자동 라벨이 bud를 모르면 bud는 빼고). 예: ['healthy_fruit', 'bud']
MODELS     = []      # (모델 비교) 여러 모델을 같은 정답과 차례로 비교 — 예: ['fruit_r261101_2class_rotate', 'James_모델'] 또는 MODEL_NAMES 번호 [3, 4]
                     #   정하면 PRED = 'model'로, 모델마다 표 + 모델별 대표 화면(SHOW_EACH_M개씩) → 마지막에 모델 비교표
                     #   GT는 보통 'export'(vf 이후 검토자 export) — 'auto'면 업로드 직전 백업(=A)이 정답이 되므로 'export'로 바꿈
SHOW_EACH_M = 3      # MODELS일 때 모델마다 경우별 그림 수 (0 = 그림 없이 표만)

import re, glob, numpy as np, matplotlib.pyplot as plt, pandas as pd
from collections import Counter, defaultdict
from IPython.display import display
TID = ctt.as_task_id(TID)
assert TID, 'TID에 TASK ID를 넣으세요'
use_cvat(LOGIN)
task = CLIENT.tasks.retrieve(TID)
tdir = task_dir(TID)
os.makedirs(tdir, exist_ok=True)

if MODELS and GT == 'auto':
    GT = 'export'
    print("MODELS 비교: GT = 'auto'는 업로드 직전 백업(A)을 정답으로 고르므로 'export'(검토자 export)로 바꿔 씁니다")
if MODELS:
    PRED = 'model'
# ---------- 1) 정답(수작업) ----------
gt_file = None
if GT in ('auto', 'backup'):
    cands = sorted(glob.glob(f'{tdir}/before_upload_*.zip') + glob.glob(f'{tdir}/before_auto_*.zip'),
                   key=lambda p: os.path.basename(p).split('_', 2)[-1])
    info = [(p, sum(len(v) for v in ctt.load_annotations(p)['boxes'].values())) for p in cands]
    print('업로드 직전 백업:', ', '.join(f'{os.path.basename(p)}(박스 {n})' for p, n in info) or f'없음 ({os.path.relpath(tdir, BASE)})')
    with_boxes = [p for p, n in info if n > 0]
    if with_boxes:
        gt_file = with_boxes[0]
    elif GT == 'backup':
        raise SystemExit('⛔ 라벨이 든 업로드 직전 백업이 없습니다 (이 앱의 5·6·10번 셀로 자동 라벨을 올린 TASK에만 생김)\n'
                         "   → GT = 'export'(14번 셀로 받은 export zip) 또는 파일 경로를 적으세요")
if gt_file is None and GT in ('auto', 'export'):
    _nm = task.name.split('_')
    names = ['_'.join(_nm[:k]) for k in range(len(_nm), max(len(_nm) - 3, 1), -1)]   # 뒤에 붙은 _AA 등이 달라도 찾게
    kg_sync_exports()
    ex = [r for r in ctt.scan_exports(f'{BASE}/exports', names=names, log=None) if str(r.get('task_id')) == str(TID)]
    ex = sorted(ex, key=lambda r: (str(r.get('updated') or ''), float(r.get('mtime') or 0)))
    if ex:
        display(pd.DataFrame([{'#': k, '폴더': r['folder'], '파일': r['file'], 'TASK 수정 시각(export 당시)': str(r.get('updated', ''))[:16]}
                              for k, r in enumerate(ex)]).set_index('#'))
        gt_file = ex[GT_PICK]['zip']
        print(f"export에서 정답 선택 (GT_PICK={GT_PICK}): {ex[GT_PICK]['folder']}/{ex[GT_PICK]['file']}"
              '  ← 수작업 라벨일 때 받은 것인지 위 시각으로 확인하세요')
    if gt_file is None:
        raise SystemExit(f'⛔ TASK {TID}의 수작업 라벨 파일을 찾지 못했습니다: 업로드 직전 백업도, exports/의 export zip도 없음\n'
                         "   → 자동 라벨을 아직 안 올렸다면 GT = 'cvat' (지금 CVAT 라벨) + PRED = 'model'\n"
                         '   → 아니면 14번 셀로 export를 받은 뒤 다시. (13번 셀 TASK 백업 zip은 형식이 달라 쓸 수 없음)')
elif GT == 'cvat':
    gt_file = ctt.export_annotations(task, f'{tdir}/compare_gt_{ctt.stamp()}.zip')
elif gt_file is None:
    gt_file = GT if os.path.isabs(GT) else f'{BASE}/{GT}'
gt = ctt.load_annotations(gt_file)
print(f"정답(수작업): {os.path.relpath(gt_file, BASE)}")

# ---------- 2) 자동 ----------
if PRED == 'cvat':
    assert GT != 'cvat', "GT와 PRED가 둘 다 'cvat'이면 같은 라벨끼리 비교합니다 → PRED = 'model' 또는 track XML 경로"
    pred_file = ctt.export_annotations(task, f'{tdir}/compare_now_{ctt.stamp()}.zip')
    pred = ctt.load_annotations(pred_file)
    pred_desc = f'지금 CVAT 라벨 ({os.path.relpath(pred_file, BASE)})'
def _load_model(pick):
    MODEL_PICK = pick
    from ultralytics import YOLO
    if MODEL_PICK is None:
        _w = BEST
    elif isinstance(MODEL_PICK, int):
        assert MODEL_PICK in MODEL_NAMES, f'MODEL_PICK {MODEL_PICK}가 0번 셀 MODEL_NAMES 번호 {list(MODEL_NAMES)}에 없습니다'
        _w = need_file(f'{BASE}/runs/{MODEL_NAMES[MODEL_PICK]}/weights/best.pt')
    elif str(MODEL_PICK).endswith('.pt'):
        _w = MODEL_PICK if os.path.isabs(MODEL_PICK) else f'{BASE}/{MODEL_PICK}'
    else:
        _w = need_file(f'{BASE}/runs/{MODEL_PICK}/weights/best.pt')
    assert os.path.exists(_w), f'모델 파일이 없습니다: {_w}'
    import datetime as _d
    _mlabel = f"{os.path.relpath(_w, f'{BASE}/runs')} ({_d.datetime.fromtimestamp(os.path.getmtime(_w)).strftime('%m-%d %H:%M')})"
    model = YOLO(_w)
    base_n = {i: (n[:-9] if n.endswith('_occluded') else n) for i, n in model.names.items()}
    occl = {i: n.endswith('_occluded') for i, n in model.names.items()}
    pred, pred_desc = {'boxes': {}, 'meta': gt['meta']}, f'모델 {_mlabel} 새 탐지 (conf {CONF})'
    return model, base_n, occl, pred, pred_desc

if PRED == 'model':
    model, base_n, occl, pred, pred_desc = _load_model(MODEL_PICK)
else:
    pred_file = PRED if os.path.isabs(PRED) else f'{BASE}/{PRED}'
    pred = ctt.load_annotations(pred_file)
    pred_desc = os.path.relpath(pred_file, BASE)

def _compare(pred, pred_desc, model=None, base_n=None, occl=None, show_each=SHOW_EACH, tag=''):
    """정답 gt와 pred를 짝지어 표·보고서·대표 화면 → (전체 요약 한 줄, 라벨별 F1) 돌려줌"""
    W0, H0 = gt['meta']['W'], gt['meta']['H']
    labeled = sorted(f for f in gt['frames'] if gt['boxes'].get(f))
    frames = labeled if not MAX_FRAMES or len(labeled) <= MAX_FRAMES else \
        [labeled[int(i)] for i in np.linspace(0, len(labeled) - 1, MAX_FRAMES)]
    get = ctt.cvat_frame_getter(task, gt['meta'])            # 라벨러가 본 CVAT 프레임
    if model is not None:
        for k, f in enumerate(frames):
            img = get(f)
            if img is None:
                continue
            img = img.convert('RGB'); sx, sy = W0 / img.width, H0 / img.height
            rr = model.predict(img, imgsz=IMGSZ, conf=CONF, verbose=False, agnostic_nms=any(occl.values()))[0]
            pred['boxes'][f] = [(f'p{k}_{j}', base_n[int(c)], occl[int(c)], (x1 * sx, y1 * sy, x2 * sx, y2 * sy), True)
                                for j, ((x1, y1, x2, y2), c) in enumerate(zip(rr.boxes.xyxy.tolist(), rr.boxes.cls.tolist()))]
    all_gt = {b[1] for v in gt['boxes'].values() for b in v}
    known = set(LABELS) if LABELS else (set(base_n.values()) if model is not None
                                        else {b[1] for v in pred['boxes'].values() for b in v} or all_gt)
    if all_gt - known:
        print(f"비교에서 뺀 라벨 (자동 쪽에 없음): {', '.join(sorted(all_gt - known))}  ← 넣으려면 LABELS에 적기")

    def iou(a, b):
        ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
        u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - ix*iy
        return ix*iy / u if u > 0 else 0

    def edges(g, p):
        """정답 g 대비 자동 p의 네 변 차이를 말로 (y는 아래로 증가)"""
        gh, gw = g[3]-g[1], g[2]-g[0]; out = []
        d = p[1] - g[1]
        if abs(d) > EDGE_TOL*gh: out.append(f"위쪽(꽃받침 쪽) {abs(d):.0f}px {'잘림' if d > 0 else '더 큼'} ({abs(d)/gh:.0%})")
        d = g[3] - p[3]
        if abs(d) > EDGE_TOL*gh: out.append(f"아래쪽(끝) {abs(d):.0f}px {'잘림' if d > 0 else '더 큼'} ({abs(d)/gh:.0%})")
        d = p[0] - g[0]
        if abs(d) > EDGE_TOL*gw: out.append(f"왼쪽 {abs(d):.0f}px {'좁음' if d > 0 else '넓음'}")
        d = g[2] - p[2]
        if abs(d) > EDGE_TOL*gw: out.append(f"오른쪽 {abs(d):.0f}px {'좁음' if d > 0 else '넓음'}")
        return ', '.join(out) or '네 변 모두 2% 안쪽'

    # ---------- 3) 짝짓기와 평가 ----------
    cases = {'위치·크기 다름': [], '놓침': [], '잘못 그림': []}
    st = defaultdict(Counter); ious, dtop, dbot, dw = [], [], [], []
    for f in frames:
        G = [b for b in gt['boxes'].get(f, []) if b[1] in known]
        P = [b for b in pred['boxes'].get(f, []) if b[1] in known]
        used, mg = set(), set()
        for v, i, j in sorted(((iou(g[3], p[3]), i, j) for i, g in enumerate(G) for j, p in enumerate(P)), reverse=True):
            if v < NEAR or i in mg or j in used:
                continue
            mg.add(i); used.add(j)
            g, p = G[i], P[j]; lb = g[1]
            st[lb]['짝'] += 1; st[lb]['같음(IoU≥%.2f)' % SAME] += v >= SAME; st[lb]['라벨 다름'] += g[1] != p[1]
            st[lb]['가림 표시 다름'] += bool(g[2]) != bool(p[2])
            gh, gw = g[3][3]-g[3][1], g[3][2]-g[3][0]
            ious.append(v); dtop.append((p[3][1]-g[3][1]) / gh); dbot.append((g[3][3]-p[3][3]) / gh)
            dw.append(((p[3][2]-p[3][0]) - gw) / gw)
            if v < SAME or g[1] != p[1]:
                cases['위치·크기 다름'].append(dict(frame=f, gt=g, pr=p, iou=v, bad=1 - v,
                    note=edges(g[3], p[3]) + ('' if g[1] == p[1] else f' · 라벨 다름 {g[1]}→{p[1]}')))
        for i, g in enumerate(G):
            if i not in mg:
                st[g[1]]['놓침'] += 1
                best = max([iou(g[3], p[3]) for p in P], default=0)
                near_p = max(P, key=lambda p: iou(g[3], p[3])) if best > 0 else None
                h = g[3][3] - g[3][1]
                cases['놓침'].append(dict(frame=f, gt=g, pr=near_p, iou=best, bad=h,
                    note=f"{g[1]} 높이 {h:.0f}px{' · 가림' if g[2] else ''}" + (f' · 자동 박스가 겹치나 너무 어긋남 (IoU {best:.2f})' if best > 0 else ' · 근처에 자동 박스 없음')))
        for j, p in enumerate(P):
            if j not in used:
                st[p[1]]['잘못 그림'] += 1
                best = max([iou(p[3], g[3]) for g in G], default=0)
                cases['잘못 그림'].append(dict(frame=f, gt=None, pr=p, iou=best, bad=p[3][3] - p[3][1],
                    note=f"{p[1]} 높이 {p[3][3]-p[3][1]:.0f}px" + (f' · 정답과 IoU {best:.2f} (겹쳐 그림?)' if best > 0 else ' · 정답에 없는 곳')))

    print(f"\nTASK {TID} '{task.name}' · 비교 프레임 {len(frames)}장 (정답 라벨이 있는 프레임)")
    print(f"정답 = {os.path.basename(gt_file)} · 자동 = {pred_desc}")
    rows = []
    for lb in sorted(st) + ['전체']:
        c = sum(st.values(), Counter()) if lb == '전체' else st[lb]
        tp, fn, fp = c['짝'], c['놓침'], c['잘못 그림']
        rows.append({'라벨': lb, '정답 박스': tp + fn, '자동 박스': tp + fp, '짝': tp, '놓침': fn, '잘못 그림': fp,
                     '재현율': round(tp / (tp + fn), 3) if tp + fn else None,
                     '정밀도': round(tp / (tp + fp), 3) if tp + fp else None,
                     'F1': round(2 * tp / (2 * tp + fn + fp), 3) if tp else 0,
                     f'같음(IoU≥{SAME})': c['같음(IoU≥%.2f)' % SAME], '라벨 다름': c['라벨 다름'], '가림 표시 다름': c['가림 표시 다름']})
    summary = pd.DataFrame(rows).set_index('라벨')
    display(summary)
    if ious:
        q = lambda a: f"평균 {np.mean(a):+.1%} · 중앙 {np.median(a):+.1%}"
        print(f"짝지은 박스 {len(ious)}개: IoU 평균 {np.mean(ious):.3f} · 중앙 {np.median(ious):.3f} · 0.75 미만 {np.mean(np.array(ious) < 0.75):.0%}")
        print(f"  위쪽(꽃받침 쪽) 차이 (+ = 자동이 잘림): {q(dtop)}")
        print(f"  아래쪽(끝) 차이     (+ = 자동이 잘림): {q(dbot)}")
        print(f"  폭 차이             (+ = 자동이 넓음): {q(dw)}")
    os.makedirs(f'{BASE}/reports', exist_ok=True)
    rep = f"{BASE}/reports/manual_vs_auto_{TID}{tag}_{ctt.stamp()}.csv"
    pd.DataFrame([{'경우': k, '프레임': c['frame'], 'IoU': round(c['iou'], 3), '설명': c['note']}
                  for k, lst in cases.items() for c in lst]).to_csv(rep, index=False, encoding='utf-8-sig')
    summary.to_csv(rep.replace('.csv', '_summary.csv'), encoding='utf-8-sig')
    print(f"저장: reports/{os.path.basename(rep)} (경우별 전체 목록), _summary.csv (표)")

    # ---------- 4) 대표 화면 ----------
    for kind, lst in cases.items():
        lst = sorted(lst, key=lambda c: -c['bad'])[:show_each]
        if not lst:
            continue
        print(f"\n■ {kind} (나쁜 것부터 {len(lst)}개)  —  흰 실선 = 수작업, 노란 점선 = 자동")
        display(pd.DataFrame([{'#': k + 1, '프레임': c['frame'], 'IoU': round(c['iou'], 2), '설명': c['note']}
                              for k, c in enumerate(lst)]).set_index('#'))
        fig, axes = plt.subplots(1, len(lst), figsize=(3.2 * len(lst), 4.2), squeeze=False)
        for k, c in enumerate(lst):
            ax = axes[0][k]; ax.axis('off')
            img = get(c['frame'])
            if img is None:
                ax.set_title(f"#{k+1} f{c['frame']}: 이미지 없음"); continue
            img = img.convert('RGB'); sx, sy = img.width / W0, img.height / H0
            bxs = [b[3] for b in (c['gt'], c['pr']) if b]
            x1 = min(b[0] for b in bxs); y1 = min(b[1] for b in bxs); x2 = max(b[2] for b in bxs); y2 = max(b[3] for b in bxs)
            m = 0.35 * max(x2 - x1, y2 - y1)                    # 박스 주변을 잘라 확대
            cx1, cy1, cx2, cy2 = max(0, x1 - m), max(0, y1 - m), min(W0, x2 + m), min(H0, y2 + m)
            ax.imshow(img.crop((cx1 * sx, cy1 * sy, cx2 * sx, cy2 * sy)))
            def rect(b, col, ls):
                ax.add_patch(plt.Rectangle(((b[0] - cx1) * sx, (b[1] - cy1) * sy), (b[2] - b[0]) * sx, (b[3] - b[1]) * sy,
                                           fill=False, ec=col, lw=2.5, ls=ls))
            if c['gt']: rect(c['gt'][3], 'white', '-')
            if c['pr']: rect(c['pr'][3], 'yellow', '--')
            ax.set_title(f"#{k+1} frame {c['frame']} IoU {c['iou']:.2f}", fontsize=9)
        plt.tight_layout(); plt.show()

    return summary, (float(np.mean(ious)) if ious else None)

# ---------- 모델 비교 (MODELS) 또는 한 번 비교 ----------
if MODELS:
    _rows, _f1 = [], {}
    for _pick in MODELS:
        _name = MODEL_NAMES.get(_pick, str(_pick)) if isinstance(_pick, int) else str(_pick)
        print(f"\n{'=' * 20} 모델 {_name} {'=' * 20}")
        _m, _bn, _oc, _pr, _pd = _load_model(_pick)
        _tag = '_' + re.sub(r'[^0-9A-Za-z_.-]', '_', str(_pick))[:40]
        _s, _iou = _compare(_pr, _pd, model=_m, base_n=_bn, occl=_oc, show_each=SHOW_EACH_M, tag=_tag)
        _t = _s.loc['전체'].to_dict()
        _rows.append({'모델': _name, **{k: _t[k] for k in ('정답 박스', '자동 박스', '짝', '놓침', '잘못 그림', '재현율', '정밀도', 'F1')},
                      f'같음(IoU≥{SAME})': _t.get(f'같음(IoU≥{SAME})'), '라벨 다름': _t.get('라벨 다름'),
                      'IoU 평균': round(_iou, 3) if _iou is not None else None})
        _f1[_name] = _s['F1'].drop('전체', errors='ignore')
    print(f"\n{'=' * 20} 모델 비교 · TASK {TID} · 정답 = {os.path.basename(gt_file)} · conf {CONF} · imgsz {IMGSZ} {'=' * 20}")
    _cmp = pd.DataFrame(_rows).set_index('모델')
    display(_cmp)
    print('라벨별 F1:'); display(pd.DataFrame(_f1))
    _rep = f"{BASE}/reports/models_vs_gt_{TID}_{ctt.stamp()}.csv"
    _cmp.to_csv(_rep, encoding='utf-8-sig')
    print(f"저장: reports/{os.path.basename(_rep)}")
else:
    _compare(pred, pred_desc, model=(model if PRED == 'model' else None), base_n=(base_n if PRED == 'model' else None),
             occl=(occl if PRED == 'model' else None))


## 업로드 기록 보기 — 5·6·10번 셀로 CVAT에 올린 것
`logs/uploads.csv`(업로드 한 건에 한 줄)를 TASK·기간·방식으로 걸러 보고, 한 건을 골라 상세 로그(`tasks/<ID>/upload_<시각>.log`)를 펼쳐 봅니다. 읽기만 합니다.


In [ ]:
# 25. 업로드 기록 보기 — 누가·언제·어떤 모델·설정으로 무엇을 올렸고 기존 라벨이 어떻게 바뀌었나 (읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
UL_TASK   = ''          # TASK ID (하나 또는 '123, 456') · '' = 전체
UL_QUERY  = ''          # TASK 이름 검색 (모든 셀 같은 규칙) · '' = 전체
UL_SINCE  = ''          # 이 날짜 이후만, 예 '2026-10-30'
UL_METHOD = ''          # 'track' / 'shape' / '' = 둘 다
UL_DETAIL = None        # 상세 로그를 펼칠 log_id (아래 표의 log_id) 또는 -1 = 표의 마지막 건

import pandas as pd
from IPython.display import display
p = f'{BASE}/logs/uploads.csv'
_ul_rows = ctt.read_sharded(p) if hasattr(ctt, 'read_sharded') else []   # v4.31.0: 세션 파일 합침
if not _ul_rows:
    print('아직 기록이 없습니다 (v4.9.6부터 5·6·10번 셀 업로드 때 생김)')
else:
    df = pd.DataFrame(_ul_rows, dtype=str).fillna('')
    if UL_TASK:
        df = df[df['task_id'].isin([str(i) for i in ctt.as_task_ids(UL_TASK)])]
    if UL_QUERY:
        df = df[[bool(ctt.match_name(n or a, UL_QUERY)[0]) for n, a in zip(df['task_name'], df['task_name_after'])]]
    if UL_SINCE:
        df = df[df['start'] >= UL_SINCE]
    if UL_METHOD:
        df = df[df['method'] == UL_METHOD]
    print(f'업로드 기록 {len(df)}건 (전체 파일 {p})')
    cols = ['log_id', 'start', 'sec', 'status', 'method', 'mode', 'task_id', 'task_name', 'operator', 'cvat_user',
            'model', 'model_time', 'before_total', 'after_total', 'uploaded']
    pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 60)
    display(df[[c for c in cols if c in df]])
    if UL_DETAIL is not None and len(df):
        r = df.iloc[-1] if UL_DETAIL == -1 else df[df['log_id'] == str(UL_DETAIL)].iloc[0]
        lp = f"{BASE}/{r['detail_json'][:-5]}.log" if r.get('detail_json') else ''
        print('=' * 70)
        print(open(lp, encoding='utf-8').read() if lp and os.path.exists(lp) else f'상세 로그 파일 없음: {lp}')


## 작업 기록 (시계열) — CVAT 작업·백업·export·학습을 한 곳에서
`logs/activity.csv`에 셀 실행 한 번이 '작업' 하나(시작·끝 줄)로, 그 안에서 TASK마다 한 줄(무엇을 했고 라벨이 몇 개였나)로 쌓입니다. 5·6·7·10·13·14·15번 셀이 기록합니다.
- `AC_VIEW = 'jobs'` 작업 목록 · `'tasks'` TASK별 줄 · `'history'` TASK 하나의 변화 이력(라벨 수 변화) · `'daily'` 날짜×작업 종류 건수
- 처음 실행하면 예전 기록(작업 대장·백업 색인·runs/)을 한 번 옮겨 옵니다. 읽기만 합니다(CVAT 접속 없음).


In [ ]:
# 26. 작업 기록 보기 — 언제·누가·무슨 작업을 했고 TASK별로 라벨이 어떻게 바뀌었나 (읽기만 함)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
AC_VIEW   = 'jobs'      # 'jobs' = 작업(셀 실행) 목록 · 'tasks' = TASK별 줄 · 'history' = TASK 하나의 변화 이력 · 'daily' = 날짜별 건수
AC_TASK   = ''          # TASK ID (하나 또는 '123, 456') · 'history'는 이것 또는 AC_QUERY 필요
AC_QUERY  = ''          # TASK 이름 검색 (모든 셀 같은 규칙)
AC_SINCE  = ''          # 이 날짜·시각 이후, 예 '2026-10-28' 또는 '2026-10-30 09:00'
AC_UNTIL  = ''          # 이 날짜·시각 이전 (그날 포함하려면 '2026-10-30 23:59')
AC_ACTION = ''          # 작업 종류 일부: '백업' 'export' 'track' '업로드' '자동 라벨' '학습' '검수' … · '' = 전부
AC_WHO    = ''          # 작업자(OPERATOR) 일부 · '' = 전부
AC_JOB    = ''          # 작업 하나(job_id)의 TASK 줄만 보기 ('jobs' 표의 job_id)
AC_IMPORT_OLD = True    # 처음 한 번: 예전 기록(작업 대장·백업 색인·runs/)을 작업 기록으로 옮김
AC_SAVE   = True        # 보이는 표를 reports/activity_<보기>_<시각>.csv로 저장

import pandas as pd, json as _json
from IPython.display import display
if AC_IMPORT_OLD:
    ctt.import_old_records(BASE)
p = f'{BASE}/logs/activity.csv'
_ac_rows = ctt.read_sharded(p) if hasattr(ctt, 'read_sharded') else []   # v4.31.0: 세션 파일 합침
assert _ac_rows, '아직 작업 기록이 없습니다 (v4.10부터 5·6·7·10·13·14·15번 셀이 기록)'
A = pd.DataFrame(_ac_rows, dtype=str).fillna('')
A = A.sort_values('time', kind='stable')
if AC_SINCE: A = A[A['time'] >= AC_SINCE]
if AC_UNTIL: A = A[A['time'] <= AC_UNTIL]
if AC_WHO:   A = A[A['who'].str.contains(AC_WHO, regex=False)]
if AC_ACTION: A = A[A['action'].str.contains(AC_ACTION, regex=False)]
pd.set_option('display.max_rows', 500); pd.set_option('display.max_colwidth', 70)
T = A[A['row'] == 'TASK']
if AC_TASK:
    T = T[T['task_id'].isin([str(i) for i in ctt.as_task_ids(AC_TASK)])]
if AC_QUERY:
    T = T[[bool(n) and bool(ctt.match_name(n, AC_QUERY)[0]) for n in T['task_name']]]
if AC_JOB:
    T = T[T['job_id'] == AC_JOB]

def _labels(s):
    try:
        d = _json.loads(s) if s.startswith('{') else None
    except Exception:
        d = None
    return ', '.join(f'{k} {v}' for k, v in d.items()) if isinstance(d, dict) else s

if AC_VIEW == 'jobs':
    J = A[A['row'].isin(['작업 시작', '작업 끝'])]
    g = []
    for jid, d in J.groupby('job_id', sort=False):
        s_ = d[d['row'] == '작업 시작']; e_ = d[d['row'] == '작업 끝']
        s0 = s_.iloc[0] if len(s_) else d.iloc[0]
        g.append({'job_id': jid, '시작': s0['time'], '끝': e_.iloc[-1]['time'] if len(e_) else '',
                  '셀': s0['cell'], '작업': s0['action'], '작업자': s0['who'], 'CVAT 계정': s0['server'],
                  '상태': e_.iloc[-1]['status'] if len(e_) else '끝 기록 없음 (중단 또는 실행 중)',
                  '초': e_.iloc[-1]['sec'] if len(e_) else '', 'TASK 줄': int((A['job_id'] == jid).sum() - len(d)),
                  'TASK': (lambda t: ', '.join(f"{a} {b}" for a, b in t[:3]) + (f' 외 {len(t) - 3}개' if len(t) > 3 else ''))(
                      list(A[(A['job_id'] == jid) & (A['row'] == 'TASK')][['task_id', 'task_name']].drop_duplicates().itertuples(index=False, name=None))),
                  '요약': e_.iloc[-1]['summary'] if len(e_) else s0['summary'], '모델': s0['model']})
    V = pd.DataFrame(g)
    old = A[(A['row'] == 'TASK') & (A['job_id'] == '')]
    print("TASK 줄을 자세히 보려면 AC_VIEW = 'tasks', AC_JOB = '<job_id>'")
    print(f'작업 {len(V)}개' + (f' · 작업 없이 기록된 TASK 줄(예전 기록 등) {len(old)}개 → AC_VIEW = "tasks"' if len(old) else ''))
elif AC_VIEW == 'tasks':
    V = T[['time', 'action', 'status', 'task_id', 'task_name', 'who', 'boxes_before', 'boxes', 'by_label', 'frames',
           'task_updated', 'file', 'summary', 'job_id', 'source']].copy()
    V['by_label'] = V['by_label'].map(_labels)
    print(f'TASK 줄 {len(V)}개')
elif AC_VIEW == 'history':
    assert AC_TASK or AC_QUERY, "'history'는 AC_TASK 또는 AC_QUERY로 TASK를 고르세요"
    rows = []
    for tid, d in T.groupby('task_id', sort=False):
        prev = None
        for _, r in d.iterrows():
            nb_ = r['boxes']
            change = ''
            if r['boxes_before'] != '' and nb_ != '':
                change = f"{r['boxes_before']}→{nb_}"
            elif nb_ != '' and prev is not None:
                change = f"{prev}→{nb_}" if prev != nb_ else '같음'
            if nb_ != '':
                prev = nb_
            rows.append({'TASK': tid, '시각': r['time'], '작업': r['action'], '상태': r['status'], '이름': r['task_name'],
                         '박스(라벨 수)': nb_, '변화': change, '라벨별': _labels(r['by_label']),
                         'CVAT 수정 시각': r['task_updated'], '작업자': r['who'], '파일': r['file'], '메모': r['summary'][:80]})
    V = pd.DataFrame(rows)
    print(f'TASK {T["task_id"].nunique()}개의 이력 {len(V)}줄 (시간순) — 박스 = 그 시점의 라벨 박스 수 (백업은 track 키프레임만)')
elif AC_VIEW == 'daily':
    T2 = T.assign(날짜=T['time'].str[:10])
    V = T2.pivot_table(index='날짜', columns='action', values='task_id', aggfunc='count', fill_value=0)
    V['합계'] = V.sum(axis=1)
    print('날짜별 TASK 줄 수 (작업 종류별)')
else:
    raise SystemExit("AC_VIEW는 'jobs' / 'tasks' / 'history' / 'daily'")
display(V)
if AC_SAVE and len(V):
    os.makedirs(f'{BASE}/reports', exist_ok=True)
    out = f'{BASE}/reports/activity_{AC_VIEW}_{ctt.stamp()}.csv'
    V.to_csv(out, encoding='utf-8-sig', index=AC_VIEW == 'daily')
    print('저장:', os.path.relpath(out, BASE))


## 앙상블 자동 라벨링 — 모델 여러 개가 함께 (shape)
같은 프레임(원본 영상 또는 CVAT)을 모델 여러 개가 각각 탐지하고, 같은 열매끼리 묶어(WBF) **여러 모델이 동의한 박스만** 위치를 평균해 한 개로 올립니다. 한 모델만 찾은 것은 검토 목록으로 남깁니다.
- 먼저 `EN_UPLOAD = False`(미리 보기)로 실행 → 모델별 기여 표·그림·검토 목록 확인 → `EN_UPLOAD = True`(다시 계산하지 않고 그 결과를 올림).
- 클래스가 다른 모델: `…_occluded`는 기본 라벨 + 가림 표시로, bud처럼 일부 모델만 아는 라벨은 아는 모델들끼리 표를 셈.
- 효과 확인: 수작업 라벨이 있는 TASK에서 24번 셀로 단일 모델과 비교.


In [ ]:
# 27. 앙상블 자동 라벨링 — 모델 여러 개가 같은 프레임을 보고, 여러 모델이 동의한 박스만 하나로 합쳐 올림 (shape)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
EN_MODELS = ['fruit_1920',
             'exp_20261025_081411_cvat_for_images_1_1',
             'fruit_r261028',
             'exp_2dirs_20261029_212022_cvat_for_images_1_1_A',
             'exp_2dirs_20261029_212022_cvat_for_images_1_1_B']    # runs/ 아래 이름 (best.pt 사용) 또는 .pt 경로
EN_TASK_IDS = []        # TASK ID (하나 또는 여러 개). 처음엔 수작업 라벨이 있는 TASK 하나로 시험 → 24번 셀로 비교
EN_QUERY    = ''        # 또는 TASK 이름 검색 (EN_TASK_IDS가 비었을 때)
LOGIN       = None
EN_MIN_VOTES = 2        # 채택 기준: 이 수 이상의 모델이 찾은 박스 (그 라벨을 아는 모델이 더 적으면 그 수)
EN_IOU      = 0.55      # 같은 열매로 묶는 IoU 기준
EN_CONF     = 0.25      # 모델마다 탐지 신뢰도 하한
EN_WEIGHTS  = None      # 모델별 가중치 {'fruit_1920': 1.0, …} · None = 모두 같게
EN_SOURCE   = 'auto'    # 프레임: 'auto' = 원본 영상이 있으면 영상(원본 화질), 없으면 CVAT · 'video' · 'cvat'
EN_FRAMES   = None      # None = TASK 전체 · 시험은 range(0, 30) 처럼
EN_UPLOAD   = False     # False = 미리 보기 (계산·검토 목록·그림을 tasks/<ID>/ensemble_*/에 저장, CVAT에 안 올림)
                        # True  = CVAT에 올림 (같은 설정으로 미리 본 결과가 있으면 다시 계산하지 않고 그것을 올림)
EN_EXISTING = 'skip'    # 기존 라벨이 있으면: 'skip' 빼기 · 'replace' 백업 후 교체 · 'add' 덧붙임
EN_SHOW     = 6         # 그림으로 볼 프레임 수 (검토 박스가 많은 프레임부터)

import json, hashlib, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
en = load_script('ensemble_annotate.py')
use_cvat(LOGIN)

# ---------- 모델 ----------
def _w(m):
    return m if str(m).endswith('.pt') and os.path.exists(m) else need_file(f'{BASE}/runs/{m}/weights/best.pt')
_paths = {m: _w(m) for m in EN_MODELS}
_miss = [m for m, p in _paths.items() if not os.path.exists(p)]
assert not _miss, f'best.pt가 없는 모델: {_miss}'
_cache = globals().setdefault('_EN_CACHE', {})
MODELS = []
for m, p in _paths.items():
    key = (p, os.path.getmtime(p))
    if key not in _cache:
        _cache[key] = en.ModelInfo(m, p)
    MODELS.append(_cache[key])
_dev = 0 if torch.cuda.is_available() else 'cpu'
print(f"모델 {len(MODELS)}개 · 장치 {'GPU' if _dev == 0 else 'CPU (느림)'} · 채택 {EN_MIN_VOTES}표 이상 · IoU {EN_IOU} · conf {EN_CONF}")
_knows = {}
for i, mi in enumerate(MODELS, 1):
    print(f"  M{i} {mi.name:<50} 라벨 {', '.join(mi.labels)}{' (가림 split)' if mi.split else ''} · md5 {ctt.file_fingerprint(mi.weights)[1]}")
    for l in mi.labels:
        _knows.setdefault(l, []).append(f'M{i}')
for l, ms in _knows.items():
    print(f"  라벨 {l}: 아는 모델 {len(ms)}개 ({', '.join(ms)}) → 채택에 {min(EN_MIN_VOTES, len(ms))}표 필요")
SHORT = {mi.name: f'M{i}' for i, mi in enumerate(MODELS, 1)}
_sig = hashlib.md5(json.dumps([[mi.name, ctt.file_fingerprint(mi.weights)[1]] for mi in MODELS]
                              + [EN_MIN_VOTES, EN_IOU, EN_CONF, EN_WEIGHTS, IMGSZ, str(EN_FRAMES)], sort_keys=True).encode()).hexdigest()[:10]

# ---------- 대상 TASK ----------
ids = ctt.as_task_ids(EN_TASK_IDS)
if not ids and EN_QUERY:
    ids = [t['task_id'] for t in ctt.select_tasks(ctt.list_tasks(CLIENT, limit=5000), EN_QUERY, label='앙상블 대상')]
assert ids, 'EN_TASK_IDS 또는 EN_QUERY를 넣으세요'
if EN_UPLOAD:
    job_start(27, 'auto', {'ensemble': [mi.name for mi in MODELS], 'min_votes': EN_MIN_VOTES, 'iou': EN_IOU, 'conf': EN_CONF,
                           'weights': EN_WEIGHTS, 'existing': EN_EXISTING, 'source': EN_SOURCE, 'sig': _sig})
summary = []
for tid in ids:
    print('=' * 80)
    task = CLIENT.tasks.retrieve(tid)
    folder = task_dir(tid); os.makedirs(folder, exist_ok=True)
    meta = task.get_meta()
    fr0 = (getattr(meta, 'frames', None) or [None])[0]
    W0, H0 = (getattr(fr0, 'width', None), getattr(fr0, 'height', None)) if fr0 is not None else (None, None)
    deleted = set(getattr(meta, 'deleted_frames', None) or [])
    todo = [k for k in (EN_FRAMES if EN_FRAMES is not None else range(task.size)) if k not in deleted]
    n_exist = ctt.annotation_count(task)
    print(f"TASK {tid} '{task.name}' · 프레임 {len(todo)}/{task.size} · 기존 라벨 {n_exist}개")
    row = {'task_id': tid, 'task_name': task.name}
    if EN_UPLOAD and n_exist and EN_EXISTING == 'skip':
        print("⏭ 기존 라벨이 있어 뺌 (EN_EXISTING = 'replace' 또는 'add')"); row['결과'] = '건너뜀(기존 라벨)'; summary.append(row); continue
    # 같은 설정으로 미리 본 결과
    saved = sorted(glob.glob(f'{folder}/ensemble_*_{_sig}/result.json'))
    if EN_UPLOAD and saved:
        res = {int(k): v for k, v in json.load(open(saved[-1], encoding='utf-8')).items()}
        out_dir = os.path.dirname(saved[-1]); src = f'미리 본 결과 {os.path.relpath(out_dir, BASE)}'
        print(f'같은 설정으로 미리 본 결과를 씀: {os.path.relpath(out_dir, BASE)}')
    else:
        vid = None
        if EN_SOURCE != 'cvat' and ctt.task_basic(task)['is_video']:
            vid = ctt.find_video(VIDEOS, '', task.name) or ctt.video_from_backups(f'{BASE}/backups', task.name, VIDEOS)[0]
            if vid is not None:
                al = ctt.check_video_alignment(task, vid, TRACK_FRAME_OFFSET, 'pyav')
                if not al['ok']:
                    print(f"⚠️ 영상과 CVAT 프레임이 {al['best_shift']:+d} 어긋남 → CVAT 프레임으로 (20번 셀 진단 권장)"); vid = None
        if EN_SOURCE == 'video' and vid is None:
            print("❌ 원본 영상을 쓸 수 없어 뺌 (EN_SOURCE = 'video')"); row['결과'] = '영상 없음'; summary.append(row); continue
        if vid is not None:
            start, step = ctt.task_frame_map(meta)
            vmap = {start + k * step - TRACK_FRAME_OFFSET: k for k in todo}
            frames_iter = ((vmap[vi], im) for vi, im in ctt.video_frames(vid, vmap, 'pyav'))
            src = f'원본 영상 {os.path.relpath(str(vid), BASE)}'
        else:
            frames_iter = en.frames_from_cvat(task, todo)
            src = 'CVAT 프레임'
        print(f'프레임 출처: {src}')
        import time as _t; t0 = _t.time()
        res = en.run_ensemble(task, MODELS, frames_iter, W0, H0, imgsz=IMGSZ, conf=EN_CONF, iou_thr=EN_IOU,
                              min_votes=EN_MIN_VOTES, weights=EN_WEIGHTS, device=_dev)
        out_dir = f'{folder}/ensemble_{ctt.stamp()}_{_sig}'
        os.makedirs(out_dir, exist_ok=True)
        json.dump({str(k): v for k, v in res.items()}, open(f'{out_dir}/result.json', 'w', encoding='utf-8'), ensure_ascii=False, default=list)
        json.dump({'models': {mi.name: {'weights': os.path.relpath(mi.weights, BASE), 'md5': ctt.file_fingerprint(mi.weights)[1],
                                        'labels': mi.labels} for mi in MODELS},
                   'min_votes': EN_MIN_VOTES, 'iou': EN_IOU, 'conf': EN_CONF, 'weights': EN_WEIGHTS, 'imgsz': IMGSZ,
                   'source': src, 'made': ctt.now(), 'who': who_am_i(), 'sec': round(_t.time() - t0, 1)},
                  open(f'{out_dir}/settings.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        pd.DataFrame(en.frame_rows(res)).to_csv(f'{out_dir}/frames.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame(en.review_rows(res)).to_csv(f'{out_dir}/review.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame(en.contribution(res, [mi.name for mi in MODELS])).to_csv(f'{out_dir}/contribution.csv', index=False, encoding='utf-8-sig')
    n_acc = sum(len(f['accepted']) for f in res.values()); n_rev = sum(len(f['review']) for f in res.values())
    lab_acc = pd.Series([d['label'] for f in res.values() for d in f['accepted']]).value_counts().to_dict()
    print(f"채택 {n_acc}개 {lab_acc} · 검토 목록 {n_rev}개 (표가 모자란 박스) · 저장 {os.path.relpath(out_dir, BASE)}")
    C = pd.DataFrame(en.contribution(res, [mi.name for mi in MODELS])); C.insert(0, '약칭', C['모델'].map(SHORT))
    display(C)
    row.update(프레임=len(res), 채택=n_acc, 검토=n_rev, 출처=src, 결과='미리 보기' if not EN_UPLOAD else '')
    # ---------- 그림 ----------
    if EN_SHOW and not EN_UPLOAD:
        pick = sorted(res, key=lambda k: (-len(res[k]['review']), -len(res[k]['accepted'])))[:EN_SHOW]
        get = ctt.cvat_frame_getter(task, {'start': 0, 'step': 1})
        fig, axes = plt.subplots(1, len(pick), figsize=(4 * len(pick), 6), squeeze=False)
        for ax, k in zip(axes[0], pick):
            ax.axis('off'); im = get(k)
            if im is None:
                ax.set_title(f'f{k}: no image'); continue
            im = im.convert('RGB'); sx, sy = im.width / (W0 or im.width), im.height / (H0 or im.height)
            ax.imshow(im)
            for d, col, ls in [(d, 'yellow', '-') for d in res[k]['accepted']] + [(d, 'red', '--') for d in res[k]['review']]:
                b = d['box']
                ax.add_patch(plt.Rectangle((b[0] * sx, b[1] * sy), (b[2] - b[0]) * sx, (b[3] - b[1]) * sy, fill=False, ec=col, lw=1.8, ls=ls))
                ax.text(b[0] * sx, b[1] * sy - 3, f"{d['votes']}/{d['n_know']}" + ('' if col == 'yellow' else ' ' + ','.join(SHORT[m] for m in d['voters'])),
                        color=col, fontsize=7, backgroundcolor=(0, 0, 0, 0.4))
            ax.set_title(f"frame {k} | accepted {len(res[k]['accepted'])} | review {len(res[k]['review'])}", fontsize=9)
        print('노란 실선 = 채택 (표/그 라벨을 아는 모델 수), 빨간 점선 = 검토 목록 (찾은 모델 약칭) — 그림 제목: accepted = 채택, review = 검토')
        plt.tight_layout(); plt.show()
    # ---------- 올리기 ----------
    if EN_UPLOAD:
        label_ids = {l.name: l.id for l in task.get_labels()}
        shapes, skipped = en.to_shapes(res, label_ids)
        L = upload_log(tid, 'shape', task_name=task.name, source=f'앙상블 {len(MODELS)}개 · {src}',
                       settings={'ensemble': {SHORT[mi.name]: mi.name for mi in MODELS}, 'min_votes': EN_MIN_VOTES,
                                 'iou': EN_IOU, 'conf': EN_CONF, 'weights': EN_WEIGHTS, 'result': os.path.relpath(out_dir, BASE)})
        L.set(model='앙상블: ' + ', '.join(mi.name for mi in MODELS), model_time='',
              model_md5=' '.join(ctt.file_fingerprint(mi.weights)[1] for mi in MODELS))
        err = None
        try:
            backup = ctt.export_annotations(task, f'{folder}/before_auto_{ctt.stamp()}.zip')
            L.counts(before_zip=backup); L.set(backup_file=os.path.relpath(backup, BASE),
                                               mode='replace' if (n_exist and EN_EXISTING == 'replace') else ('add' if n_exist else 'new'))
            from cvat_sdk.api_client import models as _M
            from cvat_sdk.core.proxies.annotations import AnnotationUpdateAction as _A
            def _write():
                for i in range(0, len(shapes), 500):
                    task.update_annotations(_M.PatchedLabeledDataRequest(shapes=shapes[i:i + 500]), action=_A.CREATE)
            if n_exist and EN_EXISTING == 'replace':
                ctt.replace_annotations(task, _write, backup, log=L.say)
            else:
                _write()
            L.set(uploaded=f'박스 {len(shapes)}개 {lab_acc}' + (f' · TASK에 없는 라벨이라 뺌 {skipped}' if skipped else ''))
            L.say(f"✅ 올림: 박스 {len(shapes)}개" + (f" (TASK에 없는 라벨 {skipped} 뺌)" if skipped else ''))
            ch = change_report(task, backup, 'after_auto')
            L.set(after_file=ch['after'], change_csv=ch['csv']); L.counts(after_zip=f"{BASE}/{ch['after']}")
            row['결과'] = f'올림 {len(shapes)}개'
            _lu = label_uploaded(task, replaced=bool(n_exist and EN_EXISTING == 'replace'), before_zip=backup, say=L.say)   # v4.31.5 이름 규칙
            if _lu['note']:
                row['결과'] += ' · ' + _lu['note']
            if _lu['renamed']:
                L.set(task_name_after=_lu['name'])
        except Exception as e:
            err = e; row['결과'] = f'오류: {ctt.api_error(e)}'; L.say(f'❌ {ctt.api_error(e)}')
        finally:
            print('   상세 로그:', os.path.relpath(str(L.finish('완료' if err is None else '오류', err)), BASE))
            ctt.append_registry(REGISTRY, run_time=ctt.now(), action='auto', task_id=tid, task_name=task.name,
                                model='앙상블 ' + '+'.join(SHORT.values()), server=CVAT_SERVER, status='완료' if err is None else '오류',
                                note=f"앙상블 {EN_MIN_VOTES}표 이상 · {row['결과']} · 결과 {os.path.relpath(out_dir, BASE)}")
    summary.append(row)
if EN_UPLOAD:
    job_end('끝', f"앙상블 {len(MODELS)}개 모델 · TASK {len(summary)}개")
print('=' * 80)
display(pd.DataFrame(summary))
print('모델 약칭:', ', '.join(f'{v} = {k}' for k, v in SHORT.items()))
if not EN_UPLOAD:
    print("검토 목록·프레임별 표는 tasks/<ID>/ensemble_*/review.csv·frames.csv. 결과가 괜찮으면 EN_UPLOAD = True로 다시 실행 (다시 계산하지 않음).")


## 28. 웹 관리 화면 연동 — 상태 파일 갱신 · 웹 요청 보기 (v4.14)
웹 관리 화면(Claude 웹 페이지)은 Drive의 `web/state.json` 하나를 읽어 TASK 생애주기·검토·백업·export·학습 상태와 모델을 보여 주고, 요청을 `requests/`에 파일로 남깁니다.
- 이 셀이 CVAT TASK 목록과 Drive 기록으로 `web/state.json`을 **새로 만듭니다** (웹 화면의 '갱신 시각'). 작업을 한 뒤 다시 실행하면 웹 화면에 반영됩니다.
- 웹 요청 목록(대기·진행 중·완료·실패·취소)을 보여 주고, 요청을 실행할 셀을 알려 줍니다.
- 요청 실행: 그 셀 맨 위 `FROM_REQUESTS = True` → 평소처럼 미리 보기 → 실행. 작업이 끝나면 요청이 '완료'로 바뀝니다 (실패하면 '실패').
  백업 13번 · export 14번 · track 생성 5번(올리기는 6번에서 사람이 승인) · 자동 라벨 shape 9번 → 10번 · 학습 15번(또는 Kaggle 학습 전용 노트북)
- **v4.15**: TASK마다 '학습 준비' — CVAT for images export가 최신인지, 이미지를 얻을 곳(export zip 안 · `videos/`의 영상 · `backups/`의 TASK 백업)이 있는지. 웹 '학습 준비' 화면에서 부족한 TASK에 export·백업을 요청하고, 준비된 TASK로 학습을 요청합니다.
- **v4.16**: TASK마다 **프레임별·클래스별 박스 수 통계**(최신 export zip, 없거나 오래됐으면 CVAT — `WEB_CVAT_STATS`개씩, `web/label_stats.json`에 기억)와 **같은 온실·촬영일 TASK와의 비교**(프레임당 박스 수가 중앙값에서 크게 벗어나면 표시), **학습 스냅숏**(`exports/_train_*`) 매니페스트를 웹 화면에 넣습니다.


In [ ]:
# 28. 웹 관리 화면 연동 — web/state.json 갱신 + 웹 요청 보기 (요청은 각 셀이 FROM_REQUESTS = True로 실행)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
assert wb is not None, 'scripts/web_bridge.py가 없습니다 (v4.14 scripts로 바꾸세요)'
LOGIN      = None      # TASK가 있는 계정 · 0번 셀 CVAT_PROFILES의 이름
WEB_STATE  = True      # True = CVAT TASK 목록 + Drive 기록으로 web/state.json을 새로 만듦
WEB_LIMIT  = 5000      # 읽을 CVAT TASK 수 (최근 수정 순)
WEB_SHOW   = 'open'    # 요청 목록: 'open' = 대기·진행 중만 · 'all' = 전체
WEB_CVAT_STATS = 40    # export가 없거나 오래된 TASK의 라벨 통계를 CVAT에서 셀 수 (한 번에, 나머지는 다음 실행 때 · 0 = 안 함)
WEB_CORRECTIONS = 200  # 검토 수정량(자동 라벨 업로드 직후 ↔ 검토 뒤 export)을 새로 셀 TASK 수 (한 번에, 나머지는 다음 실행 때 · 0 = 안 함, v4.27)

import glob
import pandas as pd
from collections import Counter
from IPython.display import display
kg_sync_exports()                                 # Kaggle: export 최신 여부·라벨 통계를 Drive 기준으로 (v4.23)
if wb is not None and hasattr(wb, 'repair_pending'):
    wb.repair_pending(BASE)
if wb is not None and hasattr(wb, 'settle'):        # '받는 중'이던 웹 요청: 다 받았으면 완료로 (v4.22)
    for _r in wb.settle(BASE, ('backup', 'export'), who=who_am_i()):
        print(f"📨 웹 요청 {_r['id']} ({wb.KINDS[_r['kind']][0]}) → {wb.STATUS_KO.get(_r['status'], _r['status'])} · {_r.get('progress', '')}")
if WEB_STATE:
    use_cvat(LOGIN)
    if ON_KAGGLE:                                   # 모델 정리 기록·데이터 서명 (v4.24.8) · 수집기 TASK 스냅숏(사라진 TASK, v4.31.8)
        drive_pull('runs/*/*', 'datasets/*.sig', 'status/cvat_tasks.json', quiet=True)   # 이 세션 뒤에 생긴 학습도 (결과·잠금·정리 기록, 가중치는 받지 않음)
    _tasks, _cat = ctt.list_tasks_all(CLIENT, limit=WEB_LIMIT)    # 웹에서 조직·프로젝트를 바꿔 볼 수 있게 보이는 것 전부 (v4.19)
    print(f"CVAT TASK {len(_tasks)}개 · 조직 {len(_cat['orgs'])}개 · 프로젝트 {len(_cat['projects'])}개 "
          f"(웹 기본 범위 = {ctt.scope_text(CVAT_SCOPE)})")
    if ON_KAGGLE:                                   # Kaggle: 영상·백업을 받아 두지 않았으므로 Drive 목록으로
        _vn, _bn = wb.media_names(BASE, remote_videos=[p for p, s, t in _kg_files('videos')],
                                  remote_backups=[p for p, s, t in _kg_files('backups')])
    else:
        _vn, _bn = wb.media_names(BASE, videos_dir=VIDEOS)
    with ctt.org_context(CLIENT, None):            # 다른 조직의 TASK도 셀 수 있게
        _stats = wb.compute_stats(BASE, _tasks, client=CLIENT if WEB_CVAT_STATS else None, cvat_max=WEB_CVAT_STATS)
    _snaps = wb.snapshots(BASE, refresh=True, log=print)
    _corr = None
    if WEB_CORRECTIONS and hasattr(wb, 'compute_corrections'):   # 검토 수정량 (v4.27) — 모델별로 웹 '학습' 탭에
        try:
            _corr = wb.compute_corrections(BASE, _tasks, model_names=[os.path.basename(d) for d in glob.glob(f'{BASE}/runs/*')]
                                           + ([k.split('/')[1] for k in kg_file_index('runs') if k.count('/') >= 2] if ON_KAGGLE else []),
                                           reviewed_rule=REVIEWED_RULE, max_new=WEB_CORRECTIONS,
                                           fetch=(lambda ps: drive_pull(*ps, quiet=True)) if ON_KAGGLE else None)
        except Exception as _e:
            print(f'(검토 수정량 계산 실패: {type(_e).__name__}: {_e})')
    WEB = wb.build_state(BASE, _tasks, operator=who_am_i(), app_version=APP_VERSION, reviewed_rule=REVIEWED_RULE,
                         val_rule=VAL_RULE, server=globals().get('CVAT_SERVER', ''), video_stems=_vn, backup_bases=_bn, stats=_stats, snaps=_snaps,
                         scope=CVAT_SCOPE, catalog=_cat,
                         files=(kg_file_index('runs', 'datasets') if ON_KAGGLE else None),   # 모델 크기·정리 (v4.24.8)
                         corrections=_corr,                                               # 검토 수정량 (v4.27)
                         uld=uld_index(), replaced=uld_lineage())   # v4.31.8: ULD가 바꿔 만든 TASK 연결                                                 # v4.31.6: ULD TASK ('사진(ULD)'·백업 필요 없음)
    _p = wb.write_state(BASE, WEB)
    print(f"저장: {os.path.relpath(_p, BASE)} ({os.path.getsize(_p) / 1024:.0f} KB) · 갱신 {WEB['generated']}")
    _po = WEB.get('pending') or []
    if _po:
        print(f"CVAT 준비 중(받을 것 남음): 백업 {sum(x['kind'] == 'backup' for x in _po)}개 · export {sum(x['kind'] == 'export' for x in _po)}개 "
              "→ 13·14번 셀 MODE = 'download' (또는 웹 '요청' 탭의 받기)")
    _s = WEB['summary']
    display(pd.DataFrame([{'단계': k, 'TASK': _s['by_stage'].get(k, 0)} for k in WEB['stages'] + ['CVAT에 없음 (백업만)']
                          if _s['by_stage'].get(k, 0) or k in WEB['stages']]).set_index('단계').T)
    print(f"검토 완료 {_s['reviewed']}개 (역할 {_s['by_role']}) · 학습 뒤 라벨 바뀜 {_s['changed_after_train']}개 · "
          f"모델 {_s['models']}개 (학습 끝 {_s['models_done']}개)")
    print(f"학습 준비 (검토 완료 TASK): {_s['prep']} · 영상 {WEB['media']['videos']}개 · 백업 {WEB['media']['backups']}개 "
          "— export = CVAT for images export가 최신이 아님, backup = 이미지를 얻을 곳(영상·백업)이 없음")
    _cm = [m for m in WEB['models'] if (m.get('corrections') or {}).get('measured')]
    if _cm:                                          # 검토 수정량 (v4.27)
        display(pd.DataFrame([{'모델': m['name'], '셈 TASK': m['corrections']['measured'], 'export 필요': len(m['corrections']['need_export']),
                               '자동 박스': m['corrections']['total']['auto'], '그대로': m['corrections']['total']['kept'],
                               '고침': m['corrections']['total']['fixed'] + m['corrections']['total']['relabeled'],
                               '지움(오탐)': m['corrections']['total']['deleted'], '추가(놓침)': m['corrections']['total']['added'],
                               '수정률': f"{m['corrections']['rate']:.1%}" if m['corrections']['rate'] is not None else ''} for m in _cm]).set_index('모델'))
    _out = [t for t in WEB['tasks'] if (t.get('peer') or {}).get('flags')]
    print(f"라벨 통계 {_s['stats']}개 TASK · 같은 온실·촬영일 TASK와 크게 다른 것 {len(_out)}개 · 검토 완료인데 백업이 최신 아님 {_s['backup_stale']}개 · 학습 스냅숏 {len(WEB['snapshots'])}개")
    if _out:
        display(pd.DataFrame([{'task_id': t['id'], 'TASK': t['name'], '묶음': t['peer']['group'], '묶음 TASK': t['peer']['n'],
                               '프레임당 박스': t['stats']['total_mean'], '묶음 중앙값': t['peer'].get('median'),
                               '다른 점': ' · '.join(t['peer']['flags'])} for t in _out]).sort_values('묶음').head(40))

_reqs = wb.list_requests(BASE)
_rows = [{'요청': r['id'], '종류': wb.KINDS.get(r['kind'], (r['kind'],))[0], '상태': wb.STATUS_KO.get(r['status'], r['status']),
          '실행 셀': wb.KINDS.get(r['kind'], ('', ''))[1], 'TASK 수': len(r.get('task_ids') or []),
          '요청자': r.get('by', ''), '요청 시각': str(r.get('created', ''))[:16], '메모': r.get('note', ''),
          '결과': str(r.get('result', ''))[:80]}
         for r in _reqs if WEB_SHOW == 'all' or r['status'] in ('pending', 'taken')]
print(f"\n■ 웹 요청 ({'대기·진행 중' if WEB_SHOW == 'open' else '전체'} {len(_rows)}건 / 전체 {len(_reqs)}건)")
if _rows:
    display(pd.DataFrame(_rows))
    for k, n in sorted(Counter(r['kind'] for r in _reqs if r['status'] in ('pending', 'taken')).items()):
        print(f"   {wb.KINDS[k][0]} {n}건 → {wb.KINDS[k][1]}번 셀 FROM_REQUESTS = True")
else:
    print('   없음')
try:                                               # v4.31.0: 오래된 세션 기록 파일을 하나로 (파일이 너무 많아지지 않게)
    ctt.compact_shards(BASE)
except Exception as _e:
    print(f'(세션 기록 합치기 실패: {_e})')


## 29. 모델 정리 — 안 쓰는 모델의 큰 파일 지우기 (v4.24.8)
학습 하나에는 `runs/<이름>/`(가중치·기록)과 `datasets/<이름>.zip`(학습 데이터, 가장 큼)이 딸려 있습니다. 폴더를 통째로 지우면 이름 중복 보호와 학습 대장·업로드 기록의 출처가 끊기므로, 이 셀은 **큰 파일만 지우고 기록은 남깁니다**.
- `CLEAN_MODELS = []`로 실행하면 모델마다 상태·크기·데이터셋을 함께 쓰는 학습·자동 라벨링에 쓴 기록을 표로 봅니다.
- `slim`: `last.pt`·중간 가중치·데이터셋 zip을 지움 — `best.pt`는 남아 계속 씀. 끝난 학습만.
- `archive`: `best.pt`까지 지우고 '보관됨'. 자동 라벨링에 쓴 모델은 출처 확인용으로 남기길 권하며, 그래도 지우려면 `ALLOW_USED = True`. 0번 셀에서 지금 고른 모델·학습 중인 모델은 지우지 않음.
- 데이터셋을 다른 학습이 `DATASET_NAME`으로 쓰고 있으면 데이터셋은 남깁니다.
- 먼저 미리 보기 → `CONFIRM = True`로 실행. 지운 파일은 Drive 휴지통에 30일 남고, 이력은 `runs/<이름>/CLEANUP.json`과 작업 기록에 남습니다.
- 웹 '학습' 탭에서 모델을 누르고 '정리 요청'을 보내면 → 이 셀을 `FROM_REQUESTS = True`로 실행.


In [ ]:
# 29. 모델 정리 — 안 쓰는 모델의 큰 파일만 지움 (기록은 남김) · 먼저 미리 보기, CONFIRM = True일 때만 지움
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
assert ctt is not None and hasattr(ctt, 'model_inventory'), 'scripts/를 v4.24.8 이상으로 바꾸세요'
CLEAN_MODELS  = []        # 정리할 학습 이름 (runs/<이름>), 예: ['fruit_r261101_2class'] · 비우면 목록만
CLEAN_ACTION  = 'slim'    # 'slim' = last.pt·중간 가중치·데이터셋 zip만 지움 (best.pt로 계속 씀) · 'archive' = best.pt까지 지우고 '보관됨'
CLEAN_REASON  = ''        # 이유 (이력에 남음), 예: '회전 오류 데이터로 학습'
ALLOW_USED    = False     # True = 자동 라벨링에 쓴 모델도 archive (출처 확인용 가중치가 사라짐)
CONFIRM       = False     # True = 실제로 지움 (Drive 휴지통으로 — 30일 안에 되돌릴 수 있음) · False = 미리 보기만
FROM_REQUESTS = False     # True = 웹 '학습' 탭의 모델 정리 요청을 받아 실행 (요청을 보낸 것이 확인이므로 바로 지움)

import pandas as pd
from IPython.display import display
if FROM_REQUESTS:
    _p = req_use('cleanup')
    CLEAN_MODELS = list(_p.get('models') or [])
    CLEAN_ACTION = _p.get('action') or CLEAN_ACTION
    ALLOW_USED = bool(_p.get('allow_used', False))
    CLEAN_REASON = _p.get('reason') or CLEAN_REASON
    CONFIRM = True
if ON_KAGGLE:                                       # Kaggle: Drive 기준으로 (지운 것·크기·잠금)
    drive_pull('runs/*/CLEANUP.json', 'runs/*/DATASET_SIG', 'datasets/*.sig', 'logs/uploads.csv', quiet=True)
_files = kg_file_index('runs', 'datasets') if ON_KAGGLE else None
_inv = ctt.model_inventory(BASE, files=_files, stale_min=LOCK_STALE_MIN)
print(f"▶ 이번 실행: {'목록만' if not CLEAN_MODELS else ('지움' if CONFIRM else '미리 보기')}"
      + (f" · {CLEAN_ACTION} ({ctt.CLEANUP_ACTIONS.get(CLEAN_ACTION, '?')}) · {', '.join(CLEAN_MODELS)}" if CLEAN_MODELS else ''))
display(pd.DataFrame([{'학습 이름': r['name'], '상태': r['status'], 'best.pt MB': r['best_mb'] or '',
                       'last.pt·중간 MB': round(r['last_mb'] + r['other_pt_mb'], 1) or '', '데이터셋 MB': r['dataset_mb'] or '',
                       '데이터셋을 함께 쓰는 학습': ', '.join(r['dataset_users']),
                       '자동 라벨에 씀': (f"업로드 {r['used_uploads']} · 작업 {r['used_jobs']}" if r['used_uploads'] + r['used_jobs'] else ''),
                       'slim으로 줄 MB': r['slim_mb'] or '', 'archive로 줄 MB': r['archive_mb'] or '',
                       '마지막 정리': (f"{r['last_cleanup']['time'][:16]} {r['last_cleanup']['action']}" if r['last_cleanup'] else '')}
                      for r in _inv]))
print(f"모델 {len(_inv)}개 · best.pt {sum(r['best_mb'] for r in _inv) / 1024:.1f} GB · last.pt·중간 {sum(r['last_mb'] + r['other_pt_mb'] for r in _inv) / 1024:.1f} GB · "
      f"데이터셋 {sum(r['dataset_mb'] for r in _inv) / 1024:.1f} GB")
_protect = [NAME] if globals().get('NAME') else []
_plans = [ctt.cleanup_plan(BASE, n, CLEAN_ACTION, files=_files, inventory=_inv, allow_used=ALLOW_USED, protect=_protect)
          for n in CLEAN_MODELS]
for _pl in _plans:
    print(f"\n■ {_pl['name']} — {ctt.CLEANUP_ACTIONS.get(_pl['action'], _pl['action'])} · 지울 것 {len(_pl['delete'])}개 {_pl['freed_mb']:.0f} MB")
    for _r, _mb in _pl['delete']:
        print(f"   🗑 {_r} ({_mb} MB)")
    for _r, _why in _pl['keep']:
        print(f"   ✓ 남김 {_r} — {_why}")
    for _b in _pl['blockers']:
        print(f"   ⛔ {_b}")

def _rm_drive(rel):
    # Colab: 파일 삭제 = Drive 휴지통 · Kaggle: Drive에서 바로 지움(rclone, 휴지통으로) + 이 세션의 복사본
    p = f'{BASE}/{rel}'
    if ON_KAGGLE:
        r = _kg_rc(['deletefile', _kg_remote(rel)], quiet=True)
        if r.returncode and _files is not None and rel in _files:
            raise RuntimeError((r.stderr or '').strip()[-200:] or 'rclone deletefile 실패')
        if os.path.exists(p):
            os.remove(p)
    elif os.path.exists(p):
        os.remove(p)

if CLEAN_MODELS and CONFIRM:
    _ok_plans = [p for p in _plans if not p['blockers']]
    job_start(29, 'cleanup', {'models': CLEAN_MODELS, 'action': CLEAN_ACTION, 'allow_used': ALLOW_USED, 'reason': CLEAN_REASON},
              summary=f"{CLEAN_ACTION} {', '.join(CLEAN_MODELS)}")
    _res = [ctt.cleanup_apply(BASE, p, remove=_rm_drive, who=who_am_i(), reason=CLEAN_REASON) for p in _ok_plans]
    for p in _plans:
        if p['blockers']:
            print(f"⛔ {p['name']}: 정리하지 않음 — {p['blockers'][0]}")
        ctt.log_task('cleanup', '', p['name'], '정리' if not p['blockers'] else '건너뜀',
                     summary=(f"{p['action']} {p['freed_mb']} MB" if not p['blockers'] else p['blockers'][0]))
    if ON_KAGGLE:
        for _k in ('runs', 'datasets'):
            _KG['ls'].pop(_k, None)                   # 다음 목록은 Drive에서 새로
    _freed = sum(r['freed_mb'] for r in _res)
    _bad = [p['name'] for p in _plans if p['blockers']] + [r['name'] for r in _res if not r['ok']]
    job_end('끝' if any(r['ok'] for r in _res) else '실패',
            f"{len([r for r in _res if r['ok']])}개 정리 · {_freed:.0f} MB" + (f" · 못 함 {', '.join(_bad)}" if _bad else ''))
    print(f"\n정리: {len(_res)}개 모델 · {_freed / 1024:.2f} GB 비움 (Drive 휴지통에 30일) · 기록: runs/<이름>/{ctt.MODEL_CLEANUP_FILE}"
          " → 웹 화면에 반영하려면 28번 셀")
elif CLEAN_MODELS:
    print("\n미리 보기입니다. 지우려면 CONFIRM = True로 다시 실행하세요.")


## 30. 작업자 — 폰 '가지 재처리' + LC앱 '가지 라벨링 관리' (v4.31.6에서 31번 LC 작업자를 합침)
**v4.31.7**: 학습(train) 요청도 받습니다 (15번 셀을 `FROM_REQUESTS`로 — 모든 GPU 줄이 빈 뒤 시작). `W_ROLE = 'gpu'`는 Kaggle GPU 워커용입니다 (`eggplant_kaggle_gpu_worker_v4.31.7.ipynb` — CPU 마스터가 시작, 할 일이 없으면 스스로 끝남).

**v4.31.6 — 한 셀로**: 이 셀 하나가 두 앱의 요청을 모두 받습니다 (31번 셀은 실행하지 않아도 됨).
- 재처리 요청은 뒤 스레드가 받아 GPU 줄·입출력 줄에 나눠 주고, LC 요청(백업·export·track·자동 라벨·정리·비교)은 그 셀을 이 셀 안에서 차례로 실행합니다. LC 셀이 오래 걸려도 폰 요청은 바로 받습니다.
- **공정 차례**: 가벼운 일 먼저 → **사람마다 돌아가며 한 건씩**(각자 가장 오래된 것부터) → 같은 차례 안에서는 오래 기다린 것 먼저. 밀린 요청이 많은 사람이 다른 사람의 새 요청을 막지 않습니다. 다른 사람이 기다릴 때 한 사람은 무거운 일을 `W_USER_MAX`건까지만 동시에 차지합니다.
- **GPU**: LC의 track·자동 라벨·비교가 도는 동안 GPU 한 줄을 맡아 두고, 재처리 라벨링은 남은 줄(`RL_GPU_WORKERS`)로.
- 요청 확인은 `W_POLL_SEC`초마다 (기본 Colab 10초 · Kaggle 20초). 통신은 지금처럼 Drive 파일입니다.
- LC 요청은 `scripts/notebook_cells.json`의 셀 기본값으로 실행합니다. 노트북에서 5번 셀의 `TRACK_AUTO_UPLOAD`를 바꿔도 작업자에는 적용되지 않으니 `LC_SET = {'5': {'TRACK_AUTO_UPLOAD': True}}`처럼 적으세요.

이름이 `uld`·`AA`로 끝나는 TASK를 골라 모델로 다시 라벨링하는 안드로이드용 웹 페이지 '가지 재처리'의 요청을 이 셀이 받아 **차례로** 실행합니다. 셀을 실행해 두면(Colab·Kaggle) 폰에서 요청한 일이 자동으로 처리됩니다.
- 상태는 Drive `eggplant_yolo/relabel/`에만 씁니다 (`state.json`·`tracked.json`·TASK별 `tasks/<ID>.json`·확인 그림 `previews/`·요청 `requests/`·작업자 `WORKER.json`). **TASK 이름은 바꾸지 않습니다** — 이름이 uld·AA가 아니게 바뀐 TASK(검토로 진척)는 폰에서 줄을 긋고, 숨길 수 있습니다.
- **shape**: 10번 셀과 같은 순서 — 라벨 백업 → 기존 라벨이 있으면 교체 → 전·후 비교·상세 로그. 원본 영상이 CVAT 프레임과 맞으면 원본 화질.
- **track**: 5번 셀과 같은 track 생성 → '확인 대기'(폰에서 확인 그림·박스 수를 보고 **올리기** 또는 **버리기**). 영상이 없으면 TASK 백업을 요청해 받은 뒤 만들고, 백업이 늦으면 '백업 대기'로 두었다가 `RL_BACKUP_POLL_MIN`마다 다시 받음. 올리기는 6번 셀과 같은 순서(직전 백업 → 교체).
- 한 번에 한 작업자만: 다른 세션의 작업자가 켜져 있으면 멈춤(`RL_TAKEOVER = True`로 넘겨받기). 작업 기록·업로드 기록·작업 대장은 다른 셀과 같은 곳에 남습니다.
- 멈추기: 셀 중지(■) 또는 `RL_HOURS`가 지나면. 하던 요청은 '진행 중'으로 남으니, 다시 켜기 전에 폰에서 확인하세요.

**v4.29.0 — 두 줄로 동시에**
- **GPU 줄**(shape·track, 한 번에 하나) + **입출력 줄**(track 올리기·백업, `RL_IO_WORKERS`건까지 동시). 긴 shape를 하는 동안에도 올리기·백업 요청이 바로 처리됩니다.
- **백업은 요청만** 하고 요청을 '받는 중'으로 둡니다. `RL_BACKUP_POLL_MIN`(기본 1분)마다 CVAT에 확인해 준비된 것을 받습니다. track을 만들려다 영상이 없던 TASK는 백업을 받은 뒤 GPU 줄이 track을 마저 만듭니다. `RL_BACKUP_GIVEUP_H`시간 안에 준비되지 않으면 포기(실패)합니다.
- **같은 TASK의 요청은 보낸 차례대로** 하나씩: 앞 요청이 끝나야 다음 요청이 시작됩니다 (예: track 만들기 → 올리기).
- **지난 track 정리**: track을 만든 뒤 shape로 CVAT 라벨이 바뀌면 그 track의 '확인 대기'를 자동으로 정리합니다 (올리면 shape 라벨을 덮어쓰므로). TASK 이력에 '지난 track 정리'로 남고, track 파일은 지우지 않습니다. 셀을 켤 때 이미 그런 TASK도 정리합니다.
- **멈추기(■)**: 새 요청은 받지 않고, 하던 TASK까지 끝낸 뒤 멈춥니다. 남은 TASK는 요청에 '대기'로 남아 다음에 이어 받습니다. 기다리는 중에 ■를 또 누르면 바로 멈추지만, 하던 일은 뒤에서 마저 끝납니다. 이때 셀을 다시 켜면 그 일이 끝난 뒤 시작합니다.
- 줄마다 CVAT에 따로 접속하고, 작업 기록(`logs/activity.csv`)도 줄마다 따로 묶입니다.

**v4.29.3 — 라벨링도 동시에**
- shape·track 라벨링을 **TASK 단위로 `RL_GPU_WORKERS`개(기본 2)까지 동시에** 합니다. 한 요청에 TASK가 여럿이면 그 TASK들을 나눠 하고, 다른 요청의 TASK와도 함께 합니다.
- GPU가 둘 이상(Kaggle T4×2)이면 줄마다 다른 GPU를 씁니다. GPU가 하나면 같은 GPU를 나눠 씁니다.
- `CUDA out of memory`가 나거나 오히려 느려지면 `RL_GPU_WORKERS = 1`로 줄이세요. Colab 무료는 CPU가 2개라 track(영상 읽기)을 둘 이상 함께 돌리면 이득이 작습니다.
- ■로 멈추면 이미 시작한 TASK까지만 끝내고, 시작하지 않은 TASK는 요청에 '대기'로 남습니다.


In [ ]:
# 30. 작업자 — 두 앱의 요청을 한 셀에서 공정하게 처리 (v4.31.6: 예전 30번 재처리 + 31번 LC 작업자를 합침 · v4.31.7: 학습 요청 · GPU 워커)
#     · AA앱(가지 재처리): uld·AA TASK의 shape·track 라벨링 · track 올리기 · 백업 → 뒤에서 여러 줄로 동시에 (GPU 줄 · 입출력 줄)
#     · LC앱 '가지 라벨링 관리': 백업 · export · track 생성 · 자동 라벨 · 모델 정리 · 모델 비교 · 학습 → 그 셀(5·9·10·13·14·15·29·32번)을 차례로 실행
#     · v4.31.7: W_ROLE = 'gpu' = Kaggle GPU 워커 (CPU 마스터가 시작) — kaggle_gpu로 배정된 GPU 요청만 받고, 일이 없으면 스스로 끝나 GPU를 돌려줌
#     · 공정 차례: 가벼운 일(새로 고침 등) 먼저 → 사람마다 돌아가며 한 건씩(각자 가장 오래된 것부터) → 같은 차례 안에서는 오래 기다린 것 먼저.
#       밀린 요청이 많은 사람이 다른 사람의 새 요청을 막지 않음 · 한 사람이 무거운 일을 W_USER_MAX건 넘게 동시에 차지하지 않음 (다른 사람이 기다릴 때)
#     · GPU: LC의 track·자동 라벨·비교가 실행되는 동안 GPU 한 줄을 맡아 두고(학습은 모든 줄), 재처리 라벨링은 남은 줄로
#     · 요청 확인은 Drive를 W_POLL_SEC초마다 (일이 끝나면 바로) · 같은 TASK의 요청은 보낸 차례대로 하나씩
#     · 멈추려면 셀 중지(■) — 새 요청은 받지 않고, 하던 TASK까지 끝낸 뒤 멈춤 (남은 TASK는 요청에 '대기'로 남아 다음에 이어 받음)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
W_LOGIN      = None     # CVAT 계정 (0번 셀 CVAT_PROFILES의 이름) · None = 지금 그대로
W_HOURS      = 8        # 이 시간 동안 켜 둠
W_POLL_SEC   = None     # 새 요청 확인 간격(초) · None = Colab 10초 · Kaggle 20초 (Kaggle은 매번 rclone으로 받음)
W_USER_MAX   = 2        # 한 사람이 동시에 차지하는 무거운 일(라벨링·올리기·백업) 수 — 다른 사람이 기다릴 때만 · 0 = 제한 없음
W_STATUS_MIN = 10       # 요청이 없을 때 이 간격(분)마다 '대기 중 · 열린 요청 n건' 한 줄
W_TAKEOVER   = False    # True = 다른 세션의 작업자가 켜져 있어도 넘겨받음
W_MAX_LOOPS  = None     # (시험용) 이만큼 돌고 멈춤
W_ROLE       = 'all'    # 'all' = 두 앱의 요청 모두 (Colab·Kaggle에서 사람이 켬) · 'gpu' = Kaggle GPU 워커 (CPU 마스터가 시작, kaggle_gpu 배정만)
W_EXIT_IDLE_MIN = None  # 할 일이 없는 채로 이 시간(분)이 지나면 끝냄 · None = W_HOURS까지 켜 둠 ('gpu'이면 5분)
# ---- 재처리 (AA앱(가지 재처리))
RL_REFRESH_MIN = 15       # CVAT TASK 목록을 새로 읽는 간격 (분) — 폰의 '새로 고침'은 바로
RL_GPU_WORKERS = 2        # shape·track 라벨링을 동시에 몇 TASK까지 (LC의 GPU 일도 이 수 안에서) · CUDA out of memory면 줄이세요
RL_IO_WORKERS  = 2        # 올리기·백업을 동시에 몇 건까지 · CVAT 서버가 느려지면 줄이세요
RL_BACKUP_POLL_MIN = 1    # '받는 중' 백업을 CVAT에 확인하는 간격 (분)
RL_BACKUP_GIVEUP_H = 6    # 이 시간이 지나도 준비되지 않은 백업은 포기 (그 TASK는 실패로 → 다시 요청)
RL_REPLACE_ANY = False    # v4.31.9 (F5): True = 이름이 uld·AA가 아닌 TASK(사람이 검토해 _vf 등으로 바꾼 것)도 기존 라벨을 교체 · False = 보류 (권장)
# ---- LC (LC앱 '가지 라벨링 관리')
LC_KINDS        = ['cleanup', 'backup', 'export', 'aa', 'track', 'compare', 'train']   # 받을 종류 (v4.31.7: 학습도 요청으로 — 15번 셀) · 차례는 위 공정 차례
LC_GPU_KINDS    = ['track', 'aa', 'compare', 'train']   # GPU를 쓰는 LC 일 (실행 중 GPU 한 줄을 맡아 둠 · 학습은 모든 줄이 빈 뒤 시작)
LC_DOWNLOAD_MIN = 10      # '받는 중' 백업·export: CVAT가 준비한 것을 이 간격(분)마다 받음
LC_STATE_MIN    = 30      # LC앱 상태(web/state.json)를 28번 셀로 새로 만드는 간격(분) — 요청을 처리한 뒤에도 새로 만듦
LC_STALE_TAKEN_H = 6      # 다른 세션이 '진행 중'으로 둔 요청이 이 시간(시간)이 지나면 끊긴 것으로 보고 다시 대기로
# 각 LC 요청은 그 셀의 코드를 그대로(FROM_REQUESTS = True) 실행 — 셀 기본설정을 따름. 예: 자동 라벨은 9번 셀 AA_EXISTING = 'skip',
# track은 5번 셀 TRACK_AUTO_UPLOAD = False (노트북에서 셀을 고쳐도 작업자는 scripts/notebook_cells.json의 기본값으로 실행).
# 바꾸려면 아래 LC_SET에 '셀 번호': {'설정 이름': 값}을 적음.
LC_SET          = {}      # 예: {'5': {'TRACK_AUTO_UPLOAD': True}, '9': {'AA_EXISTING': 'replace'}, '13': {'BK_MAX_WAIT': 900}}

if W_POLL_SEC is None:
    W_POLL_SEC = 20 if ON_KAGGLE else 10
RL_LOGIN = LC_LOGIN = W_LOGIN
RL_HOURS = LC_HOURS = W_HOURS
RL_POLL_SEC = LC_POLL_SEC = W_POLL_SEC
RL_STATUS_MIN = LC_STATUS_MIN = W_STATUS_MIN
RL_TAKEOVER = LC_TAKEOVER = W_TAKEOVER
RL_MAX_LOOPS = LC_MAX_LOOPS = W_MAX_LOOPS
RL_STALE_TAKEN_H = LC_STALE_TAKEN_H
W_SLOT = '_gpu' if W_ROLE == 'gpu' else ''        # v4.31.7: 작업자 기록 파일 (GPU 워커는 web/worker_gpu.json · relabel/WORKER_gpu.json)
REQ_HOST = 'kaggle_gpu' if W_ROLE == 'gpu' else None   # 0번 셀 req_use(셀이 요청을 받을 때)의 배정 확인
if W_ROLE == 'gpu':
    LC_KINDS = [k for k in LC_KINDS if k in LC_GPU_KINDS]
    if W_EXIT_IDLE_MIN is None:
        W_EXIT_IDLE_MIN = 5
assert wb is not None and hasattr(wb, 'uld_scan'), 'scripts/를 v4.31.9로 바꾸세요'
from collections import Counter
_W = {'stop': False, 'lc_gpu': 0, 'aa_err': None, 'aa_done': False, 'served': [], 'idle_since': None}

def _w_mine(rid, host):
    # 이 작업자가 받을 요청인가 (배정 requests/assign) · GPU 워커는 배정 파일이 있는 것만 (사람이 켠 30번 셀 몫을 가져가지 않게)
    if W_ROLE == 'gpu' and not os.path.exists(f'{BASE}/requests/assign/ASSIGN_{rid}.json'):
        return False
    return wb._assigned_to_me(BASE, rid, host)
W_FAIR_MIN = 60          # 돌아가며: 최근 이 시간(분) 동안 차례를 받은 횟수만큼 그 사람을 뒤로

def _w_served():
    # 최근 W_FAIR_MIN분 동안 사람마다 맡긴 무거운 일 수 (재처리 + LC)
    t = _time.time() - W_FAIR_MIN * 60
    _W['served'] = [(x, u) for x, u in _W['served'] if x >= t]
    return Counter(u for _, u in _W['served'])

def _w_served_add(u):
    _W['served'].append((_time.time(), u))

# =============== 재처리 (예전 30번 셀) ===============
import glob, json, threading, time as _time, traceback, importlib
from concurrent.futures import ThreadPoolExecutor, wait as _fwait
import torch
import relabel_bridge as rb
importlib.reload(rb)

_old = globals().get('_RL_POOLS')                # 앞 실행을 ■로 멈췄을 때 아직 끝나지 않은 일 → 끝날 때까지 기다림 (같은 요청을 두 번 하지 않게)
if _old:
    _left = [f for f in list(_old['futs']) if not f.done()]
    if _left:
        print(f"⏳ 앞 실행에서 하던 일 {len(_left)}건이 끝나기를 기다립니다 …")
        _fwait(_left)
    for _p in ('gpu', 'io', 'task'):
        if _p in _old:
            _old[_p].shutdown(wait=False)
use_cvat(RL_LOGIN)
_RL = {'stop': False, 'heart_stop': False, 'started': ctt.now(), 'host': 'kaggle_gpu' if W_ROLE == 'gpu' else 'Kaggle' if ON_KAGGLE else 'Colab', 'refresh_t': 0,
       'bk_t': 0, 'state_t': 0, 'dirty': False, 'gpu': {}, 'io': {}, 'tasks': {}, 'ready': {}}
_RL_LOCK = threading.RLock()                    # 요청 파일·작업 대장·Drive 동기화 (줄끼리 겹치지 않게)
_RL_BK_LOCK = threading.Lock()                  # 백업 스크립트는 한 번에 하나 (설정·대기 목록을 같이 씀)
_RL_EV = threading.Event()                      # 일이 끝나면 바로 다음 요청을 보게
_RL_TL = threading.local()
_RL_POOLS = {'gpu': ThreadPoolExecutor(max(1, int(RL_GPU_WORKERS)), thread_name_prefix='rl-gpu'),   # 요청을 맡아 TASK를 나눠 주는 쪽
             'task': ThreadPoolExecutor(max(1, int(RL_GPU_WORKERS)), thread_name_prefix='rl-gput'),  # 라벨링 TASK를 실제로 하는 줄 (v4.29.3)
             'io': ThreadPoolExecutor(max(1, int(RL_IO_WORKERS)), thread_name_prefix='rl-io'), 'futs': {}}
_RL_FUNC = f'{SCRIPTS}/cvat_yolo_function.py'
_RL_NGPU = torch.cuda.device_count() if torch.cuda.is_available() else 0

def _rl_dev():
    # 이 줄이 쓸 GPU — GPU가 여럿(Kaggle T4×2)이면 줄마다 나눠 씀, 없으면 CPU (v4.29.3)
    if not _RL_NGPU:
        return 'cpu'
    try:
        i = int(threading.current_thread().name.rsplit('_', 1)[-1])
    except ValueError:
        i = 0
    return str(i % _RL_NGPU)
_RL_GPU, _RL_IO, _RL_LIGHT = ('shape', 'track'), ('upload', 'backup'), ('refresh', 'hide', 'unhide')
_RL_OPEN_BK = ('요청됨', '준비 중')
rb.ensure(BASE)
if ON_KAGGLE:
    drive_pull('relabel/**', quiet=True)
_RL_PREV = rb.acquire_worker(BASE, who_am_i(), SESSION_ID, takeover=RL_TAKEOVER, slot=W_SLOT) or {}   # 앞 작업자 (v4.31.4: 그 세션이 남긴 '진행 중'만 되돌림)

def _rl_push(*rels):
    # Kaggle: 바뀐 것을 Drive로 (Colab은 Drive에 바로 씀)
    if ON_KAGGLE:
        drive_push(*(rels or ('relabel/**',)), quiet=True)

def _rl_cl():
    # 줄(스레드)마다 따로 CVAT 접속 — 동시에 써도 섞이지 않게 (주 스레드는 CLIENT)
    if threading.current_thread() is threading.main_thread():
        return CLIENT
    if getattr(_RL_TL, 'cl', None) is None:
        _RL_TL.cl = cvat_client(CVAT_LOGIN_MODE)
    return _RL_TL.cl

def _rl_reg(**row):
    with _RL_LOCK:
        ctt.append_registry(REGISTRY, **row)

def _rl_busy():
    # 폰에 보일 '지금 하는 일': GPU 줄 + 입출력 줄
    g, io = [dict(v) for v in list(_RL['gpu'].values())], [dict(v) for v in list(_RL['io'].values())]
    top = dict(g[0]) if g else dict(io[0]) if io else {'what': '대기 중'}
    if _RL.get('closing'):                     # ■ 뒤 하던 일만 마무리 — 새 요청은 받지 않음 (v4.29.4)
        top = {'what': f"멈추는 중 · 하던 일 {len(g) + len(io)}건 마무리", 'closing': True}
    top['lanes'] = {'gpu': g[0] if g else None, 'gpus': g, 'io': io}
    top['loop_at'] = _RL.get('loop_at')          # 요청 확인 루프가 마지막으로 돈 시각 (v4.29.4)
    return top

def _rl_beat(what=None, stopping=False):
    b = {'what': what} if what else _rl_busy()
    rb.worker_beat(BASE, who_am_i(), SESSION_ID, _RL['host'], busy=b, started=_RL['started'], stopping=stopping, slot=W_SLOT)
    _rl_push(f'relabel/WORKER{W_SLOT}.json')

def _rl_lane(info):
    # 이 줄이 지금 하는 일 (None = 끝)
    me = threading.current_thread().name
    lane = _RL['gpu'] if me.startswith('rl-gpu') else _RL['io']
    if info is None:
        lane.pop(me, None)
    else:
        lane[me] = info
    try:
        _rl_beat()
    except Exception:
        pass

def _rl_heart():
    while not _RL['heart_stop']:                 # 오래 걸리는 TASK 중에도 폰에 '켜짐'으로 보이게
        try:
            _rl_beat()
        except Exception:
            pass
        for _ in range(60):
            if _RL['heart_stop']:
                return
            _time.sleep(1)

def _rl_status(req, status, **f):
    with _RL_LOCK:                               # 이름 바꾸기 → Drive에 올리기 → 예전 이름 지우기를 한 번에 (요청 받아 오기와 겹치지 않게)
        old = req.get('_path')
        new = rb.set_status(BASE, req, status, by=who_am_i(), **f)
        if ON_KAGGLE:                            # Drive에 남은 예전 상태 파일 지움 (폰이 같은 요청을 둘로 보지 않게)
            drive_push('relabel/requests/' + os.path.basename(new['_path']), quiet=True)
            if old and os.path.basename(old) != os.path.basename(new['_path']):
                _kg_rc(['deletefile', _kg_remote('relabel/requests/' + os.path.basename(old))], quiet=True)
        return new

def _rl_req(rid):
    # 요청 파일을 다시 읽음 (그 사이 상태가 바뀌었을 수 있음)
    return next((r for r in rb.list_requests(BASE) if r['id'] == rid), None)

def _rl_model(name):
    if not name:
        raise RuntimeError('요청에 모델이 없습니다')
    p = f'{BASE}/runs/{name}/weights/best.pt'
    if ON_KAGGLE and not os.path.exists(p):
        drive_pull(f'runs/{name}/weights/best.pt', f'runs/{name}/*', quiet=True)
    if not os.path.exists(p):
        raise RuntimeError(f'모델 없음: runs/{name}/weights/best.pt')
    import datetime as _dt
    return p, f"{name}/best.pt ({_dt.datetime.fromtimestamp(os.path.getmtime(p)).strftime('%m-%d %H:%M')})"

def _rl_media():
    if ON_KAGGLE:
        _KG['ls'].pop('videos', None); _KG['ls'].pop('backups', None)
        return wb.media_names(BASE, remote_videos=[p for p, s, t in _kg_files('videos')],
                              remote_backups=[p for p, s, t in _kg_files('backups')])
    return wb.media_names(BASE, videos_dir=VIDEOS)

def _rl_refresh(force=False):
    if not force and _time.time() - _RL['refresh_t'] < RL_REFRESH_MIN * 60:
        return
    tasks = ctt.list_tasks(_rl_cl(), limit=5000)          # v4.31.6: 뒤 스레드에서도 (주 스레드는 LC 셀이 CLIENT를 씀)
    if ON_KAGGLE:
        drive_pull('status/cvat_tasks.json', quiet=True)   # v4.31.8: 사라진 TASK (수집기 기록)
    tr = rb.update_tracked(BASE, tasks)
    _RL['media'] = _rl_media()
    _RL['models'], _RL['models_t'] = rb.model_rows(BASE, files=kg_file_index('runs') if ON_KAGGLE else None), _time.time()   # v4.31.6: Kaggle은 Drive 목록으로
    st = rb.build_state(BASE, tr, models=_RL['models'], media=_RL['media'], uld=uld_index(), replaced=uld_lineage(), server=CVAT_SERVER, scope_text=ctt.scope_text(CVAT_SCOPE),
                        defaults={'conf': AA_CONF, 'imgsz': IMGSZ, 'model': globals().get('NAME', '')})
    _RL['refresh_t'] = _RL['state_t'] = _time.time()
    _RL['dirty'] = False
    _rl_push('relabel/**')
    s = st['summary']
    print(f"[{ctt.now()[11:16]}] 목록: 대상 {s['targets']} · 진척 {s['progressed']} · 숨김 {s['hidden']} · 확인 대기 track {s['pending_track']} · 모델 {len(st['models'])}개")

def _rl_state(force=False):
    # 폰 화면 상태 다시 쓰기 — 줄들이 바꾼 것을 모아 몇 초에 한 번 (주 스레드만)
    if not (force or _RL['dirty']) or (not force and _time.time() - _RL['state_t'] < 5):
        return
    _RL['dirty'] = False
    if _RL.get('models') is not None and (_RL.get('closing') or _time.time() - _RL.get('models_t', 0) < RL_REFRESH_MIN * 60):
        _models = _RL['models']                    # v4.31.6: 모델 목록은 목록 새로 고침(RL_REFRESH_MIN) 때만 새로 — 상태를 쓸 때마다 만들지 않음
    else:
        _models = _RL['models'] = rb.model_rows(BASE, files=kg_file_index('runs') if ON_KAGGLE else None)
        _RL['models_t'] = _time.time()
    rb.build_state(BASE, models=_models, media=_RL.get('media') or _rl_media(), uld=uld_index(), replaced=uld_lineage(), server=CVAT_SERVER, scope_text=ctt.scope_text(CVAT_SCOPE),
                   defaults={'conf': AA_CONF, 'imgsz': IMGSZ, 'model': globals().get('NAME', '')})
    _RL['state_t'] = _time.time()
    _rl_push('relabel/**')

def _rl_video(task, name, source=''):
    v = ctt.find_video(VIDEOS, source, name)
    if v is None:
        v = ctt.video_from_backups(f'{BASE}/backups', name, VIDEOS)[0]
    return v

def _rl_backup(ids, mode):
    # 13번 셀과 같은 백업 (TASK 영상 + 라벨) · mode 'request' = 요청만 / 'download' = 준비된 것 받기 (한 번 확인) — 기다리지 않음
    with _RL_BK_LOCK:
        cb = load_script('cvat_backup.py')
        prof = CVAT_PROFILES[CVAT_LOGIN_MODE]
        cb.HOST, cb.AUTH_MODE, cb.BASE = CVAT_HOST, prof_login(CVAT_LOGIN_MODE), Path(BASE)
        cb.CREDENTIALS = cvat_auth(CVAT_LOGIN_MODE)[1]
        cb.ORGANIZATION_SLUG, cb.PROJECT_ID = CVAT_SCOPE.get('org'), CVAT_SCOPE.get('project_id')
        cb.NAME_QUERY, cb.TASK_IDS, cb.PREVIEW = '', list(ids), False
        cb.FORCE, cb.UPDATE_BOARD, cb.BOARD_MODE, cb.ONLY_STATES = False, True, 'file', None
        cb.MODE, cb.MAX_WAIT = mode, 0
        st = cb.run_backup() or []
        _RL['media'] = _rl_media()               # 받은 백업 → 영상을 얻을 곳
        _rl_push('backups/**', 'backup_index.csv', 'backup_status.csv', 'reports/**')
    want = {str(t) for t in ids}                  # 백업 스크립트는 작업보드용으로 모든 TASK의 백업 상태를 돌려줌 → 요청한 TASK만 (v4.27)
    return {str(r.get('task_id')): r.get('backup_state', '') for r in st if str(r.get('task_id')) in want}

def _rl_bk_left(ids, st):
    # 아직 받지 못한 TASK: 이 서버의 백업 요청이 열려 있음(요청됨·준비 중)
    op = {str(r.get('task_id')) for r in wb.pending_rows(BASE, 'backup')
          if r.get('status') in _RL_OPEN_BK and r.get('server', '') in ('', CVAT_SERVER)}
    return [t for t in ids if str(t) in op or st.get(str(t)) in _RL_OPEN_BK]

def _rl_clear_stale(tid, req=None):
    # shape로 CVAT 라벨이 바뀌면 그 전에 만든 track(확인 대기)은 지난 것 → 올리지 않게 정리 (파일은 남음) — v4.29
    rec = rb.task_record(BASE, tid)
    p = rec.get('track_pending')
    if not p:
        return False
    rb.set_pending_track(BASE, tid, None)
    rb.add_history(BASE, tid, kind='discard', status='지난 track 정리', request=(req or {}).get('id', ''), who=who_am_i(),
                   note=f"shape로 라벨이 바뀌어 {str(p.get('made', ''))[:16]}에 만든 track({p.get('tracks')}개)은 올리지 않음 · 파일 {p.get('xml')}")
    print(f'   [{tid}] 지난 track 확인 대기 정리 (shape로 라벨이 바뀜)')
    return True

def _rl_shape_one(tid, mpath, mlabel, conf, req):
    # 10번 셀과 같은 순서: 백업 → (기존 라벨 있으면 교체) 자동 라벨 → 전·후 비교 → 상세 로그 · 이름은 바꾸지 않음
    folder = task_dir(tid); os.makedirs(folder, exist_ok=True)
    ts = ctt.stamp()
    row = dict(run_time=ctt.now(), action='auto', task_id=tid, model=mlabel, server=CVAT_SERVER)
    L, err, res = None, None, {}
    try:
        task = _rl_cl().tasks.retrieve(tid)
        row['task_name'] = task.name
        vid, off = None, (TRACK_FRAME_OFFSET if globals().get('AA_VIDEO_OFFSET') is None else AA_VIDEO_OFFSET)
        if ctt.task_basic(task)['is_video']:
            v = _rl_video(task, task.name)
            try:                                 # 9번 셀과 같은 확인: 영상과 CVAT 프레임이 같은 장면이면 원본 화질로, 아니면 CVAT 프레임
                if v is not None and ctt.check_video_alignment(task, v, off, 'pyav')['ok']:
                    vid = v
            except Exception as _e:
                print(f'   [{tid}] (영상 맞춤 확인 실패 → CVAT 프레임: {ctt.api_error(_e)})')
        L = upload_log(tid, 'shape', model_path=mpath, echo=lambda m: print(f'   [{tid}] {m}'), task_name=task.name,
                       source=(f'원본 영상 {os.path.relpath(str(vid), BASE)} (pyav, 보정 {off:+d})' if vid else 'CVAT 프레임 (original)'),
                       settings={'imgsz': IMGSZ, 'conf': conf, 'existing': 'replace', 'app': 'relabel', 'request': req['id']})
        backup = ctt.export_annotations(task, f'{folder}/before_auto_{ts}.zip')
        row['backup_file'] = os.path.relpath(backup, BASE)
        _replace = ctt.annotation_count(task) > 0
        if _replace and not (rb.is_target(task.name) or RL_REPLACE_ANY):   # v4.31.9 (F5): 요청 뒤 사람이 검토해 이름이 바뀐 TASK는 교체하지 않음
            row.update(status='보류', note=f"이름이 {task.name} (uld·AA 아님) — 사람 라벨일 수 있어 교체하지 않음 · 백업 {os.path.basename(backup)} (RL_REPLACE_ANY 또는 10번 셀)")
            L.set(mode='보류'); L.say(row['note'])
            return row
        L.counts(before_zip=backup)
        L.set(backup_file=row['backup_file'], mode='replace' if _replace else 'new')
        L.say(f"백업 {os.path.basename(backup)} · 기존 라벨 박스 {L.rec.get('before_total', 0)}개")
        r = ctt.auto_annotate_frames(task, _RL_FUNC, mpath, imgsz=IMGSZ, device=_rl_dev(), conf=conf, tile=False, quality='original',
                                     frames=None, log=L.say, prefetch=4, replace=_replace, backup=backup,
                                     video=vid, video_offset=off, decoder='pyav')
        res = {'boxes': r['boxes'], 'per_label': r['per_label'], 'frames': r['frames'], 'replaced': r.get('replaced')}
        row.update(status='완료(교체)' if r.get('replaced') is not None else '완료',
                   note=f"재처리 앱 · 박스 {r['boxes']}개 {r['per_label']} · 프레임 {r['frames']}장")
        L.set(uploaded=f"박스 {r['boxes']}개 {r['per_label']}", frames_done=r['frames'], quality_used=r['quality'])
        _lu = label_uploaded(task, replaced=r.get('replaced') is not None, before_zip=backup, say=L.say)   # v4.31.5 이름 규칙 (앱 작업자도 같음)
        if _lu['note']:
            row['note'] += ' · ' + _lu['note']
        if _lu['renamed']:
            row['task_name'] = _lu['name']; L.set(task_name_after=_lu['name'])
        try:
            _ch = change_report(task, backup, 'after_auto', show=False)
            row['note'] += f" · {_ch['text']}"; res['change'] = _ch['text']
            L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
        except Exception as _e:
            L.say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
    except Exception as e:
        err = e
        row.update(status='오류', note=ctt.api_error(e)[:300])
        print(f'   [{tid}] ❌ {row["note"]}')
    finally:
        _rl_reg(**row)
        if L is not None:
            try:
                L.finish(row.get('status', '?'), err)
            except Exception:
                pass
        rb.add_history(BASE, tid, kind='shape', model=mlabel, conf=conf, status=row.get('status'), note=row.get('note', ''),
                       backup_file=row.get('backup_file', ''), request=req['id'], who=who_am_i(), **res)
        if str(row.get('status', '')).startswith('완료'):
            _rl_clear_stale(tid, req)
        _rl_push(f'tasks/{tid}/**', 'task_registry.csv', 'logs/**', f'relabel/tasks/{tid}.json')
    return row

def _rl_track_one(tid, mpath, mlabel, conf, req):
    # 5번 셀과 같은 track 생성 (올리지 않음) → 확인 대기 + 폰에서 볼 그림
    folder = task_dir(tid); os.makedirs(folder, exist_ok=True)
    ts = ctt.stamp()
    row = dict(run_time=ctt.now(), action='track', task_id=tid, model=mlabel, server=CVAT_SERVER)
    pend = None
    try:
        task = _rl_cl().tasks.retrieve(tid)
        info = ctt.task_basic(task)
        row.update(task_name=info['task_name'], media=info['media'], frames=info['frames'])
        _uld, _fm = None, None
        if not info['is_video']:
            _uld = uld_task(tid)                           # v4.31.5: ULD 사진 TASK는 clip으로 track
            _why = ('' if uld_is_images(_uld) else 'ULD 기록 없음') or uld_mismatch(task, _uld)
            if _why:
                row.update(status='건너뜀', note=f'이미지 TASK → track 불가 ({_why} · shape를 쓰세요)')
                return row
            _fm = [int(x) for x in _uld['frame_map']]
        backup = ctt.export_annotations(task, f'{folder}/backup_{ts}.zip')
        meta = ctt.read_export(backup)
        row.update(step=meta['step'], existing_labels=meta['existing_boxes'], backup_file=os.path.relpath(backup, BASE))
        if _fm:
            _clip = uld_clip(_uld)
            if _clip is None:
                row.update(status='오류', note='ULD clip을 받지 못함 (CVAT_UpLoader/clips)')
                return row
            video = Path(_clip)
        else:
            video = _rl_video(task, info['task_name'], meta['source'])
        _uvid = False
        if video is None and not _fm:                  # v4.31.8: ULD가 영상으로 올린 TASK → 원본은 ULD clip
            _uc = uld_video(tid)
            if _uc:
                video, _uvid = Path(_uc), True
        if video is None:
            row.update(status='영상 없음', note='백업(영상)이 아직 없음')
            return row
        row['video'] = video.name
        out_dir = f'{folder}/track_{ts}'
        log = f'{folder}/run_{ts}_track.log'
        rc = run([sys.executable, f'{SCRIPTS}/yolo_track_to_cvat.py', '--model', mpath, '--video', video,
                  '--reference', backup, '--imgsz', IMGSZ, '--conf', conf, '--fill-gaps', TRACK_FILL_GAPS,
                  '--track-buffer', TRACK_BUFFER, '--decoder', 'pyav' if (_fm or _uvid) else TRACK_DECODER,
                  '--frame-offset', 0 if (_fm or _uvid) else TRACK_FRAME_OFFSET, *(['--frame-map', ','.join(map(str, _fm))] if _fm else []),
                  '--box-source', TRACK_BOX_SOURCE, '--near', TRACK_NEAR, '--device', _rl_dev(), '--out', out_dir], log=log)
        if rc != 0:
            raise RuntimeError(f'track 생성 실패 (로그 {os.path.relpath(log, BASE)})')
        _mt, _md5 = ctt.file_fingerprint(mpath)
        json.dump({'made': ctt.now(), 'operator': who_am_i(), 'session': SESSION_ID, 'server': CVAT_SERVER,
                   'model': os.path.relpath(mpath, BASE), 'model_time': _mt, 'model_md5': _md5,
                   'video': os.path.relpath(str(video), BASE), 'reference_backup': os.path.relpath(backup, BASE), 'app_version': APP_VERSION,
                   'settings': {'imgsz': IMGSZ, 'conf': conf, 'fill_gaps': TRACK_FILL_GAPS, 'track_buffer': TRACK_BUFFER,
                                'decoder': 'pyav' if (_fm or _uvid) else TRACK_DECODER, 'frame_offset': 0 if (_fm or _uvid) else TRACK_FRAME_OFFSET,   # v4.31.9 (F20)
                                'box_source': TRACK_BOX_SOURCE,
                                'near': TRACK_NEAR, 'app': 'relabel'}},
                  open(f'{out_dir}/track_settings.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        xml_path = f'{out_dir}/tracks_{video.stem}.xml'
        summ = rb.track_summary(xml_path)
        prev = f'{BASE}/relabel/previews/{tid}_{ts}.jpg'
        try:
            _vg = ctt.video_frame_getter(video)
            pth, _fr = rb.preview_image(xml_path, (lambda k: _vg(_fm[k]) if 0 <= k < len(_fm) else None) if _fm else _vg, prev,
                                        offset=0 if (_fm or _uvid) else TRACK_FRAME_OFFSET)
        except Exception as _e:
            pth = None
            print(f'   [{tid}] (확인 그림을 만들지 못함: {_e})')
        row.update(tracks=summ['tracks'], result_file=os.path.relpath(xml_path, BASE),
                   status='완료 · 확인 대기' if summ['tracks'] else '완료 · track 없음',
                   note=f"재처리 앱 · track {summ['tracks']}개 · 박스 {summ['boxes']}개 {summ['by_label']}")
        if _fm or _uvid:
            uld_clip_done(video)
        if summ['tracks']:
            pend = {'made': ctt.now(), 'model': mlabel, 'model_path': os.path.relpath(mpath, BASE), 'conf': conf,
                    'xml': os.path.relpath(xml_path, BASE), 'video': os.path.relpath(str(video), BASE),
                    'preview': os.path.relpath(pth, BASE) if pth else None, 'request': req['id'], **summ}
    except Exception as e:
        row.update(status='오류', note=ctt.api_error(e)[:300])
        print(f'   [{tid}] ❌ {row["note"]}')
    finally:
        _rl_reg(**row)
        if pend:
            rb.set_pending_track(BASE, tid, pend)
        rb.add_history(BASE, tid, kind='track', model=mlabel, conf=conf, status=row.get('status'), note=row.get('note', ''),
                       request=req['id'], who=who_am_i(), tracks=row.get('tracks'))
        _rl_push(f'tasks/{tid}/**', 'task_registry.csv', 'logs/**', f'relabel/tasks/{tid}.json', 'relabel/previews/**')
    return row

def _rl_upload_one(tid, req):
    # 6번 셀과 같은 업로드: 직전 백업 → 기존 라벨이 있으면 교체 → 전·후 비교 · 이름 규칙 (v4.31.5: …_uld → …_uld_AA)
    rec = rb.task_record(BASE, tid)
    pend = rec.get('track_pending')
    row = dict(run_time=ctt.now(), action='upload', task_id=tid, server=CVAT_SERVER)
    L, err = None, None
    if not pend:                                   # v4.28.2: 같은 track을 또 올리라는 요청(두 번 누름) → 오류가 아니라 '이미 올림'
        last = [h for h in (rec.get('history') or []) if h.get('kind') == 'upload']
        if last and str(last[-1].get('status', '')).startswith('업로드 완료'):
            row.update(status='이미 올림', note=f"{str(last[-1].get('time', ''))[:16]}에 올림 — 다시 올리지 않음")
            rb.add_history(BASE, tid, kind='upload', status='이미 올림', request=req['id'], who=who_am_i(), note=row['note'])
            print(f"   [업로드 {tid}] 이미 올린 track — 다시 올리지 않음")
            return row
    try:
        if not pend:
            raise RuntimeError('확인 대기 중인 track이 없습니다')
        row.update(model=pend.get('model', ''), tracks=pend.get('tracks', ''), result_file=pend['xml'])
        xml_path = f"{BASE}/{pend['xml']}"
        if ON_KAGGLE and not os.path.exists(xml_path):
            drive_pull(os.path.dirname(pend['xml']), quiet=True)
        ts_ = track_settings_of(pend['xml'])
        L = upload_log(tid, 'track', model_path=f"{BASE}/{pend['model_path']}" if pend.get('model_path') else None,
                       echo=lambda m: print(f'   [업로드 {tid}] {m}'), result_file=pend['xml'], source=pend.get('video', ''),
                       settings={**(ts_.get('settings') or {}), 'request': req['id']}, track_made=pend.get('made', ''))
        task = _rl_cl().tasks.retrieve(tid)
        row['task_name'] = task.name
        L.set(task_name=task.name)
        b = ctt.export_annotations(task, f'{task_dir(tid)}/before_upload_{ctt.stamp()}.zip')
        meta_ = ctt.read_export(b)
        row.update(existing_labels=meta_['existing_boxes'], backup_file=os.path.relpath(b, BASE))
        L.set(backup_file=row['backup_file'], mode='replace' if meta_['existing_boxes'] else 'new')
        L.counts(before_zip=b)
        if meta_['existing_boxes'] > 0 and not (rb.is_target(task.name) or RL_REPLACE_ANY):   # v4.31.9 (F5)
            row.update(status='보류', note=f"이름이 {task.name} (uld·AA 아님) — 사람 라벨일 수 있어 올리지 않음 · track은 확인 대기에 그대로 (6번 셀에서 ALLOW_REPLACE)")
            L.set(mode='보류'); L.say(row['note'])
            return row
        ctt.import_tracks(task, xml_path, replace=meta_['existing_boxes'] > 0, backup=b, log=L.say)
        row.update(status='업로드 완료', note=('기존 라벨 교체(백업 있음) · ' if meta_['existing_boxes'] else '') + '재처리 앱')
        L.set(uploaded=f"track {pend.get('tracks')}개")
        _lu = label_uploaded(task, replaced=meta_['existing_boxes'] > 0, before_zip=b, say=L.say)   # v4.31.5 이름 규칙
        if _lu['note']:
            row['note'] += ' · ' + _lu['note']
        if _lu['renamed']:
            row['task_name'] = _lu['name']; L.set(task_name_after=_lu['name'])
        try:
            _ch = change_report(task, b, 'after_upload', show=False)
            row['note'] += f" · {_ch['text']}"
            L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
        except Exception as _e:
            L.say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
        rb.set_pending_track(BASE, tid, None)
    except Exception as e:
        err = e
        row.update(status='오류', note=ctt.api_error(e)[:300])
        print(f'   [업로드 {tid}] ❌ {row["note"]}')
    finally:
        _rl_reg(**row)
        if L is not None:
            try:
                L.finish(row.get('status', '?'), err)
            except Exception:
                pass
        rb.add_history(BASE, tid, kind='upload', model=row.get('model', ''), status=row.get('status'), note=row.get('note', ''),
                       backup_file=row.get('backup_file', ''), request=req['id'], who=who_am_i())
        _rl_push(f'tasks/{tid}/**', 'task_registry.csv', 'logs/**', f'relabel/tasks/{tid}.json')
    return row

_RL_ACTION = {'shape': 'auto', 'track': 'track', 'upload': 'upload', 'backup': 'backup'}
_OK = ('완료', '업로드 완료', '이미 올림')

def _rl_ids(req, key='task_ids'):
    return [int(t) for t in (req.get(key) or []) if str(t).strip().isdigit()]

def _rl_short(note):
    # 요청에 남길 TASK 결과 한 줄 (라벨 변화 상세는 TASK 이력에)
    s = str(note or '').replace('재처리 앱 · ', '').replace('재처리 앱', '').strip(' ·')
    return s.split(' · 기존 객체')[0][:140]

def _rl_res(rows):
    # TASK별 결과 → 요청 파일의 results (폰 '요청' 탭이 TASK마다 무엇을 했는지 보여 줌) — v4.29.1
    return [{'task_id': r['task_id'], 'name': r.get('task_name', ''), 'status': r.get('status', ''), 'note': _rl_short(r.get('note'))} for r in rows]

def _rl_merge(req, new):
    # 앞서 남긴 TASK 결과에 새 결과를 덮어 합침 (백업 받은 뒤 track 등)
    old = {str(x.get('task_id')): x for x in (req.get('results') or [])}
    for x in new:
        old[str(x['task_id'])] = {**old.get(str(x['task_id']), {}), **x}
    return list(old.values())

def _rl_do(req):
    # 요청 하나 처리 → (상태, 요약, 추가 칸) · 상태 'pending' = 작업자가 멈춰 남은 TASK를 다시 대기로
    kind, p = req['kind'], dict(req.get('params') or {})
    ids = _rl_ids(req)
    if kind == 'refresh':
        _rl_refresh(force=True); return 'done', '목록을 새로 읽음', {}
    if kind in ('hide', 'unhide'):
        n = rb.set_hidden(BASE, ids, kind == 'hide'); return 'done', f'{n}개 {rb.KINDS[kind]}', {}
    if kind == 'discard':
        for t in ids:
            rb.set_pending_track(BASE, t, None)
            rb.add_history(BASE, t, kind='discard', status='버림', request=req['id'], who=who_am_i())
        return 'done', f'track {len(ids)}개 버림', {'results': [{'task_id': t, 'status': '버림'} for t in ids]}
    if not ids:
        return 'failed', 'TASK가 없습니다', {}
    job_start(30, _RL_ACTION[kind], {'request': req['id'], 'ids': ids, **p}, summary=f"재처리 앱 {rb.KINDS[kind]}")
    rows, extra, left = [], {}, []
    try:
        if kind == 'backup':                       # 요청만 → '받는 중' (받기는 받기 줄이 RL_BACKUP_POLL_MIN마다)
            _rl_lane({'what': '백업 요청', 'request': req['id'], 'tasks': ids, 'since': ctt.now()})
            st = _rl_backup(ids, 'request')
            wait_ = _rl_bk_left(ids, st)
            for t in ids:
                rb.add_history(BASE, t, kind='backup', status='요청함 · 받는 중' if t in wait_ else (st.get(str(t)) or '?'),
                               request=req['id'], who=who_am_i())
            summ = (f"CVAT에 요청함 · 받는 중 {len(wait_)}개" if wait_ else '') + \
                   (' · ' if wait_ and len(wait_) < len(ids) else '') + \
                   ', '.join(f'{k} {v}' for k, v in st.items() if int(k) not in wait_)
            job_end('끝', summ or '바뀐 것 없음')
            res = [{'task_id': t, 'status': '받는 중 (CVAT 준비 중)' if t in wait_ else (st.get(str(t)) or '?')} for t in ids]
            if wait_:
                return 'waiting', summ, {'remaining': wait_, 'waiting_t': _time.time(), 'results': res}
            return 'done', summ or '바뀐 것 없음', {'results': res}
        conf = float(p.get('conf') or AA_CONF)
        if kind in ('shape', 'track'):
            mpath, mlabel = _rl_model(p.get('model'))
        if kind in ('shape', 'track'):             # v4.29.3: TASK를 라벨링 줄들에 나눠 동시에 (다른 요청의 TASK와 함께 RL_GPU_WORKERS개까지)
            job = ctt.current_job()
            def _one(i, t):
                if _RL['stop']:                    # ■: 시작하지 않은 TASK는 다시 대기
                    return None
                ctt._ACT['job'] = job               # 이 줄의 작업 기록을 요청의 작업에 묶음
                try:
                    _rl_lane({'what': rb.KINDS[kind], 'request': req['id'], 'task': t, 'n': i, 'of': len(ids), 'since': ctt.now(), 'device': _rl_dev()})
                    print(f"[{ctt.now()[11:16]}] {rb.KINDS[kind]} {i}/{len(ids)} · TASK {t} · {threading.current_thread().name.replace('rl-gput_', '라벨링 줄 ')}"
                          + (f" (GPU {_rl_dev()})" if _RL_NGPU > 1 else ''))
                    r = (_rl_shape_one if kind == 'shape' else _rl_track_one)(t, mpath, mlabel, conf, req)
                    _RL['dirty'] = True
                    return r
                finally:
                    ctt._ACT['job'] = None
                    _rl_lane(None)
            futs = [_RL_POOLS['task'].submit(_one, i, t) for i, t in enumerate(ids, 1)]
            _fwait(futs)
            for t, f in zip(ids, futs):
                r = f.result()
                if r is None:
                    left.append(t)
                else:
                    rows.append(r)
        else:
            for i, t in enumerate(ids, 1):
                if _RL['stop']:                    # ■: 하던 TASK까지 끝냄 → 남은 TASK는 다시 대기
                    left = ids[i - 1:]
                    break
                _rl_lane({'what': rb.KINDS[kind], 'request': req['id'], 'task': t, 'n': i, 'of': len(ids), 'since': ctt.now()})
                print(f"[{ctt.now()[11:16]}] {rb.KINDS[kind]} {i}/{len(ids)} · TASK {t}")
                rows.append(_rl_upload_one(t, req))
                _RL['dirty'] = True
        if left:
            summ = f"작업자가 멈춰 다시 대기 — {len(rows)}/{len(ids)}개 끝남, 남은 {len(left)}개"
            job_end('멈춤', summ)
            return 'pending', summ, {'task_ids': left, 'done_ids': [r['task_id'] for r in rows], 'results': _rl_merge(req, _rl_res(rows))}
        if kind == 'track':                       # 영상(백업)이 없던 TASK: 백업을 요청만 하고 '받는 중' → 준비되면 GPU 줄이 마저 만듦
            need = [r['task_id'] for r in rows if r.get('status') == '영상 없음']
            if need and p.get('auto_backup', True):
                print(f'   영상이 없는 TASK {len(need)}개 → 백업 요청 (받는 동안 다른 요청을 처리, 준비되면 track을 마저 만듦)')
                _rl_lane({'what': '백업 요청 (track용)', 'request': req['id'], 'tasks': need, 'since': ctt.now()})
                st = _rl_backup(need, 'request')
                wait_ = _rl_bk_left(need, st)
                for t in [x for x in need if x not in wait_]:     # 이미 백업이 있었음 → 바로 다시
                    rows = [r for r in rows if r['task_id'] != t] + [_rl_track_one(t, mpath, mlabel, conf, req)]
                if wait_:
                    extra = {'remaining': wait_, 'waiting_t': _time.time()}
                    for r in rows:
                        if r['task_id'] in wait_:
                            r.update(status='백업 받는 중', note='영상이 없어 TASK 백업을 요청함 → 받으면 track을 만듦')
        ok = sum(1 for r in rows if str(r.get('status', '')).startswith(_OK))
        summ = f"{ok}/{len(ids)}개 완료" + (f" · 백업 받는 중 {len(extra['remaining'])}개" if extra.get('remaining') else '') \
               + ''.join(f" · {r['task_id']} {r.get('status')}" for r in rows if not str(r.get('status', '')).startswith(_OK)
                         and r['task_id'] not in (extra.get('remaining') or []))[:400]
        job_end('끝' if ok or extra else '실패', summ)
        extra['results'] = _rl_res(rows)
        return ('waiting' if extra.get('remaining') else 'done' if ok else 'failed'), summ, extra
    except Exception as e:
        job_end('실패', ctt.api_error(e)[:300])
        raise

def _rl_run(req):
    # 줄(스레드)에서 요청 하나 → 상태 기록
    try:
        st_, summ, extra = _rl_do(req)
    except Exception as e:
        st_, summ, extra = 'failed', ctt.api_error(e)[:400], {}
        traceback.print_exc()
    if st_ == 'pending':
        _rl_status(req, 'pending', result=summ, **extra)
    else:
        _rl_status(req, st_, result=summ, done_at=ctt.now() if st_ != 'waiting' else '', **extra)
    print(f"   → {req['id']} {wb.STATUS_KO.get(st_, st_)} · {summ}")
    return st_

def _rl_bk_poll():
    # '받는 중' 요청의 백업을 받음 (입출력 줄) — 백업 요청은 다 받으면 완료, track 요청은 받은 TASK를 GPU 줄로 넘김
    reqs = [r for r in rb.waiting_requests(BASE) if r['id'] not in _RL['ready']]
    need = sorted({t for r in reqs for t in _rl_ids(r, 'remaining')})
    if not need:
        return
    _rl_lane({'what': '백업 받는 중', 'tasks': need, 'since': ctt.now()})
    try:
        st = _rl_backup(need, 'download')
        left_all = set(_rl_bk_left(need, st))
        for r in reqs:
            rem = _rl_ids(r, 'remaining')
            got, left = [t for t in rem if t not in left_all], [t for t in rem if t in left_all]
            since = float(r.get('waiting_t') or 0) or _time.time()
            late = left and _time.time() - since > RL_BACKUP_GIVEUP_H * 3600
            if r['kind'] == 'backup':
                for t in got:
                    rb.add_history(BASE, t, kind='backup', status=st.get(str(t)) or '받음', request=r['id'], who=who_am_i())
                res = _rl_merge(r, [{'task_id': t, 'status': st.get(str(t)) or '받음'} for t in got]
                                + ([{'task_id': t, 'status': '포기 (준비 안 됨)'} for t in left] if late else []))
                if late:
                    for t in left:
                        rb.add_history(BASE, t, kind='backup', status='포기 (준비 안 됨)', request=r['id'], who=who_am_i())
                    _rl_status(r, 'failed', remaining=left, done_at=ctt.now(), results=res,
                               result=(str(r.get('result', '')) + f" · {RL_BACKUP_GIVEUP_H}시간 넘게 준비 안 됨 {len(left)}개 → 다시 요청하세요")[:500])
                elif not left:
                    _rl_status(r, 'done', remaining=[], done_at=ctt.now(), results=res, result=(str(r.get('result', '')) + f" · 모두 받음")[:500])
                elif got:
                    _rl_status(r, 'waiting', remaining=left, waiting_t=since, results=res, result=(str(r.get('result', '')) + f" · {len(got)}개 받음")[:500])
                elif not r.get('waiting_t'):
                    _rl_status(r, 'waiting', waiting_t=since)
            elif got or late:                     # track: 받은 TASK(와 포기한 TASK) → GPU 줄이 마저 함
                _RL['ready'][r['id']] = {'got': got, 'gave_up': left if late else []}
            elif not r.get('waiting_t'):
                _rl_status(r, 'waiting', waiting_t=since)
            _RL['dirty'] = True
    finally:
        _rl_lane(None)

def _rl_track_resume(rid):
    # 백업을 받은 TASK의 track을 마저 만듦 (GPU 줄)
    info = _RL['ready'].get(rid) or {}
    req = _rl_req(rid)
    try:
        if not req or req['status'] != 'waiting':
            return 'skip'
        p = dict(req.get('params') or {})
        got, gave = info.get('got') or [], info.get('gave_up') or []
        rem = [t for t in _rl_ids(req, 'remaining') if t not in got and t not in gave]
        mpath, mlabel = _rl_model(p.get('model'))
        conf = float(p.get('conf') or AA_CONF)
        bad, rows = [], []
        for i, t in enumerate(got, 1):
            _rl_lane({'what': 'track 만들기 (백업 받은 뒤)', 'request': rid, 'task': t, 'n': i, 'of': len(got), 'since': ctt.now()})
            print(f"[{ctt.now()[11:16]}] track 만들기 (백업 받은 뒤) {i}/{len(got)} · TASK {t}")
            r = _rl_track_one(t, mpath, mlabel, conf, req)
            rows.append(r)
            if not str(r.get('status', '')).startswith('완료'):
                bad.append(f"{t} {r.get('status')}")
            _RL['dirty'] = True
        note = f" · 백업 뒤 {len(got) - len(bad)}개 track 완료" + ''.join(f" · {b}" for b in bad) \
               + (f" · 백업 포기 {len(gave)}개" if gave else '')
        res = _rl_merge(req, _rl_res(rows) + [{'task_id': t, 'status': '백업 포기 (준비 안 됨)', 'note': ''} for t in gave])
        if rem:
            _rl_status(req, 'waiting', remaining=rem, results=res, result=(str(req.get('result', '')) + note)[:500])
        else:
            _rl_status(req, 'done' if len(got) > len(bad) or not gave and not bad else 'failed', remaining=[], done_at=ctt.now(), results=res,
                       result=(str(req.get('result', '')) + note)[:500])
        return 'done'
    finally:
        _RL['ready'].pop(rid, None)

# ---------- 차례 정하기 ----------
def _rl_done(fut, lane, ids):
    with _RL_LOCK:
        for t in ids:
            _RL['tasks'].pop(t, None)
        _RL['dirty'] = True
    _RL_EV.set()

def _rl_submit(lane, fn, arg, ids, label, who=None):
    with _RL_LOCK:
        for t in ids:
            _RL['tasks'][t] = label
    def _go():
        try:
            return fn(arg)
        finally:
            _rl_lane(None)
    f = _RL_POOLS[lane].submit(_go)
    _RL_POOLS['futs'][f] = (lane, label)
    _RL_POOLS.setdefault('who', {})[f] = who        # v4.31.6: 한 사람이 무거운 일을 W_USER_MAX건 넘게 차지하지 않게
    f.add_done_callback(lambda f_, l=lane, i=list(ids): _rl_done(f_, l, i))
    return f

def _rl_user_running(u):
    return sum(1 for f, w in list(_RL_POOLS.get('who', {}).items()) if w == u and not f.done())

def _rl_running(lane):
    return [f for f, (l, _) in list(_RL_POOLS['futs'].items()) if l == lane and not f.done()]

def _rl_dispatch(poll=True):
    # 대기 요청을 차례대로 보며 맡길 수 있는 것을 맡김 → 맡긴 수
    n = 0
    reqs = [r for r in rb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in rb.KINDS
            and _w_mine(r['id'], _RL['host'])]   # v4.31.4: 배정(requests/assign) 따르기
    reqs = wb.fair_order(reqs, _RL_LIGHT, _w_served())      # v4.31.6: 공정 차례 (가벼운 일 → 사람마다 돌아가며 → 오래 기다린 것)
    busy, blocked = set(_RL['tasks']), set()
    gpu_free = max(1, int(RL_GPU_WORKERS)) - len(_rl_running('gpu')) - int(_W['lc_gpu'] or 0)   # LC가 GPU 줄을 맡아 둠 (학습은 모두)
    io_free = max(1, int(RL_IO_WORKERS)) - len(_rl_running('io'))
    users = {wb._user_of(r) for r in reqs}
    for rid in list(_RL['ready']):                # 백업을 받은 track 요청 (먼저 보낸 요청이므로 먼저)
        rq = _rl_req(rid)
        ids = set(_RL['ready'][rid].get('got') or [])
        if rq is None or gpu_free <= 0 or ids & busy:
            if rq is None:
                _RL['ready'].pop(rid, None)
            blocked |= ids
            continue
        _rl_submit('gpu', _rl_track_resume, rid, sorted(ids), f'track 마저 {rid}'); n += 1
        gpu_free -= 1; busy |= ids
    for r in reqs:
        k, ids = r['kind'], set(_rl_ids(r))
        if k in _RL_LIGHT or (k == 'discard' and not ids & (busy | blocked)):
            print(f"\n📨 [{ctt.now()[11:16]}] {rb.KINDS.get(k, k)} · {r.get('by', '')} · TASK {len(ids)}개 · {r['id']}")
            r = _rl_status(r, 'taken', taken_by=who_am_i(), taken_session=SESSION_ID, taken_at=ctt.now(), taken_host=_RL['host'], host=_RL['host'])
            _rl_run(r); n += 1; _RL['dirty'] = True
            continue
        lane = 'gpu' if k in _RL_GPU else 'io' if k in _RL_IO else None
        if lane is None or ids & (busy | blocked) or (lane == 'gpu' and gpu_free <= 0) or (lane == 'io' and io_free <= 0):
            blocked |= ids                        # 같은 TASK의 뒤 요청은 이 요청이 끝난 뒤에
            continue
        u = wb._user_of(r)
        if W_USER_MAX and len(users) > 1 and _rl_user_running(u) >= int(W_USER_MAX):
            blocked |= ids                        # v4.31.6: 이 사람은 이미 W_USER_MAX건 — 다른 사람 요청 먼저
            continue
        print(f"\n📨 [{ctt.now()[11:16]}] {rb.KINDS.get(k, k)} · {r.get('by', '')} · TASK {len(ids)}개 · {r['id']} → {'GPU 줄' if lane == 'gpu' else '입출력 줄'}")
        r = _rl_status(r, 'taken', taken_by=who_am_i(), taken_session=SESSION_ID, taken_at=ctt.now(), taken_host=_RL['host'], host=_RL['host'])
        _rl_submit(lane, _rl_run, r, sorted(ids), f"{rb.KINDS.get(k, k)} {r['id']}", who=u); n += 1
        _w_served_add(u)
        busy |= ids
        if lane == 'gpu':
            gpu_free -= 1
        else:
            io_free -= 1
    if poll and io_free > 0 and _time.time() - _RL['bk_t'] >= RL_BACKUP_POLL_MIN * 60 \
            and not any(lbl == '백업 받기' for f, (l, lbl) in list(_RL_POOLS['futs'].items()) if not f.done()):
        if any(_rl_ids(r, 'remaining') for r in rb.waiting_requests(BASE) if r['id'] not in _RL['ready']):
            _RL['bk_t'] = _time.time()
            f = _RL_POOLS['io'].submit(_rl_bk_poll)
            _RL_POOLS['futs'][f] = ('io', '백업 받기')
            f.add_done_callback(lambda f_: _RL_EV.set())
            n += 1
    return n

def _rl_wait_all():
    _fwait([f for f in list(_RL_POOLS['futs']) if not f.done()])
    _rl_tidy()

def _rl_tidy():
    for f in [f for f in list(_RL_POOLS['futs']) if f.done()]:
        _RL_POOLS['futs'].pop(f, None)
        _RL_POOLS.get('who', {}).pop(f, None)

_rl_left = []
for _r in [r for r in rb.list_requests(BASE) if r['status'] == 'taken']:   # 앞 작업자가 멈추며 남긴 '진행 중' → 다시 대기 (처음부터 다시 함, 올리기 전 백업이 있음)
    if (_r.get('taken_session') == SESSION_ID or (_RL_PREV.get('session') and _r.get('taken_session') == _RL_PREV.get('session'))
            or wb.age_hours(_r.get('taken_at')) >= RL_STALE_TAKEN_H):          # v4.31.4: 다른 세션(CPU 마스터 등)이 진행 중인 것은 그대로
        _rl_status(_r, 'pending', result='작업자가 멈춰 다시 대기')
        print(f"↺ 멈춘 요청을 다시 대기로: {_r['id']} ({rb.KINDS.get(_r['kind'], _r['kind'])})")
    else:
        _rl_left.append(_r)
if _rl_left:
    print(f"ℹ️ 다른 세션이 진행 중인 요청 {len(_rl_left)}건은 그대로 둡니다: " + ', '.join(f"{r['id']}({r.get('taken_by', '?')})" for r in _rl_left)
          + f" — 끊긴 것이면 {RL_STALE_TAKEN_H}시간 뒤 다시 대기로")
for _t in rb.stale_pending(BASE):                  # v4.29: shape로 라벨이 바뀐 뒤에도 남은 track 확인 대기 → 정리
    _rl_clear_stale(_t)


# =============== LC (예전 31번 셀) ===============
import json, threading, time as _time, traceback, re as _re
from IPython import get_ipython
assert wb is not None and hasattr(wb, 'lc_worker_beat'), 'scripts/를 v4.26.0 이상으로 바꾸세요'
_lc_cells_p = f'{SCRIPTS}/notebook_cells.json'
assert os.path.exists(_lc_cells_p), 'scripts/notebook_cells.json이 없습니다 — v4.26.0 scripts zip을 모두 올리세요'
_LC_CELLS = json.load(open(_lc_cells_p, encoding='utf-8'))
if _LC_CELLS.get('version') not in SCRIPTS_OK:
    print(f"⚠️ notebook_cells.json 버전 {_LC_CELLS.get('version')} ≠ 노트북 {APP_VERSION} — scripts를 노트북과 같은 버전으로 맞추세요")
_LC = {'busy': None, 'stop': False, 'started': ctt.now(), 'host': 'kaggle_gpu' if W_ROLE == 'gpu' else 'Kaggle' if ON_KAGGLE else 'Colab', 'dl_t': 0, 'state_t': 0, 'dirty': False}
if ON_KAGGLE:
    drive_pull('requests/*.json', f'web/worker{W_SLOT}.json', quiet=True)
_LC_PREV = wb.lc_worker_acquire(BASE, who_am_i(), SESSION_ID, takeover=LC_TAKEOVER, slot=W_SLOT) or {}   # 앞 작업자 (멈춘 요청을 되돌릴 때 씀)

def _lc_src(cell, set_=None):
    # 셀 코드 + 설정 바꾸기 (그 셀의 '설정 이름 = …' 첫 줄을 바꿈)
    src = _LC_CELLS['cells'][str(cell)]
    for k, v in {**(set_ or {}), **(LC_SET.get(str(cell)) or {})}.items():
        src, n = _re.subn(rf'(?m)^({_re.escape(k)}\s*=\s*)[^#\n]*?(\s*(#.*)?)$', lambda m: m.group(1) + repr(v) + ('  ' + m.group(3) if m.group(3) else ''), src, count=1)
        if not n:
            print(f'⚠️ {cell}번 셀에 {k} 설정이 없습니다 (무시)')
    return src

def _lc_run(cell, set_=None):
    # 셀 하나를 실행 (Kaggle은 셀마다 Drive와 주고받음) → 오류 (없으면 None) · 셀 중지(■)는 작업자도 멈춤
    print(f"\n━━━━ {cell}번 셀 실행 ({ctt.now()[11:16]}) ━━━━")
    r = get_ipython().run_cell(_lc_src(cell, set_), store_history=False)
    err = r.error_in_exec or r.error_before_exec
    if isinstance(err, KeyboardInterrupt):
        raise err
    return err

def _lc_kg_begin():
    # Kaggle: 요청 받기 + 지금 상태 기억 (작업자가 직접 바꾼 요청 파일만 Drive에 올리고, 이름이 바뀐 예전 파일은 Drive에서 지움)
    if ON_KAGGLE:
        drive_pull('requests/*.json', quiet=True)
        _KG['snap'], _KG['snap_t'] = _kg_snapshot(), _time.time()

def _lc_kg_end():
    if ON_KAGGLE and _KG.get('snap') is not None:
        drive_sync_now(echo=False)

def _lc_beat(busy=None, stopping=False):
    if busy is not None:
        _LC['busy'] = busy
    wb.lc_worker_beat(BASE, who_am_i(), SESSION_ID, _LC['host'], busy=dict(_LC['busy'] or {}, loop_at=_LC.get('loop_at')),   # loop_at: 요청 확인 루프가 마지막으로 돈 시각 (v4.29.4)
                      started=_LC['started'], stopping=stopping, slot=W_SLOT)
    if ON_KAGGLE:
        drive_push(f'web/worker{W_SLOT}.json', quiet=True)

def _lc_heart(st=_LC, beat=_lc_beat, sleep=_time.sleep):
    while not st['stop']:
        try:
            beat()
        except Exception:
            pass
        for _ in range(60):
            if st['stop']:
                return
            sleep(1)

def _lc_fail_left(ids, why):
    # 셀이 요청을 끝내지 못하고 멈췄을 때 (오류·대상 없음): '진행 중'으로 남은 요청을 '실패'로 + 이유
    for r in wb.list_requests(BASE):
        if r['id'] in ids and r['status'] == 'taken':
            wb.set_status(BASE, r, 'failed', result=why[:400], done_at=ctt.now(), by=who_am_i(), summary=why[:200])
            print(f"   → 요청 {r['id']} 실패로: {why[:120]}")

_LC_CELL = {'backup': ['13'], 'export': ['14'], 'track': ['5'], 'aa': ['9', '10'], 'cleanup': ['29'], 'compare': ['32'], 'train': ['15']}

def _lc_do(kind):
    ids = [r['id'] for r in wb.list_requests(BASE) if r['kind'] == kind and r['status'] in ('pending', 'taken')
           and (r.get('taken_session') == SESSION_ID if r['status'] == 'taken' else _w_mine(r['id'], _LC['host']))]   # 셀이 받을 것 (v4.31.7: 다른 세션이 진행 중인 것은 빼고)
    _lc_beat({'what': wb.KINDS[kind][0], 'cell': _LC_CELL[kind][0], 'requests': len(ids), 'since': ctt.now()})
    err = None
    for c in _LC_CELL[kind]:
        err = _lc_run(c, {'FROM_REQUESTS': True} if c != '10' else {'RUN_AA': True})
        if err is not None:
            break
    if err is not None:
        why = f"{type(err).__name__}: {err}" if not isinstance(err, SystemExit) else str(err)
    elif kind == 'aa' and not globals().get('AA_READY'):
        why = '자동 라벨을 올릴 TASK가 없음 (9번 셀 점검에서 모두 빠짐 — 기존 라벨이 있거나 영상·프레임 문제)'
    else:
        why = '셀은 끝났지만 요청을 마치지 않음 (출력 확인)'
    _lc_kg_begin()
    _lc_fail_left(ids, why)
    _lc_kg_end()
    _LC['dirty'] = True

def _lc_waiting():
    if W_ROLE == 'gpu':                            # v4.31.7: GPU 워커는 받기·상태 만들기를 하지 않음 (CPU 마스터·30번 셀 몫)
        return
    # '받는 중' 백업·export: CVAT가 준비한 것을 받음 (13·14번 셀 받기 방식) → 다 받으면 요청이 완료로
    if _time.time() - _LC['dl_t'] < LC_DOWNLOAD_MIN * 60:
        return
    _LC['dl_t'] = _time.time()
    w = {r['kind'] for r in wb.list_requests(BASE) if r['status'] == 'waiting'}
    p = {x['kind'] for x in (wb.pending_open(BASE) if hasattr(wb, 'pending_open') else [])}
    if 'backup' in w | p:
        _lc_beat({'what': '백업 받기 (준비된 것)', 'cell': '13', 'since': ctt.now()})
        _lc_run('13', {'FROM_REQUESTS': False, 'BK_MODE': 'download', 'BK_PREVIEW': False, 'BK_QUERY': '', 'BK_TASK_IDS': None})
        _LC['dirty'] = True
    if 'export' in w | p:
        _lc_beat({'what': 'export 받기 (준비된 것)', 'cell': '14', 'since': ctt.now()})
        _lc_run('14', {'FROM_REQUESTS': False, 'EX_MODE': 'download', 'EX_PREVIEW': False, 'EX_QUERY': '', 'EX_IDS': []})
        _LC['dirty'] = True

def _lc_state(force=False):
    if W_ROLE == 'gpu':
        return
    if not (force or _LC['dirty'] or _time.time() - _LC['state_t'] >= LC_STATE_MIN * 60):
        return
    _lc_beat({'what': 'LC앱 상태 새로 만들기', 'cell': '28', 'since': ctt.now()})
    _lc_run('28', {'LOGIN': LC_LOGIN})
    _LC['state_t'], _LC['dirty'] = _time.time(), False

_lc_kg_begin()
def _lc_age_h(t):
    return wb.age_hours(t)                     # v4.31.4: 시간대 없는 KST 시각('2026-11-07 15:00:00')도 바르게
# '진행 중'으로 남은 요청: 이 세션·앞 LC 작업자가 받다 멈춘 것, 또는 LC_STALE_TAKEN_H시간이 지난 것만 다시 대기로
# (다른 사람이 셀을 직접 실행해 진행 중인 요청은 그대로 둠 — 두 번 실행하지 않게)
_left = []
for _r in [r for r in wb.list_requests(BASE) if r['status'] == 'taken' and r['kind'] in LC_KINDS]:
    if (_r.get('taken_session') == SESSION_ID or (_LC_PREV.get('session') and _r.get('taken_session') == _LC_PREV.get('session'))
            or _lc_age_h(_r.get('taken_at')) >= LC_STALE_TAKEN_H):
        wb.set_status(BASE, _r, 'pending', result='작업자가 멈춰 다시 대기', by=who_am_i())
        print(f"↺ 멈춘 요청을 다시 대기로: {_r['id']} ({wb.KINDS.get(_r['kind'], (_r['kind'],))[0]})")
    else:
        _left.append(_r)
if _left:
    print(f"ℹ️ 다른 세션이 진행 중인 요청 {len(_left)}건은 그대로 둡니다: " + ', '.join(f"{r['id']}({r.get('taken_by', '?')})" for r in _left)
          + f" — 끊긴 것이면 {LC_STALE_TAKEN_H}시간 뒤 다시 대기로 (또는 LC앱에서 확인)")
_lc_kg_end()


# ---------- 돌기 (v4.31.6): 재처리는 뒤 스레드가 요청을 나눠 주고, LC 셀은 이 스레드에서 차례로 ----------
def _w_now_utc():
    import datetime as _d
    return _d.datetime.now(_d.timezone.utc).isoformat(timespec='seconds')

def _w_aa_loop():
    # 재처리 요청 확인 (예전 30번 셀의 돌기) — LC 셀이 오래 걸려도 폰 요청은 바로 받음
    n = 0
    try:
        while not _W['stop'] and (W_MAX_LOOPS is None or n < W_MAX_LOOPS):
            n += 1
            _RL['loop_at'] = _w_now_utc()
            _RL_EV.clear()
            try:
                if ON_KAGGLE:
                    with _RL_LOCK:
                        drive_pull('relabel/requests/*.json', 'requests/assign/*.json', quiet=True)
                got = _rl_dispatch()
                if W_MAX_LOOPS is not None:            # 시험: 맡긴 일이 끝나기를 기다리고, 그 사이 풀린 요청도 이어서
                    while got:
                        _rl_wait_all(); _rl_state(force=True)
                        got = _rl_dispatch(poll=False)
                _rl_state()
                running = [f for f in list(_RL_POOLS['futs']) if not f.done()]
                _rl_tidy()
                if not running and not got:
                    _rl_refresh()
                _RL['errs'] = 0
            except Exception as _e:                    # 요청 확인 중 오류 한 번으로 멈추지 않게 — 알리고 다음 차례에 다시
                _RL['errs'] = _RL.get('errs', 0) + 1
                print(f"⚠ [{ctt.now()[11:16]}] 재처리 요청 확인 중 오류 — 계속 ({_RL['errs']}번 연속): {type(_e).__name__}: {_e}"[:400])
                if _RL['errs'] >= 10:                    # 계속 실패하면 멈춤 (Drive 연결이 끊겼을 때 등)
                    _W['aa_err'] = _e
                    return
            if not _W['stop'] and (W_MAX_LOOPS is None or n < W_MAX_LOOPS):
                _RL_EV.wait(W_POLL_SEC)
    finally:
        _W['aa_done'] = True

def _w_wait_gpu(kind=''):
    # LC의 GPU 일: 한 줄을 맡아 두고(재처리는 새 GPU 일을 덜 받음) 빈 줄이 날 때까지 기다림 · 학습은 모든 줄 (v4.31.7)
    n = max(1, int(RL_GPU_WORKERS))
    _W['lc_gpu'] = n if kind == 'train' else 1
    t0, said = _time.time(), False
    while len(_rl_running('gpu')) > n - _W['lc_gpu'] and not _W['stop']:
        if not said:
            print(f"   (GPU 줄 {len(_rl_running('gpu'))}개가 재처리 라벨링 중 — 하나가 끝나면 시작)")
            said = True
        _lc_beat({'what': 'GPU 줄이 비기를 기다림', 'since': ctt.now()})
        _time.sleep(3)
    if said:
        print(f"   (GPU 줄 비움 — {_time.time() - t0:.0f}초 기다림)")

def _w_status():
    if _time.time() - _W.get('status_t', 0) < W_STATUS_MIN * 60:
        return
    _W['status_t'] = _time.time()
    _a = [r for r in rb.list_requests(BASE) if r['status'] in ('pending', 'taken', 'waiting')]
    _l = [r for r in wb.list_requests(BASE) if r['status'] in ('pending', 'taken', 'waiting')]
    _run = [lbl for f, (l, lbl) in list(_RL_POOLS['futs'].items()) if not f.done()]
    print(f"[{ctt.now()[11:16]}] 대기 중 · 열린 요청 재처리 {len(_a)}건 · LC {len(_l)}건 (학습 대기 {sum(r['kind'] == 'train' and r['status'] == 'pending' for r in _l)}"
          f" · 받는 중 {sum(r['status'] == 'waiting' for r in _a + _l)})" + (f" · 하는 일 {len(_run)}건" if _run else '')
          + f" — 요청 파일 {len(glob.glob(f'{BASE}/relabel/requests/REQ_*.json'))}개 (폰에서 보낸 요청이 안 잡히면 Drive 연결·폴더를 확인)")

_t_end, _wn = _time.time() + W_HOURS * 3600, 0
_aa_th = threading.Thread(target=_w_aa_loop, name='aa-loop', daemon=True)
try:
    threading.Thread(target=_rl_heart, daemon=True).start()   # 멈출 때(finally) 함께 멈춤
    threading.Thread(target=_lc_heart, daemon=True).start()
    _lc_beat({'what': '대기 중'})
    print(f"▶ 작업자 켬 · {_RL['host']} · {who_am_i()} · 최대 {W_HOURS}시간 · 요청 확인 {W_POLL_SEC}초마다 · CVAT {CVAT_SERVER} ({ctt.scope_text(CVAT_SCOPE)})\n"
          f"   재처리(AA앱(가지 재처리)): 라벨링 줄 {max(1, int(RL_GPU_WORKERS))}" + (f" (GPU {_RL_NGPU}개에 나눔)" if _RL_NGPU > 1 else "")
          + f" + 입출력 줄 {max(1, int(RL_IO_WORKERS))}\n"
          f"   LC(LC앱): {', '.join(wb.KINDS[k][0] for k in LC_KINDS)}" + (f" · GPU 워커: 할 일이 없으면 {W_EXIT_IDLE_MIN}분 뒤 끝남" if W_ROLE == 'gpu' else '') + "\n"
          f"   공정 차례: 가벼운 일 → 사람마다 돌아가며 한 건씩(오래 기다린 것 먼저) · 다른 사람이 기다릴 때 한 사람 동시 {W_USER_MAX or '제한 없음'}건")
    _rl_refresh(force=True)
    _aa_th.start()
    while _time.time() < _t_end and (W_MAX_LOOPS is None or _wn < W_MAX_LOOPS):
        _wn += 1
        if _W['aa_err'] is not None:
            raise RuntimeError(f"재처리 요청 확인이 10번 연속 실패해 멈춥니다: {_W['aa_err']}")
        _LC['loop_at'] = _w_now_utc()
        if ON_KAGGLE:
            drive_pull('requests/*.json', 'requests/assign/*.json', 'backups/pending_backups.csv', 'exports/pending_exports.csv',
                       *(['jobs/STOP_GPU.json'] if W_ROLE == 'gpu' else []), quiet=True)
        if W_ROLE == 'gpu' and os.path.exists(f'{BASE}/jobs/STOP_GPU.json'):
            print(f"🛑 [{ctt.now()[11:16]}] CPU 마스터의 정지 신호(jobs/STOP_GPU.json) → 하던 일만 마치고 끝냄")
            break
        _pend = [dict(r, _lc=True) for r in wb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in LC_KINDS
                 and _w_mine(r['id'], _LC['host'])]   # 다른 곳(kaggle_cpu 등)으로 배정된 요청은 건너뜀
        if _pend:
            _aa_pend = [r for r in rb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in rb.KINDS]
            _first = next(r for r in wb.fair_order(_pend + _aa_pend, list(_RL_LIGHT) + ['cleanup'], _w_served()) if r.get('_lc'))
            _k = _first['kind']
            _w_served_add(wb._user_of(_first))                        # 그 사람의 차례인 LC 요청의 종류 → 그 셀이 같은 종류의 요청을 모두 처리
            print(f"\n📨 [{ctt.now()[11:16]}] LC {wb.KINDS[_k][0]} 요청 {sum(1 for r in _pend if r['kind'] == _k)}건 (차례: {_first.get('by', '?')} · {_first['id']}) → {', '.join(_LC_CELL[_k])}번 셀")
            try:
                if _k in LC_GPU_KINDS:
                    _w_wait_gpu(_k)
                _lc_do(_k)
            except KeyboardInterrupt:
                raise
            except Exception as _e:
                traceback.print_exc()
                _lc_kg_begin()
                _lc_fail_left([r['id'] for r in _pend if r['kind'] == _k], f'작업자 오류: {_e}')
                _lc_kg_end()
            finally:
                _W['lc_gpu'] = 0
                _RL_EV.set()                           # 재처리 쪽이 바로 다음 GPU 일을 보게
            _lc_state(force=True)
            _lc_beat({'what': '대기 중'})
            _W['idle_since'] = None
            continue
        if W_EXIT_IDLE_MIN:                            # v4.31.7: 할 일이 없으면 끝냄 (GPU 워커 → GPU 반환)
            _op = {str(x.get('task_id')) for x in wb.pending_rows(BASE, 'backup') if x.get('status') in _RL_OPEN_BK}
            _busy = [f for f in list(_RL_POOLS['futs']) if not f.done()] or _RL['ready'] or [
                r for r in rb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in rb.KINDS and _w_mine(r['id'], _RL['host'])] or [
                r for r in rb.list_requests(BASE) if r['status'] == 'waiting' and r.get('taken_session') == SESSION_ID    # v4.31.9 (F10): 백업이 준비된 '받는 중' track은 마저 처리
                and any(str(t) not in _op for t in (r.get('remaining') or []))]
            if _busy:
                _W['idle_since'] = None
            elif _W['idle_since'] is None:
                _W['idle_since'] = _time.time()
            elif _time.time() - _W['idle_since'] >= float(W_EXIT_IDLE_MIN) * 60:
                print(f"✅ [{ctt.now()[11:16]}] 할 일이 {W_EXIT_IDLE_MIN}분 동안 없어 끝냅니다" + (' (GPU 반환)' if W_ROLE == 'gpu' else ''))
                break
        _lc_waiting()
        _lc_state()
        _lc_beat({'what': '대기 중'})
        _w_status()
        if W_MAX_LOOPS is None or _wn < W_MAX_LOOPS:
            _time.sleep(W_POLL_SEC)
    if W_MAX_LOOPS is not None:                        # 시험: 재처리 쪽도 다 돌기를 기다림
        while not _W['aa_done']:
            _time.sleep(0.2)
finally:
    _W['stop'] = True
    _RL_EV.set()
    _LC['stop'] = True
    try:
        _lc_beat({'what': '꺼짐'}, stopping=True)
    except Exception:
        pass
    try:
        if _aa_th.is_alive():
            _aa_th.join(timeout=120)               # 지금 나눠 주던 차례만 마치고 끝남
    except BaseException:
        pass
    _RL['stop'] = True
    _RL['loop_at'], _RL['closing'] = None, True
    _left = [f for f in list(_RL_POOLS['futs']) if not f.done()]
    try:
        _rl_beat()                                 # 폰에 '멈추는 중' 바로
    except Exception:
        pass
    try:
        if _left:
            print(f"⏸ 하던 일 {len(_left)}건 — 지금 TASK까지 끝내고 멈춥니다 (남은 TASK는 요청에 '대기'로 남김) …")
            _fwait(_left)
        try:
            _rl_state(force=True)
        except KeyboardInterrupt:                  # v4.31.4: 마지막 상태 파일 쓰기 중 ■ → 조용히 건너뜀 (기록은 이미 Drive에)
            print('ℹ️ 마지막 화면 상태 파일은 건너뜀 — 다음에 켤 때 다시 씀')
        except Exception as _e:
            print(f'⚠ 마지막 상태 파일 쓰기 실패 (무시): {_e}')
    except BaseException:
        print("⚠ 기다리지 않고 멈춤 — 하던 일은 뒤에서 마저 끝나고, 30번 셀을 다시 켜면 그것이 끝난 뒤 시작합니다")
        raise
    finally:
        if W_ROLE == 'gpu':                       # v4.31.7: CPU 마스터가 상태를 새로 만들도록 끝 표시
            try:
                os.makedirs(f'{BASE}/jobs', exist_ok=True)
                json.dump({'session': SESSION_ID, 'time': ctt.now(), 'who': who_am_i()}, open(f'{BASE}/jobs/DONE_{SESSION_ID}.json', 'w'))
                drive_push(f'jobs/DONE_{SESSION_ID}.json', quiet=True)
            except Exception as _e:
                print(f'(끝 표시 실패: {_e})')
        if not [f for f in list(_RL_POOLS['futs']) if not f.done()]:
            _RL['heart_stop'] = True
            try:
                _rl_beat('꺼짐', stopping=True)
            except Exception:
                pass
            print(f"■ 작업자 끔 ({ctt.now()[11:16]})")


## 31. (v4.31.6) 30번 셀로 합침
LC 작업자는 30번 셀로 합쳐졌습니다. **30번 셀 하나만** 켜면 두 앱의 요청을 모두 처리합니다. LC 설정(`LC_KINDS`·`LC_SET` 등)도 30번 셀 위쪽에 있습니다.


In [ ]:
# 31. (v4.31.6) LC 작업자는 30번 셀로 합쳤습니다 — 30번 셀 하나만 켜면 두 앱('가지 재처리' · LC앱 '가지 라벨링 관리')의 요청을 모두 처리합니다.
#     LC 설정(LC_KINDS · LC_SET 등)도 30번 셀 위쪽에 있습니다. 이 셀은 실행하지 않아도 됩니다.
print('31번 셀은 30번 셀로 합쳐졌습니다 → 30번 셀을 실행하세요 (두 앱의 요청을 함께 처리)')


## 32. 모델 비교 — 같은 TASK를 여러 모델로 탐지해 비교 (v4.28.0, CVAT에 올리지 않음)
이미 자동 라벨을 올린 TASK를 다른 모델로 다시 탐지해, **지금 CVAT 라벨**을 기준으로 견줍니다. CVAT의 라벨은 바꾸지 않습니다.
- 기준: TASK 이름이 검토 완료(`REVIEWED_RULE`)면 검토자가 고친 라벨 = 정답 → 정밀도 P · 재현율 R · F1. 검토 전이면 지금 CVAT 라벨(올린 자동 라벨) 기준 **일치도**일 뿐이라 어느 모델이 나은지는 알 수 없습니다.
- 대상: `CMP_MODELS`의 모델마다 프레임을 새로 탐지 + (`CMP_UPLOADED`) 이 TASK에 마지막으로 올린 자동 라벨(업로드 직후 기록). 같은 프레임(`CMP_FRAMES`장, TASK 전체에서 고르게)으로 셉니다.
- 셈: 맞힘 = 같은 라벨 IoU ≥ 0.5 (그중 위치가 다른 것은 따로) · 라벨 다름 · 잘못 그림(오탐) · 놓침 — 검토 수정량과 같은 짝짓기.
- 기록: `tasks/<ID>/compare_<시각>.json`(전체, 라벨별) · `logs/model_compare.csv`(TASK × 대상) · 작업 기록('모델 비교') → LC앱 TASK 상세, 재처리 앱 TASK 이력. 웹 TASK 탭에서 **모델 비교 요청**도 됩니다 (LC 작업자가 켜져 있으면 자동).
- 프레임별 탐지(shape)로 비교합니다. track으로 올린 TASK도 프레임마다 박스를 펼쳐 같은 방식으로 셉니다.


In [ ]:
# 32. 모델 비교 — 같은 TASK를 여러 모델로 탐지해 지금 CVAT 라벨과 비교 (CVAT에 아무것도 올리지 않음) · 결과는 TASK 기록에
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
CMP_TASK_IDS = []         # 비교할 TASK ID 목록, 예: [2700651, 2700652]
CMP_QUERY    = ''         # 또는 TASK 이름 검색 (CMP_TASK_IDS가 비었을 때). 예: 'G1 261019 vf* tag=AA*'
CMP_MODELS   = []         # 비교할 모델 (runs/ 이름), 예: ['fruit_r261101_2class', 'fruit_r261101_2class_rotate'] · 비우면 0번 셀에서 고른 모델
CMP_UPLOADED = True       # True = 이 TASK에 마지막으로 올린 자동 라벨(업로드 직후 기록)도 함께 비교 — 검토 완료 TASK일 때만 의미 있음
CMP_CONF     = 0.25       # 모델 탐지 신뢰도 하한
CMP_FRAMES   = 40         # TASK마다 비교할 프레임 수 (TASK 전체에서 고르게, 박스 없는 프레임 포함) · None = 전부 (느림)
CMP_LABELS   = None       # 비교할 라벨, 예: ['healthy_fruit'] · None = 모델이 아는 라벨
FROM_REQUESTS = False     # True = 웹 화면의 요청(모델 비교)을 받아 그 TASK·모델로 실행 (끝나면 요청이 '완료'로)
LOGIN        = None       # None = 지금 접속 중인 서버 그대로 · 또는 0번 셀 CVAT_PROFILES의 이름
# 기준 = 지금 CVAT 라벨. 이름이 검토 완료(0번 셀 REVIEWED_RULE)면 검토자가 고친 라벨 = 정답 → 정밀도·재현율·F1.
# 검토 전이면 지금 CVAT 라벨(올린 자동 라벨) 기준 '일치도'일 뿐 — 어느 모델이 더 좋은지는 알 수 없음.
# 맞힘 = 같은 라벨 IoU ≥ 0.5 · 잘못 그림(오탐) · 놓침 · 라벨 다름은 따로 셈 (검토 수정량과 같은 짝짓기)

import json, pandas as pd
from IPython.display import display
_CMP_REQ = ''
if FROM_REQUESTS:
    CMP_TASK_IDS, CMP_QUERY = req_use('compare'), ''
    _CMP_REQ = ','.join((_REQ.get('next') or ('', []))[1])            # 요청 번호 (작업이 시작되면 작업 기록으로 옮겨 감)
    _p = _REQ['params']
    CMP_MODELS = list(_p.get('models') or CMP_MODELS)
    CMP_CONF = float(_p.get('conf') or CMP_CONF)
    CMP_FRAMES = int(_p.get('frames') or CMP_FRAMES or 0) or None
    CMP_UPLOADED = bool(_p.get('uploaded', CMP_UPLOADED))
CMP_TASK_IDS = ctt.as_task_ids(CMP_TASK_IDS)
use_cvat(LOGIN)
if not CMP_TASK_IDS and CMP_QUERY:
    CMP_TASK_IDS = [t['task_id'] for t in ctt.select_tasks(ctt.list_tasks(CLIENT, limit=5000), CMP_QUERY, display=display, label='비교 대상')]
assert CMP_TASK_IDS, 'CMP_TASK_IDS 또는 CMP_QUERY를 적으세요'
assert hasattr(wb, 'compare_models'), 'scripts/를 v4.28.0 이상으로 바꾸세요'

def _cmp_model(name):
    # runs/<이름>/weights/best.pt (Kaggle은 Drive에서 받음) → (경로, 표시)
    import datetime as _d
    if not name:
        return BEST, model_label()
    w = f'{BASE}/runs/{name}/weights/best.pt'
    if ON_KAGGLE and not os.path.exists(w):
        drive_pull(f'runs/{name}/weights/best.pt', quiet=True)
    assert os.path.exists(w), f'모델 파일이 없습니다: runs/{name}/weights/best.pt (보관(archive)한 모델은 가중치가 없음)'
    return w, f"{name}/best.pt ({_d.datetime.fromtimestamp(os.path.getmtime(w)).strftime('%m-%d %H:%M')})"

_models = [(n or os.path.basename(os.path.dirname(os.path.dirname(BEST))),) + _cmp_model(n) for n in (CMP_MODELS or [None])]
print(f"▶ TASK {len(CMP_TASK_IDS)}개 × 모델 {len(_models)}개 ({', '.join(m[0] for m in _models)})" + (' + 올린 자동 라벨' if CMP_UPLOADED else '')
      + f" · conf {CMP_CONF} · 프레임 {CMP_FRAMES or '전부'}장씩 · CVAT에 올리지 않음")
try:
    import relabel_bridge as _rbm
except Exception:
    _rbm = None
_pred = {n: wb.yolo_predictor(w, imgsz=IMGSZ, conf=CMP_CONF) for n, w, lab in _models}
if CMP_LABELS is None and len(_models) > 1:     # v4.31.4: 모델마다 아는 라벨이 다르면 공통 라벨로만 (기준 박스 수가 모델마다 달라지지 않게)
    from ultralytics import YOLO as _Y
    _ln = [{x[:-len('_occluded')] if x.endswith('_occluded') else x for x in _Y(w).names.values()} for n, w, lab in _models]
    _common = set.intersection(*_ln)
    if _common != set.union(*_ln):
        CMP_LABELS = sorted(_common)
        print(f"ℹ️ 모델마다 라벨이 달라 공통 라벨 {CMP_LABELS}로만 비교합니다 (빠진 것: {sorted(set.union(*_ln) - _common)}) — 바꾸려면 CMP_LABELS")
job_start(32, 'compare', {'ids': CMP_TASK_IDS, 'models': [m[0] for m in _models], 'conf': CMP_CONF, 'frames': CMP_FRAMES, 'uploaded': CMP_UPLOADED},
          summary='모델 비교 (올리지 않음)')
_rows, _err = [], 0
for _i, _tid in enumerate(CMP_TASK_IDS, 1):
    try:
        task = CLIENT.tasks.retrieve(_tid)
        _rev = ctt.is_reviewed(task.name, REVIEWED_RULE)
        _dir = task_dir(_tid); os.makedirs(_dir, exist_ok=True)
        _ts = ctt.stamp()
        print(f"\n[{_i}/{len(CMP_TASK_IDS)}] TASK {_tid} {task.name} · 기준 = {'검토 뒤 라벨 ✅' if _rev else '검토 전 CVAT 라벨 (일치도만)'}")
        ref = ctt.export_annotations(task, f'{_dir}/compare_ref_{_ts}.zip')
        _src = [{'name': n, 'kind': 'model', 'predict': _pred[n], 'model_label': lab, 'conf': CMP_CONF} for n, w, lab in _models]
        _up = wb.last_upload(BASE, _tid) if CMP_UPLOADED else None
        if _up and _rev:
            if ON_KAGGLE:
                drive_pull(_up['_src'], quiet=True)
            if os.path.exists(f"{BASE}/{_up['_src']}"):
                _un = wb._upload_model(_up.get('model'), [m[0] for m in _models])
                _src.append({'name': f'올린 자동({_un})', 'kind': 'uploaded', 'file': f"{BASE}/{_up['_src']}", 'model_label': _up.get('model', '')})
        elif _up and not _rev:
            print('   (올린 자동 라벨 = 지금 CVAT 라벨이라 함께 비교하지 않음 — 검토 뒤에 다시 비교하세요)')
        res = wb.compare_models(ref, _src, ctt.cvat_frame_getter(task, ctt.load_annotations(ref)['meta']), max_frames=CMP_FRAMES,
                                labels=CMP_LABELS, reviewed=_rev, log=lambda m: print('  ', m))
        res.update(task_id=_tid, task_name=task.name, time=ctt.now(), ref_file=os.path.relpath(ref, BASE), who=who_am_i(),
                   request=_CMP_REQ)
        _jp = f'{_dir}/compare_{_ts}.json'
        open(_jp, 'w', encoding='utf-8').write(json.dumps(res, ensure_ascii=False, indent=1, default=str))
        wb.write_compare(BASE, _tid, task.name, res, os.path.relpath(ref, BASE), who=who_am_i(), request=res['request'], detail=os.path.relpath(_jp, BASE))
        _sum = wb.compare_summary(res)
        ctt.log_task('compare', _tid, task.name, '완료', model=' vs '.join(s['name'] for s in res['sources']), summary=_sum[:300],
                     file=os.path.relpath(_jp, BASE))
        if _rbm is not None and str(_tid) in _rbm.load_tracked(BASE):   # 재처리 앱(uld·AA) TASK면 그 이력에도
            _rbm.add_history(BASE, _tid, kind='compare', status='완료', who=who_am_i(), note=_sum[:240],
                           model=' vs '.join(s['name'] for s in res['sources']))
        display(pd.DataFrame([{'대상': s['name'], '종류': '모델 새 탐지' if s['kind'] == 'model' else '올린 자동 라벨', '자동 박스': s['total']['auto'],
                               '기준 박스': s['total']['ref_boxes'], '맞힘': s['total']['tp'], '그중 위치 다름': s['total']['fixed'],
                               '라벨 다름': s['total']['relabeled'], '잘못 그림': s['total']['fp'], '놓침': s['total']['fn'],
                               '정밀도 P': s['total']['precision'], '재현율 R': s['total']['recall'], 'F1': s['total']['f1']}
                              for s in res['sources']]).set_index('대상'))
        if res['best'] and _rev:
            print(f"   → 이 TASK에서 F1이 가장 높은 것: {res['best']}")
        _rows += [{'TASK': _tid, '대상': s['name'], 'F1': s['total']['f1'], 'P': s['total']['precision'], 'R': s['total']['recall'],
                   '검토 완료': _rev} for s in res['sources']]
        print(f"   저장: {os.path.relpath(_jp, BASE)} · logs/shards/model_compare/{ctt.SHARD['id'] or '(한 파일)'}.csv (세션별 기록 — LC앱·wb.read_compares가 합쳐 읽음) · 작업 기록")
    except Exception as e:
        _err += 1
        print(f"   ⚠️ TASK {_tid} 비교 실패: {ctt.api_error(e) if hasattr(ctt, 'api_error') else e}")
        ctt.log_task('compare', _tid, '', '오류', summary=str(e)[:300])
if len(CMP_TASK_IDS) > 1 and _rows:
    _df = pd.DataFrame(_rows)
    print('\n■ 모델별 평균 (검토 완료 TASK만 — 검토 전 TASK는 일치도라 빼고)')
    _d2 = _df[_df['검토 완료']]
    display(_d2.groupby('대상')[['F1', 'P', 'R']].mean().round(3) if len(_d2) else pd.DataFrame())
job_end('끝' if not _err else '일부 실패', f"TASK {len(CMP_TASK_IDS) - _err}개 비교" + (f" · 실패 {_err}" if _err else ''))


## 33. 학습 계획 연구 (v4.31.8) — 고정 시험 TASK · TASK 지문 · 모델 성적(개수 오차) · 관계 · 분할안 시뮬레이션
진화하는 학습의 잣대와 도구입니다. CVAT에는 아무것도 올리지 않습니다.
- **① 고정 시험 TASK**: 후보(기본 U세트 261008·261013)가 어느 모델 학습에 쓰였는지와 온실·촬영일 분포를 보고, `TEST_SAVE = True`로 `config/test_tasks.json`에 고정합니다. 15번 셀은 이 TASK와 **같은 그루·촬영일** TASK를 학습·val에서 자동으로 뺍니다.
- **② 지문**: 검토 완료 TASK마다 프레임 `SIM_FRAMES`장을 CVAT에서 받아 배경(박스 밖 밝기·어두운 비율·초록 비율·바닥 밝기·채도·색상)과 개체(프레임당 박스·가림·작은 박스·겹침·박스 크기)를 잽니다.
- **③ 성적**: 같은 프레임을 모델로 탐지해 검토 뒤 라벨과 비교 — 정밀도·재현율·F1과 **개수 오차**(프레임당 개수 차이, 정답 대비 비율, +는 많이 셈/−는 덜 셈).
- **④ 관계**: 성적이 지문(어두움·밀도 …)과 '학습에 쓰였는지'에 따라 얼마나 달라지는지 — 지문 1 표준편차당 변화와 90% 구간(✅ = 0을 넘지 않는 확실한 효과). 온실별 평균 표로 관찰(G6·G5 어두움, G6 조밀, G2 개체 많음)을 숫자로 확인합니다.
- **⑤ 분할안**: train·val·시험을 정하면 val·시험 TASK마다 가장 가까운 train TASK와의 거리(⚠️ = train에 비슷한 환경 없음)와 예상 재현율, 넣으면 덮임이 가장 넓어지는 TASK를 보여 줍니다.
- 지문·성적은 TASK마다 한 번 저장(`sim/`)하고 다음 실행에 이어서 합니다. 관계는 TASK가 30개 이상일 때 믿을 만합니다.


In [ ]:
# 33. 학습 계획 연구 (v4.31.8) — 고정 시험 TASK · TASK 지문 · 모델 성적(개수 오차 포함) · 관계 · 분할안 시뮬레이션 (CVAT에 아무것도 올리지 않음)
#     ① test: 시험 TASK 후보(U세트 등)가 어느 모델 학습에 쓰였는지 · 온실·촬영일 분포 → TEST_SAVE = True면 config/test_tasks.json에 고정
#            (고정한 TASK와 같은 그루·촬영일 TASK는 15번 셀이 학습·val에서 자동으로 뺌)
#     ② fingerprint: 검토 완료 TASK마다 프레임 SIM_FRAMES장을 CVAT에서 받아 배경(밝기·초록 비율·바닥 색)과 개체(밀도·가림·작은 박스·겹침)를 숫자로
#     ③ evaluate: 같은 프레임을 모델로 탐지해 검토 뒤 라벨과 비교 — 클래스별 정밀도·재현율·F1 + 개수 오차(프레임당 개수 차이·비율·많이/덜 셈)
#     ④ relation: 성적이 지문과 '학습에 쓰였는지'에 따라 얼마나 달라지는지 (지문 1 표준편차당 변화 · 90% 구간) + 온실별 표
#     ⑤ plan: train/val/시험을 정하면 val·시험 TASK마다 가장 가까운 train TASK와의 거리(덮임)·예상 재현율 + 넣으면 덮임이 가장 넓어지는 TASK
#     결과: sim/fingerprints.csv · sim/eval.csv · sim/relation.json · sim/plan_<시각>.json (지문·성적은 TASK마다 한 번 — 다시 하려면 SIM_REDO)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
SIM_STEPS     = ['test', 'fingerprint', 'evaluate', 'relation', 'plan']   # 할 단계 (빼면 건너뜀)
SIM_QUERY     = '검토'       # 연구 대상 TASK (모든 셀 같은 검색 규칙) · '검토' = 검토 완료 전부
SIM_FRAMES    = 12           # TASK마다 볼 프레임 수 (고르게, 박스 없는 프레임 포함)
SIM_MODELS    = []           # 성적을 잴 모델 (runs/ 이름) · 비우면 0번 셀 모델
SIM_CONF      = 0.25         # 탐지 신뢰도 하한 (32번 셀과 같게)
SIM_REDO      = False        # True = 이미 만든 지문·성적도 다시
SIM_MAX_NEW   = 60           # 한 번에 새로 처리할 TASK 수 (CVAT에서 프레임을 받으므로 — 나머지는 다음 실행에 이어서)
SIM_LABEL     = '_all'       # 관계·예상에 쓸 라벨: '_all' = 모든 라벨 합 · 예 'healthy_bud'
# ---- 고정 시험 TASK
TEST_QUERY    = 'date=261008|261013'   # 시험 TASK 후보 (U세트 — 모델 개입 없음) · 바꿔서 다른 후보도 볼 수 있음
TEST_SAVE     = False        # True = 위 후보를 config/test_tasks.json에 고정 (바꾸면 기록이 남음 — 바꾸기 전후 성적은 서로 비교하지 않음)
TEST_NOTE     = 'U세트 (모델 개입 없음)'
# ---- 분할안
PLAN_TRAIN    = ''           # train 검색어 · '' = 연구 대상 중 시험 TASK·같은 그루·촬영일을 뺀 전부
PLAN_VAL      = ''           # val 검색어 · '' = 0번 셀 VAL_RULE (train 후보 안에서)
PLAN_SUGGEST  = 10           # 넣으면 덮임이 가장 넓어지는 TASK 몇 개 보여 줄지 (train 밖 TASK 중)
LOGIN         = None

import json, glob, importlib, pandas as pd
from IPython.display import display
import train_sim as ts
importlib.reload(ts)
use_cvat(LOGIN)
_t0_sim = __import__('time').time()
_tasks_all = ctt.list_tasks(CLIENT, limit=5000)
_by_id = {str(t['task_id']): t for t in _tasks_all}
_hits = ctt.select_tasks(_tasks_all, SIM_QUERY, display=None, show=lambda *a: None, label='연구 대상')
_hit_ids = [str(t['task_id']) for t in _hits]
_info = lambda tid: (ctt.parse_task_name(_by_id.get(str(tid), {}).get('name', '')) or {})
print(f"▶ 연구 대상 '{SIM_QUERY}' TASK {len(_hit_ids)}개 · 단계 {SIM_STEPS} · CVAT에 올리지 않음")
_ledger = ctt.read_train_tasks(BASE)
_TEST = ctt.read_test_tasks(BASE)
os.makedirs(f'{BASE}/{ts.SIM_DIR}', exist_ok=True)

# ---------- ① 고정 시험 TASK
if 'test' in SIM_STEPS:
    _c = ctt.select_tasks(_tasks_all, TEST_QUERY, display=None, show=lambda *a: None, label='시험 후보') if TEST_QUERY else []
    _cands = [{'task_id': t['task_id'], 'name': t['name'], 'gh': (ctt.parse_task_name(t['name']) or {}).get('gh', ''),
               'date': (ctt.parse_task_name(t['name']) or {}).get('date', '')} for t in _c]
    _runs = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(f'{BASE}/runs/*/TRAINING_DONE'))
    _rows, _models_seen = ts.test_candidate_table(_cands, _ledger)
    print(f"\n■ ① 시험 TASK 후보 '{TEST_QUERY}' {len(_cands)}개 — 어느 모델 학습(train·val)에 쓰였는지 (logs/train_tasks.csv)")
    if _rows:
        display(pd.DataFrame(_rows).set_index('task_id'))
        _d = pd.DataFrame(_cands)
        print('온실 × 촬영일 TASK 수 (빈 칸 = 후보 없음 → 시험이 그 환경을 재지 못함)')
        display(pd.crosstab(_d['gh'], _d['date']))
        _all_gh = sorted({_info(t).get('gh', '') for t in _hit_ids} - {''})
        _miss = [g for g in _all_gh if g not in set(_d['gh'])]
        if _miss:
            print(f"⚠️ 연구 대상에 있는데 후보에 없는 온실: {_miss} → 그 온실 TASK를 시험에 보충할지 정하세요 (TEST_QUERY에 더하기)")
    _noledger = [r for r in _runs if r not in set(_models_seen)]
    if _noledger:
        print(f"ℹ️ 학습 대장(train_tasks)이 없는 모델 {len(_noledger)}개 — 이 모델들은 어떤 TASK로 학습했는지 알 수 없어 '없음'이 확실하지 않음: {_noledger[:8]}")
    _used = [r for r in _rows if r['학습에 쓴 모델'] != '없음']
    if _used:
        print(f"⚠️ 후보 중 {len(_used)}개는 이미 학습에 쓰였습니다 — 그 모델과 비교하는 시험에는 쓸 수 없음 (빼거나, 그 모델은 시험 비교에서 제외)")
    if TEST_SAVE:
        assert _cands, 'TEST_QUERY에 맞는 TASK가 없습니다'
        _p = ctt.write_test_tasks(BASE, {str(c['task_id']): c['name'] for c in _cands}, by=who_am_i(), note=TEST_NOTE)
        _TEST = ctt.read_test_tasks(BASE)
        print(f"✅ 고정 시험 TASK {len(_TEST['ids'])}개 저장: {os.path.relpath(str(_p), BASE)} — 15번 셀이 이 TASK와 같은 그루·촬영일 TASK를 학습·val에서 뺍니다")
    print(f"지금 고정 시험 TASK: {len(_TEST['ids'])}개" + (f" ({_TEST['note']}, {_TEST['updated'][:16]})" if _TEST['ids'] else ' (없음 — TEST_SAVE = True로 저장)'))

# ---------- ②·③ 지문 · 성적 (TASK마다 프레임을 한 번 받아 둘 다)
def _sim_models():
    out = []
    for n in (SIM_MODELS or [None]):
        if not n:
            out.append((os.path.basename(os.path.dirname(os.path.dirname(BEST))), BEST))
        else:
            w = need_file(f'{BASE}/runs/{n}/weights/best.pt')
            assert os.path.exists(w), f'모델 파일이 없습니다: runs/{n}/weights/best.pt'
            out.append((n, w))
    return out

if 'fingerprint' in SIM_STEPS or 'evaluate' in SIM_STEPS:
    kg_sync_exports()
    _scan = ctt.scan_exports(f'{BASE}/exports', names=[_by_id[t]['name'] for t in _hit_ids if t in _by_id], log=None) if os.path.exists(f'{BASE}/exports') else []
    _fp = ts.read_fingerprints(BASE)
    _done_ev = {(str(r['task_id']), r['model']) for r in ts.read_eval(BASE)}
    _models = _sim_models() if 'evaluate' in SIM_STEPS else []
    _preds = {}
    if _models:
        from ultralytics import YOLO as _Y
        for n, w in _models:
            _preds[n] = (wb.yolo_predictor(w, imgsz=IMGSZ, conf=SIM_CONF),
                         {x[:-9] if x.endswith('_occluded') else x for x in _Y(w).names.values()})
    _used_by = {}
    for r in _ledger:
        _used_by.setdefault((r.get('model', ''), str(r.get('task_id'))), r.get('role', ''))
    _todo = [t for t in _hit_ids if SIM_REDO or t not in _fp or any((t, n) not in _done_ev for n, _ in _models)]
    print(f"\n■ ②③ 지문·성적: 새로 할 TASK {len(_todo)}개 (이번에 {min(len(_todo), SIM_MAX_NEW)}개) · 모델 {[n for n, _ in _models] or '없음(지문만)'}")
    _n_ok = 0
    for _i, _tid in enumerate(_todo[:SIM_MAX_NEW], 1):
        try:
            t = _by_id[_tid]
            task = CLIENT.tasks.retrieve(int(_tid))
            ex, st = ctt.best_export(_scan, _tid, t.get('updated_utc'), kind='images') if _scan else (None, '없음')
            if ex and st == '최신' and os.path.exists(ex['zip']):
                ref = ex['zip']
            else:                                       # 최신 export가 없으면 라벨만 받음 (이미지 없이)
                ref = ctt.export_annotations(task, f'{task_dir(_tid)}/sim_ref_{ctt.stamp()}.zip')
            ann = ctt.load_annotations(ref)
            W, H = ann['meta'].get('W'), ann['meta'].get('H')
            frames = ts.pick_frames(ann, SIM_FRAMES)
            get = ctt.cvat_frame_getter(task, ann['meta'])
            imgs = {f: get(f) for f in frames}
            frames = [f for f in frames if imgs.get(f) is not None]
            if not frames:                              # v4.31.9 (F18): 프레임을 못 받으면 저장하지 않고 다음 실행에 다시
                print(f"  [{_i}] {task.name} · 프레임을 하나도 받지 못함 — 저장하지 않고 다음 실행에 다시 시도")
                continue
            f_ = ctt.parse_task_name(task.name) or {}
            base_row = {'task_id': _tid, 'task_name': task.name, 'gh': f_.get('gh', ''), 'date': f_.get('date', ''),
                        'row': f_.get('row', ''), 'plant': f_.get('plant', ''), 'ref_file': os.path.relpath(ref, BASE), 'time': ctt.now()}
            if 'fingerprint' in SIM_STEPS and (SIM_REDO or _tid not in _fp):
                _row = dict(base_row, **ts.fingerprint(ann, frames, imgs, W, H))
                ts.save_fingerprint(BASE, _row)
                _fp[_tid] = _row
            ev_rows = []
            for n, _ in _models:
                if not SIM_REDO and (_tid, n) in _done_ev:
                    continue
                pred_fn, known = _preds[n]
                preds = {f: pred_fn(imgs[f], W, H) for f in frames}
                res = ts.evaluate(ann, frames, preds, labels=known)
                role = _used_by.get((n, _tid), '')
                for lab, sc in res.items():
                    ev_rows.append(dict(base_row, model=n, label=lab, in_train=role, **{k: sc.get(k) for k in ts.EVAL_COLS if k in sc}))
            if ev_rows:
                ts.save_eval(BASE, ev_rows)
            _a = [r for r in ev_rows if r['label'] == '_all']
            print(f"  [{_i}/{min(len(_todo), SIM_MAX_NEW)}] {task.name} · 프레임 {len(frames)} · 밝기 {(_fp.get(_tid) or {}).get('bg_luma', '–')} · "
                  f"박스/프레임 {(_fp.get(_tid) or {}).get('boxes_per_frame', '–')}"
                  + ''.join(f" · {r['model']} R {r['recall']} 개수오차 {r['count_rel']}" for r in _a))
            _n_ok += 1
        except Exception as e:
            print(f"  ⚠️ TASK {_tid}: {ctt.api_error(e) if hasattr(ctt, 'api_error') else e}")
    print(f"지문 {len(ts.read_fingerprints(BASE))}개 · 성적 {len({(r['task_id'], r['model']) for r in ts.read_eval(BASE)})}건 저장됨 (sim/)"
          + (f" · 남은 TASK {len(_todo) - SIM_MAX_NEW}개는 다시 실행하면 이어서" if len(_todo) > SIM_MAX_NEW else ''))

# ---------- ④ 관계
_fp, _ev = ts.read_fingerprints(BASE), ts.read_eval(BASE)
_rels = {}
if 'relation' in SIM_STEPS:
    _ms = [n for n, _ in _sim_models()] if not SIM_MODELS else list(SIM_MODELS)
    for n in _ms:
        print(f"\n■ ④ 관계 — 모델 {n} · 라벨 {SIM_LABEL}")
        for metric, ko in (('recall', '재현율'), ('count_rel', '개수 오차 비율')):
            rel = ts.relation(_fp, _ev, n, SIM_LABEL, metric)
            _rels[(n, metric)] = rel
            if not rel['coef']:
                print(f"  {ko}: {rel['warn']}")
                continue
            print(f"  {ko} (평균 {rel['mean']:.3f} · TASK {rel['n']}개 · 설명력 R² {rel['r2']}){' — ' + rel['warn'] if rel['warn'] else ''}")
            display(pd.DataFrame([{'지문': c['ko'], '1 표준편차(원래 단위)': c['sd'] if c['sd'] is not None else '쓰임/안 쓰임',
                                   f'{ko} 변화': c['effect'], '90% 구간': f"{c['lo']} ~ {c['hi']}", '확실': '✅' if c['sure'] else ''}
                                  for c in rel['coef']]).set_index('지문'))
        print('  온실별 평균 (관찰과 맞는지: G6·G5 어두움, G6 조밀, G2 개체 많음)')
        display(pd.DataFrame(ts.group_table(_fp, _ev, n, SIM_LABEL)).set_index('gh'))
    json.dump({f'{k[0]}|{k[1]}': v for k, v in _rels.items()}, open(f'{BASE}/{ts.SIM_DIR}/relation.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)

# ---------- ⑤ 분할안
if 'plan' in SIM_STEPS:
    _cand = [t for t in _hit_ids if not ctt.test_conflict(_TEST, t, _by_id.get(t, {}).get('name', ''))]
    if PLAN_TRAIN:
        _pt = {str(t['task_id']) for t in ctt.select_tasks(_tasks_all, PLAN_TRAIN, display=None, show=lambda *a: None, label='train')}
        _cand = [t for t in _cand if t in _pt]
    if PLAN_VAL:
        _pv = {str(t['task_id']) for t in ctt.select_tasks(_tasks_all, PLAN_VAL, display=None, show=lambda *a: None, label='val')}
        _val = [t for t in _cand if t in _pv]
    else:
        _val = [t for t in _cand if task_role(_by_id[t]['name']) == 'val']
    _train = [t for t in _cand if t not in set(_val)]
    _test = sorted(_TEST['ids'])
    _roles = {**{t: 'val' for t in _val}, **{t: '시험' for t in _test}}
    _m0 = (SIM_MODELS or [os.path.basename(os.path.dirname(os.path.dirname(BEST)))])[0]
    _rel = _rels.get((_m0, 'recall')) or ts.relation(_fp, _ev, _m0, SIM_LABEL, 'recall')
    sim = ts.simulate(_fp, _train, _val + _test, rel=_rel if _rel.get('b') else None, roles=_roles)
    print(f"\n■ ⑤ 분할안 — train {len(_train)} · val {len(_val)} · 시험 {len(_test)} (지문 있는 train {sim.get('train_n', 0)}개) · 예상 재현율 = 모델 {_m0} 관계로")
    if sim.get('error'):
        print('  ' + sim['error'])
    else:
        display(pd.DataFrame([{'온실': g, 'train TASK': v['tasks'], 'train 박스': v['boxes']} for g, v in sorted(sim['train_by_gh'].items())]).set_index('온실'))
        _df = pd.DataFrame(sim['rows'])
        if len(_df):
            _df['덮임'] = _df.get('covered').map({True: '✅', False: '⚠️ train에 비슷한 환경 없음'}) if 'covered' in _df else ''
            display(_df[[c for c in ['task_id', 'task_name', 'role', 'gh', 'distance', '덮임', 'far_keys', 'nearest_name', 'pred_recall', 'note'] if c in _df]]
                    .rename(columns={'role': '역할', 'distance': '가장 가까운 train과 거리', 'far_keys': '가장 다른 지문', 'nearest_name': '가장 가까운 train',
                                     'pred_recall': '예상 재현율'}).set_index('task_id'))
        print(f"  덮임 기준: 거리 ≤ {sim['threshold']} (train끼리 가장 가까운 거리의 90%값)")
        _pool = [t for t in _hit_ids if t not in set(_train) and not ctt.test_conflict(_TEST, t, _by_id.get(t, {}).get('name', ''))]
        _sug = ts.suggest_additions(_fp, _train, _pool, k=PLAN_SUGGEST) if PLAN_SUGGEST else []
        if _sug:
            print(f'  train에 넣으면 덮는 범위가 가장 넓어지는 TASK (train 밖 · 시험과 겹치지 않는 것) {len(_sug)}개')
            display(pd.DataFrame(_sug).set_index('task_id'))
        _pp = f"{BASE}/{ts.SIM_DIR}/plan_{ctt.stamp()}.json"
        json.dump({'time': ctt.now(), 'who': who_am_i(), 'train': _train, 'val': _val, 'test': _test, 'model': _m0, **sim, 'suggest': _sug},
                  open(_pp, 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
        print(f"  저장: {os.path.relpath(_pp, BASE)}")
print(f"\n(33번 셀 {__import__('time').time() - _t0_sim:.0f}초)")
